# Inverted dropout kernel from a noise tensor

Dropout zeroes each activation with probability $p$ during training. **Inverted** dropout also scales the survivors by $\frac{1}{1-p}$ at training time, so the expected value of every output equals its input and inference needs no rescaling at all.

To keep the kernel testable, the randomness is handed to you: `u` holds one uniform sample per element. (A production kernel would draw it inside the kernel with `tl.rand(seed, offsets)`; see the follow-ups.)

Write `dropout(x, u, p)`.

## Inputs
- `x`: contiguous `float32` CUDA tensor, any shape (including 0 elements).
- `u`: contiguous `float32` CUDA tensor of the same shape, values in $[0, 1)$.
- `p`: Python float, $0 \le p \le 1$.

## Output
A new `float32` tensor of the shape of `x`:

$$y_i = \begin{cases} x_i \cdot \dfrac{1}{1-p} & \text{if } u_i \ge p \quad\text{(kept)}\\[4pt] 0 & \text{if } u_i < p \quad\text{(dropped)}\end{cases}$$

## Rules
- Ties: an element with $u_i = p$ exactly is **kept**.
- $p = 0$ returns a copy of `x`. $p = 1$ drops everything and returns zeros; compute the scale on the host as `1 / (1 - p)` if `p < 1` else `0.0`, so nothing divides by zero.
- Flat 1-D grid of `triton.cdiv(n, BLOCK)` programs, masked loads and stores; no launch for an empty tensor.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from an equivalent PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def dropout(x, u, p):
    """x, u: float32 CUDA tensors of one shape, u in [0, 1). Keep where u >= p, scale survivors by 1/(1-p)."""
    raise NotImplementedError


@triton.jit
def _dropout_kernel(x_ptr, u_ptr, out_ptr, n, p, scale, BLOCK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [1.0, 2.0, 3.0, 4.0], "dtype": "float32"}, {"__nd__": [0.8999999761581421, 0.10000000149011612, 0.5, 0.699999988079071], "dtype": "float32"}, 0.5], "e": {"__nd__": [2.0, 0.0, 6.0, 8.0], "dtype": "float32"}}, {"a": [{"__nd__": [[-2.0, 6.0], [1.5, 0.0]], "dtype": "float32"}, {"__nd__": [[0.30000001192092896, 0.20000000298023224], [0.800000011920929, 0.9900000095367432]], "dtype": "float32"}, 0.25], "e": {"__nd__": [[-2.6666667461395264, 0.0], [2.0, 0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [0.0341927669942379, 1.3597475290298462, 1.224721074104309, -0.5103070735931396, -0.2979695200920105, -0.5273842215538025, 0.5697263479232788, -0.056064438074827194, 0.7468855977058411, -1.8473248481750488, 1.5665488243103027, -0.0964321568608284, 0.6803784370422363, -0.13656634092330933, -0.3790985643863678, 0.46311014890670776, 0.8245135545730591, -0.2025298774242401, -0.15278618037700653, 0.6856986284255981, -0.8703406453132629, -1.5143835544586182, 0.39498186111450195, -0.6705658435821533, -1.9203405380249023, -0.8140536546707153, -0.4675975441932678, -1.1932024955749512, -1.4924638271331787, 0.03663782775402069, 0.8972492814064026, -0.23313207924365997, -0.7435960173606873, 0.3849938213825226, 0.717235803604126, -0.3000105917453766, 0.5446677803993225], "dtype": "float32"}, {"__nd__": [0.2022160291671753, 0.9014310836791992, 0.21714825928211212, 0.03307468816637993, 0.20076894760131836, 0.3457478880882263, 0.4689081609249115, 0.9061343669891357, 0.6973611116409302, 0.33932065963745117, 0.01687721535563469, 0.1598237007856369, 0.9964358806610107, 0.45971599221229553, 0.6910399198532104, 0.054668061435222626, 0.03405027836561203, 0.8458901047706604, 0.5878819227218628, 0.3087097406387329, 0.31737664341926575, 0.0892372578382492, 0.17266960442066193, 0.024586107581853867, 0.839124858379364, 0.4663031995296478, 0.12720291316509247, 0.7392468452453613, 0.1956528276205063, 0.06192023679614067, 0.598392128944397, 0.8957577347755432, 0.02694341167807579, 0.80513596534729, 0.190170019865036, 0.09290142357349396, 0.017962034791707993], "dtype": "float32"}, 0.0], "e": {"__nd__": [0.0341927669942379, 1.3597475290298462, 1.224721074104309, -0.5103070735931396, -0.2979695200920105, -0.5273842215538025, 0.5697263479232788, -0.056064438074827194, 0.7468855977058411, -1.8473248481750488, 1.5665488243103027, -0.0964321568608284, 0.6803784370422363, -0.13656634092330933, -0.3790985643863678, 0.46311014890670776, 0.8245135545730591, -0.2025298774242401, -0.15278618037700653, 0.6856986284255981, -0.8703406453132629, -1.5143835544586182, 0.39498186111450195, -0.6705658435821533, -1.9203405380249023, -0.8140536546707153, -0.4675975441932678, -1.1932024955749512, -1.4924638271331787, 0.03663782775402069, 0.8972492814064026, -0.23313207924365997, -0.7435960173606873, 0.3849938213825226, 0.717235803604126, -0.3000105917453766, 0.5446677803993225], "dtype": "float32"}}, {"a": [{"__nd__": [0.6319571733474731, 1.2597551345825195, 1.7911754846572876, -1.5735763311386108, 0.8831318020820618, 0.465068519115448, -0.09386078268289566, -1.0066649913787842, 1.2571885585784912, -1.2617379426956177, 0.566945493221283, 1.301867961883545, -1.599669337272644, -0.302517831325531, -1.309216856956482, 0.24405410885810852, 1.51437509059906, 2.0235559940338135, -1.7781144380569458, -0.5749490261077881, 0.703545093536377, 1.5793726444244385, 0.4212106466293335, -0.7461519837379456, 0.2971315085887909, -0.016619201749563217, -0.20374071598052979, -0.7344710826873779, 0.38725975155830383, 0.30787962675094604, -0.09298412501811981, -0.2216879427433014, -1.2849165201187134, -0.48617565631866455, 1.2064496278762817, -0.19055825471878052, -1.439697265625], "dtype": "float32"}, {"__nd__": [0.35721972584724426, 0.06088670343160629, 0.870384931564331, 0.6363614201545715, 0.15974824130535126, 0.4982685446739197, 0.07866975665092468, 0.6109797954559326, 0.23166322708129883, 0.03863893449306488, 0.11523750424385071, 0.5552626252174377, 0.6369805932044983, 0.32480064034461975, 0.6434524059295654, 0.3520660996437073, 0.13065901398658752, 0.3152102828025818, 0.3952682316303253, 0.9126388430595398, 0.11566037684679031, 0.08616389334201813, 0.5615020394325256, 0.9633927941322327, 0.9072551131248474, 0.7002440690994263, 0.06675074994564056, 0.8064014315605164, 0.6833747625350952, 0.14393024146556854, 0.4649437367916107, 0.04932265356183052, 0.8018759489059448, 0.7186341881752014, 0.8046280741691589, 0.7604156136512756, 0.26725757122039795], "dtype": "float32"}, 1.0], "e": {"__nd__": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "dtype": "float32"}}, {"a": [{"__nd__": [], "dtype": "float32"}, {"__nd__": [], "dtype": "float32"}, 0.5], "e": {"__nd__": [], "dtype": "float32"}}, {"a": [{"__nd__": [-0.33889079093933105, -1.9236828088760376, -0.07228295505046844, 0.22534579038619995, 1.0844756364822388, 0.577863872051239, -0.6435607075691223, -0.7237765789031982, 2.0105974674224854, 0.7566379308700562, 1.8314398527145386, 2.1294445991516113, -0.818083643913269, 0.38526931405067444, 0.45816701650619507, 0.5596122145652771, 0.5419022440910339, 0.20205463469028473, 0.17412924766540527, -1.5024999380111694, -0.16540119051933289, -0.7476196885108948, 0.12608730792999268, -0.4675462543964386, 0.6185207366943359, 0.8190757036209106, 0.3086923658847809, 0.3161681294441223, 0.09294668585062027, -0.44779789447784424, -0.16450126469135284, -0.49564605951309204, 0.3879871368408203, 0.014113412238657475, 0.5812911987304688, -1.328648328781128, 0.8877789378166199, -0.7626962661743164, -0.7342821955680847, -0.19744297862052917, -0.5633845925331116, 0.29112520813941956, -0.5741647481918335, -1.0694775581359863, -0.8458027839660645, 1.311983346939087, 0.04425833001732826, -1.1675302982330322, 0.008377403020858765, -1.5559464693069458, 1.7895665168762207, -1.5244781970977783, 0.4787527024745941, 0.5436256527900696, -1.4500077962875366, 0.300276517868042, 0.9971720576286316, 0.4675752818584442, 0.26101627945899963, 0.9490966796875, 0.16090454161167145, 0.3364252746105194, -0.12671154737472534, 0.6318057179450989, -0.9413841366767883, 0.7917632460594177, -0.5053015351295471, -1.090653419494629, 0.3652489483356476, 1.692955493927002, 0.9616782069206238, -0.5156498551368713, 0.6980596780776978, -0.4542091190814972, -0.12401945888996124, 0.08952849358320236, -2.31721830368042, 0.1916147619485855, -1.029609203338623, -0.6973986029624939, -1.4743903875350952, -1.5165581703186035, -0.9432644248008728, 0.8255972862243652, 1.666041374206543, -0.025205103680491447, 1.0917909145355225, -0.2639806568622589, -1.9119728803634644, 0.1499166190624237, 0.4456482529640198, -0.42861655354499817, 0.3022242486476898, 0.5725562572479248, -0.863407552242279, -1.4768340587615967, -0.2212541699409485, -0.2110605388879776, -0.35266321897506714, 0.9872467517852783, 1.7255887985229492, -0.4166470170021057, 0.6988866925239563, 0.9409311413764954, 0.7131564021110535, 1.0474863052368164, -0.3892439603805542, 0.7398063540458679, 2.024482011795044, 0.8036655783653259, -0.6198936700820923, 0.6077125072479248, 1.2623802423477173, 0.3689862787723541, -0.5634006857872009, 1.5430805683135986, -1.2497670650482178, 0.5034250617027283, -0.01617557369172573, -1.1936657428741455, 1.2194496393203735, 0.3478876054286957, -1.1667982339859009, 0.6017283797264099, -0.431600421667099, -1.902788519859314, -0.6889051198959351, 0.2635514438152313, 0.635702908039093, 0.1655649095773697, 0.045165009796619415, 0.4699900150299072, -0.21097610890865326, 1.1769546270370483, 0.1486416459083557, 0.293921560049057, 0.5407989621162415, -1.0571688413619995, -0.7092268466949463, 1.5285791158676147, 0.335308700799942, -0.2826160788536072, 0.34148353338241577, -0.48162224888801575, 0.27856630086898804, -0.6658710837364197, 0.27601882815361023, -1.577702522277832, 1.3309197425842285, -0.5129578709602356, -1.581241250038147, -0.22384169697761536, -0.3709962069988251, 0.15239842236042023, -1.1452505588531494, 0.3974655866622925, 3.623567581176758, -1.278667688369751, 0.3377155661582947, -0.3107188642024994, 0.1926082968711853, -1.8118090629577637, -1.151009440422058, 0.45965954661369324, -0.03841802105307579, 1.6352381706237793, -0.7047973871231079, 0.1744995266199112, 2.913121461868286, -0.7655614614486694, -0.9038335084915161, -0.035130083560943604, -0.044562678784132004, 0.8441672921180725, 0.12957587838172913, -0.7567393183708191, 0.21142154932022095, 2.638596773147583, 1.2717736959457397, -2.7821691036224365, -0.1420169174671173, -0.8750905394554138, 0.6222625374794006, -0.17326347529888153, 1.932695746421814, 0.9312355518341064, 0.8904029130935669, 0.18657106161117554, -0.040297653526067734, 0.3401799201965332, 1.2952797412872314, 0.5343897938728333, -0.3511325716972351, 1.2750846147537231, 0.19211037456989288, -0.07761505991220474, -0.7913768291473389, -0.5659922957420349, -0.6360467672348022, -2.7923572063446045, 1.0168461799621582, 0.5664775967597961, -0.21427059173583984, 1.0459837913513184, 0.4379877746105194, 0.3715674877166748, -2.4747636318206787, -0.23301592469215393, 0.532189130783081, 1.6096868515014648, 1.8402948379516602, 1.4258382320404053, -1.1616095304489136, -2.197671413421631, 0.5704695582389832, 0.027615835890173912, 0.9625532627105713, -0.12588436901569366, 0.21975496411323547, -1.4999890327453613, 0.4480096697807312, -0.44489845633506775, 0.34998536109924316, 0.4594059884548187, -0.1172221451997757, 0.31680113077163696, -0.013607030734419823, 0.7591995000839233, 0.501742959022522, -0.5472163558006287, -0.5879018902778625, 1.7437387704849243, -0.10159549862146378, 0.991807758808136, 0.39338573813438416, -0.5641124844551086, -0.7489737868309021, 0.12012568861246109, 0.25665441155433655, 0.6143637299537659, -1.2669334411621094, -0.5946832895278931, -0.09486620128154755, 1.6007494926452637, -2.3601784706115723, 0.4391116499900818, -0.5400458574295044, 0.4313773512840271, -0.4919789135456085, 0.3645510673522949, 1.5646380186080933, 0.3541964888572693, 0.4333999454975128, 0.3728578984737396, -0.7503135204315186, 0.5430178642272949, -0.20834656059741974, -0.8117136359214783, -0.038547925651073456, -0.22728846967220306, 0.7534797787666321, -0.04283922538161278, -0.5294420719146729, 1.455971598625183, 0.9883877635002136, 1.269729495048523, 0.6076673865318298, -0.10969899594783783, -0.41909340023994446, -0.279339998960495, 0.9476906061172485, 0.11190328001976013, -1.4828431606292725, -0.03908279910683632, -1.4161041975021362, 0.16586583852767944, -1.2038196325302124, -1.263936161994934, 0.1185794547200203, -0.07821432501077652, -1.6617616415023804, 0.7482237219810486, 2.1630966663360596, -1.4813756942749023, 1.8198497295379639, -0.6877894401550293, -0.49409061670303345, -0.17888721823692322, -1.445277214050293, 1.299127221107483, 0.49269363284111023, -0.7514541745185852, -0.27707716822624207, -1.5163594484329224, -0.3477577269077301, 0.9362131357192993, -1.715289831161499, -0.4362403154373169, 0.1467917412519455, 1.0482393503189087, 1.3493305444717407, -0.45725294947624207, -0.7403068542480469, 0.4503996670246124, -0.3632547855377197, -0.3100193738937378, 1.0801836252212524, 1.6897379159927368, 0.26586756110191345, -0.19221477210521698, 0.5712619423866272, -1.8875157833099365, -0.018249062821269035, -1.0605767965316772, 2.389427900314331, 0.7201999425888062, 0.14136984944343567, -0.19436462223529816, -2.514955997467041, 0.7128044962882996, 1.6293689012527466, -0.7288199067115784, 0.6834045052528381, 0.7225431799888611, -1.5481276512145996, 0.9357403516769409, 1.3547810316085815, 0.22848011553287506, -0.06318199634552002, 0.7879635095596313, -1.540753960609436, 0.09328070282936096, 1.0543419122695923, -1.242435336112976, 0.992571234703064, -0.4182296693325043, -2.080826997756958, -0.10574795305728912, -0.9718965888023376, 0.3192596435546875, -1.172141194343567, 1.0502759218215942, -0.7388303279876709, 0.35635748505592346, -0.5939799547195435, -0.3229053020477295, 1.1917600631713867, 0.21527844667434692, -0.3950320780277252, -1.2515079975128174, 1.1599513292312622, 0.7328111529350281, -0.32653558254241943, 0.24331101775169373, -1.0295774936676025, -1.0148448944091797, -0.8261877298355103, 0.3534272313117981, 0.2958458960056305, -0.7453029155731201, 0.7377746105194092, 0.7846935391426086, 0.17290815711021423, 1.0383886098861694, -0.2873624563217163, -0.1130211353302002, 0.4728586673736572, -0.03404400870203972, 0.7938991785049438, -1.2526681423187256, 1.425765037536621, -0.7812714576721191, -0.16469785571098328, -1.3775427341461182, 0.6002223491668701, 1.2738513946533203, 1.0710469484329224, 0.8333749771118164, -1.557143211364746, -0.7821680307388306, -0.4625629782676697, -0.9347342848777771, 1.1226952075958252, 0.1844843178987503, -0.024477804079651833, 0.4875357449054718, -0.951619029045105, -0.9083507061004639, -0.8814619779586792, 0.49762094020843506, 0.3415963649749756, 0.5780924558639526, 1.0783830881118774, -1.357300877571106, -0.5604943633079529, 0.14753809571266174, -1.3487430810928345, -0.5321752429008484, 1.0140289068222046, 1.1026440858840942, 1.076706886291504, 0.043017685413360596, -0.21196754276752472, 1.0795246362686157, -0.3726460039615631, 0.33823898434638977, 1.325278878211975, -0.36757078766822815, -0.6105523109436035, -0.009371273219585419, 0.5983791947364807, 0.4824090301990509, 0.15585777163505554, 0.5564201474189758, -1.59959876537323, -0.8592659831047058, 0.16322611272335052, -0.9286615252494812, -1.3416744470596313, -1.2078198194503784, -0.9181638956069946, -0.17847488820552826, -1.0226258039474487, 0.23489883542060852, 1.6962240934371948, -0.14762631058692932, 1.2746988534927368, 0.28843173384666443, -1.1198234558105469, -0.25640958547592163, -0.9818347692489624, 0.7679353952407837, -1.1530953645706177, 1.3225727081298828, 0.5781879425048828, -2.0223100185394287, 0.7843413949012756, 1.5530163049697876, -0.7905237078666687, 0.9852522015571594, -1.7983816862106323, 0.2521566152572632, -1.6613893508911133, 0.48239099979400635, -1.787962794303894, -2.6328701972961426, 0.3960931599140167, 2.2878897190093994, -0.5956695675849915, 0.8301655650138855, 0.11071109771728516, -0.3235047459602356, 0.6153842806816101, 1.642405390739441, -1.2351280450820923, 0.5638969540596008, 0.3451354205608368, -1.42434561252594, 1.614710807800293, 1.9493587017059326, -0.9527719616889954, 1.7379004955291748, 0.05614611878991127, -0.16324834525585175, 1.5034489631652832, 0.22743286192417145, 0.32317590713500977, 1.5176749229431152, 1.288219928741455, 0.8931494355201721, -1.4005498886108398, -1.5723903179168701, 1.1261411905288696, -0.9267195463180542, 0.148483544588089, -1.074072003364563, -0.1836865246295929, -1.2786951065063477, 1.6301740407943726, -1.018912434577942, -0.1219942644238472, -0.1881994754076004, 1.2060978412628174, -1.4198975563049316, -0.03334203362464905, -0.20923426747322083, 0.529925525188446, 0.4720167815685272, -0.4637465476989746, 0.2395859956741333, -0.37463119626045227, 0.4681152403354645, -0.10258016735315323, -0.6555690765380859, -0.027557022869586945, 0.36378395557403564, 2.2658824920654297, -0.49123185873031616, 0.3066580593585968, 1.4630367755889893, -2.0781686305999756, -1.7430381774902344, -1.1886593103408813, -0.11423654109239578, 2.439976692199707, 0.39136695861816406, 1.2080752849578857, -0.2220393568277359, -2.463723659515381, -2.31363844871521, 1.018619179725647, 0.18485954403877258, -1.2984992265701294, 1.1173206567764282, -1.5389549732208252, 1.1342421770095825, 1.4513407945632935, -0.7648233771324158, -1.1081169843673706, 0.8443270325660706, -1.095288634300232, 0.8618864417076111, -1.1952388286590576, 0.8075785636901855, 0.09386694431304932, -0.43277838826179504, -0.8956846594810486, 0.2375747710466385, -1.8926606178283691, -0.2759641706943512, -0.39615416526794434, 0.6783599257469177, -1.2538203001022339, 1.7134381532669067, -0.3115807771682739, -1.1381675004959106, 1.3860787153244019, 1.6629310846328735, -0.1877720057964325, -0.8910728096961975, 0.0160529688000679, -0.6904117465019226, -0.3120335340499878, 1.5584293603897095, 0.3919009864330292, 0.14539381861686707, 0.3460122048854828, -0.058089807629585266, 0.841567873954773, 0.8792901039123535, -0.36971673369407654, 0.10508839040994644, 2.413607597351074, 0.4215991497039795, -0.18585337698459625, 2.049691677093506, 0.009432554244995117, -1.097301959991455, 0.6745808720588684, -0.899826169013977, -0.49722251296043396, 0.326325386762619, -0.12676692008972168, -0.7634866237640381, -0.5509566068649292, -0.5933759808540344, 1.1661864519119263, -0.7210001945495605, -0.5722190737724304, 2.0383543968200684, -0.5537643432617188, 1.4894176721572876, 1.3157322406768799, -0.152751624584198, -1.419391393661499, 0.4530141353607178, 0.6726751327514648, 0.967804491519928, 0.07189423590898514, 1.2045575380325317, 1.390358567237854, -1.054382085800171, 1.4294883012771606, 0.1745423525571823, -0.5657440423965454, -0.3186277151107788, 1.8775721788406372, 0.9840299487113953, -1.7890459299087524, 0.08345750719308853, -0.1398499459028244, -0.1755872666835785, -0.5131552219390869, -1.4563014507293701, 0.8616443276405334, -0.9396897554397583, -0.3700411021709442, -0.6023852825164795, -0.7278238534927368, 0.7432366609573364, 0.38377416133880615, -0.6741616129875183, -0.944810688495636, -0.4694144129753113, 0.9670242071151733, 0.125921368598938, 0.23308369517326355, -0.32816365361213684, -0.25788041949272156, 0.21654373407363892, -1.8434250354766846, 1.612058162689209, -0.787643551826477, -0.5683143734931946, 1.3384970426559448, 0.22243373095989227, 0.04208355396986008, -0.5833193063735962, -1.5611693859100342, 0.18494535982608795, 1.1952613592147827, -0.8996552228927612, 1.3627548217773438, -1.127142071723938, -0.7340111136436462, 0.5058915615081787, 6.522834883071482e-05, 0.690887987613678, -0.7417380809783936, -1.031761646270752, -1.8670626878738403, -1.2037112712860107, -1.0761077404022217, -1.426965594291687, 0.30846118927001953, -0.6638851761817932, -1.988295078277588, 2.483482599258423, -0.8348536491394043, -1.255509376525879, 1.239788293838501, 1.1654338836669922, -1.1513409614562988, -1.4739818572998047, -1.0000667572021484, 1.129485845565796, -0.31126075983047485, -0.8086978793144226, -0.02837241254746914, 1.2041914463043213, -1.3895503282546997, -0.38380178809165955, -1.383475661277771, 2.1721553802490234, 1.81831955909729, 0.9511620998382568, -0.13340158760547638, 0.6591266393661499, -2.10201358795166, 0.276426762342453, -1.336165189743042, -1.324772834777832, -0.7047259211540222, -0.35021674633026123, 0.5742192268371582, 0.7006787657737732, 0.2626725733280182, -0.9443365931510925, 0.6759867668151855, -0.1209469810128212, 1.18022882938385, -0.9171306490898132, -2.7632250785827637, -0.4767092168331146, 2.151263475418091, 0.4689873456954956, -0.194329172372818, 0.8836362957954407, 0.5031824707984924, 0.4613412916660309, -0.5135456323623657, 0.07617104053497314, 0.2011920064687729, -0.6426617503166199, -1.1187301874160767, 0.13902266323566437, 0.968450129032135, 0.6473042964935303, 0.4939775764942169, 0.5043149590492249, 2.584002733230591, 0.25988373160362244, 1.042038083076477, 1.9571000337600708, 0.053063105791807175, 0.2645300626754761, 1.0440465211868286, 1.6876633167266846, 0.3131053149700165, 0.36301782727241516, 0.09083957970142365, -1.2055991888046265, 1.1176679134368896, -0.2524987459182739, -1.7187526226043701, -0.1441367119550705, -1.7270299196243286, -1.004696011543274, 0.021083984524011612, -0.9169257879257202, 1.0753227472305298, -1.4641393423080444, 0.2691328823566437, 0.5642681121826172, 1.467471718788147, 0.9299717545509338, 0.24120445549488068, 0.08867354691028595, -0.05003194883465767, 1.1474753618240356, -0.647798478603363, 0.5302385687828064, -0.08412697911262512, 0.25039854645729065, -0.2523863911628723, 2.1453747749328613, -0.6467331647872925, 0.2638084888458252, 0.2868878245353699, 0.6192367672920227, 0.4617531895637512, 0.43603435158729553, 0.9934980869293213, -1.4932655096054077, 0.021734239533543587, -1.2216815948486328, 0.24459578096866608, -1.1706072092056274, -0.9478177428245544, -0.048246245831251144, 1.4088457822799683, 0.5899841785430908, 0.30932170152664185, 1.4161463975906372, -0.9322607517242432, -2.0790228843688965, -0.4107004404067993, -0.26495981216430664, 1.2480376958847046, 2.152547836303711, -1.3163338899612427, 2.4909064769744873, 1.5780632495880127, 0.8522765636444092, 0.3818080425262451, 2.4398858547210693, 0.09437491744756699, -0.4598529636859894, 0.15297850966453552, -1.3553317785263062, 0.48598402738571167, -0.09581363946199417, 0.3445990979671478, 0.37423524260520935, -1.0447908639907837, 1.3303216695785522, -1.0006709098815918, 0.9023624062538147, -1.5790756940841675, 0.3858772814273834, -0.8055641651153564, 0.05724462494254112, -0.6367143392562866, -2.1684751510620117, -1.0997687578201294, 0.0837288349866867, 0.6391448378562927, -1.453936219215393, -2.1098473072052, -0.8568490743637085, -0.19603924453258514, -0.6178174614906311, -0.7942455410957336, 0.8660274147987366, 0.9813162684440613, 0.7571935653686523, -0.922544538974762, 0.09168804436922073, -0.17823214828968048, -1.4120382070541382, 0.8979007601737976, 0.10336226224899292, -0.4695143401622772, 1.4329354763031006, 0.7792069911956787, 0.8713791966438293, 1.2153939008712769, 0.7331652641296387, -1.2500243186950684, -0.37364959716796875, 1.0182842016220093, -0.7538830041885376, 0.05186493694782257, 0.4124680161476135, 1.4561649560928345, 1.313318133354187, 0.9128286838531494, -0.17215624451637268, 0.9290069341659546, 0.5420740842819214, -0.1479790210723877, 1.3713910579681396, 0.5081032514572144, -0.7426578402519226, 1.2794140577316284, -0.24214041233062744, 0.6064618229866028, -1.5118073225021362, -0.007087988778948784, -0.9341909289360046, -0.9610256552696228, -1.899191975593567, 2.1544809341430664, -0.139062762260437, 0.24888040125370026, 1.118491768836975, -0.11150328814983368, 0.5236306190490723, -1.3491207361221313, 0.20616845786571503, -0.40428897738456726, 0.510039746761322, 0.1903166025876999, -1.3764606714248657, -1.8428354263305664, -1.0060802698135376, 0.7459077835083008, 0.4336598515510559, 0.4115520417690277, -2.0111770629882812, -2.681530475616455, -0.61628657579422, -0.32557496428489685, -0.5966591835021973, 0.3610243797302246, -0.16507138311862946, -1.1802427768707275, -0.8745474815368652, 0.1481192708015442, 0.07739625126123428, -0.0698157250881195, 2.2715704441070557, -0.2587800920009613, 0.12378431111574173, -0.992669403553009, -0.35602903366088867, 0.5269494652748108, 0.5694980621337891, -0.15533877909183502, 0.3967377245426178, 0.05658234655857086, -1.8801957368850708, -3.27447509765625, 2.311171531677246, 0.8309165835380554, -0.056763678789138794, -1.7380383014678955, -1.2754769325256348, 1.3559104204177856, -2.788177490234375, -0.3243311643600464, 0.8685643672943115, 0.14323113858699799, 0.7547928094863892, -0.131249338388443, 2.393451690673828, -1.0279959440231323, -1.3622270822525024, 0.27453234791755676, 1.4295806884765625, -0.05589188262820244, 1.0490049123764038, -0.5308372974395752, -1.1852744817733765, 1.3999205827713013, 1.5687130689620972, 1.5331289768218994, -0.4382525384426117, -1.1552072763442993, -1.1693729162216187, -0.22463519871234894, 0.5824378132820129, 0.509171724319458, -1.4344056844711304, 0.19517281651496887, 0.017730241641402245, 0.09012774378061295, -0.023697257041931152, -0.20192024111747742, 0.5932751297950745, 0.12685956060886383, 0.2254108190536499, -0.6907558441162109, 0.39577779173851013, -0.12115202099084854, -2.115161180496216, 0.4836100935935974, -0.2643015682697296, 0.9262818694114685, -0.16209876537322998, -0.09574252367019653, 0.21168993413448334, -0.01185640785843134, -0.44692981243133545, -1.7932004928588867, 1.8426883220672607, -1.5787862539291382, -0.3415219485759735, 0.653138279914856, 0.3800170123577118, 0.5672684907913208, 0.7691897749900818, 1.2633726596832275, 0.2972833514213562, 2.9066426753997803, -1.0134117603302002, 0.22737890481948853, -0.3608529269695282, -0.5451372861862183, 0.28726518154144287, 1.4582406282424927, -0.7102085947990417, -0.7109557390213013, 0.7244888544082642, 0.5439940690994263, 1.9577568769454956, 1.0278799533843994, 0.22959129512310028, -0.3293730318546295, 0.21526280045509338, -0.17521701753139496, -1.3650909662246704, 0.008234765380620956, -0.1751221865415573, -0.2325407862663269, -0.42930877208709717, 1.6900694370269775, 0.6962661743164062, 0.9368627667427063, -0.6076712012290955, -1.022621512413025, -0.2828836441040039, 0.24407178163528442, -0.3453289270401001, -0.18822795152664185, 0.9614195227622986, 0.2992241084575653, -0.9840149283409119, -0.4279992878437042, -0.657539963722229, 0.18093839287757874, -1.167251706123352, -0.17874735593795776, 0.6926417350769043, 0.028723962604999542, -0.913876473903656, -0.6164140701293945, 0.12435099482536316, 0.48308154940605164, 0.8370703458786011, -0.14521142840385437, -1.3172497749328613, 1.424992561340332, 0.29788631200790405, 0.40607187151908875, 0.30551663041114807, -1.0374903678894043, -0.31930452585220337, 0.2572585344314575, 1.032628059387207, 0.06018184870481491, -3.119948387145996, -0.24135127663612366, 0.3526540994644165, 1.2938891649246216, 0.7113162279129028, 2.490208864212036, 1.4243422746658325, 1.3476637601852417, 0.9427873492240906, -1.3824353218078613, 0.8470504879951477, 0.5994841456413269, -0.5260205268859863, 0.5866459608078003, 0.418066143989563, -0.4650195837020874, -0.1233912855386734, -0.2632948160171509, 0.23047909140586853, -0.4245753884315491, 0.5449131727218628, -1.9175878763198853, -0.2819282114505768, -1.6846873760223389, -0.48289939761161804, 0.5169365406036377, 0.02161206305027008, -0.11063522100448608, 0.06879237294197083, 1.5695009231567383, -1.1062519550323486, 0.8877331614494324, -1.512596845626831, -0.46744096279144287, 0.1635974645614624, -0.8516802787780762, 0.20513737201690674, 0.7286924123764038, -1.1937426328659058, -0.07737187296152115, -0.37656623125076294, -1.940383791923523, -0.18707415461540222, -0.7977706789970398], "dtype": "float32"}, {"__nd__": [0.7057249546051025, 0.854948878288269, 0.0934964045882225, 0.49988988041877747, 0.2446163296699524, 0.285042941570282, 0.7467974424362183, 0.9770298600196838, 0.27890753746032715, 0.5425630211830139, 0.31031233072280884, 0.062139227986335754, 0.6024695038795471, 0.15535324811935425, 0.6876676082611084, 0.7472129464149475, 0.14234432578086853, 0.5962836146354675, 0.20836345851421356, 0.7631357908248901, 0.285936176776886, 0.6895727515220642, 0.6478928327560425, 0.5751852989196777, 0.21764692664146423, 0.8342790007591248, 0.3548252582550049, 0.7705613374710083, 0.474972128868103, 0.44807907938957214, 0.7122223377227783, 0.3408597409725189, 0.7634636759757996, 0.44492077827453613, 0.740540623664856, 0.6975468993186951, 0.9453259110450745, 0.40864297747612, 0.3489898443222046, 0.9948298335075378, 0.5163481831550598, 0.4184684753417969, 0.7007313370704651, 0.25013023614883423, 0.9577507376670837, 0.7538508176803589, 0.6490341424942017, 0.2918492257595062, 0.809325635433197, 0.9191300868988037, 0.7914100885391235, 0.9122895002365112, 0.09219107031822205, 0.7412245869636536, 0.8663061261177063, 0.4761693775653839, 0.08014252036809921, 0.5425769090652466, 0.24253864586353302, 0.032602787017822266, 0.9357964396476746, 0.8323116302490234, 0.6344637274742126, 0.7485085129737854, 0.6714454293251038, 0.46283382177352905, 0.121271513402462, 0.2887426018714905, 0.5060701370239258, 0.5820850729942322, 0.3015058636665344, 0.1272195428609848, 0.3423459231853485, 0.18563446402549744, 0.930577278137207, 0.22096243500709534, 0.3940616548061371, 0.5418621897697449, 0.25229477882385254, 0.4949019253253937, 0.5527174472808838, 0.0491056889295578, 0.9474173188209534, 0.5379729270935059, 0.14605048298835754, 0.9824180006980896, 0.16824278235435486, 0.7274504899978638, 0.482444167137146, 0.25755053758621216, 0.3796198070049286, 0.48119622468948364, 0.370919406414032, 0.24702127277851105, 0.41608214378356934, 0.4358029067516327, 0.6877507567405701, 0.19755306839942932, 0.04547934606671333, 0.8337687253952026, 0.06929394602775574, 0.6314336061477661, 0.548957347869873, 0.975030779838562, 0.18992795050144196, 0.8876760601997375, 0.1302955597639084, 0.17150548100471497, 0.6538016200065613, 0.16566655039787292, 0.5865099430084229, 0.6750147938728333, 0.125433087348938, 0.7901574969291687, 0.42309069633483887, 0.9670588970184326, 0.011436053551733494, 0.07859750837087631, 0.6201355457305908, 0.25720879435539246, 0.4026874899864197, 0.9088809490203857, 0.34384045004844666, 0.051247332245111465, 0.14302778244018555, 0.5555664300918579, 0.7747755646705627, 0.7577711343765259, 0.4715495705604553, 0.2396315634250641, 0.12850837409496307, 0.22325661778450012, 0.20037440955638885, 0.9074196219444275, 0.4077524244785309, 0.11668526381254196, 0.6381484866142273, 0.4572353959083557, 0.02907915599644184, 0.9797609448432922, 0.9462941884994507, 0.2625889182090759, 0.8329894542694092, 0.4551050364971161, 0.6736708879470825, 0.653353214263916, 0.7318907380104065, 0.44049590826034546, 0.1465798169374466, 0.8043003082275391, 0.44042158126831055, 0.4420555830001831, 0.7592571377754211, 0.6853834390640259, 0.9889789819717407, 0.3879208266735077, 0.6348629593849182, 0.0837678536772728, 0.11255142092704773, 0.2547001242637634, 0.2799467146396637, 0.446199893951416, 0.2288137674331665, 0.936138391494751, 0.4279947280883789, 0.3823525905609131, 0.9020341038703918, 0.7957159280776978, 0.770178496837616, 0.9140985608100891, 0.7365683317184448, 0.30561649799346924, 0.9613307118415833, 0.02714688889682293, 0.897101104259491, 0.41941630840301514, 0.3376644253730774, 0.47457706928253174, 0.39980366826057434, 0.8604415655136108, 0.806104838848114, 0.7466607093811035, 0.23270653188228607, 0.43989264965057373, 0.07504522055387497, 0.7425045967102051, 0.04215057194232941, 0.3959467113018036, 0.1511133462190628, 0.8520627021789551, 0.9185346364974976, 0.28950977325439453, 0.8723992705345154, 0.17803563177585602, 0.7111397385597229, 0.25066524744033813, 0.44552358984947205, 0.6110087037086487, 0.3511492908000946, 0.19784513115882874, 0.5878562927246094, 0.7939566373825073, 0.9958166480064392, 0.3223147690296173, 0.25761690735816956, 0.07643157988786697, 0.2047143280506134, 0.025933435186743736, 0.24969251453876495, 0.725985050201416, 0.11469917744398117, 0.646344780921936, 0.29489225149154663, 0.22886034846305847, 0.4835885763168335, 0.4362388849258423, 0.4467865824699402, 0.11758997291326523, 0.5646038055419922, 0.9021064639091492, 0.009289282374083996, 0.08099506050348282, 0.5859522223472595, 0.31856223940849304, 0.2519151568412781, 0.9879576563835144, 0.30818745493888855, 0.9208565354347229, 0.6168197989463806, 0.030290525406599045, 0.26589417457580566, 0.07573599368333817, 0.537943422794342, 0.9083301424980164, 0.052621886134147644, 0.02802717685699463, 0.029325997456908226, 0.27298420667648315, 0.1662149280309677, 0.9682813286781311, 0.7750470638275146, 0.939283013343811, 0.5798512697219849, 0.8210633993148804, 0.7088701128959656, 0.29211312532424927, 0.6526764631271362, 0.289256751537323, 0.46424344182014465, 0.8227136135101318, 0.22070321440696716, 0.17490419745445251, 0.851578414440155, 0.4632019102573395, 0.17382685840129852, 0.3178371787071228, 0.28056851029396057, 0.8962182402610779, 0.47277313470840454, 0.9201880693435669, 0.47028031945228577, 0.9394099116325378, 0.9165433049201965, 0.5675777792930603, 0.786361813545227, 0.2871455252170563, 0.11547257006168365, 0.6042254567146301, 0.05519085377454758, 0.4674983620643616, 0.6200212836265564, 0.7675755023956299, 0.8966580033302307, 0.2040802538394928, 0.865459680557251, 0.496288001537323, 0.622783899307251, 0.46199652552604675, 0.8477858901023865, 0.2999999225139618, 0.008125674910843372, 0.8292075991630554, 0.6616435647010803, 0.7165061831474304, 0.12419383972883224, 0.20616522431373596, 0.625433623790741, 0.7471646070480347, 0.6611494421958923, 0.5959951281547546, 0.6547896862030029, 0.1478193998336792, 0.4507160186767578, 0.32950615882873535, 0.7361946105957031, 0.5174099206924438, 0.7200554013252258, 0.6948289275169373, 0.9052076935768127, 0.8562219142913818, 0.051626138389110565, 0.6677396297454834, 0.37347081303596497, 0.10085657238960266, 0.4783187806606293, 0.8432270288467407, 0.7964068651199341, 0.1058022677898407, 0.20233073830604553, 0.6949390172958374, 0.968195915222168, 0.07718166708946228, 0.08874581009149551, 0.695624053478241, 0.6159710884094238, 0.8110842704772949, 0.19826899468898773, 0.4942558705806732, 0.8009063005447388, 0.3823811113834381, 0.39959821105003357, 0.03283292427659035, 0.7978233098983765, 0.4466037154197693, 0.37799641489982605, 0.6554109454154968, 0.8529432415962219, 0.8662599325180054, 0.342448890209198, 0.6082830429077148, 0.38746583461761475, 0.8388484716415405, 0.7935746312141418, 0.6617377400398254, 0.8235282897949219, 0.5098089575767517, 0.5710331797599792, 0.1626371443271637, 0.7079200148582458, 0.28816962242126465, 0.5618203282356262, 0.002071334980428219, 0.3974110186100006, 0.6158143877983093, 0.6307198405265808, 0.5637473464012146, 0.25201863050460815, 0.6543802618980408, 0.6612261533737183, 0.5899431705474854, 0.37706294655799866, 0.15208807587623596, 0.5566045641899109, 0.05190025269985199, 0.916222333908081, 0.3525911569595337, 0.2829148769378662, 0.450684130191803, 0.10383766144514084, 0.8841753602027893, 0.9795714616775513, 0.8232486248016357, 0.5737705230712891, 0.9407293200492859, 0.3304104208946228, 0.4651155173778534, 0.1715383231639862, 0.4247169494628906, 0.927596926689148, 0.6633753180503845, 0.5456851720809937, 0.5359809994697571, 0.635047435760498, 0.6099027991294861, 0.45251816511154175, 0.5108402371406555, 0.10134122520685196, 0.44344356656074524, 0.6747488379478455, 0.22334395349025726, 0.27692070603370667, 0.80482417345047, 0.13588517904281616, 0.19546177983283997, 0.8184902667999268, 0.12094831466674805, 0.3005170226097107, 0.9103597402572632, 0.4129326641559601, 0.8174883127212524, 0.7645699977874756, 0.019972797483205795, 0.2196066677570343, 0.9118710160255432, 0.32130560278892517, 0.40389716625213623, 0.934796154499054, 0.10969013720750809, 0.9716711640357971, 0.6467198729515076, 0.6285250186920166, 0.0008457356016151607, 0.6307234168052673, 0.8221396803855896, 0.39406928420066833, 0.07310543209314346, 0.13204365968704224, 0.07758726179599762, 0.41498062014579773, 0.8170669078826904, 0.05869324877858162, 0.40575456619262695, 0.40083280205726624, 0.3980712890625, 0.5267326235771179, 0.5352633595466614, 0.20406104624271393, 0.7944493293762207, 0.5170342922210693, 0.35208195447921753, 0.9641202092170715, 0.7058357000350952, 0.16302597522735596, 0.9691534638404846, 0.5782573223114014, 0.3402772545814514, 0.23743653297424316, 0.06784328818321228, 0.24393202364444733, 0.28902044892311096, 0.34413132071495056, 0.9879720211029053, 0.5924455523490906, 0.1921703815460205, 0.37058761715888977, 0.4323784112930298, 0.08314815908670425, 0.46839606761932373, 0.5082892775535583, 0.9096124172210693, 0.4332273602485657, 0.6886699795722961, 0.6796634197235107, 0.17496024072170258, 0.5671796202659607, 0.4490261375904083, 0.1626632660627365, 0.02393132448196411, 0.9958374500274658, 0.09051088988780975, 0.8866096138954163, 0.3665370047092438, 0.08170259743928909, 0.9646189212799072, 0.660266637802124, 0.40453699231147766, 0.837957501411438, 0.7077130079269409, 0.4155561029911041, 0.7178558707237244, 0.7378352880477905, 0.17218756675720215, 0.7163593173027039, 0.9165647625923157, 0.6876439452171326, 0.7993513345718384, 0.9899770617485046, 0.7963619828224182, 0.4697687327861786, 0.562157392501831, 0.7668201923370361, 0.6323398351669312, 0.07091642916202545, 0.06015748158097267, 0.27264729142189026, 0.8182665705680847, 0.22160041332244873, 0.31900954246520996, 0.8231207728385925, 0.12893220782279968, 0.2798326313495636, 0.2564210295677185, 0.04736597090959549, 0.7426241636276245, 0.3206166625022888, 0.4076133370399475, 0.3330325782299042, 0.06582058221101761, 0.8448415994644165, 0.13449645042419434, 0.2855237126350403, 0.7208223342895508, 0.34836626052856445, 0.5068077445030212, 0.3301902115345001, 0.5601713061332703, 0.3173162341117859, 0.4564225971698761, 0.008890286087989807, 0.015199950896203518, 0.5068934559822083, 0.817022442817688, 0.9986169934272766, 0.8031591176986694, 0.5244194269180298, 0.6443382501602173, 0.6714072227478027, 0.7296179533004761, 0.991834282875061, 0.3436763286590576, 0.4463500380516052, 0.7789769768714905, 0.21602179110050201, 0.09595028311014175, 0.8989338874816895, 0.01595199853181839, 0.8939353227615356, 0.914818286895752, 0.1656104028224945, 0.7551727890968323, 0.28882864117622375, 0.27108484506607056, 0.9836257100105286, 0.8869724869728088, 0.7643600702285767, 0.4360775947570801, 0.9858700633049011, 0.570486843585968, 0.9800450801849365, 0.7213862538337708, 0.04681255295872688, 0.7982065081596375, 0.837978184223175, 0.8810876607894897, 0.6220525503158569, 0.037615276873111725, 0.8674815893173218, 0.42198681831359863, 0.19658102095127106, 0.20848040282726288, 0.20156258344650269, 0.8156508803367615, 0.07268578559160233, 0.9182376265525818, 0.016871241852641106, 0.722892165184021, 0.23057694733142853, 0.5147964954376221, 0.17729757726192474, 0.8996950387954712, 0.15589496493339539, 0.4668366312980652, 0.6085731387138367, 0.45320799946784973, 0.2161933183670044, 0.2899831235408783, 0.934506356716156, 0.12522634863853455, 0.017576081678271294, 0.9087177515029907, 0.8862205743789673, 0.6851661205291748, 0.4719628393650055, 0.9519408941268921, 0.4131331443786621, 0.012975938618183136, 0.43521395325660706, 0.529211163520813, 0.5483018755912781, 0.8881709575653076, 0.06773213297128677, 0.854145348072052, 0.5222989916801453, 0.7321836948394775, 0.4137372374534607, 0.2961517870426178, 0.3357003331184387, 0.27450406551361084, 0.4535571038722992, 0.5978543162345886, 0.8309438228607178, 0.17442786693572998, 0.45006951689720154, 0.5320907831192017, 0.818175196647644, 0.7102118134498596, 0.623180091381073, 0.6743481755256653, 0.4481610953807831, 0.8065134286880493, 0.8373637199401855, 0.5662263035774231, 0.1170128881931305, 0.4262271523475647, 0.2647109627723694, 0.1917954534292221, 0.29669639468193054, 0.8193408250808716, 0.34779229760169983, 0.4296596348285675, 0.38277921080589294, 0.1845981776714325, 0.5713686347007751, 0.41673311591148376, 0.9365896582603455, 0.6184582710266113, 0.9779860377311707, 0.8588871955871582, 0.49076777696609497, 0.016397440806031227, 0.21293310821056366, 0.11933683604001999, 0.7375672459602356, 0.13775663077831268, 0.6451295614242554, 0.32253652811050415, 0.48051929473876953, 0.7566872835159302, 0.29833826422691345, 0.17172303795814514, 0.2880389988422394, 0.9436244368553162, 0.5528345108032227, 0.2871234118938446, 0.14571654796600342, 0.06822140514850616, 0.33400726318359375, 0.6140109896659851, 0.45681998133659363, 0.812220573425293, 0.9947057962417603, 0.5702311396598816, 0.34493395686149597, 0.7947162389755249, 0.08871473371982574, 0.7595641613006592, 0.8816657662391663, 0.7760822772979736, 0.09866859763860703, 0.5810592770576477, 0.8021994233131409, 0.41158363223075867, 0.844047486782074, 0.1660054624080658, 0.5287381410598755, 0.5581948757171631, 0.7882542610168457, 0.41202181577682495, 0.25825250148773193, 0.7121237516403198, 0.11424089223146439, 0.36785608530044556, 0.4680415093898773, 0.7877964377403259, 0.4247654378414154, 0.5621921420097351, 0.6783180236816406, 0.6755548119544983, 0.9635146856307983, 0.463181734085083, 0.5717672109603882, 0.01940631866455078, 0.19185444712638855, 0.24877899885177612, 0.2111942172050476, 0.5191103219985962, 0.1993263065814972, 0.5700426697731018, 0.82584148645401, 0.07262935489416122, 0.24289967119693756, 0.4967051148414612, 0.046154171228408813, 0.14325778186321259, 0.5749762654304504, 0.39950788021087646, 0.9037125110626221, 0.5704394578933716, 0.757084310054779, 0.34587085247039795, 0.3770763874053955, 0.8534501791000366, 0.6152943968772888, 0.9144701957702637, 0.25240564346313477, 0.11460940539836884, 0.6772404909133911, 0.12509943544864655, 0.07598291337490082, 0.39041703939437866, 0.6864987015724182, 0.3603608310222626, 0.7530660629272461, 0.45529431104660034, 0.28491291403770447, 0.5342788696289062, 0.604515552520752, 0.14993968605995178, 0.5915592908859253, 0.1685125231742859, 0.9681732058525085, 0.5315338373184204, 0.20543724298477173, 0.042533330619335175, 0.2145373821258545, 0.31830233335494995, 0.5734805464744568, 0.6635378003120422, 0.7115905284881592, 0.44110944867134094, 0.630198061466217, 0.5535587668418884, 0.2964804768562317, 0.2189837098121643, 0.3992635905742645, 0.6896237134933472, 0.13449080288410187, 0.6359798908233643, 0.020734209567308426, 0.18368595838546753, 0.6697959899902344, 0.8080883622169495, 0.7671982049942017, 0.6834942102432251, 0.2540881037712097, 0.9878446459770203, 0.6009690165519714, 0.9101248979568481, 0.9364516735076904, 0.23867252469062805, 0.9698075652122498, 0.08120868355035782, 0.6373606324195862, 0.256687194108963, 0.09408686310052872, 0.5694360136985779, 0.21255524456501007, 0.7864004373550415, 0.4615944027900696, 0.269769549369812, 0.011786538176238537, 0.6920069456100464, 0.6364942193031311, 0.8051963448524475, 0.38320231437683105, 0.9149069786071777, 0.023592988029122353, 0.9188183546066284, 0.7647116780281067, 0.10237264633178711, 0.9871436953544617, 0.0897718071937561, 0.7510737776756287, 0.24438197910785675, 0.6909843683242798, 0.8370243906974792, 0.11793873459100723, 0.32452958822250366, 0.8951908946037292, 0.4774826169013977, 0.15308305621147156, 0.08685710281133652, 0.12300166487693787, 0.10579770058393478, 0.35687386989593506, 0.7765896916389465, 0.452919602394104, 0.0983489379286766, 0.35259291529655457, 0.4837037920951843, 0.4481954276561737, 0.64126056432724, 0.6805934309959412, 0.7477760314941406, 0.5358681082725525, 0.029946843162178993, 0.7696399688720703, 0.7831279635429382, 0.7294098138809204, 0.4375581443309784, 0.8868927955627441, 0.09633012115955353, 0.7401564121246338, 0.5963119268417358, 0.7001442909240723, 0.0672585666179657, 0.9681426286697388, 0.03023293986916542, 0.17687897384166718, 0.5166168212890625, 0.5728997588157654, 0.1779341995716095, 0.2874286472797394, 0.9151440858840942, 0.32132163643836975, 0.4452156722545624, 0.11151249706745148, 0.5689390897750854, 0.7009254693984985, 0.740875780582428, 0.23261195421218872, 0.3692445755004883, 0.654403805732727, 0.9729508757591248, 0.7698057293891907, 0.21924962103366852, 0.06551530957221985, 0.3281269073486328, 0.5293447971343994, 0.27202385663986206, 0.7898894548416138, 0.25653788447380066, 0.12388630956411362, 0.6024402976036072, 0.7601385712623596, 0.20496442914009094, 0.33378830552101135, 0.25978830456733704, 0.02878272533416748, 0.6827777624130249, 0.05082406476140022, 0.6119324564933777, 0.6271377801895142, 0.5301708579063416, 0.04246621951460838, 0.03735196590423584, 0.5150322914123535, 0.49059587717056274, 0.015480260364711285, 0.01076376624405384, 0.7733554244041443, 0.3244816064834595, 0.7169274687767029, 0.19858723878860474, 0.2656373381614685, 0.8982089757919312, 0.19771826267242432, 0.4082777202129364, 0.49863559007644653, 0.711885392665863, 0.3207509219646454, 0.6493085026741028, 0.2222355306148529, 0.6813405752182007, 0.021222030743956566, 0.18681462109088898, 0.06101437285542488, 0.10201912373304367, 0.7482159733772278, 0.4640614986419678, 0.7832897901535034, 0.9749870300292969, 0.1860867142677307, 0.7689496874809265, 0.9719066023826599, 0.3093525469303131, 0.07720945030450821, 0.46176764369010925, 0.542599081993103, 0.6880753636360168, 0.5708144903182983, 0.30175575613975525, 0.16243141889572144, 0.3400014340877533, 0.4688442647457123, 0.18558086454868317, 0.2986828684806824, 0.8840554356575012, 0.42350608110427856, 0.12530498206615448, 0.6655415296554565, 0.3549523949623108, 0.5151125192642212, 0.39272356033325195, 0.1147596463561058, 0.7370633482933044, 0.568810760974884, 0.9889789819717407, 0.0047342656180262566, 0.5772925615310669, 0.42574813961982727, 0.43477535247802734, 0.7225639224052429, 0.24240891635417938, 0.399951308965683, 0.13332246243953705, 0.5667563676834106, 0.4373795688152313, 0.0426395945250988, 0.23035790026187897, 0.3953455686569214, 0.5318840742111206, 0.09288071095943451, 0.6193492412567139, 0.8176847100257874, 0.3328525125980377, 0.06674116849899292, 0.1466177999973297, 0.7629795074462891, 0.5322260856628418, 0.13611043989658356, 0.16791120171546936, 0.8331626653671265, 0.6794942617416382, 0.6956266164779663, 0.14040905237197876, 0.9904685020446777, 0.25393664836883545, 0.35399889945983887, 0.23394906520843506, 0.9426423907279968, 0.6315892338752747, 0.5532799363136292, 0.414213627576828, 0.7171370387077332, 0.46856847405433655, 0.46165990829467773, 0.1518438160419464, 0.8235776424407959, 0.06617023795843124, 0.9074894189834595, 0.9164444208145142, 0.5233621597290039, 0.16334819793701172, 0.21788714826107025, 0.1396980881690979, 0.7008470296859741, 0.3627524971961975, 0.982330322265625, 0.3413195312023163, 0.32973188161849976, 0.6767038106918335, 0.8782767057418823, 0.5266700983047485, 0.12219949811697006, 0.1707080751657486, 0.7150632739067078, 0.7657718062400818, 0.9898869395256042, 0.5732265114784241, 0.7820449471473694, 0.674830973148346, 0.549789547920227, 0.8082476258277893, 0.20379213988780975, 0.15313157439231873, 0.3097216486930847, 0.38497793674468994, 0.9494956731796265, 0.8249403834342957, 0.7443501949310303, 0.8915281891822815, 0.3834291696548462, 0.9847447276115417, 0.9076462388038635, 0.18519054353237152, 0.37773221731185913, 0.30795904994010925, 0.1930071860551834, 0.9372280836105347, 0.3906494677066803, 0.8690319061279297, 0.5965050458908081, 0.05979764461517334, 0.4335363805294037, 0.7800418138504028, 0.318947434425354, 0.2927455008029938, 0.09353697299957275, 0.4740464687347412, 0.9424002766609192, 0.018715722486376762, 0.8867527842521667, 0.5997104644775391, 0.2803289592266083, 0.8983492255210876, 0.9275097250938416, 0.9731449484825134, 0.4626820385456085, 0.41444990038871765, 0.6646049618721008, 0.8888010382652283, 0.34997817873954773, 0.33196693658828735, 0.4086766242980957, 0.02103586494922638, 0.8400704264640808, 0.4576227068901062, 0.4667685627937317, 0.6328137516975403, 0.8089245557785034, 0.7054206132888794, 0.3335813879966736, 0.2946935296058655, 0.3331200182437897, 0.5321716070175171, 0.8107723593711853, 0.007968691177666187, 0.6052743196487427, 0.8438578844070435, 0.561456024646759, 0.38106927275657654, 0.4740556478500366, 0.5145765542984009, 0.6270368099212646, 0.07510644942522049, 0.7924485802650452, 0.5516571998596191, 0.2771161198616028, 0.9155441522598267, 0.39853814244270325, 0.7748503088951111, 0.29753750562667847, 0.5481941103935242, 0.18987177312374115, 0.2736159861087799, 0.3292052149772644, 0.14361225068569183, 0.7605120539665222, 0.45064741373062134, 0.3470838665962219, 0.5850711464881897, 0.22989676892757416, 0.4378507733345032, 0.5677094459533691], "dtype": "float32"}, 0.1], "e": {"__nd__": [-0.37654533982276917, -2.137425422668457, 0.0, 0.25038421154022217, 1.2049729824066162, 0.642071008682251, -0.7150675058364868, -0.8041962385177612, 2.233997344970703, 0.84070885181427, 2.03493332862854, 0.0, -0.9089818596839905, 0.4280770421028137, 0.5090745091438293, 0.6217913627624512, 0.6021136045455933, 0.22450515627861023, 0.19347694516181946, -1.6694444417953491, -0.18377910554409027, -0.8306885957717896, 0.1400970220565796, -0.5194958448410034, 0.6872453093528748, 0.9100841879844666, 0.34299153089523315, 0.35129794478416443, 0.10327409952878952, -0.4975532293319702, -0.1827791929244995, -0.5507178902626038, 0.4310968518257141, 0.015681570395827293, 0.6458791494369507, -1.4762760400772095, 0.9864211082458496, -0.8474403619766235, -0.815869152545929, -0.21938109397888184, -0.625982940196991, 0.3234724700450897, -0.6379608511924744, -1.1883084774017334, -0.9397808909416199, 1.4577593803405762, 0.049175925552845, -1.2972559928894043, 0.00930822640657425, -1.7288295030593872, 1.9884073734283447, -1.6938647031784058, 0.0, 0.6040285229682922, -1.6111198663711548, 0.33364060521125793, 0.0, 0.5195280909538269, 0.29001811146736145, 0.0, 0.17878283560276031, 0.3738058805465698, -0.14079061150550842, 0.7020063996315002, -1.0459824800491333, 0.8797369599342346, -0.5614461898803711, -1.2118371725082397, 0.4058321714401245, 1.8810617923736572, 1.0685313940048218, -0.5729442834854126, 0.7756218910217285, -0.5046768188476562, -0.13779941201210022, 0.09947610646486282, -2.5746872425079346, 0.21290530264377594, -1.144010305404663, -0.7748873829841614, -1.6382116079330444, 0.0, -1.0480716228485107, 0.9173303842544556, 1.8511571884155273, -0.028005672618746758, 1.2131010293960571, -0.29331186413764954, -2.1244144439697266, 0.16657403111457825, 0.49516475200653076, -0.4762406349182129, 0.3358047306537628, 0.636173665523529, -0.9593417644500732, -1.6409268379211426, -0.2458379715681076, -0.23451171815395355, 0.0, 1.0969408750534058, 0.0, -0.46294113993644714, 0.7765408158302307, 1.0454790592193604, 0.792396068572998, 1.1638736724853516, -0.43249329924583435, 0.8220071196556091, 2.249424457550049, 0.8929618000984192, -0.6887707710266113, 0.675236165523529, 1.4026447534561157, 0.4099847674369812, -0.6260008215904236, 1.714534044265747, 0.0, 0.0, -0.017972860485315323, -1.3262953758239746, 1.354944109916687, 0.38654181361198425, -1.2964425086975098, 0.0, -0.47955605387687683, -2.1142096519470215, -0.7654501795768738, 0.29283493757247925, 0.7063366174697876, 0.18396101891994476, 0.05018334835767746, 0.5222111344337463, -0.23441791534423828, 1.3077274560928345, 0.16515739262104034, 0.3265795409679413, 0.6008877754211426, -1.1746320724487305, 0.0, 1.6984213590621948, 0.37256523966789246, -0.3140178918838501, 0.37942615151405334, -0.5351358652114868, 0.30951812863349915, -0.7398567795753479, 0.306687593460083, -1.7530028820037842, 1.478799819946289, -0.569953203201294, -1.7569347620010376, -0.24871300160884857, -0.41221803426742554, 0.1693315953016281, -1.2725006341934204, 0.44162845611572266, 4.026186466217041, 0.0, 0.3752395212650299, -0.3452431857585907, 0.21400922536849976, -2.0131213665008545, -1.2788994312286377, 0.5107328295707703, -0.042686693370342255, 1.8169313669204712, -0.7831082344055176, 0.1938883662223816, 3.236801862716675, -0.8506239056587219, -1.004259467124939, -0.0390334278345108, -0.04951408877968788, 0.0, 0.1439732015132904, -0.8408215045928955, 0.23491284251213074, 2.931774377822876, 1.4130820035934448, -3.091299057006836, -0.15779657661914825, -0.9723228812217712, 0.6914028525352478, -0.19251498579978943, 0.0, 1.0347062349319458, 0.0, 0.20730118453502655, -0.04477517306804657, 0.3779776990413666, 1.4391998052597046, 0.593766450881958, -0.39014732837677, 1.416760802268982, 0.21345597505569458, -0.08623895794153214, -0.8793076276779175, -0.6288803815841675, -0.7067186832427979, -3.102619171142578, 1.1298291683197021, 0.6294195652008057, -0.23807844519615173, 1.1622042655944824, 0.4866531193256378, 0.0, -2.749737501144409, 0.0, 0.5913212895393372, 1.7885410785675049, 2.044772148132324, 1.5842647552490234, -1.2906773090362549, -2.44185733795166, 0.6338551044464111, 0.03068426437675953, 1.069503664970398, -0.1398715227842331, 0.24417220056056976, -1.6666545867919922, 0.0, 0.0, 0.3888726532459259, 0.5104511380195618, -0.1302468329668045, 0.35200127959251404, -0.015118923969566822, 0.8435550332069397, 0.557492196559906, 0.0, -0.6532243490219116, 0.0, -0.11288389563560486, 1.1020087003707886, 0.0, 0.0, 0.0, 0.1334729939699173, 0.2851715683937073, 0.6826264262199402, -1.4077038764953613, -0.6607592701911926, -0.10540689527988434, 1.7786105871200562, -2.6224205493927, 0.48790186643600464, -0.6000509858131409, 0.47930818796157837, -0.5466432571411133, 0.40505677461624146, 1.7384867668151855, 0.39355167746543884, 0.4815555214881897, 0.4142865836620331, -0.8336817026138306, 0.6033532023429871, -0.2314961850643158, -0.9019041061401367, -0.04283102974295616, -0.25254276394844055, 0.8371998071670532, -0.04759914055466652, -0.5882689952850342, 1.617746353149414, 1.0982086658477783, 1.4108105897903442, 0.675186038017273, -0.12188778072595596, 0.0, -0.3103777766227722, 1.0529896020889282, 0.12433698028326035, -1.6476036310195923, -0.04342533275485039, -1.5734491348266602, 0.1842953860759735, -1.33757746219635, -1.4043735265731812, 0.13175494968891144, -0.08690480887889862, 0.0, 0.8313597440719604, 2.4034409523010254, -1.6459730863571167, 2.0220553874969482, -0.7642105221748352, -0.5489895939826965, -0.19876357913017273, -1.6058636903762817, 1.4434747695922852, 0.5474373698234558, -0.8349491357803345, -0.3078635334968567, -1.684843897819519, -0.38639748096466064, 1.0402368307113647, -1.905877709388733, -0.48471149802207947, 0.1631019413471222, 1.1647104024887085, 0.0, -0.5080588459968567, -0.8225632309913635, 0.5004441142082214, -0.40361645817756653, -0.3444660007953644, 1.2002041339874268, 1.8774867057800293, 0.2954084277153015, -0.21357198059558868, 0.6347355246543884, 0.0, 0.0, -1.178418755531311, 2.6549201011657715, 0.8002222180366516, 0.1570776253938675, -0.21596069633960724, -2.794395685195923, 0.7920050621032715, 1.8104100227355957, 0.0, 0.75933837890625, 0.8028258085250854, -1.720141887664795, 1.039711594581604, 1.5053123235702515, 0.2538668215274811, -0.07020222395658493, 0.8755150437355042, -1.7119488716125488, 0.10364522784948349, 1.171491026878357, -1.3804837465286255, 1.102856993675232, -0.4646996557712555, -2.312030076980591, -0.11749773472547531, -1.0798851251602173, 0.3547329604625702, -1.3023791313171387, 0.0, -0.8209226131439209, 0.3959527909755707, -0.6599777340888977, -0.35878369212150574, 1.324177861213684, 0.23919828236103058, -0.43892455101013184, -1.3905645608901978, 1.2888349294662476, 0.8142346739768982, -0.36281731724739075, 0.0, -1.143975019454956, -1.1276054382324219, -0.9179863929748535, 0.39269694685935974, 0.3287176787853241, -0.82811439037323, 0.8197495937347412, 0.8718817234039307, 0.19212017953395844, 1.1537652015686035, -0.3192916214466095, -0.12557904422283173, 0.5253985524177551, -0.03782667964696884, 0.8821102380752563, -1.3918535709381104, 1.5841834545135498, -0.868079423904419, -0.18299762904644012, -1.5306031703948975, 0.6669137477874756, 1.4153904914855957, 1.1900522708892822, 0.9259722232818604, -1.7301591634750366, -0.8690756559371948, -0.5139588713645935, -1.0385936498641968, 1.2474391460418701, 0.20498257875442505, -0.027197562158107758, 0.541706383228302, -1.0573545694351196, -1.009278655052185, -0.9794022440910339, 0.552912175655365, 0.3795515298843384, 0.0, 1.198203444480896, -1.5081121921539307, -0.6227715611457825, 0.1639312207698822, -1.4986034631729126, -0.5913058519363403, 1.1266988515853882, 1.2251601219177246, 1.1963410377502441, 0.0, -0.23551949858665466, 1.1994718313217163, -0.4140511453151703, 0.0, 1.4725321531295776, 0.0, -0.6783915162086487, -0.010412526316940784, 0.0, 0.5360100865364075, 0.17317530512809753, 0.6182446479797363, -1.777332067489624, -0.954740047454834, 0.1813623607158661, -1.0318461656570435, -1.490749478340149, -1.3420220613479614, -1.0201821327209473, -0.19830544292926788, -1.1362509727478027, 0.2609987258911133, 1.8846935033798218, -0.16402924060821533, 1.4163321256637573, 0.0, -1.244248390197754, -0.2848995625972748, -1.0909276008605957, 0.8532615900039673, -1.281217098236084, 1.4695253372192383, 0.6424310803413391, -2.247011184692383, 0.0, 1.7255737781524658, -0.8783597350120544, 1.0947247743606567, -1.9982019662857056, 0.28017401695251465, -1.8459882736206055, 0.5359899997711182, -1.9866254329681396, -2.9254114627838135, 0.44010353088378906, 0.0, -0.6618551015853882, 0.0, 0.12301233410835266, -0.35944974422454834, 0.0, 1.8248950242996216, -1.3723645210266113, 0.6265522241592407, 0.3834838271141052, -1.582606315612793, 1.7941231727600098, 2.165954113006592, -1.0586355924606323, 1.9310005903244019, 0.062384579330682755, -0.1813870519399643, 1.6704989671707153, 0.2527031898498535, 0.3590843677520752, 1.6863055229187012, 1.431355595588684, 0.9923883080482483, -1.556166648864746, -1.7471004724502563, 0.0, 0.0, 0.16498172283172607, -1.1934133768081665, -0.20409615337848663, -1.4207724332809448, 1.8113045692443848, -1.1321250200271606, -0.13554918766021729, -0.2091105431318283, 0.0, -1.5776640176773071, -0.037046704441308975, -0.23248253762722015, 0.58880615234375, 0.0, -0.5152739882469177, 0.26620668172836304, -0.4162569046020508, 0.520128071308136, -0.11397796869277954, -0.7284101247787476, -0.03061891533434391, 0.40420442819595337, 2.5176472663879395, -0.5458132028579712, 0.0, 0.0, -2.3090763092041016, -1.9367091655731201, -1.320732593536377, -0.12692949175834656, 2.711085319519043, 0.4348521828651428, 1.3423058986663818, -0.246710404753685, -2.737470865249634, -2.570709466934204, 1.1317991018295288, 0.2053994983434677, -1.4427770376205444, 0.0, -1.7099500894546509, 0.0, 1.6126009225845337, -0.8498038053512573, -1.2312411069869995, 0.9381411671638489, -1.2169873714447021, 0.957651674747467, -1.3280432224273682, 0.8973095417022705, 0.10429660975933075, -0.48086491227149963, -0.9952052235603333, 0.2639719843864441, -2.1029562950134277, -0.3066268563270569, 0.0, 0.7537332773208618, -1.3931337594985962, 1.903820276260376, -0.3462008833885193, 0.0, 1.5400875806808472, 1.8477013111114502, -0.20863556861877441, -0.9900809526443481, 0.01783663220703602, -0.7671241760253906, 0.0, 1.7315882444381714, 0.0, 0.1615486890077591, 0.3844580352306366, -0.06454423069953918, 0.9350754618644714, 0.9769890308380127, -0.4107964038848877, 0.11676488071680069, 2.681786298751831, 0.46844351291656494, -0.206503763794899, 2.277435302734375, 0.010480616241693497, -1.219224452972412, 0.0, -0.9998068809509277, -0.5524694919586182, 0.36258378624916077, -0.14085213840007782, -0.8483185172080994, -0.6121740341186523, 0.0, 1.2957627773284912, -0.8011113405227661, -0.635798990726471, 2.264838218688965, 0.0, 1.6549086570739746, 1.4619247913360596, -0.16972403228282928, -1.5771015882492065, 0.5033490657806396, 0.7474168539047241, 1.075338363647461, 0.07988248765468597, 1.3383973836898804, 1.5448429584503174, -1.1715357303619385, 1.5883203744888306, 0.19393596053123474, -0.628604531288147, -0.3540308177471161, 2.086191415786743, 1.0933666229248047, -1.9878288507461548, 0.0927305668592453, -0.15538883209228516, -0.1950969696044922, -0.5701724886894226, -1.6181128025054932, 0.9573826193809509, -1.0440998077392578, -0.4111568033695221, -0.6693170070648193, -0.8086932301521301, 0.8258185386657715, 0.42641574144363403, -0.7490684986114502, -1.0497896671295166, -0.5215715765953064, 1.074471354484558, 0.13991263508796692, 0.2589818835258484, -0.36462628841400146, -0.28653380274772644, 0.0, -2.048250198364258, 1.7911758422851562, -0.8751595616340637, -0.631460428237915, 1.487218976020813, 0.24714860320091248, 0.046759504824876785, -0.6481326222419739, -1.7346327304840088, 0.20549485087394714, 1.3280682563781738, -0.9996169805526733, 1.514172077178955, -1.252380132675171, -0.8155679702758789, 0.0, 7.247594476211816e-05, 0.7676533460617065, -0.824153482913971, -1.1464018821716309, -2.074514150619507, -1.3374570608139038, -1.1956753730773926, -1.585517406463623, 0.0, -0.7376502156257629, -2.209216833114624, 2.759425163269043, 0.0, -1.3950104713439941, 1.3775426149368286, 1.294926643371582, -1.2792677879333496, -1.637757658958435, -1.1111853122711182, 1.2549843788146973, -0.3458453118801117, -0.8985532522201538, -0.0315249040722847, 1.3379905223846436, -1.5439448356628418, -0.42644643783569336, -1.5371952056884766, 2.413506031036377, 2.020355224609375, 1.0568468570709229, -0.14822399616241455, 0.7323629856109619, -2.335570812225342, 0.30714085698127747, -1.4846280813217163, 0.0, -0.7830288410186768, -0.38912972807884216, 0.6380214095115662, 0.7785320281982422, 0.291858434677124, -1.0492628812789917, 0.7510964274406433, 0.0, 1.3113654851913452, -1.0190341472625732, 0.0, -0.5296769142150879, 2.3902928829193115, 0.5210970640182495, -0.2159213125705719, 0.9818181395530701, 0.5590916872024536, 0.5126014351844788, -0.5706062912940979, 0.08463449031114578, 0.22354668378829956, -0.7140686511993408, -1.2430336475372314, 0.15446963906288147, 1.0760557651519775, 0.7192270159721375, 0.0, 0.5603500008583069, 2.8711142539978027, 0.288759708404541, 1.1578201055526733, 2.174555778503418, 0.05895901098847389, 0.2939223051071167, 1.160051703453064, 1.8751815557479858, 0.3478948175907135, 0.40335315465927124, 0.10093287378549576, -1.3395546674728394, 1.2418532371520996, 0.0, -1.9097251892089844, -0.1601519137620926, -1.9189221858978271, -1.1163289546966553, 0.02342665009200573, -1.0188064575195312, 1.1948031187057495, -1.626821517944336, 0.29903656244277954, 0.6269646286964417, 1.6305241584777832, 1.0333019495010376, 0.2680049538612366, 0.09852616488933563, 0.0, 1.274972677230835, -0.7197760939598083, 0.5891540050506592, -0.09347442537546158, 0.27822062373161316, -0.28042933344841003, 2.3837499618530273, -0.7185924649238586, 0.2931205630302429, 0.31876426935195923, 0.6880409121513367, 0.513059139251709, 0.0, 1.1038868427276611, -1.6591839790344238, 0.0, -1.357424020767212, 0.271773099899292, -1.3006747961044312, -1.053130865097046, -0.05360694229602814, 0.0, 0.655538022518158, 0.34369078278541565, 1.5734961032867432, -1.0358452796936035, -2.310025453567505, 0.0, -0.29439979791641235, 1.3867086172103882, 2.3917198181152344, -1.4625933170318604, 0.0, 1.753403663635254, 0.946973979473114, 0.4242311716079712, 2.710984468460083, 0.10486102104187012, -0.5109477639198303, 0.1699761301279068, -1.5059242248535156, 0.539982259273529, 0.0, 0.38288789987564087, 0.4158169627189636, -1.1608787775039673, 1.4781352281570435, -1.1118565797805786, 0.0, -1.7545286417007446, 0.4287525415420532, -0.8950713276863098, 0.06360514461994171, -0.7074604034423828, -2.409416913986206, -1.2219653129577637, 0.0, 0.7101609706878662, -1.6154847145080566, -2.3442749977111816, -0.9520545601844788, -0.2178213894367218, 0.0, -0.882495105266571, 0.9622527360916138, 1.0903514623641968, 0.0, -1.0250495672225952, 0.0, -0.19803573191165924, -1.5689314603805542, 0.9976675510406494, 0.11484696716070175, -0.521682620048523, 1.5921505689620972, 0.8657855987548828, 0.9681991338729858, 1.3504377603530884, 0.8146281242370605, -1.388916015625, -0.4151662290096283, 1.1314269304275513, -0.8376477956771851, 0.057627711445093155, 0.45829781889915466, 1.6179611682891846, 1.4592424631118774, 0.0, -0.19128473103046417, 1.032230019569397, 0.6023045778274536, -0.16442114114761353, 1.5237679481506348, 0.5645592212677002, -0.825175404548645, 1.4215712547302246, -0.2690449059009552, 0.6738464832305908, -1.679785966873169, 0.0, -1.0379899740219116, 0.0, -2.1102135181427, 2.3938677310943604, -0.15451419353485107, 0.0, 0.0, -0.12389254570007324, 0.5818118453025818, 0.0, 0.0, -0.449209988117218, 0.5667108297348022, 0.21146290004253387, -1.5294008255004883, -2.0475950241088867, -1.1178669929504395, 0.8287864923477173, 0.48184430599212646, 0.45728006958961487, -2.2346413135528564, -2.979478359222412, -0.6847628951072693, -0.3617499768733978, -0.6629546880722046, 0.0, -0.18341265618801117, 0.0, -0.9717194437980652, 0.16457697749137878, 0.08599583804607391, -0.0775730311870575, 2.5239672660827637, -0.2875334620475769, 0.1375381350517273, -1.102966070175171, -0.39558783173561096, 0.0, 0.6327756643295288, -0.17259864509105682, 0.44081971049308777, 0.06286928057670593, -2.089106559753418, -3.638305902481079, 2.5679686069488525, 0.9232407212257385, -0.06307075917720795, -1.9311537742614746, -1.4171966314315796, 1.5065672397613525, -3.097975254058838, -0.3603679835796356, 0.9650715589523315, 0.15914571285247803, 0.8386586904525757, -0.1458325982093811, 2.659390926361084, -1.142217755317688, -1.5135856866836548, 0.0, 1.5884230136871338, -0.062102094292640686, 1.1655610799789429, -0.5898192524909973, -1.3169716596603394, 1.5554673671722412, 1.7430145740509033, 1.7034766674041748, -0.48694729804992676, 0.0, -1.2993032932281494, -0.24959467351436615, 0.6471531391143799, 0.0, -1.593784213066101, 0.21685869991779327, 0.019700270146131516, 0.0, -0.026330286636948586, -0.22435583174228668, 0.6591946482658386, 0.14095507562160492, 0.25045648217201233, -0.7675065398216248, 0.43975311517715454, -0.13461336493492126, -2.3501791954040527, 0.5373445749282837, -0.2936684191226959, 1.0292021036148071, -0.18010975420475006, -0.10638058930635452, 0.23521104454994202, -0.013173786923289299, -0.49658870697021484, -1.9924451112747192, 2.047431468963623, -1.7542070150375366, -0.3794688582420349, 0.7257092595100403, 0.0, 0.630298376083374, 0.8546553254127502, 1.4037474393844604, 0.33031484484672546, 3.2296030521392822, -1.126013159751892, 0.2526432275772095, -0.4009477198123932, -0.605708122253418, 0.3191835582256317, 1.620267391204834, -0.7891206741333008, -0.7899508476257324, 0.8049876689910889, 0.6044378876686096, 2.175285577774048, 1.1420888900756836, 0.25510144233703613, -0.36597004532814026, 0.23918090760707855, -0.19468557834625244, -1.5167678594589233, 0.009149739518761635, -0.19458021223545074, -0.25837865471839905, -0.47700977325439453, 1.8778550624847412, 0.7736291289329529, 1.0409586429595947, -0.675190269947052, -1.1362462043762207, -0.31431517004966736, 0.271190881729126, -0.38369882106781006, -0.20914217829704285, 1.0682439804077148, 0.33247125148773193, -1.0933499336242676, -0.47555479407310486, -0.7305999994277954, 0.20104266703128815, -1.296946406364441, -0.1986081898212433, 0.0, 0.031915515661239624, -1.0154184103012085, -0.6849045753479004, 0.13816778361797333, 0.0, 0.9300782084465027, -0.16134603321552277, 0.0, 1.5833251476287842, 0.33098480105400085, 0.4511909782886505, 0.33946293592453003, -1.1527671813964844, -0.3547828197479248, 0.2858428359031677, 1.147364616394043, 0.06686872243881226, -3.466609477996826, -0.2681680917739868, 0.3918378949165344, 1.437654733657837, 0.0, 2.7668988704681396, 1.582602620124817, 1.4974042177200317, 1.0475414991378784, -1.5360393524169922, 0.9411672353744507, 0.6660935282707214, -0.5844672918319702, 0.6518288850784302, 0.4645179510116577, -0.5166884660720825, 0.0, -0.29254981875419617, 0.25608789920806885, -0.4717504680156708, 0.6054590940475464, -2.1306533813476562, -0.31325358152389526, -1.8718749284744263, 0.0, 0.5743739604949951, 0.024013403803110123, -0.12292803078889847, 0.07643597573041916, 1.7438900470733643, -1.2291688919067383, 0.9863702058792114, -1.6806632280349731, -0.5193789005279541, 0.18177497386932373, -0.9463114738464355, 0.22793042659759521, 0.8096582889556885, -1.326380729675293, -0.0859687551856041, -0.4184069335460663, -2.15598201751709, -0.20786018669605255, -0.8864119052886963], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.6482142210006714, 0.4650425612926483, 1.7209542989730835, 1.5779813528060913, -0.8211223483085632, -1.219365119934082, -1.3614988327026367, 0.7885647416114807, -0.32300296425819397, 0.07014422118663788, 0.3197571635246277, 0.4688089191913605, 0.5914093255996704, 0.1775089055299759, -1.5537521839141846, 1.5059634447097778, -2.697430372238159, 1.2834581136703491, -0.5565227270126343, -0.15229374170303345, -0.4782603085041046, -0.32656779885292053, 0.020421765744686127, -0.23861096799373627, -1.0573861598968506, 0.8892454504966736, -0.5405218005180359, 1.3817715644836426, -0.11449361592531204, -0.667657732963562, 0.47478678822517395, 0.925800621509552, -0.886284589767456, -0.24611565470695496, 1.464553713798523, -0.9611939787864685, -0.8127199411392212, 0.20388488471508026, -1.1775338649749756, -2.427327871322632, -1.6307036876678467, 0.411334753036499, 0.691758930683136, 0.6292776465415955, 0.42758119106292725, -1.1959296464920044, 0.7144575119018555, 1.189182162284851, -0.660447895526886, 0.9798712134361267, -0.3294444978237152, 0.2280895859003067, 0.4057886302471161, -1.447401523590088, 1.5044094324111938, -0.8167674541473389, -0.06182324141263962, -0.4543454945087433, 0.4977503716945648, 0.4717484712600708, 0.06107362359762192, 1.4539319276809692, 0.32284218072891235, -0.3588906228542328, -2.839308977127075, -0.24120305478572845, -0.24952559173107147, -0.8302331566810608, -0.8196539878845215, 0.6341829299926758, 1.0859260559082031, 1.935760498046875, 1.1834930181503296, -0.6665120720863342, 0.2973428964614868, -1.3222918510437012, 0.7718214392662048, 0.39883729815483093, -0.5694808959960938, -0.0642252266407013, -0.20915745198726654, -0.5952231884002686, 1.3621968030929565, 3.1466548442840576, 0.8654130697250366, 1.0998727083206177, 0.20223990082740784, -0.3074783980846405, 2.237739086151123, -0.3233644962310791, 0.4386433959007263, -0.07648865878582001, 0.6335176825523376, -0.0064241401851177216, 0.6360477805137634, -1.2237902879714966, -0.058222025632858276, 1.4131340980529785, -0.990725576877594, -0.6247061491012573, 1.4133023023605347, 0.447041392326355, 1.1113066673278809, -2.1952526569366455, 0.18756863474845886, -1.2339162826538086, -1.350220799446106, 0.4528181850910187, 1.225705623626709, -1.456261396408081, -1.0676881074905396, -0.5444940328598022, -0.08709894865751266, -1.2927582263946533, 2.301841974258423, 0.6270906329154968, -0.3229014575481415, -0.48251843452453613, 2.3657214641571045, 2.281644821166992, -0.45462456345558167, 0.38317325711250305, 1.2971643209457397, -1.1542186737060547, -0.5561326146125793, 1.3869233131408691, -0.6646700501441956, -0.4027581214904785, -1.3222121000289917, -0.7345042824745178, 0.7160645723342896, -0.799659788608551, -1.285730242729187, -0.6070308685302734, 1.8023414611816406, 0.19950492680072784, -0.5427983999252319, 0.17368720471858978, 0.46392592787742615, 0.1400620937347412, -1.79904043674469, 0.3165435791015625, -0.19950278103351593, 0.8622168302536011, -1.34687340259552, 1.0669448375701904, -0.48700228333473206, -1.7070789337158203, 1.348166584968567, -0.5101401805877686, 0.38389095664024353, -0.15568023920059204, 1.1064070463180542, 1.1178319454193115, 1.1274888515472412, 0.7235288023948669, -1.6225941181182861, -1.690760612487793, 0.20374658703804016, -0.9588313102722168, 0.4703589975833893, 0.16768096387386322, -1.424636721611023, -1.0448960065841675, -1.1435818672180176, -0.8955109119415283, -0.20996807515621185, 0.6851730942726135, 1.3483935594558716, -0.9804847240447998, -0.5700996518135071, 1.5655848979949951, 0.9001184701919556, 0.23382458090782166, -2.3701975345611572, -1.018689513206482, 0.1574651151895523, -0.5980643033981323, 2.00570011138916, 0.08852750808000565, 0.2830216884613037, -1.112587809562683, -1.0569053888320923, -0.6760910749435425, -0.6353604793548584, -0.23953711986541748, 0.3740549385547638, 0.2362360954284668, 1.008105754852295, -1.3694227933883667, 0.20906014740467072, 0.23849253356456757, 1.6002615690231323, -0.22421187162399292, 0.32561933994293213, -0.8282291293144226, 0.42369726300239563, 0.39780059456825256, -0.6384897828102112, -0.44736188650131226, -0.0361575111746788, 0.5822706818580627, -1.1652668714523315, 0.8901387453079224, -0.3394482433795929, 1.9155681133270264, 1.472556233406067, -0.41231629252433777, -0.6368224620819092, 2.480792999267578, 0.6333624124526978, 0.5705969333648682, -0.4310245215892792, -0.13425950706005096, 0.8111648559570312, 0.6196345686912537, -0.8394138813018799, 1.2539678812026978, 0.3089892268180847, -0.9637929201126099, 1.0033364295959473, -0.17963877320289612, 0.7891740202903748, -0.5618761777877808, 0.3638172745704651, -1.6426016092300415, 0.3749770522117615, -1.5376657247543335, 0.16305863857269287, -2.563204288482666, -0.06671608984470367, 0.7280970215797424, 0.48788177967071533, 1.2062944173812866, -1.0858370065689087, 1.581565499305725, 1.6194497346878052, -1.2144516706466675, -0.8053379058837891, -0.5791904926300049, -0.02570410817861557, 0.1850467026233673, -2.4942197799682617, 0.596307635307312, -0.2603791654109955, -1.030979037284851, 0.9555270075798035, -0.3512997329235077, 1.1433916091918945, 0.5343224406242371, -0.32676175236701965, 1.1792329549789429, 0.8093041777610779, -0.5139932036399841, -1.6307717561721802, -0.5559564232826233, 0.617825448513031, -0.2447885423898697, 1.7744909524917603, 0.25833660364151, 0.1915307193994522, 0.40203744173049927, -0.8954833149909973, -0.12842339277267456, -0.6719511151313782, -2.3730626106262207, -1.7985494136810303, 0.7385280728340149, -0.452958345413208, -0.128556489944458, -1.495591640472412, -0.598751962184906, 0.7842103242874146, 0.09205117076635361, -0.5287432074546814, 1.5371147394180298, -0.06640288978815079, -0.3585336208343506, 1.148478388786316, -1.7547096014022827, 0.5501161217689514, -1.1294233798980713, -0.8003106117248535, -0.6952264904975891, 0.1244712844491005, 0.10253830999135971, 0.3002135455608368, 0.20457011461257935, 1.4074671268463135, 1.698273777961731, 0.7091894149780273, 0.7574627995491028, -0.38367772102355957, -1.1135311126708984, 0.5908296704292297, 0.03098517283797264, 0.8594216704368591, -1.8438467979431152, 2.694462537765503, -1.2855801582336426, 0.031222041696310043, 0.9947347044944763, -1.398305058479309, -0.5612946152687073, -2.0390918254852295, -1.4449849128723145, -1.9561554193496704, 0.10157186537981033, 0.29320111870765686, -0.3226666748523712, 1.2703768014907837, 1.3966221809387207, -0.009404872544109821, -0.22463564574718475, -0.27840563654899597, -0.4582413136959076, 1.3145108222961426, 0.23799565434455872, -0.3356803059577942, -0.2301177978515625, 0.5490220189094543, -0.402199923992157, 0.31610995531082153, -0.03657497093081474, 1.3647024631500244, 1.167984127998352, -1.4270228147506714, -0.16109751164913177, 0.7173812985420227, -0.45146143436431885, 0.817314863204956, 0.33916544914245605, 0.5985380411148071], [-0.06178319826722145, 0.3080354332923889, 0.5292964577674866, -1.478916049003601, -0.920261561870575, -1.0231167078018188, 1.696533203125, 0.30714666843414307, 0.3775489032268524, 0.17713363468647003, -0.5190394520759583, -0.7472490668296814, 0.8573623895645142, -0.37268099188804626, 1.5932174921035767, 0.20073410868644714, 0.09621468931436539, -0.216475710272789, -0.07016262412071228, 0.03887861967086792, 1.535563349723816, 2.3539040088653564, -1.1552504301071167, 0.26218220591545105, 0.7177766561508179, -0.9020490050315857, -0.6302981972694397, -0.45350155234336853, -2.412590265274048, 0.6603590846061707, -0.039578210562467575, -0.08629558235406876, 0.08977711200714111, 0.25816571712493896, 2.3323473930358887, 0.19026359915733337, 0.8464248180389404, -0.5133787989616394, 0.36325931549072266, -0.30626705288887024, -0.14036114513874054, -0.897028923034668, -0.8092706799507141, 0.468296080827713, 0.7596321105957031, 0.4735875427722931, -0.6624654531478882, 0.32943063974380493, -2.5324158668518066, 0.9592615365982056, -0.02498801425099373, 0.2856321632862091, 1.183023452758789, -0.40822097659111023, 0.9779146313667297, -0.42469099164009094, 0.5633623600006104, -0.3907078802585602, 0.32348349690437317, 1.4750739336013794, -0.6330736875534058, -0.5921738147735596, -0.3069300651550293, 0.32372167706489563, 2.780587673187256, -0.5567612051963806, -2.0234248638153076, -0.6308740377426147, -1.562705636024475, -0.5349490642547607, 0.0441179983317852, -0.9342491626739502, -0.6554226279258728, -1.8456737995147705, 0.21011044085025787, -0.7217122316360474, 1.1187175512313843, 1.083809494972229, -0.8411478996276855, -0.3400620222091675, 0.08587205410003662, 0.7922604084014893, -1.6442711353302002, -0.03848643600940704, 1.902430772781372, -0.05428394302725792, 0.2577248811721802, 0.03738292679190636, -1.1582998037338257, 0.9905984401702881, 0.307858943939209, 0.2028823047876358, 0.7682207822799683, 0.8294957280158997, -1.8590651750564575, 0.5245935320854187, -0.5427536368370056, 0.2026815563440323, -0.6727867722511292, -0.8245437741279602, 0.9993339776992798, 0.03741529956459999, -0.23675380647182465, -0.24125292897224426, 0.1293821781873703, 0.03302696719765663, -2.321737289428711, -0.8222453594207764, -0.18391641974449158, 0.8020210862159729, 1.420545220375061, -0.8791860342025757, 2.000393867492676, 0.15557146072387695, 0.552997350692749, -0.47050216794013977, -0.5402604341506958, 1.5075055360794067, -0.14957670867443085, -0.03158063814043999, 1.2337077856063843, 1.800978660583496, -1.4267395734786987, -0.11174982786178589, 0.06755655258893967, 1.8978948593139648, 2.552579164505005, 0.0323796309530735, 0.45858868956565857, 0.953726053237915, 1.2416282892227173, -0.23042821884155273, 0.9499623775482178, -0.6084501147270203, -2.5878918170928955, -1.2882450819015503, 0.3118617534637451, 0.8828957676887512, -2.1646974086761475, -1.7091127634048462, 0.07181792706251144, -3.2815754413604736, 0.6665022373199463, 1.1449239253997803, -2.877190351486206, 0.8823738098144531, -0.07774955779314041, 1.5386884212493896, 0.4015059173107147, -0.11826805770397186, 0.24354371428489685, 1.0665901899337769, 0.3174889087677002, 0.16904112696647644, -1.3166314363479614, 0.6156789064407349, 0.16377033293247223, 1.9784257411956787, 0.12279610335826874, 0.04993902146816254, -0.48984628915786743, 0.8642653226852417, -0.06461617350578308, -0.5657132267951965, 1.7479660511016846, 1.9280235767364502, -0.3112655580043793, -0.8833087682723999, 1.275760531425476, -0.692431628704071, 2.9780070781707764, 0.010257071815431118, 1.113154649734497, -1.1538735628128052, 0.1131681576371193, 1.2460542917251587, 0.5907927751541138, -0.38924640417099, 2.612948179244995, 0.8928096890449524, -0.07827665656805038, -0.4551759362220764, -0.11897178739309311, 1.5090630054473877, -0.622319757938385, 0.18558983504772186, -0.6707249879837036, 1.4132933616638184, -0.010023768059909344, 1.019897222518921, 0.7236582636833191, -0.42924052476882935, 0.8578779697418213, 1.0984857082366943, -0.9981207847595215, 0.8612942099571228, 3.7315614223480225, -0.14360080659389496, 0.7977492809295654, -1.1870315074920654, 0.6180363893508911, -0.4364781677722931, 0.927480936050415, 0.10431388765573502, -1.0289359092712402, 1.2630255222320557, -0.061862099915742874, -0.8665631413459778, 0.22234874963760376, -0.8175748586654663, 0.554183840751648, 0.0918857678771019, -0.2174685299396515, 0.8278314471244812, -1.131846308708191, 1.1356791257858276, -1.8655941486358643, 1.7766350507736206, 0.3393910229206085, -0.11015508323907852, -0.5863906741142273, 0.7082918882369995, -0.00501395994797349, -0.023528946563601494, 1.1979174613952637, -0.2821219563484192, -0.0822412446141243, 0.5899243950843811, 1.239526391029358, 0.00463500153273344, 0.3174443244934082, 0.2028447985649109, 0.9067575335502625, 1.6621297597885132, -0.7878056764602661, 0.02139906771481037, 0.10863234102725983, 0.7525575757026672, 1.6729707717895508, 0.13902229070663452, 0.37124207615852356, 1.3003253936767578, -1.102310299873352, -0.6227874159812927, -0.3002778887748718, 0.1478458195924759, 0.38759082555770874, -0.3702648878097534, 1.9278615713119507, -0.5610166192054749, -0.051333047449588776, 1.252914547920227, 0.9350707530975342, 0.13813425600528717, 0.5327526330947876, 0.9803134799003601, 0.13819383084774017, 0.8075012564659119, -0.779331624507904, 0.2204638570547104, -1.5162256956100464, -0.7580425143241882, -1.0123255252838135, 1.4113726615905762, -0.32663920521736145, 0.44429153203964233, -0.7966078519821167, -0.06581418961286545, -0.14455008506774902, -0.7305445671081543, 1.0067445039749146, -1.5516319274902344, 0.7694779634475708, 2.830980062484741, -0.7020774483680725, 0.16027118265628815, -1.45541512966156, 0.09707231819629669, -1.6412662267684937, 0.7560213804244995, -0.38492822647094727, -1.7640080451965332, -1.206512212753296, -1.507468342781067, -0.1873469054698944, 0.9670549631118774, -0.30815380811691284, -1.1049864292144775, -0.7608441710472107, 0.2052489072084427, -2.395297050476074, 0.6749802231788635, 0.48739349842071533, -2.428635835647583, 1.272812843322754, -1.6010899543762207, -0.15765823423862457, 1.2599846124649048, -1.612291932106018, -1.8301770687103271, -1.090431571006775, -1.056141972541809, -0.09440289437770844, -0.49091628193855286, -0.4633454978466034, -0.449800044298172, -0.1450023055076599, -0.8323647975921631, 0.37418612837791443, 0.03658729046583176, -0.9992756247520447, 1.9010521173477173, -1.2582014799118042, 0.7911444306373596, -0.7201827764511108, 0.4103832244873047, 0.5766889452934265, 0.9240045547485352, 1.497646450996399, -0.811020016670227, 2.9690170288085938, 0.46652936935424805, -0.7746854424476624, -1.3289947509765625, 2.4545185565948486, 1.7419911623001099, 0.04683196172118187, 0.84437495470047, 1.4128462076187134, 0.17107224464416504, -0.36114242672920227, 0.037007544189691544, 0.9312546253204346], [0.09691762924194336, -1.1812466382980347, 0.8473201394081116, -1.6881656646728516, -0.6903966069221497, 1.3428090810775757, 0.32018133997917175, 0.31588080525398254, 0.18588955700397491, 1.1320579051971436, 0.024892080575227737, -1.917344093322754, 0.32688918709754944, -3.0583109855651855, -2.0212533473968506, 1.6940560340881348, 1.1376917362213135, -0.4000489413738251, -0.002641988219693303, 0.1612658053636551, 1.0980855226516724, -0.7170817852020264, -0.828022301197052, -0.5698844790458679, 0.21354511380195618, 0.3417341709136963, -0.3313666582107544, 0.8488039970397949, -0.9742937684059143, -0.006832235027104616, -0.6365848779678345, -0.9579819440841675, 1.3259512186050415, -0.2553349435329437, -0.9579455852508545, 0.11225531250238419, -0.9533947706222534, 0.681079626083374, 0.9405848979949951, 1.1205435991287231, -1.0107654333114624, -0.07789767533540726, 1.9580249786376953, -0.14679288864135742, 1.3662078380584717, 1.6337683200836182, -1.338371753692627, 1.312495470046997, 0.8442769050598145, 0.9711555242538452, 1.8030033111572266, 0.545741617679596, 0.15593600273132324, 0.7005673050880432, -0.27663296461105347, 0.5827591419219971, -1.98159921169281, 1.8438711166381836, -0.46741488575935364, 0.6279374361038208, 0.39442524313926697, 0.1818164587020874, 0.18916313350200653, 0.14098410308361053, -1.6009570360183716, -0.48411014676094055, 1.2950345277786255, -0.97103351354599, -1.004866600036621, 2.2358157634735107, 1.3074817657470703, 1.670384168624878, 0.0599958635866642, 0.3475346267223358, 0.8507393002510071, 0.6462746262550354, -0.01834901235997677, -1.4242703914642334, -0.6796587109565735, -1.4715572595596313, 1.6734185218811035, -0.34498104453086853, -0.1220492571592331, 0.3507508337497711, 0.07647678256034851, 0.5965756773948669, -0.5798034071922302, -1.4996758699417114, 1.840981364250183, -0.674163281917572, -0.43399178981781006, -0.7617133259773254, -0.5108789205551147, 0.06703842431306839, 0.807339608669281, -1.028593897819519, 0.33794504404067993, -0.6439771056175232, 1.2637972831726074, -1.21353280544281, 0.14069072902202606, 0.7635555267333984, -1.3458417654037476, 0.14700302481651306, 0.18284353613853455, -0.7776884436607361, -0.18773971498012543, 1.0372114181518555, 0.2796303629875183, -0.39101898670196533, -0.1826649159193039, 0.4814785122871399, 1.3136781454086304, 0.832439124584198, 0.7663241028785706, 1.6097573041915894, -1.539671540260315, -1.4308322668075562, -2.2779011726379395, 0.8639148473739624, -2.2946741580963135, -0.02588176541030407, 0.7796458005905151, -1.3941924571990967, -0.651979923248291, -0.25091034173965454, 0.7503045797348022, -1.5189157724380493, 0.3340268135070801, 0.714794933795929, 0.15476712584495544, 0.1694377362728119, 1.26852548122406, 0.23221240937709808, -0.014102834276854992, -0.32921504974365234, 0.6634801030158997, -0.28362545371055603, 0.6649001836776733, 0.10762582719326019, 1.1424052715301514, -0.789470911026001, 0.0811527669429779, 0.3565989136695862, 0.4209122061729431, 1.1428831815719604, 0.401082307100296, 0.9141574501991272, 1.9749577045440674, 0.8516463041305542, -1.0576419830322266, -1.4282631874084473, 0.6770605444908142, -0.3739376366138458, 0.2961777448654175, -0.1699894666671753, 1.3158645629882812, 1.2074010372161865, -1.1269677877426147, 0.27778682112693787, -0.2981376051902771, -1.309740662574768, 0.6526275873184204, -0.1407569795846939, 0.4472343921661377, 2.485701084136963, 1.3343160152435303, 1.1094746589660645, -0.4059571325778961, -1.1821799278259277, 1.2535203695297241, -0.30139681696891785, 0.04708219692111015, -0.37475278973579407, 0.10233020037412643, 2.413410186767578, -0.3137982189655304, -0.28109806776046753, 0.2041698843240738, 0.28638285398483276, -1.202077865600586, -1.778314471244812, -0.5849977731704712, -0.21608120203018188, -0.79278963804245, -1.0095555782318115, -0.9501887559890747, -0.4063320457935333, -0.13346071541309357, -1.647688865661621, -0.5666359066963196, 0.33674925565719604, -0.14454317092895508, -1.3815933465957642, -0.37948182225227356, -0.0428592823445797, 0.7446359992027283, -1.6513692140579224, 0.6585149168968201, -0.48899030685424805, 0.17597737908363342, 0.6547057628631592, -0.898847222328186, -0.8354224562644958, -0.14355330169200897, 0.694297730922699, -0.5751674771308899, -1.4445794820785522, -0.7283405661582947, 0.34901970624923706, 1.1230100393295288, -0.5424818396568298, -0.36144569516181946, -0.21739360690116882, -0.17390502989292145, -0.9853690266609192, -0.481656014919281, 0.1726529747247696, 0.06017927825450897, -0.020851507782936096, -1.046106219291687, -0.5911703109741211, -0.7544742822647095, 1.0033730268478394, 1.4044266939163208, 0.29719316959381104, 1.5445647239685059, -0.3351368308067322, -0.33033132553100586, -1.3330366611480713, 0.18769939243793488, -0.493558406829834, 0.0515587255358696, 0.444592148065567, -0.2801675796508789, -1.1552380323410034, -0.08271728456020355, -0.10981988906860352, -0.009394236840307713, 0.6335258483886719, -0.2488531619310379, -0.30063396692276, 1.6856043338775635, 1.0317844152450562, -0.12253812700510025, -0.9580948352813721, -0.47220534086227417, 0.3491304814815521, 0.808112621307373, 2.359581708908081, -0.2671704590320587, 0.1296033412218094, -0.3725273013114929, 0.06182975694537163, -0.3956781029701233, -1.073636770248413, 0.9984791874885559, 0.11461351811885834, 1.2612003087997437, -0.5078645348548889, 0.2521410286426544, 0.6437727808952332, -1.8629844188690186, -1.3099573850631714, 0.739809513092041, 0.09756849706172943, -1.8161866664886475, -0.729424774646759, -1.0374466180801392, -0.4743015766143799, 1.5312931537628174, -0.561677873134613, 2.0101864337921143, -0.24069060385227203, -0.7160159945487976, 1.5586175918579102, 0.9481443762779236, 1.3761337995529175, 1.9362514019012451, 1.068121314048767, 1.2268521785736084, -1.6780548095703125, -0.792407214641571, 0.4610411822795868, 0.2282714545726776, 0.7399961352348328, -0.4409095048904419, 1.130455493927002, 2.0286448001861572, 0.6144060492515564, 0.8655135631561279, 0.5415249466896057, 0.9599432349205017, 0.25572291016578674, -0.45315638184547424, 0.2410992830991745, 0.15411239862442017, -1.4628673791885376, 0.27894359827041626, 0.07423841208219528, 0.2536517083644867, -0.4942910075187683, 0.524606466293335, 0.6650620102882385, -0.6518340110778809, 0.3663484752178192, 0.8180758357048035, 0.7810273766517639, -0.8664475679397583, -0.9697191715240479, 0.06440911442041397, -0.9983781576156616, 2.5062406063079834, -0.771392285823822, -0.7605314254760742, 0.6614928245544434, -0.4455181062221527, -2.302992820739746, -0.13359269499778748, -0.9692337512969971, -0.5155580639839172, -1.5983200073242188, -0.9006591439247131, -0.17415618896484375, -0.09317313879728317, -1.122118592262268, -0.7341514825820923, -0.8720816373825073, 0.10444768518209457, -0.18465843796730042, -1.2741892337799072, 0.5916423797607422, -0.7347812056541443], [0.33672648668289185, -0.9540027976036072, 0.2423943430185318, -0.17098164558410645, 8.989984780782834e-05, 0.4417066276073456, 0.37684836983680725, 0.10646731406450272, 0.10403823107481003, -1.3122111558914185, -0.12673696875572205, -2.220201253890991, -2.2701117992401123, -0.36600780487060547, -0.2782774865627289, 1.4172511100769043, 0.599136233329773, 1.037062406539917, -0.36115050315856934, 0.382651150226593, 1.0956499576568604, 0.4703393578529358, 0.3003329336643219, 0.05541110411286354, -0.9524925947189331, -0.924770712852478, -1.454416275024414, 0.3635725975036621, -0.3050912022590637, -0.702267050743103, 0.3163476586341858, -0.5995518565177917, 0.9995893239974976, 1.8537801504135132, 0.4167470633983612, -0.7090855240821838, -1.5036991834640503, 1.9609185457229614, 0.4202286899089813, -1.1853913068771362, 1.0366289615631104, -1.6701457500457764, -0.45861124992370605, -0.2578120529651642, -0.378357470035553, -1.1483043432235718, 2.34578013420105, -0.41476842761039734, 0.3581903576850891, 0.01417465414851904, -1.6026089191436768, -0.5629239678382874, 0.18757687509059906, 0.9025432467460632, -0.3604457676410675, 0.34475094079971313, -0.35311031341552734, 0.5650457739830017, -2.4678571224212646, 0.5782606601715088, -0.6324611306190491, -0.5902531147003174, -2.8341124057769775, -1.0694005489349365, 0.5933617949485779, 2.277731418609619, -0.7112997770309448, -0.6684938073158264, 0.4742656946182251, 0.27268582582473755, -0.5948666334152222, -0.17811283469200134, -0.3911433815956116, 0.6043050289154053, 0.7784029841423035, 1.7382376194000244, 0.9029248952865601, 0.6549893021583557, 0.14874768257141113, 0.9244657158851624, -0.9842801690101624, 2.137793779373169, 0.2749103605747223, 1.3995137214660645, -0.7949187159538269, 1.021384835243225, 0.39555126428604126, -0.18759670853614807, -0.667049765586853, -0.8775473833084106, 1.7873941659927368, 1.2127339839935303, 0.11351118981838226, 0.1935553401708603, -0.7237750291824341, -1.8281699419021606, -1.7834604978561401, -0.6058821678161621, -0.1543627381324768, -1.167389154434204, 0.7936605215072632, 0.513680100440979, 0.15997323393821716, -1.7534781694412231, -0.21468757092952728, -0.03045160509645939, -1.258015513420105, -2.1798903942108154, 1.9102146625518799, 0.6187108755111694, -0.38444140553474426, 1.5935776233673096, 1.1083375215530396, 0.4465538561344147, 0.22484081983566284, 0.777571976184845, 1.4601149559020996, 0.06918942183256149, -0.2309146374464035, 0.5808848142623901, -2.815406322479248, 0.087220199406147, -0.4073428213596344, -0.018013065680861473, -0.18478216230869293, 0.33644339442253113, 0.10391255468130112, 0.06692289561033249, -0.13302789628505707, 0.4537537097930908, -0.5079919099807739, 0.9817184805870056, -1.6831303834915161, 0.2030455619096756, 0.4515121877193451, -1.2713053226470947, -0.43046581745147705, 1.3234368562698364, -0.1320183128118515, -0.8825887441635132, -2.9810738563537598, 0.34010446071624756, 0.10180404037237167, 0.6098385453224182, 2.049147605895996, -0.0040780627168715, 0.13358832895755768, -0.6863937377929688, -0.06172189861536026, -0.029064428061246872, 0.7549787163734436, 0.5538395047187805, -0.7639915347099304, -0.475504606962204, 1.3786042928695679, -0.959780752658844, 0.7472248077392578, 1.6240487098693848, 0.6535729765892029, 0.25313571095466614, 1.3646888732910156, 0.35354137420654297, -0.009042670950293541, -0.20306143164634705, 1.1481717824935913, 0.6951559782028198, 1.544500470161438, -0.3358617424964905, -0.31296682357788086, -0.43597984313964844, 0.2799172103404999, -0.6925589442253113, 0.9256901144981384, 0.33869653940200806, -0.9817950129508972, -2.964954137802124, -1.8264200687408447, 1.4378979206085205, -0.061856709420681, 0.1466788798570633, -0.4258829951286316, 0.3928937017917633, -1.9222831726074219, -2.002235174179077, -1.6381443738937378, 0.06467795372009277, 0.7016574144363403, 0.4721813499927521, -0.1290326714515686, -1.2838741540908813, 0.7933192849159241, -0.285183310508728, -0.02359299175441265, -0.07748734951019287, -0.4723057448863983, -0.7276017069816589, -0.5950741171836853, -1.2959738969802856, 0.24430808424949646, -0.17786580324172974, 1.0133464336395264, -1.8160256147384644, 1.1678755283355713, -0.07612062245607376, -0.16703282296657562, -0.8101052045822144, -0.8216758966445923, -1.4985029697418213, 0.8942722678184509, -0.21845722198486328, 0.7130377888679504, 0.8232052326202393, -0.503943145275116, -0.24234654009342194, -0.34927889704704285, 1.0229249000549316, 0.12761645019054413, 0.16498404741287231, 0.7173846364021301, -0.07892610132694244, 2.040977954864502, -0.9308256506919861, 1.2515534162521362, 1.4022560119628906, -1.1440343856811523, 0.8609968423843384, 0.2793208360671997, 0.8696194291114807, -1.2024965286254883, 0.24294538795948029, -0.980441153049469, -0.3318025469779968, 1.4197955131530762, -0.1349439024925232, -2.252091646194458, -1.3989219665527344, -1.1400413513183594, 1.0157252550125122, -0.697831392288208, -1.5571390390396118, 1.116661548614502, 0.2505347430706024, 0.34230363368988037, -0.7989553809165955, 1.427563190460205, -0.1228242814540863, -1.9737814664840698, -1.1379519701004028, -0.8425267338752747, -0.16109620034694672, 0.7645562291145325, -0.7270549535751343, -0.6008740067481995, 2.2042877674102783, 0.08263807743787766, -0.5287814736366272, 0.03295746073126793, -1.2521106004714966, 0.5350187420845032, 0.9682777523994446, -0.5823999047279358, 1.001400351524353, -0.2907167971134186, 0.28820428252220154, -0.7415891289710999, -0.8008134365081787, -1.4992759227752686, 0.21967051923274994, -0.284157395362854, -0.253517210483551, -1.1289037466049194, -1.168033242225647, -0.6322441697120667, 0.4695322513580322, 0.30108892917633057, -0.682431697845459, 0.293691486120224, 1.7576667070388794, 0.10578486323356628, 2.0218212604522705, 0.4334779679775238, 1.3312336206436157, -0.5287891626358032, 1.1737130880355835, -1.1846307516098022, 0.24306249618530273, -0.3349968492984772, -1.996754765510559, -1.3650058507919312, 1.1313519477844238, 0.48965585231781006, 1.2943719625473022, 0.194626122713089, -2.9038097858428955, 0.24875813722610474, -0.5710901618003845, -0.5055790543556213, -0.45187368988990784, -0.7687228322029114, -0.9624055624008179, 1.2389930486679077, 0.44339078664779663, -0.6569135785102844, -0.4258054792881012, 0.9700149297714233, 0.37686508893966675, 0.19077087938785553, -1.2309021949768066, -0.4393526315689087, -1.1405385732650757, -1.9223990440368652, 0.8881905674934387, -0.3344660699367523, -0.41889679431915283, -0.9673193693161011, 0.7496910095214844, 2.0852651596069336, 0.5772936940193176, 0.9217863082885742, -1.2532308101654053, 0.16800040006637573, 0.21051430702209473, 1.4820555448532104, 0.0012638290645554662, -0.9520331621170044, -0.07471924275159836, -0.44319120049476624, 1.4372583627700806, 0.656079113483429, 0.33216527104377747, 0.17029815912246704, 0.8344084620475769, -0.5005050897598267], [-0.9282617568969727, -1.8915799856185913, 1.1760345697402954, 0.49501854181289673, 0.6308279037475586, 0.030497251078486443, 1.6500556468963623, -0.9940615892410278, 1.1907883882522583, -0.8571158051490784, -0.46620243787765503, -1.231430172920227, 0.07511146366596222, -0.30585697293281555, 0.41496598720550537, -0.6028551459312439, -1.4105578660964966, 0.5491094589233398, 1.2685896158218384, 0.716909646987915, -0.07406593859195709, 0.38392940163612366, -1.7194175720214844, -0.355699360370636, 1.13895845413208, 0.43790045380592346, -0.876461923122406, 0.8336467146873474, -0.15533308684825897, 0.7078014612197876, -1.4311529397964478, -0.550011932849884, 0.3017818331718445, 0.27948325872421265, 0.5302591919898987, -0.034040093421936035, -0.221464142203331, 0.44825735688209534, 0.840988039970398, -1.5372000932693481, -0.3442014455795288, -0.22628863155841827, 0.1351175159215927, 0.9886977672576904, -1.349581241607666, 0.5097571611404419, -1.7515203952789307, -0.7751612663269043, -0.543375551700592, -0.2922438681125641, -0.6331840753555298, -0.5350913405418396, -0.36264464259147644, -0.29198166728019714, -1.8613362312316895, -1.4194210767745972, -0.45796117186546326, 0.5045005679130554, 0.9786188006401062, 2.821652889251709, 0.08592081815004349, -0.8056232929229736, 0.15616782009601593, 0.6572337746620178, 0.11707638949155807, -1.0769072771072388, -0.12140487879514694, -0.06220824643969536, -1.4232293367385864, -1.2231498956680298, 1.1735126972198486, -2.2392706871032715, 0.14744719862937927, -0.2796500325202942, 1.3603951930999756, 0.587039053440094, -0.8801161050796509, 1.2331300973892212, -0.5781127214431763, 1.9601165056228638, 1.6275118589401245, -0.0986751988530159, 0.7502515316009521, 0.16435883939266205, -0.9154852032661438, 1.039635181427002, 0.4061304032802582, 0.8289175033569336, 1.0480411052703857, 1.3908355236053467, 1.016243815422058, -0.10041145235300064, 0.07587991654872894, 0.3996839225292206, -0.1292012631893158, -0.2431120127439499, 1.8391282558441162, 0.47833317518234253, 1.8738027811050415, -0.769766628742218, -0.04692312702536583, -0.4369194507598877, -0.3922710120677948, -0.2918502390384674, -0.1267402619123459, -0.4356306791305542, 1.7829262018203735, 0.6660987734794617, -0.7414745688438416, -0.543912947177887, 0.7349713444709778, 0.5363830924034119, -0.6069660782814026, 0.9131186008453369, -1.4210996627807617, -0.47018560767173767, 1.7080482244491577, -2.5829570293426514, -0.20373046398162842, 0.9763892292976379, -0.1452770084142685, -0.775070071220398, -0.8828583359718323, -2.062086820602417, -1.3144760131835938, 0.06382085382938385, -0.5653773546218872, -0.014137234538793564, -0.1496659219264984, 0.2805001139640808, -1.645377278327942, -0.3065566122531891, -0.33688804507255554, -0.30988478660583496, -0.8810281157493591, -2.1720564365386963, 0.6230443120002747, 0.8045369982719421, 2.3449432849884033, 0.7212111353874207, 1.624613642692566, 0.5879198908805847, -0.3776054382324219, 0.48945310711860657, -1.1567161083221436, 0.23902206122875214, -0.10927663743495941, -1.0456702709197998, 1.208757758140564, 1.1063584089279175, 0.4567049741744995, 0.8073647618293762, 1.630489706993103, -0.13134847581386566, -1.3383187055587769, -0.9422166347503662, 0.691810131072998, 0.2293209731578827, -1.4700064659118652, 1.3110976219177246, 0.2155350297689438, 1.6018511056900024, 1.122945785522461, -0.42598533630371094, 0.10018844157457352, -0.7216942310333252, -0.8259448409080505, -0.2879478931427002, 0.5039294362068176, 1.3655567169189453, -0.874052882194519, 0.7056276798248291, 0.05041689798235893, -0.20513999462127686, 1.1332314014434814, -0.3363068997859955, 1.3834214210510254, -0.31176987290382385, -0.8947950601577759, 0.7819488048553467, -1.424142599105835, -1.1536705493927002, -0.16120316088199615, 0.027106819674372673, -0.2412717640399933, -1.9341212511062622, -1.2996138334274292, 0.535790205001831, 0.5636754035949707, -0.9744518399238586, 0.29373034834861755, 0.23416434228420258, 1.0398789644241333, -0.6978287696838379, 1.4795671701431274, -0.26994115114212036, 1.466697335243225, 0.9519497752189636, -0.9718633890151978, -0.2752710282802582, 0.31801337003707886, 1.0444213151931763, -1.3427398204803467, -0.4625294804573059, 0.6534935235977173, 0.5103999972343445, -1.0111573934555054, -0.858955442905426, 0.46131688356399536, -0.9847958087921143, 0.2652566730976105, 0.9452515840530396, 0.0493972972035408, -2.6347193717956543, 1.2016222476959229, 1.1790821552276611, 1.4442825317382812, 0.8887757062911987, -1.641516923904419, 1.4436768293380737, 0.37715205550193787, 0.9693018198013306, -0.1236313208937645, -0.2084580808877945, -0.2007625848054886, 1.6236132383346558, 0.3074965476989746, -0.1676214337348938, -1.2406405210494995, 0.5028992295265198, 0.04624354466795921, 0.19035740196704865, 1.0182217359542847, 0.7557967901229858, 0.038763582706451416, 0.2505192756652832, 0.1252444088459015, 0.7503341436386108, 0.723770797252655, 0.1756080687046051, -0.8176755309104919, 0.8764512538909912, 0.9326688647270203, -1.0314170122146606, 0.4211631417274475, -0.35138922929763794, -2.145371437072754, -0.012192795053124428, 0.9786068797111511, -0.6975687146186829, -0.6733545064926147, 1.0459505319595337, -1.0695396661758423, -0.9115452170372009, -0.2910284698009491, 0.6466947793960571, 0.6512537598609924, -0.5009593963623047, -0.2833631932735443, 0.1133001297712326, -0.3637593984603882, -0.8378428816795349, -1.0664342641830444, 0.7334561944007874, -0.19586530327796936, 2.877272129058838, 0.6445407271385193, -0.7266354560852051, 0.024293014779686928, 0.36272764205932617, 0.4193783700466156, 0.46707475185394287, 0.47377917170524597, -0.1155509427189827, 0.6700873374938965, 2.201589345932007, -0.41716235876083374, -0.35812288522720337, 0.7286757826805115, 1.2639271020889282, 0.8367475867271423, 1.2345575094223022, -0.02733219601213932, -0.20809952914714813, 0.6737865209579468, 0.6932280659675598, 1.688090205192566, 1.5749238729476929, -1.797497034072876, 0.7962563037872314, -0.646194338798523, -2.079765796661377, 0.41048964858055115, -0.14912179112434387, -0.32846853137016296, -1.261722445487976, -0.9946390390396118, -0.19745495915412903, -1.4748075008392334, -0.09386183321475983, 0.12649624049663544, -0.13951566815376282, 0.31009232997894287, -0.05652330815792084, -1.1473429203033447, -1.0197298526763916, 0.14359150826931, 0.970978856086731, 0.15821526944637299, -0.4926706552505493, 0.1732914000749588, -1.796041488647461, -0.9853021502494812, -0.37087172269821167, -0.05861297994852066, -0.31981855630874634, -0.26624393463134766, -1.1803808212280273, -0.5688968300819397, 0.24971599876880646, -1.8341885805130005, -1.1963540315628052, -0.3464243710041046, 1.477643609046936, 0.6742927432060242, 1.5593222379684448, 1.3378148078918457, -0.4471452832221985, -0.06952418386936188, -1.1711076498031616, 1.1711742877960205, 0.7267274260520935, 0.5899269580841064], [0.6610268950462341, 0.459516316652298, 0.20844526588916779, 0.27548643946647644, 1.2079347372055054, -0.784766674041748, -1.0409573316574097, 0.0043632895685732365, 1.1576441526412964, 0.2407122105360031, -0.2029840350151062, 0.8326268792152405, 0.1133594661951065, -1.8911601305007935, -1.0007869005203247, -2.081385850906372, -0.21703986823558807, -0.4427882432937622, 0.44086599349975586, 2.10782527923584, -0.5905892252922058, 0.011556691490113735, -0.3760651648044586, -2.7969255447387695, -0.7543385028839111, -0.7121173143386841, 0.27814245223999023, -0.8302106261253357, -1.280784249305725, 0.5918357372283936, -0.05957604944705963, -0.8857067227363586, -0.7490122318267822, -0.29342010617256165, 0.8215857148170471, 1.4221367835998535, -1.2956907749176025, 1.1675392389297485, 0.06091490015387535, 0.3662532866001129, 0.9743555188179016, -0.9532046914100647, 0.8625930547714233, -1.2931723594665527, -0.4570690989494324, -0.8390726447105408, 1.6873645782470703, 1.6413826942443848, 0.7633056044578552, 0.5220226645469666, -0.5193854570388794, 1.7433983087539673, 1.4015320539474487, 0.9070159792900085, -0.09902406483888626, -0.3340286910533905, -0.0700199082493782, 1.337058186531067, -0.9247221350669861, 0.13360027968883514, 1.6432576179504395, 0.8986158967018127, 1.7527201175689697, 0.434911847114563, 0.45056620240211487, 1.3995741605758667, 0.6754961609840393, 0.3696436583995819, -0.476951539516449, -1.4414089918136597, 1.512297511100769, -0.37981897592544556, -0.7045789361000061, 0.8972569704055786, -0.8428066372871399, -1.301943063735962, -0.5252231955528259, -0.020824899896979332, 0.3120388686656952, 0.550483226776123, 1.0048354864120483, -0.07128024846315384, 0.4073561429977417, -0.06490328162908554, -0.5875574350357056, 0.44551795721054077, 0.3973197340965271, 1.8828667402267456, 0.6411940455436707, -0.6422591209411621, 0.6393157243728638, -0.1991991251707077, 1.6104682683944702, -1.6380594968795776, -0.6057244539260864, -0.603274941444397, 0.16866974532604218, -0.07839787751436234, -1.5221112966537476, -1.792394995689392, 0.2304980307817459, 1.4993095397949219, 1.4872324466705322, -0.3439878523349762, 0.505285382270813, -0.8047406077384949, -1.3764070272445679, 1.2627737522125244, 0.7201958894729614, -0.2772785723209381, -0.3160712718963623, 0.9587565660476685, 0.3704436719417572, -0.27814242243766785, 0.33229920268058777, 0.0446915365755558, -0.34783831238746643, -0.6527727246284485, 0.967522144317627, -0.41044434905052185, -1.1074810028076172, 1.8385587930679321, 1.285901665687561, 0.5724612474441528, -0.008130398578941822, 0.2498309314250946, -1.3727625608444214, 0.03709707781672478, 0.3254106342792511, -0.14271889626979828, 0.48250797390937805, -0.7016567587852478, 0.2306303232908249, -1.4936020374298096, 0.023617632687091827, -0.5567931532859802, -1.2596068382263184, -0.3461359441280365, 0.06463553011417389, -0.542325496673584, 0.954192578792572, -0.6441678404808044, 0.961017906665802, 0.7560918927192688, -0.5547707080841064, 1.4463680982589722, -1.2994589805603027, 0.5610623359680176, 0.10125749558210373, 0.7859514951705933, -1.5030546188354492, -1.135616421699524, -0.7843849658966064, 0.04822622239589691, 0.10981037467718124, 0.042992670089006424, -0.23148420453071594, -0.6707863807678223, -1.322285771369934, -0.4013601243495941, 0.5368198156356812, 1.0133622884750366, -0.479574054479599, 1.0667321681976318, -1.3118555545806885, 0.7753884196281433, -1.0511894226074219, 0.2523414194583893, -0.9237934947013855, -0.9378179907798767, -0.5221214890480042, 1.4029531478881836, -0.6405518651008606, 0.6203612089157104, 0.16344881057739258, 0.3073389530181885, 0.3177020847797394, 0.626953125, -1.1639026403427124, 0.05474785715341568, 0.35343673825263977, -0.5583035945892334, 0.36097487807273865, -0.9858989119529724, 0.7219184637069702, -0.6737388968467712, -0.2609383761882782, 0.78687983751297, 1.0691113471984863, -0.5424123406410217, -0.6112111210823059, 0.9794551134109497, -2.8045058250427246, 1.3213846683502197, 1.276041030883789, -1.0300652980804443, 1.1767940521240234, 0.36646151542663574, 0.4598233997821808, -2.2539126873016357, -0.4693569540977478, -0.17763902246952057, -0.39008045196533203, -0.36896610260009766, -0.2696385979652405, -2.793569326400757, -0.18606296181678772, -0.6690100431442261, -0.07762499898672104, -0.4773363769054413, 0.004799084737896919, 0.2847839891910553, 1.7443534135818481, -0.7985907793045044, 1.659325361251831, -0.2303473949432373, -0.8684769868850708, -1.4114176034927368, 0.8510198593139648, -1.4784823656082153, 0.8971815705299377, 0.3436129093170166, -0.5518722534179688, -0.6482755541801453, 0.47890815138816833, 0.15464362502098083, 0.729897141456604, 0.0018462856533005834, -0.49906736612319946, 0.5288677215576172, -1.1699007749557495, 0.8308131098747253, 0.0551152229309082, -1.092481255531311, -0.9944207072257996, -0.31672191619873047, -1.7806817293167114, 1.3658782243728638, 0.9493183493614197, -1.0718709230422974, -0.11035330593585968, 1.664217233657837, 0.39620935916900635, -0.9956503510475159, -2.1168346405029297, -1.6099786758422852, 1.1713718175888062, 0.3115244507789612, -1.287138819694519, 0.6965139508247375, -0.3570626676082611, -0.6571287512779236, -0.5136642456054688, -0.4574911892414093, 1.64710533618927, -0.9321240186691284, -0.3967629373073578, 1.31564462184906, 1.4851804971694946, -0.9678979516029358, 2.040414810180664, 1.0893259048461914, -1.4661964178085327, 1.7859396934509277, -1.4604315757751465, 0.6741138100624084, -1.137724757194519, -1.760111927986145, 0.8433619737625122, 0.4189917743206024, -0.08051396161317825, 0.19092826545238495, -0.10012504458427429, -0.2549878656864166, -1.0574874877929688, 0.5794012546539307, 0.02322559244930744, -0.24043165147304535, -0.5617735981941223, -0.27698174118995667, -1.5026531219482422, -0.30528923869132996, -0.9749119281768799, 1.2746031284332275, -0.9428203701972961, -0.4453776478767395, 0.01692257449030876, 0.7897602915763855, 0.45833203196525574, 0.0167620237916708, 2.1433939933776855, -0.015096597373485565, 0.3216009736061096, 2.7114744186401367, -0.8813253045082092, 0.9568053483963013, 0.44158169627189636, -1.304658055305481, -0.6166298985481262, 0.8281942009925842, -1.4377083778381348, -0.18270033597946167, 0.9267576336860657, -0.234514981508255, 0.266166090965271, -0.6621248722076416, -0.6185137629508972, -0.60929936170578, -0.34789201617240906, 0.6275887489318848, 0.0763772651553154, 1.9565480947494507, 0.5472909212112427, 0.2760450839996338, 2.230491876602173, -0.10839453339576721, -0.8005086779594421, 0.1522584855556488, -1.081290602684021, 0.5126543045043945, 1.3545887470245361, 0.6332600116729736, -0.40561601519584656, -0.14324447512626648, 0.479897677898407, 1.171344518661499, -0.3964679539203644, -0.4118032455444336, 1.0794106721878052, 0.21820247173309326, 0.3431844711303711, 0.4810938537120819, -0.7660545110702515]], "dtype": "float32"}, {"__nd__": [[0.2579793632030487, 0.207331582903862, 0.8845612406730652, 0.2859463393688202, 0.15874716639518738, 0.15093736350536346, 0.725324273109436, 0.430298775434494, 0.38807249069213867, 0.7046705484390259, 0.6317011117935181, 0.461929053068161, 0.6197314262390137, 0.3083208501338959, 0.43196579813957214, 0.6758690476417542, 0.02179115265607834, 0.4081903100013733, 0.38205763697624207, 0.32313430309295654, 0.5010659098625183, 0.6891348361968994, 0.7389214038848877, 0.7705004215240479, 0.6201828718185425, 0.015507572330534458, 0.24679650366306305, 0.6678922176361084, 0.1173594743013382, 0.580011785030365, 0.015327123925089836, 0.8389666080474854, 0.16709288954734802, 0.43468716740608215, 0.12075520306825638, 0.17661605775356293, 0.47100916504859924, 0.5215066075325012, 0.8636270761489868, 0.2074863314628601, 0.8820586204528809, 0.2370673418045044, 0.39399266242980957, 0.9590667486190796, 0.5140812397003174, 0.25455477833747864, 0.4529690444469452, 0.8224987387657166, 0.8705319166183472, 0.11281177401542664, 0.5155284404754639, 0.37534838914871216, 0.08328761905431747, 0.43942853808403015, 0.29489555954933167, 0.7384839057922363, 0.016261398792266846, 0.4526436924934387, 0.3444765508174896, 0.42085814476013184, 0.43341323733329773, 0.9787257313728333, 0.6855444312095642, 0.38429901003837585, 0.33720946311950684, 0.43694594502449036, 0.9339410662651062, 0.5967455506324768, 0.9537162780761719, 0.20549276471138, 0.5210180878639221, 0.7212436199188232, 0.8251705765724182, 0.10381587594747543, 0.40594589710235596, 0.8721954822540283, 0.613632082939148, 0.14189110696315765, 0.420538067817688, 0.501807689666748, 0.0018704597605392337, 0.4209490120410919, 0.47416943311691284, 0.9626440405845642, 0.5179644823074341, 0.5270266532897949, 0.07645176351070404, 0.03740818426012993, 0.45993664860725403, 0.6304895877838135, 0.2180892378091812, 0.5733131170272827, 0.09534292668104172, 0.2740572988986969, 0.5216003656387329, 0.3073064386844635, 0.44474732875823975, 0.7126684784889221, 0.17846590280532837, 0.6556335687637329, 0.9613915085792542, 0.8249989748001099, 0.3752124607563019, 0.6387562155723572, 0.1128559485077858, 0.7217093110084534, 0.7480537295341492, 0.22109319269657135, 0.379911333322525, 0.6047608256340027, 0.6580710411071777, 0.6202763915061951, 0.7486639618873596, 0.8506323099136353, 0.4695621430873871, 0.015084738843142986, 0.8062147498130798, 0.1093313992023468, 0.5300770401954651, 0.7136270403862, 0.8072691559791565, 0.5303871035575867, 0.6330094933509827, 0.9645200967788696, 0.770165205001831, 0.2842334806919098, 0.9347658157348633, 0.6781083941459656, 0.30215179920196533, 0.276064932346344, 0.4429455101490021, 0.5095111727714539, 0.8449265956878662, 0.456657350063324, 0.7111745476722717, 0.3350728154182434, 0.46905338764190674, 0.3242906928062439, 0.39218828082084656, 0.6992383599281311, 0.23169749975204468, 0.48947611451148987, 0.7419692277908325, 0.01020693127065897, 0.4088374674320221, 0.7726573348045349, 0.6563610434532166, 0.8149323463439941, 0.25708118081092834, 0.6144607663154602, 0.30817219614982605, 0.0844058245420456, 0.785987913608551, 0.2987942397594452, 0.7329147458076477, 0.7710994482040405, 0.8059435486793518, 0.2900068461894989, 0.8017067909240723, 0.24433420598506927, 0.6646433472633362, 0.008630525320768356, 0.7854048609733582, 0.06299508363008499, 0.7182780504226685, 0.6325806975364685, 0.06269384175539017, 0.5829792022705078, 0.7062718868255615, 0.8076038360595703, 0.9813479781150818, 0.23977364599704742, 0.42576247453689575, 0.3667891025543213, 0.6621814370155334, 0.18987472355365753, 0.5596038103103638, 0.2383880764245987, 0.20804357528686523, 0.7542637586593628, 0.5483749508857727, 0.7785181999206543, 0.6626361012458801, 0.8507164120674133, 0.4762320816516876, 0.0126518364995718, 0.9997555613517761, 0.22964389622211456, 0.8236502408981323, 0.1992928385734558, 0.5106165409088135, 0.2886914610862732, 0.6569910049438477, 0.07932191342115402, 0.072081059217453, 0.46159887313842773, 0.848987877368927, 0.466014564037323, 0.11797632277011871, 0.992976725101471, 0.10572612285614014, 0.3102180063724518, 0.5657753944396973, 0.26942265033721924, 0.6992303133010864, 0.41926005482673645, 0.7947131395339966, 0.08248402923345566, 0.8224289417266846, 0.9027858972549438, 0.2540225386619568, 0.23755665123462677, 0.882152259349823, 0.4137130677700043, 0.3977853059768677, 0.5844590663909912, 0.5489285588264465, 0.81281578540802, 0.6697506904602051, 0.7056506872177124, 0.43042659759521484, 0.5137199759483337, 0.8549718260765076, 0.8665494322776794, 0.6115064024925232, 0.5910772681236267, 0.3031640946865082, 0.48032185435295105, 0.19622954726219177, 0.48574915528297424, 0.05997467413544655, 0.2577209770679474, 0.28531113266944885, 0.47368863224983215, 0.6821367144584656, 0.09779097139835358, 0.6406919360160828, 0.7752304077148438, 0.16580182313919067, 0.13897068798542023, 0.1013670414686203, 0.6323894262313843, 0.3964296579360962, 0.038471076637506485, 0.11642958968877792, 0.3634629547595978, 0.05207562819123268, 0.7929448485374451, 0.1936141699552536, 0.8103676438331604, 0.03534315526485443, 0.33798453211784363, 0.9296635985374451, 0.15173228085041046, 0.036552172154188156, 0.8376949429512024, 0.7968242168426514, 0.49639439582824707, 0.1261565238237381, 0.2932107448577881, 0.4444045424461365, 0.9630046486854553, 0.21369893848896027, 0.6607457995414734, 0.7070181965827942, 0.8705142736434937, 0.6740798354148865, 0.37467363476753235, 0.4422233998775482, 0.4752616882324219, 0.5885958075523376, 0.5320678949356079, 0.10258948802947998, 0.7731111645698547, 0.36442917585372925, 0.6248050332069397, 0.3110261857509613, 0.44739672541618347, 0.9958270192146301, 0.014048890210688114, 0.6394975781440735, 0.18857668340206146, 0.9547568559646606, 0.8206179738044739, 0.8320720791816711, 0.8375911116600037, 0.3438447117805481, 0.5062382817268372, 0.9222469925880432, 0.47115814685821533, 0.9397982358932495, 0.813818633556366, 0.48823612928390503, 0.5270284414291382, 0.5199353098869324, 0.2687769830226898, 0.34990739822387695, 0.42757102847099304, 0.533126175403595, 0.8758601546287537, 0.48459669947624207, 0.5585671067237854, 0.7069954872131348, 0.7993971109390259, 0.9727157950401306, 0.7387970685958862, 0.5987541675567627, 0.7170587182044983, 0.12223289906978607, 0.7391888499259949, 0.21914944052696228, 0.8804131746292114, 0.8722050189971924, 0.6282647848129272, 0.9380940198898315, 0.051131293177604675, 0.07591959089040756, 0.4958520233631134, 0.19831401109695435, 0.022410882636904716, 0.9596068859100342, 0.41891226172447205, 0.8400207161903381, 0.285489946603775, 0.6319930553436279, 0.06912998110055923, 0.6191803216934204, 0.7061749696731567, 0.6171972155570984, 0.4809798002243042, 0.7384523749351501, 0.22528979182243347, 0.6747691035270691], [0.28692713379859924, 0.06520818918943405, 0.3744406998157501, 0.7525026798248291, 0.1656099557876587, 0.7750082015991211, 0.357491672039032, 0.49196362495422363, 0.0695013776421547, 0.16731344163417816, 0.6661268472671509, 0.9140288829803467, 0.2669532597064972, 0.6883271932601929, 0.8798079490661621, 0.8461396098136902, 0.9515554308891296, 0.10805568844079971, 0.6684588193893433, 0.29154568910598755, 0.04266728460788727, 0.49914100766181946, 0.9574552774429321, 0.3451099693775177, 0.26174452900886536, 0.9909095764160156, 0.671578586101532, 0.15715394914150238, 0.020686592906713486, 0.08227512985467911, 0.08241908252239227, 0.886620044708252, 0.7027868628501892, 0.18169160187244415, 0.4334673583507538, 0.4492020010948181, 0.8636953830718994, 0.9734612703323364, 0.008602441288530827, 0.02694655768573284, 0.6131097674369812, 0.05467238649725914, 0.41444131731987, 0.6376237273216248, 0.5161823034286499, 0.18573574721813202, 0.2869839370250702, 0.5397359728813171, 0.15253660082817078, 0.9296717643737793, 0.25358036160469055, 0.8446953892707825, 0.19599655270576477, 0.499417781829834, 0.8196479678153992, 0.6643087863922119, 0.13502798974514008, 0.2584463059902191, 0.38302579522132874, 0.5055316686630249, 0.9658187627792358, 0.34907975792884827, 0.3609772026538849, 0.5373751521110535, 0.2281344085931778, 0.8305624127388, 0.00019040420011151582, 0.7510932087898254, 0.913651704788208, 0.3311188519001007, 0.764485776424408, 0.7264196276664734, 0.7785335183143616, 0.6655599474906921, 0.5467259287834167, 0.2838054299354553, 0.42492663860321045, 0.1155325323343277, 0.635614275932312, 0.5146297812461853, 0.06618134677410126, 0.11100170016288757, 0.3279643654823303, 0.19639401137828827, 0.19660130143165588, 0.7740129232406616, 0.3280939757823944, 0.06477812677621841, 0.9199623465538025, 0.95359867811203, 0.826554000377655, 0.4522235691547394, 0.9643361568450928, 0.9079455733299255, 0.6475633978843689, 0.6970379948616028, 0.7216652035713196, 0.547347366809845, 0.8806391358375549, 0.8639018535614014, 0.7520309686660767, 0.707471489906311, 0.4654307961463928, 0.33729588985443115, 0.9663614630699158, 0.5791378021240234, 0.9397416114807129, 0.22225302457809448, 0.16562563180923462, 0.7373189330101013, 0.6672008633613586, 0.41565874218940735, 0.5177655816078186, 0.9292502403259277, 0.680613100528717, 0.5501813888549805, 0.2804989814758301, 0.1694098263978958, 0.37948283553123474, 0.3047376275062561, 0.7066887617111206, 0.9925227165222168, 0.8606355786323547, 0.28199246525764465, 0.30083122849464417, 0.5982134342193604, 0.8491247296333313, 0.3049432933330536, 0.28076767921447754, 0.47860807180404663, 0.7713455557823181, 0.4550536870956421, 0.013019297271966934, 0.686676025390625, 0.8880800604820251, 0.6919599771499634, 0.03345126658678055, 0.2712666392326355, 0.5236623883247375, 0.4962524175643921, 0.07925251871347427, 0.28467825055122375, 0.4050706923007965, 0.5036053657531738, 0.9834820628166199, 0.7172552943229675, 0.5278702974319458, 0.00930959265679121, 0.03224645182490349, 0.7742170095443726, 0.5662176609039307, 0.6752477884292603, 0.12847934663295746, 0.9760090112686157, 0.5140868425369263, 0.5741744041442871, 0.6818036437034607, 0.6327570676803589, 0.7868500351905823, 0.20995956659317017, 0.06200133636593819, 0.42264097929000854, 0.7798914909362793, 0.7545000314712524, 0.3890123665332794, 0.7468118071556091, 0.7754124999046326, 0.7951440215110779, 0.9611406922340393, 0.46213746070861816, 0.730076014995575, 0.9154694676399231, 0.29087576270103455, 0.9846647381782532, 0.33799049258232117, 0.6650415658950806, 0.1380731463432312, 0.19026780128479004, 0.09133540093898773, 0.6381505727767944, 0.026194967329502106, 0.4831792712211609, 0.743855893611908, 0.27135616540908813, 0.19304300844669342, 0.9627269506454468, 0.20304498076438904, 0.5855889916419983, 0.4636213481426239, 0.9064098596572876, 0.6308901906013489, 0.48235663771629333, 0.2539636194705963, 0.8376057744026184, 0.31356891989707947, 0.8110498785972595, 0.6985701322555542, 0.5534795522689819, 0.9962347745895386, 0.6447452306747437, 0.5982842445373535, 0.7063862681388855, 0.16728654503822327, 0.43916070461273193, 0.00024410313926637173, 0.13619033992290497, 0.03482004255056381, 0.5737119913101196, 0.32577380537986755, 0.17311078310012817, 0.3995926082134247, 0.9500538110733032, 0.05602826178073883, 0.11166588962078094, 0.8766801953315735, 0.21773135662078857, 0.666167676448822, 0.08791042864322662, 0.037165768444538116, 0.617270827293396, 0.7420315742492676, 0.20186910033226013, 0.4632418155670166, 0.1596314013004303, 0.8284072279930115, 0.13482695817947388, 0.5627540349960327, 0.9455013275146484, 0.211316779255867, 0.7290102243423462, 0.4936765134334564, 0.8608038425445557, 0.8642008304595947, 0.49933627247810364, 0.4704028367996216, 0.7691221237182617, 0.7131527066230774, 0.33007746934890747, 0.9440901279449463, 0.5036263465881348, 0.2466406226158142, 0.3954753279685974, 0.4012646973133087, 0.4994591474533081, 0.3450269103050232, 0.45844733715057373, 0.9734466671943665, 0.9928969144821167, 0.2561552822589874, 0.3087804317474365, 0.37791386246681213, 0.23116803169250488, 0.2220071703195572, 0.4852278232574463, 0.4167824983596802, 0.0776529386639595, 0.520811915397644, 0.4737348258495331, 0.9518841505050659, 0.851845383644104, 0.6420314311981201, 0.5236219167709351, 0.11281438171863556, 0.644385814666748, 0.9330692887306213, 0.9660479426383972, 0.9292566776275635, 0.7382588982582092, 0.34444963932037354, 0.18136852979660034, 0.22252511978149414, 0.39431294798851013, 0.11619335412979126, 0.30547571182250977, 0.9723085761070251, 0.7461925148963928, 0.7033811211585999, 0.15809082984924316, 0.5066729187965393, 0.491458535194397, 0.9257228970527649, 0.6699705719947815, 0.30169039964675903, 0.25920918583869934, 0.10706984996795654, 0.49883997440338135, 0.6275568008422852, 0.3990577161312103, 0.9386727213859558, 0.7522314786911011, 0.5497003793716431, 0.39595428109169006, 0.6568796634674072, 0.6911807060241699, 0.7355496883392334, 0.3139567971229553, 0.4030535817146301, 0.23539285361766815, 0.1378508061170578, 0.30918988585472107, 0.2599186897277832, 0.2836800515651703, 0.4792720377445221, 0.38374748826026917, 0.732452392578125, 0.2576272785663605, 0.7151909470558167, 0.8898884057998657, 0.4997911751270294, 0.8063247799873352, 0.0775412991642952, 0.2616601586341858, 0.0867619588971138, 0.49338671565055847, 0.3791600167751312, 0.48270294070243835, 0.2307976335287094, 0.034788090735673904, 0.7595230340957642, 0.9379448890686035, 0.19106808304786682, 0.7384559512138367, 0.35123616456985474, 0.7415264844894409, 0.6015549898147583, 0.7556101679801941, 0.9902021884918213, 0.28204452991485596, 0.8794876337051392, 0.2055140733718872, 0.7950683236122131, 0.15785595774650574, 0.2005780190229416], [0.430020809173584, 0.20727849006652832, 0.8072442412376404, 0.037505727261304855, 0.34598082304000854, 0.5713973641395569, 0.8006781339645386, 0.9755572080612183, 0.5362966060638428, 0.1925501972436905, 0.22895322740077972, 0.61086106300354, 0.14728647470474243, 0.8466960191726685, 0.8422708511352539, 0.7244114279747009, 0.9711930155754089, 0.45017683506011963, 0.9597342014312744, 0.1997687965631485, 0.2566474676132202, 0.531429648399353, 0.8879612684249878, 0.009430769830942154, 0.856073260307312, 0.373008131980896, 0.8631445169448853, 0.8573013544082642, 0.8870662450790405, 0.563801109790802, 0.12033550441265106, 0.3014896810054779, 0.35428664088249207, 0.6499993205070496, 0.3393896818161011, 0.4906466007232666, 0.3612080514431, 0.6210029125213623, 0.15282794833183289, 0.38893723487854004, 0.8787738680839539, 0.28579485416412354, 0.486983984708786, 0.7777359485626221, 0.6938441395759583, 0.46641430258750916, 0.07384567707777023, 0.8672577738761902, 0.9199715256690979, 0.7908905148506165, 0.7655586004257202, 0.8846514821052551, 0.8737118244171143, 0.47470301389694214, 0.3920162618160248, 0.4555235207080841, 0.13360412418842316, 0.19201122224330902, 0.688297688961029, 0.7376689314842224, 0.692673921585083, 0.5788929462432861, 0.15480473637580872, 0.3777100443840027, 0.9509391188621521, 0.9526332020759583, 0.29495126008987427, 0.3744526505470276, 0.18463845551013947, 0.3591172397136688, 0.44769254326820374, 0.28228071331977844, 0.9069678783416748, 0.07697264105081558, 0.4386189877986908, 0.07003746181726456, 0.3592091500759125, 0.7634172439575195, 0.1007799431681633, 0.5794978141784668, 0.5724019408226013, 0.5686197280883789, 0.5023679733276367, 0.6185713410377502, 0.26415127515792847, 0.26429077982902527, 0.7759628295898438, 0.1743319034576416, 0.8237887024879456, 0.8976368308067322, 0.5693147778511047, 0.28016525506973267, 0.43674978613853455, 0.5117327570915222, 0.696871280670166, 0.9395566582679749, 0.4268031716346741, 0.20533931255340576, 0.30903470516204834, 0.9490150809288025, 0.21229352056980133, 0.25270745158195496, 0.032009489834308624, 0.612373411655426, 0.471620112657547, 0.8246209621429443, 0.6416214108467102, 0.6530384421348572, 0.6845768094062805, 0.45790523290634155, 0.2966424524784088, 0.018643096089363098, 0.3967496156692505, 0.012681480497121811, 0.21346643567085266, 0.3767392337322235, 0.5221259593963623, 0.38041815161705017, 0.2155052125453949, 0.7531785368919373, 0.3172941505908966, 0.35378968715667725, 0.22326302528381348, 0.07068492472171783, 0.7553520202636719, 0.454876184463501, 0.5529696345329285, 0.8719900846481323, 0.8824750781059265, 0.578937828540802, 0.9462696313858032, 0.5425934195518494, 0.8811001181602478, 0.470367431640625, 0.2658708393573761, 0.49867045879364014, 0.16096964478492737, 0.380737841129303, 0.22944599390029907, 0.45277127623558044, 0.09237683564424515, 0.4346342384815216, 0.5702418088912964, 0.9895857572555542, 0.9969745874404907, 0.6589239835739136, 0.4547007381916046, 0.7493973970413208, 0.9222409725189209, 0.6910229325294495, 0.19139188528060913, 0.6090787053108215, 0.4981289207935333, 0.9155688881874084, 0.6661022305488586, 0.970022439956665, 0.046345893293619156, 0.6982426047325134, 0.21745634078979492, 0.46780869364738464, 0.8439748287200928, 0.9951592087745667, 0.22211824357509613, 0.8607926368713379, 0.13146482408046722, 0.305340975522995, 0.2176465541124344, 0.9783453345298767, 0.10651073604822159, 0.8016905784606934, 0.5235501527786255, 0.72112637758255, 0.08682327717542648, 0.461422860622406, 0.5542535185813904, 0.2911684513092041, 0.6503299474716187, 0.08322471380233765, 0.08047571033239365, 0.6520721316337585, 0.2640087604522705, 0.8584319949150085, 0.565543532371521, 0.8812727928161621, 0.8500000834465027, 0.7060083746910095, 0.9811123609542847, 0.24655646085739136, 0.5622190833091736, 0.1290954202413559, 0.7109172344207764, 0.82014399766922, 0.8954952955245972, 0.5296651721000671, 0.9262061715126038, 0.38113757967948914, 0.35439303517341614, 0.7344405651092529, 0.9193987846374512, 0.08018621802330017, 0.5092326998710632, 0.44646701216697693, 0.03313719853758812, 0.11049315333366394, 0.8204286694526672, 0.9465748071670532, 0.5701441764831543, 0.24053044617176056, 0.9796893000602722, 0.4232541024684906, 0.13322344422340393, 0.7877817749977112, 0.6998392939567566, 0.6903592944145203, 0.3837158977985382, 0.28223514556884766, 0.22876480221748352, 0.6026307344436646, 0.3137528896331787, 0.7262490391731262, 0.914114236831665, 0.845904529094696, 0.034360673278570175, 0.6467256546020508, 0.05180485546588898, 0.29710853099823, 0.29838281869888306, 0.5555736422538757, 0.0433976948261261, 0.22967319190502167, 0.9343235492706299, 0.02970273420214653, 0.34430328011512756, 0.49020886421203613, 0.10872887820005417, 0.2918298840522766, 0.6244209408760071, 0.3304654359817505, 0.5581113696098328, 0.9440836310386658, 0.09643686562776566, 0.6710283756256104, 0.024719661101698875, 0.38959819078445435, 0.7181106209754944, 0.956476092338562, 0.4630621373653412, 0.6291904449462891, 0.48777198791503906, 0.11857497692108154, 0.09007877111434937, 0.5559265613555908, 0.8743963241577148, 0.2563348412513733, 0.7474157214164734, 0.6280227899551392, 0.3368074595928192, 0.6571456789970398, 0.06386490911245346, 0.2548171281814575, 0.3814086318016052, 0.6421136856079102, 0.036213282495737076, 0.042344726622104645, 0.9270373582839966, 0.5295102596282959, 0.8966782689094543, 0.9968037009239197, 0.16669076681137085, 0.3775535523891449, 0.9866167306900024, 0.007660854607820511, 0.9822471141815186, 0.9997047781944275, 0.7548112273216248, 0.3894321024417877, 0.6769243478775024, 0.02082850970327854, 0.35210320353507996, 0.03600070998072624, 0.42772746086120605, 0.6835224628448486, 0.019876644015312195, 0.895582914352417, 0.08906082063913345, 0.2366315871477127, 0.38927409052848816, 0.11196590214967728, 0.7481252551078796, 0.6795761585235596, 0.5895518064498901, 0.4294305741786957, 0.5955045223236084, 0.37594565749168396, 0.6658005118370056, 0.8949156403541565, 0.7731550931930542, 0.36662575602531433, 0.8856433629989624, 0.31222283840179443, 0.6794395446777344, 0.7394136786460876, 0.4112803637981415, 0.860826849937439, 0.9735723733901978, 0.12014488130807877, 0.04554859548807144, 0.29375067353248596, 0.026266874745488167, 0.9870815277099609, 0.2690941393375397, 0.1565324068069458, 0.5856927633285522, 0.955388069152832, 0.8725603818893433, 0.8754807710647583, 0.19776107370853424, 0.2681354880332947, 0.08818994462490082, 0.6984471082687378, 0.3559258282184601, 0.6630595922470093, 0.03317487612366676, 0.7419624328613281, 0.12853308022022247, 0.9876512289047241, 0.26865869760513306, 0.488491952419281, 0.8214971423149109, 0.6314445734024048, 0.08084391802549362, 0.9256485104560852, 0.7350147366523743], [0.7535449862480164, 0.5113141536712646, 0.16440211236476898, 0.6552965044975281, 0.31087812781333923, 0.8332025408744812, 0.3283444941043854, 0.6703680157661438, 0.5414521098136902, 0.1537604033946991, 0.6825070381164551, 0.969494640827179, 0.059792410582304, 0.5877082347869873, 0.6477670669555664, 0.8387341499328613, 0.318136990070343, 0.8298478722572327, 0.7407869100570679, 0.1907794326543808, 0.7462912797927856, 0.4484204947948456, 0.20417745411396027, 0.2429603487253189, 0.3346216678619385, 0.7389843463897705, 0.15294630825519562, 0.642848551273346, 0.8162521123886108, 0.5180686712265015, 0.9081215262413025, 0.7193546295166016, 0.5156102180480957, 0.9665672779083252, 0.5506469011306763, 0.14846938848495483, 0.21444080770015717, 0.8236505389213562, 0.8551543354988098, 0.3940122127532959, 0.3600645363330841, 0.20599403977394104, 0.8467820882797241, 0.9867057204246521, 0.9399945735931396, 0.44292688369750977, 0.8925713300704956, 0.20262500643730164, 0.07084610313177109, 0.560756266117096, 0.3443903923034668, 0.4804803729057312, 0.07254683971405029, 0.17426763474941254, 0.4885534644126892, 0.5063747763633728, 0.7152000665664673, 0.7383880019187927, 0.3846123516559601, 0.45150190591812134, 0.2818889021873474, 0.1558251678943634, 0.40154317021369934, 0.5814195871353149, 0.9978304505348206, 0.7394775748252869, 0.811509907245636, 0.8253385424613953, 0.933222770690918, 0.9183368682861328, 0.17698319256305695, 0.3180347979068756, 0.26660364866256714, 0.8030181527137756, 0.48699015378952026, 0.9453171491622925, 0.5731173157691956, 0.028239019215106964, 0.24247731268405914, 0.4031029939651489, 0.13388074934482574, 0.5614939332008362, 0.04722219705581665, 0.2434142827987671, 0.011753730475902557, 0.1194332018494606, 0.7065916657447815, 0.503231406211853, 0.958759069442749, 0.13203123211860657, 0.19020341336727142, 0.06072349473834038, 0.19906052947044373, 0.2949884235858917, 0.42956140637397766, 0.24913901090621948, 0.16995744407176971, 0.3801227807998657, 0.05046938359737396, 0.8236263990402222, 0.5701900124549866, 0.5091890692710876, 0.4664566218852997, 0.5552051663398743, 0.9934502840042114, 0.38283345103263855, 0.5679815411567688, 0.20933674275875092, 0.3947118818759918, 0.09480325132608414, 0.865246057510376, 0.6803109645843506, 0.7686507701873779, 0.7620403170585632, 0.13366685807704926, 0.6353782415390015, 0.14671890437602997, 0.6181390881538391, 0.13390447199344635, 0.1577085554599762, 0.12378554046154022, 0.5371906161308289, 0.11801814287900925, 0.36412930488586426, 0.4249703288078308, 0.9874014258384705, 0.7805427312850952, 0.5850542187690735, 0.7327951192855835, 0.01111982949078083, 0.368289053440094, 0.28691035509109497, 0.6611459255218506, 0.12954360246658325, 0.9447691440582275, 0.3240184783935547, 0.8733043074607849, 0.8495312929153442, 0.46660563349723816, 0.0025432531256228685, 0.11921897530555725, 0.38636648654937744, 0.7380461096763611, 0.4765915274620056, 0.9371113181114197, 0.9061200618743896, 0.325570672750473, 0.9891213774681091, 0.528005063533783, 0.07393039762973785, 0.03666314482688904, 0.3680742383003235, 0.41504350304603577, 0.1805591881275177, 0.9002841114997864, 0.30931445956230164, 0.7728369235992432, 0.21750789880752563, 0.43430718779563904, 0.38278937339782715, 0.766554594039917, 0.3656732439994812, 0.2225600630044937, 0.8373969793319702, 0.6501225829124451, 0.17004290223121643, 0.23791728913784027, 0.09889668226242065, 0.8332201242446899, 0.41903558373451233, 0.8306111693382263, 0.583634078502655, 0.7937557101249695, 0.7561871409416199, 0.46588462591171265, 0.25614261627197266, 0.3903423547744751, 0.5313000679016113, 0.9968966841697693, 0.28651997447013855, 0.7359548807144165, 0.25638508796691895, 0.6323515772819519, 0.7351441383361816, 0.0008455485221929848, 0.7285113334655762, 0.8686726689338684, 0.9048526883125305, 0.8740898966789246, 0.38180020451545715, 0.1468554586172104, 0.3413504660129547, 0.8808906674385071, 0.6759511828422546, 0.5049788951873779, 0.006145765073597431, 0.07199118286371231, 0.5548295974731445, 0.8713198304176331, 0.610103964805603, 0.3607301115989685, 0.0059814914129674435, 0.21057601273059845, 0.6155188083648682, 0.292432576417923, 0.5368995070457458, 0.040247272700071335, 0.5397986769676208, 0.4603394865989685, 0.022053133696317673, 0.8207181692123413, 0.7870582342147827, 0.11512762308120728, 0.6260072588920593, 0.8588316440582275, 0.5127977728843689, 0.7527575492858887, 0.12297075241804123, 0.05960074067115784, 0.23551298677921295, 0.7669026255607605, 0.28901106119155884, 0.8442255258560181, 0.06693888455629349, 0.4822765588760376, 0.4363592267036438, 0.8113841414451599, 0.6701014637947083, 0.7430853247642517, 0.6072028875350952, 0.15803126990795135, 0.7607400417327881, 0.7197344899177551, 0.7312731742858887, 0.45978325605392456, 0.4237270653247833, 0.14738845825195312, 0.8153489232063293, 0.43512067198753357, 0.9544827938079834, 0.2716640830039978, 0.48355841636657715, 0.3515886068344116, 0.6074634194374084, 0.9670428037643433, 0.2223561555147171, 0.38512852787971497, 0.2953197658061981, 0.09450680762529373, 0.0361190065741539, 0.9926519989967346, 0.33036771416664124, 0.7069787979125977, 0.7830181121826172, 0.6408859491348267, 0.2651127874851227, 0.6707323789596558, 0.9208170771598816, 0.45766833424568176, 0.8916789889335632, 0.8611966371536255, 0.7716070413589478, 0.6962323784828186, 0.9590990543365479, 0.5012273192405701, 0.9195650219917297, 0.08801872283220291, 0.7090291380882263, 0.11884942650794983, 0.37018755078315735, 0.5849614143371582, 0.6925796270370483, 0.6824446320533752, 0.1791974902153015, 0.08883563429117203, 0.14250409603118896, 0.6185733675956726, 0.17890942096710205, 0.9162400364875793, 0.6798473000526428, 0.3016386032104492, 0.6307585835456848, 0.07318232208490372, 0.6047093272209167, 0.1549958437681198, 0.632396936416626, 0.27901390194892883, 0.5533112287521362, 0.702294111251831, 0.2197130024433136, 0.058224957436323166, 0.142136350274086, 0.8376265168190002, 0.9517247080802917, 0.5764890909194946, 0.08420988917350769, 0.5984582901000977, 0.3382578194141388, 0.1730482578277588, 0.7460067272186279, 0.8651094436645508, 0.5334192514419556, 0.5740729570388794, 0.9464249610900879, 0.002799672307446599, 0.8565170168876648, 0.9292303323745728, 0.8485234975814819, 0.551000714302063, 0.8910717964172363, 0.4966062903404236, 0.36769410967826843, 0.4981803297996521, 0.04316326975822449, 0.030057154595851898, 0.51868736743927, 0.5896717309951782, 0.06935777515172958, 0.7634913325309753, 0.15012899041175842, 0.21315820515155792, 0.47604793310165405, 0.7744302749633789, 0.48867806792259216, 0.9003833532333374, 0.75777268409729, 0.561802327632904, 0.8691651225090027, 0.34677907824516296, 0.5794800519943237, 0.6439725160598755, 0.09112608432769775, 0.2631385326385498], [0.9818997979164124, 0.3056069314479828, 0.9094067215919495, 0.05062861740589142, 0.24986138939857483, 0.23178231716156006, 0.0020815483294427395, 0.7244261503219604, 0.23370178043842316, 0.5853907465934753, 0.09587395191192627, 0.5735207796096802, 0.49507319927215576, 0.47137758135795593, 0.789029061794281, 0.35106247663497925, 0.7153439521789551, 0.8068069219589233, 0.24364735186100006, 0.297124981880188, 0.6123047471046448, 0.18278111517429352, 0.0050978874787688255, 0.607944130897522, 0.7038357853889465, 0.10687804222106934, 0.5702106952667236, 0.4744841456413269, 0.510541558265686, 0.9363977313041687, 0.04327410086989403, 0.1845359355211258, 0.08244242519140244, 0.6194214820861816, 0.8616489768028259, 0.5373979210853577, 0.03184416517615318, 0.5032141208648682, 0.11007754504680634, 0.08659587800502777, 0.30353134870529175, 0.8513335585594177, 0.6958308219909668, 0.9847428798675537, 0.48206573724746704, 0.186160609126091, 0.36315786838531494, 0.08443683385848999, 0.40024179220199585, 0.6027835011482239, 0.6564081311225891, 0.9700679779052734, 0.37829622626304626, 0.08751579374074936, 0.8633459806442261, 0.1747620701789856, 0.32483741641044617, 0.14325310289859772, 0.7102025151252747, 0.8828636407852173, 0.21087145805358887, 0.5793043971061707, 0.024439210072159767, 0.6851965188980103, 0.14485998451709747, 0.31270983815193176, 0.5528505444526672, 0.46090880036354065, 0.774366557598114, 0.033333804458379745, 0.21049925684928894, 0.19884370267391205, 0.38769960403442383, 0.8496996164321899, 0.9640805721282959, 0.9627425074577332, 0.1861339509487152, 0.2048204094171524, 0.2872549593448639, 0.5900028944015503, 0.8349010944366455, 0.5086230039596558, 0.037303660064935684, 0.6817032694816589, 0.6194190382957458, 0.37776488065719604, 0.5359187126159668, 0.23985838890075684, 0.299911767244339, 0.5867577791213989, 0.3709050416946411, 0.34227892756462097, 0.14367787539958954, 0.8768070340156555, 0.5170491933822632, 0.8644374012947083, 0.5925307869911194, 0.07384765148162842, 0.03587408363819122, 0.17561830580234528, 0.36950963735580444, 0.17145609855651855, 0.35000601410865784, 0.9166214466094971, 0.01676901988685131, 0.07521180808544159, 0.7123664021492004, 0.731927752494812, 0.25715747475624084, 0.07505720853805542, 0.3381712734699249, 0.6949251890182495, 0.27636611461639404, 0.5070415139198303, 0.43626177310943604, 0.8935046792030334, 0.616369366645813, 0.345745712518692, 0.3939795196056366, 0.12525926530361176, 0.7956650257110596, 0.1906309425830841, 0.11345252394676208, 0.32529622316360474, 0.6635484099388123, 0.5271508693695068, 0.009410340338945389, 0.5818213224411011, 0.7405933141708374, 0.7322611808776855, 0.20512303709983826, 0.07556654512882233, 0.3732077181339264, 0.21131791174411774, 0.956706702709198, 0.6820374131202698, 0.3066806495189667, 0.8237460255622864, 0.6003679037094116, 0.04121534526348114, 0.2172500044107437, 0.2578508257865906, 0.9159240126609802, 0.6398462057113647, 0.18855822086334229, 0.8471785187721252, 0.38836491107940674, 0.9913443326950073, 0.5535097718238831, 0.012364435940980911, 0.4035574793815613, 0.6709698438644409, 0.43398910760879517, 0.6600087285041809, 0.09370474517345428, 0.015200094319880009, 0.093471959233284, 0.6380972266197205, 0.222098708152771, 0.8170803189277649, 0.7907429337501526, 0.12874457240104675, 0.05807476118206978, 0.6679570078849792, 0.6995212435722351, 0.8872329592704773, 0.3498162627220154, 0.02670256793498993, 0.7553675174713135, 0.5978493094444275, 0.9541160464286804, 0.303841233253479, 0.009794087149202824, 0.05628827214241028, 0.8846683502197266, 0.35373449325561523, 0.0742889866232872, 0.028716009110212326, 0.6389127373695374, 0.3603839874267578, 0.7421257495880127, 0.6078440546989441, 0.23108164966106415, 0.6082029342651367, 0.3948761522769928, 0.671179473400116, 0.24601863324642181, 0.3029305338859558, 0.16878925263881683, 0.13380946218967438, 0.40322571992874146, 0.6135350465774536, 0.586026668548584, 0.1903316080570221, 0.38814929127693176, 0.43738237023353577, 0.2178705483675003, 0.6208027005195618, 0.3407086431980133, 0.09840484708547592, 0.6837342381477356, 0.17288583517074585, 0.3526690602302551, 0.7715925574302673, 0.12737815082073212, 0.5504318475723267, 0.590687096118927, 0.27007046341896057, 0.11336556077003479, 0.4046056568622589, 0.2326718270778656, 0.3840305507183075, 0.14021672308444977, 0.3999054431915283, 0.9206194877624512, 0.00892342533916235, 0.5726542472839355, 0.41701698303222656, 0.845354437828064, 0.7683830261230469, 0.4520314335823059, 0.25584810972213745, 0.9431735277175903, 0.8204721212387085, 0.8828602433204651, 0.08583784103393555, 0.3932872414588928, 0.4678167998790741, 0.2947070002555847, 0.22302456200122833, 0.3254753351211548, 0.17087970674037933, 0.956358790397644, 0.9013692140579224, 0.6212666630744934, 0.5727525949478149, 0.9903175830841064, 0.4987879991531372, 0.7618570923805237, 0.03256041184067726, 0.42441919445991516, 0.19164815545082092, 0.08467286080121994, 0.8424705862998962, 0.7313635349273682, 0.3549219071865082, 0.12190280854701996, 0.7979897856712341, 0.3522093594074249, 0.7816787958145142, 0.6058093309402466, 0.7209429740905762, 0.5976496338844299, 0.9608692526817322, 0.240519717335701, 0.8721349835395813, 0.6244351863861084, 0.6956682205200195, 0.9840487837791443, 0.03397634997963905, 0.26806432008743286, 0.7247425317764282, 0.7889485359191895, 0.24956384301185608, 0.4713665843009949, 0.8591054081916809, 0.9739177823066711, 0.9717263579368591, 0.2092052847146988, 0.3796248733997345, 0.4302310645580292, 0.48502007126808167, 0.8598423004150391, 0.4843408763408661, 0.2934298813343048, 0.6318197250366211, 0.7885231375694275, 0.6062023639678955, 0.9277468323707581, 0.9808526635169983, 0.7470989227294922, 0.8880435228347778, 0.0870116725564003, 0.581099808216095, 0.02382129244506359, 0.2703688144683838, 0.6424302458763123, 0.6868426203727722, 0.8287448883056641, 0.8385953903198242, 0.27155622839927673, 0.44257301092147827, 0.7712008953094482, 0.20903582870960236, 0.8801287412643433, 0.03150499612092972, 0.37734904885292053, 0.04198654368519783, 0.28184089064598083, 0.8420733213424683, 0.983711302280426, 0.4263196885585785, 0.23671597242355347, 0.32346510887145996, 0.8393024206161499, 0.1934644877910614, 0.768661618232727, 0.30950620770454407, 0.24925687909126282, 0.8964983820915222, 0.21977174282073975, 0.9415706396102905, 0.4431432783603668, 0.9963507652282715, 0.730964183807373, 0.8351238965988159, 0.5438597202301025, 0.9701956510543823, 0.7147973775863647, 0.43729716539382935, 0.056982047855854034, 0.190230593085289, 0.8658610582351685, 0.9421530961990356, 0.031119046732783318, 0.4107027053833008, 0.4475417137145996, 0.16101261973381042, 0.033442575484514236, 0.8615226745605469, 0.8304508924484253, 0.8846862316131592, 0.29755300283432007], [0.8871021866798401, 0.9745954275131226, 0.7963814735412598, 0.3461383581161499, 0.37488222122192383, 0.500789999961853, 0.9144977927207947, 0.4998738765716553, 0.42658737301826477, 0.06148224696516991, 0.3252156376838684, 0.2114073783159256, 0.4731934666633606, 0.4761776328086853, 0.19474375247955322, 0.08558259904384613, 0.6337425708770752, 0.2419072836637497, 0.6635344624519348, 0.5684802532196045, 0.8697792887687683, 0.2630116045475006, 0.971799910068512, 0.8457497358322144, 0.4947783946990967, 0.34818750619888306, 0.20947636663913727, 0.5684096217155457, 0.18427495658397675, 0.35493341088294983, 0.06343213468790054, 0.6680117845535278, 0.6018686294555664, 0.6797294020652771, 0.8437749743461609, 0.5496266484260559, 0.3715398907661438, 0.2517816424369812, 0.5212951302528381, 0.43733319640159607, 0.4386844336986542, 0.267839252948761, 0.19669076800346375, 0.7946711182594299, 0.9751600623130798, 0.6684079170227051, 0.6040694713592529, 0.17066167294979095, 0.4458518922328949, 0.9408890008926392, 0.9614742398262024, 0.07380066812038422, 0.8417747020721436, 0.21675580739974976, 0.9451765418052673, 0.225847527384758, 0.7719963788986206, 0.9572601914405823, 0.04680434986948967, 0.012762889266014099, 0.5339301824569702, 0.4312657117843628, 0.9412370324134827, 0.2925896644592285, 0.020325446501374245, 0.8695546984672546, 0.6014174818992615, 0.5146411657333374, 0.6305662989616394, 0.904169499874115, 0.2620513141155243, 0.25100335478782654, 0.9890413284301758, 0.5182539224624634, 0.03392895683646202, 0.7607439160346985, 0.5643287897109985, 0.0464811734855175, 0.3742644786834717, 0.5936237573623657, 0.8646987080574036, 0.660973072052002, 0.6372513771057129, 0.7458204627037048, 0.4210100769996643, 0.8118094205856323, 0.7245047092437744, 0.09521625936031342, 0.7248634696006775, 0.8932344317436218, 0.9434528946876526, 0.4415283501148224, 0.12155349552631378, 0.5681042671203613, 0.09895724058151245, 0.573434591293335, 0.747644305229187, 0.5935482978820801, 0.6936858892440796, 0.7491610646247864, 0.4788719713687897, 0.2578950524330139, 0.7878939509391785, 0.508748471736908, 0.6991021633148193, 0.38818517327308655, 0.9044980406761169, 0.39490219950675964, 0.6001904606819153, 0.6341468095779419, 0.529198169708252, 0.7944867014884949, 0.26846185326576233, 0.06225118041038513, 0.15530617535114288, 0.42696693539619446, 0.7015729546546936, 0.15361809730529785, 0.42009276151657104, 0.6298820376396179, 0.20687039196491241, 0.644298255443573, 0.6505099534988403, 0.22967898845672607, 0.8922696113586426, 0.9503487944602966, 0.9301228523254395, 0.2272467017173767, 0.32480260729789734, 0.5436763763427734, 0.8145245909690857, 0.023237518966197968, 0.25219404697418213, 0.6516146659851074, 0.2267005294561386, 0.22025583684444427, 0.9967694878578186, 0.6081114411354065, 0.6897470951080322, 0.08900945633649826, 0.297678679227829, 0.12200556695461273, 0.2483023703098297, 0.13399289548397064, 0.11293146759271622, 0.7679668664932251, 0.2765757143497467, 0.895186185836792, 0.010385282337665558, 0.3204948306083679, 0.3018329441547394, 0.40306517481803894, 0.5794306397438049, 0.5040972828865051, 0.3474913239479065, 0.33987757563591003, 0.7096824049949646, 0.3163009583950043, 0.6956108808517456, 0.5027393698692322, 0.23876287043094635, 0.7543101906776428, 0.5392941236495972, 0.6362565755844116, 0.07644444704055786, 0.4592682421207428, 0.5131965279579163, 0.4120819866657257, 0.37245017290115356, 0.13296839594841003, 0.4967903196811676, 0.7769731879234314, 0.5831562876701355, 0.30913642048835754, 0.192853182554245, 0.37307071685791016, 0.3147319257259369, 0.19012537598609924, 0.41039004921913147, 0.3571668267250061, 0.5324676632881165, 0.8304714560508728, 0.357461541891098, 0.6739798188209534, 0.17042024433612823, 0.2379010170698166, 0.8602471947669983, 0.745328426361084, 0.7140266299247742, 0.8204612135887146, 0.8930073380470276, 0.9375597834587097, 0.7266704440116882, 0.9292128682136536, 0.2686272859573364, 0.4773823022842407, 0.18965959548950195, 0.10557673126459122, 0.39618119597435, 0.33938780426979065, 0.631264328956604, 0.4069339334964752, 0.15573403239250183, 0.4956801235675812, 0.32984524965286255, 0.7674102187156677, 0.688622772693634, 0.9556388258934021, 0.9655774831771851, 0.5751008987426758, 0.2609840929508209, 0.5635061264038086, 0.29829102754592896, 0.5202740430831909, 0.7597640156745911, 0.7238454818725586, 0.45865076780319214, 0.30508551001548767, 0.029544459655880928, 0.858718991279602, 0.8322072625160217, 0.6057921648025513, 0.8085129261016846, 0.19375362992286682, 0.6594136357307434, 0.2921437919139862, 0.8214941620826721, 0.4003485143184662, 0.4801754951477051, 0.3647879660129547, 0.427356094121933, 0.8715562224388123, 0.9730017781257629, 0.267910897731781, 0.8021843433380127, 0.9573656320571899, 0.5185125470161438, 0.5988451838493347, 0.8228565454483032, 0.6620703935623169, 0.536645770072937, 0.8959537148475647, 0.5600439310073853, 0.7867327928543091, 0.9159662127494812, 0.31793779134750366, 0.8335440158843994, 0.18825703859329224, 0.5413767099380493, 0.6550678610801697, 0.8578516244888306, 0.31589624285697937, 0.9695016741752625, 0.7510452270507812, 0.922879695892334, 0.25536400079727173, 0.546999990940094, 0.45243412256240845, 0.9613521695137024, 0.8127783536911011, 0.1645299196243286, 0.9107787013053894, 0.3620229959487915, 0.47325193881988525, 0.05469055101275444, 0.5105841755867004, 0.9056203961372375, 0.46345648169517517, 0.03251352533698082, 0.8435874581336975, 0.47013187408447266, 0.10156356543302536, 0.6972218155860901, 0.18485498428344727, 0.6101553440093994, 0.4190536439418793, 0.24808742105960846, 0.4341006278991699, 0.2986932694911957, 0.4702276885509491, 0.5848143100738525, 0.5046787261962891, 0.05549817159771919, 0.5706266164779663, 0.9453578591346741, 0.8530839681625366, 0.4225025475025177, 0.7719478011131287, 0.16989411413669586, 0.7101381421089172, 0.17730580270290375, 0.08019202947616577, 0.6627210378646851, 0.3558642268180847, 0.5161483883857727, 0.04993654415011406, 0.22715120017528534, 0.0759567990899086, 0.9277580976486206, 0.11558835208415985, 0.9205822348594666, 0.06934553384780884, 0.20209679007530212, 0.7102745771408081, 0.6321900486946106, 0.050600815564394, 0.3327714800834656, 0.1287257820367813, 0.3372960388660431, 0.35780468583106995, 0.53821861743927, 0.75009685754776, 0.031901028007268906, 0.676792562007904, 0.9300218820571899, 0.3351019322872162, 0.661972165107727, 0.44743165373802185, 0.295157790184021, 0.5321283936500549, 0.05859080329537392, 0.6790412068367004, 0.5383374691009521, 0.2651929259300232, 0.031535226851701736, 0.9305022954940796, 0.6546061038970947, 0.04528861865401268, 0.7890992760658264, 0.10949552059173584, 0.45709630846977234, 0.12012013792991638, 0.919910728931427]], "dtype": "float32"}, 0.5], "e": {"__nd__": [[0.0, 0.0, 3.441908597946167, 0.0, 0.0, 0.0, -2.7229976654052734, 0.0, 0.0, 0.14028844237327576, 0.6395143270492554, 0.0, 1.1828186511993408, 0.0, 0.0, 3.0119268894195557, 0.0, 0.0, 0.0, 0.0, -0.9565206170082092, -0.6531355977058411, 0.04084353148937225, -0.47722193598747253, -2.114772319793701, 0.0, 0.0, 2.763543128967285, 0.0, -1.335315465927124, 0.0, 1.851601243019104, 0.0, 0.0, 0.0, 0.0, 0.0, 0.4077697694301605, -2.355067729949951, 0.0, -3.2614073753356934, 0.0, 0.0, 1.258555293083191, 0.8551623821258545, 0.0, 0.0, 2.378364324569702, -1.320895791053772, 0.0, -0.6588889956474304, 0.0, 0.0, 0.0, 0.0, -1.6335349082946777, 0.0, 0.0, 0.0, 0.0, 0.0, 2.9078638553619385, 0.6456843614578247, 0.0, 0.0, 0.0, -0.49905118346214294, -1.6604663133621216, -1.639307975769043, 0.0, 2.1718521118164062, 3.87152099609375, 2.366986036300659, 0.0, 0.0, -2.6445837020874023, 1.5436428785324097, 0.0, 0.0, -0.1284504532814026, 0.0, 0.0, 0.0, 6.293309688568115, 1.7308261394500732, 2.1997454166412354, 0.0, 0.0, 0.0, -0.6467289924621582, 0.0, -0.15297731757164001, 0.0, 0.0, 1.2720955610275269, 0.0, 0.0, 2.826268196105957, 0.0, -1.2494122982025146, 2.8266046047210693, 0.89408278465271, 0.0, -4.390505313873291, 0.0, -2.467832565307617, -2.700441598892212, 0.0, 0.0, -2.912522792816162, -2.135376214981079, -1.0889880657196045, -0.17419789731502533, -2.5855164527893066, 0.0, 0.0, -0.645802915096283, 0.0, 4.731442928314209, 4.563289642333984, -0.9092491269111633, 0.7663465142250061, 2.5943286418914795, -2.3084373474121094, -1.1122652292251587, 0.0, -1.3293401002883911, -0.805516242980957, 0.0, 0.0, 0.0, -1.599319577217102, -2.571460485458374, 0.0, 3.6046829223632812, 0.0, 0.0, 0.0, 0.0, 0.2801241874694824, 0.0, 0.0, -0.39900556206703186, 0.0, 0.0, 2.133889675140381, -0.9740045666694641, -3.4141578674316406, 0.0, -1.020280361175537, 0.0, 0.0, 2.2128140926361084, 0.0, 2.2549777030944824, 1.4470576047897339, -3.2451882362365723, 0.0, 0.4074931740760803, 0.0, 0.9407179951667786, 0.0, -2.849273443222046, 0.0, -2.287163734436035, -1.7910218238830566, 0.0, 1.370346188545227, 2.696787118911743, -1.9609694480895996, -1.1401993036270142, 0.0, 0.0, 0.0, -4.7403950691223145, 0.0, 0.3149302303791046, 0.0, 0.0, 0.1770550161600113, 0.5660433769226074, -2.225175619125366, -2.1138107776641846, -1.352182149887085, 0.0, 0.0, 0.7481098771095276, 0.0, 2.01621150970459, 0.0, 0.41812029480934143, 0.0, 3.2005231380462646, 0.0, 0.0, 0.0, 0.8473945260047913, 0.0, 0.0, -0.8947237730026245, 0.0, 0.0, -2.330533742904663, 0.0, -0.6788964867591858, 0.0, 2.945112466812134, 0.0, -1.2736449241638184, 4.961585998535156, 0.0, 0.0, -0.8620490431785583, 0.0, 0.0, 1.2392691373825073, -1.6788277626037598, 2.5079357624053955, 0.6179784536361694, -1.9275858402252197, 0.0, -0.35927754640579224, 1.5783480405807495, -1.1237523555755615, 0.7276345491409302, -3.285203218460083, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -2.1716740131378174, 0.0, 3.2388994693756104, -2.428903341293335, 0.0, 0.0, 0.0, 0.3700934052467346, 0.0, 0.0, 0.0, 0.0, 0.0, -0.7025994658470154, 0.0, 1.0686448812484741, 0.0, 0.0, 1.6186083555221558, 0.0, 0.0, -1.1119128465652466, 1.235650897026062, 0.0, 0.0, 0.0, 0.0, 0.8040748834609985, 0.0, -0.2568467855453491, -1.3439022302627563, -4.746125221252441, -3.5970988273620605, 0.0, 0.0, 0.0, -2.991183280944824, -1.197503924369812, 0.0, 0.18410234153270721, 0.0, 3.0742294788360596, 0.0, 0.0, 2.296956777572632, 0.0, 1.1002322435379028, 0.0, -1.600621223449707, -1.3904529809951782, 0.248942568898201, 0.20507661998271942, 0.0, 0.4091402292251587, 2.814934253692627, 0.0, 1.4183788299560547, 1.5149255990982056, 0.0, -2.227062225341797, 1.1816593408584595, 0.0, 0.0, 0.0, 5.388925075531006, -2.571160316467285, 0.0, 1.9894694089889526, -2.796610116958618, -1.1225892305374146, -4.078183650970459, -2.889969825744629, -3.912310838699341, 0.20314373075962067, 0.0, -0.6453333497047424, 0.0, 2.7932443618774414, -0.018809745088219643, -0.4492712914943695, -0.5568112730979919, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0980440378189087, 0.0, 0.6322199106216431, 0.0, 2.729404926300049, 0.0, -2.8540456295013428, -0.32219502329826355, 1.4347625970840454, 0.0, 1.634629726409912, 0.0, 1.1970760822296143], [0.0, 0.0, 0.0, -2.957832098007202, 0.0, -2.0462334156036377, 0.0, 0.0, 0.0, 0.0, -1.0380789041519165, -1.4944981336593628, 0.0, -0.7453619837760925, 3.1864349842071533, 0.4014682173728943, 0.19242937862873077, 0.0, -0.14032524824142456, 0.0, 0.0, 0.0, -2.3105008602142334, 0.0, 0.0, -1.8040980100631714, -1.2605963945388794, 0.0, 0.0, 0.0, 0.0, -0.1725911647081375, 0.17955422401428223, 0.0, 0.0, 0.0, 1.6928496360778809, -1.0267575979232788, 0.0, 0.0, -0.2807222902774811, 0.0, 0.0, 0.936592161655426, 1.5192642211914062, 0.0, 0.0, 0.6588612794876099, 0.0, 1.9185230731964111, 0.0, 0.5712643265724182, 0.0, 0.0, 1.9558292627334595, -0.8493819832801819, 0.0, 0.0, 0.0, 2.950147867202759, -1.2661473751068115, 0.0, 0.0, 0.6474433541297913, 0.0, -1.1135224103927612, 0.0, -1.2617480754852295, -3.12541127204895, 0.0, 0.0882359966635704, -1.8684983253479004, -1.3108452558517456, -3.691347599029541, 0.42022088170051575, 0.0, 0.0, 0.0, -1.682295799255371, -0.680124044418335, 0.0, 0.0, 0.0, 0.0, 0.0, -0.10856788605451584, 0.0, 0.0, -2.3165996074676514, 1.9811968803405762, 0.615717887878418, 0.0, 1.5364415645599365, 1.6589914560317993, -3.718130350112915, 1.0491870641708374, -1.0855072736740112, 0.4053631126880646, -1.3455735445022583, -1.6490875482559204, 1.9986679553985596, 0.07483059912919998, 0.0, 0.0, 0.2587643563747406, 0.06605393439531326, -4.643474578857422, 0.0, 0.0, 1.6040421724319458, 2.841090440750122, 0.0, 4.000787734985352, 0.3111429214477539, 1.105994701385498, -0.9410043358802795, 0.0, 0.0, 0.0, 0.0, 2.4674155712127686, 3.601957321166992, -2.8534791469573975, 0.0, 0.0, 3.7957897186279297, 5.10515832901001, 0.0, 0.0, 0.0, 2.4832565784454346, 0.0, 0.0, -1.2169002294540405, -5.175783634185791, -2.5764901638031006, 0.0, 0.0, -4.329394817352295, 0.0, 0.0, 0.0, 0.0, 2.2898478507995605, -5.754380702972412, 1.7647476196289062, -0.15549911558628082, 0.0, 0.0, -0.23653611540794373, 0.4870874285697937, 2.1331803798675537, 0.0, 0.3380822539329529, -2.633262872695923, 1.2313578128814697, 0.32754066586494446, 3.9568514823913574, 0.24559220671653748, 0.0, 0.0, 0.0, -0.12923234701156616, -1.131426453590393, 0.0, 3.8560471534729004, -0.6225311160087585, -1.7666175365447998, 2.551521062850952, 0.0, 5.956014156341553, 0.020514143630862236, 0.0, -2.3077471256256104, 0.0, 2.4921085834503174, 0.0, 0.0, 0.0, 1.7856193780899048, 0.0, 0.0, -0.23794357478618622, 0.0, 0.0, 0.3711796700954437, 0.0, 2.8265867233276367, 0.0, 2.039794445037842, 1.4473165273666382, 0.0, 0.0, 2.1969714164733887, 0.0, 1.7225884199142456, 7.463122844696045, -0.2872016131877899, 1.5954985618591309, -2.374063014984131, 1.2360727787017822, -0.8729563355445862, 0.0, 0.0, 0.0, 0.0, 0.0, -1.7331262826919556, 0.0, 0.0, 0.0, 0.1837715357542038, 0.0, 0.0, -2.263692617416382, 0.0, -3.7311882972717285, 0.0, 0.0, -0.22031016647815704, -1.1727813482284546, 0.0, 0.0, 0.0, 2.3958349227905273, 0.0, -0.1644824892282486, 1.1798487901687622, 0.0, 0.00927000306546688, 0.0, 0.4056895971298218, 1.813515067100525, 0.0, 0.0, 0.04279813542962074, 0.21726468205451965, 0.0, 3.3459415435791016, 0.27804458141326904, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.7751816511154175, -0.7405297756195068, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.27638766169548035, 0.0, -1.558663249015808, 0.4409277141094208, -3.0324513912200928, -1.5160850286483765, 0.0, 2.8227453231811523, -0.6532784104347229, 0.8885830640792847, -1.5932157039642334, -0.1316283792257309, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -1.404154896736145, 0.3205423653125763, -2.91083025932312, 0.0, -3.2825324535369873, 0.0, -0.7698564529418945, -3.5280160903930664, 0.0, 0.0, 0.0, 0.0, -0.6163076162338257, 0.0, -1.5216883420944214, 0.4104978144168854, -4.790594100952148, 0.0, 0.9747869968414307, -4.857271671295166, 2.545625686645508, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.9266909956932068, 0.0, -0.2900046110153198, -1.6647295951843262, 0.0, 0.07317458093166351, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.995292901992798, -1.622040033340454, 0.0, 0.9330587387084961, 0.0, -2.657989501953125, 4.909037113189697, 3.4839823246002197, 0.09366392344236374, 0.0, 2.8256924152374268, 0.0, -0.7222848534584045, 0.0, 0.0], [0.0, 0.0, 1.6946402788162231, 0.0, 0.0, 2.6856181621551514, 0.6403626799583435, 0.6317616105079651, 0.37177911400794983, 0.0, 0.0, -3.834688186645508, 0.0, -6.116621971130371, -4.042506694793701, 3.3881120681762695, 2.275383472442627, 0.0, -0.005283976439386606, 0.0, 0.0, -1.4341635704040527, -1.656044602394104, 0.0, 0.42709022760391235, 0.0, -0.6627333164215088, 1.6976079940795898, -1.9485875368118286, -0.013664470054209232, 0.0, 0.0, 0.0, -0.5106698870658875, 0.0, 0.0, 0.0, 1.362159252166748, 0.0, 0.0, -2.021530866622925, 0.0, 0.0, -0.29358577728271484, 2.7324156761169434, 0.0, 0.0, 2.624990940093994, 1.688553810119629, 1.9423110485076904, 3.606006622314453, 1.091483235359192, 0.3118720054626465, 0.0, 0.0, 0.0, 0.0, 0.0, -0.9348297715187073, 1.2558748722076416, 0.7888504862785339, 0.3636329174041748, 0.0, 0.0, -3.201914072036743, -0.9682202935218811, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1199917271733284, 0.0, 0.0, 0.0, 0.0, -2.848540782928467, 0.0, -2.9431145191192627, 3.346837043762207, -0.6899620890617371, -0.2440985143184662, 0.7015016674995422, 0.0, 0.0, -1.1596068143844604, 0.0, 3.681962728500366, -1.348326563835144, -0.8679835796356201, 0.0, 0.0, 0.13407684862613678, 1.614679217338562, -2.057187795639038, 0.0, 0.0, 0.0, -2.42706561088562, 0.0, 0.0, 0.0, 0.2940060496330261, 0.0, -1.5553768873214722, -0.37547942996025085, 2.074422836303711, 0.5592607259750366, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -3.07934308052063, 0.0, 0.0, 1.7278296947479248, 0.0, 0.0, 0.0, 0.0, -1.303959846496582, 0.0, 1.5006091594696045, -3.0378315448760986, 0.6680536270141602, 1.429589867591858, 0.3095342516899109, 0.3388754725456238, 2.53705096244812, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1623055338859558, 0.7131978273391724, 0.8418244123458862, 2.285766363143921, 0.0, 1.8283149003982544, 3.9499154090881348, 1.7032926082611084, 0.0, -2.8565263748168945, 0.0, -0.7478752732276917, 0.592355489730835, -0.3399789333343506, 0.0, 2.414802074432373, 0.0, 0.0, -0.5962752103805542, -2.619481325149536, 0.0, -0.2815139591693878, 0.0, 0.0, 0.0, 2.218949317932129, 0.0, -2.3643598556518555, 2.5070407390594482, -0.6027936339378357, 0.0, 0.0, 0.20466040074825287, 0.0, -0.6275964379310608, 0.0, 0.0, 0.5727657079696655, 0.0, -3.556628942489624, -1.1699955463409424, -0.43216240406036377, -1.5855792760849, -2.019111156463623, -1.9003775119781494, 0.0, -0.26692143082618713, 0.0, -1.1332718133926392, 0.6734985113143921, -0.28908634185791016, -2.7631866931915283, -0.7589636445045471, 0.0, 0.0, -3.3027384281158447, 1.3170298337936401, 0.0, 0.35195475816726685, 0.0, 0.0, 0.0, -0.28710660338401794, 1.388595461845398, -1.1503349542617798, 0.0, -1.4566811323165894, 0.0, 0.0, -1.0849636793136597, -0.7228913903236389, -0.43478721380233765, 0.0, 0.0, 0.0, 0.3453059494495392, 0.0, -0.04170301556587219, -2.092212438583374, -1.1823406219482422, 0.0, 2.0067460536956787, 0.0, 0.0, 0.0, -0.6702736616134644, 0.0, 0.0, 0.37539878487586975, 0.0, 0.0, 0.0, 0.0, 0.0, -0.1654345691204071, 0.0, -0.018788473680615425, 1.2670516967773438, 0.0, -0.60126793384552, 0.0, 0.0, -0.2450762540102005, -1.9161896705627441, 0.0, 0.6982609629631042, 0.0, 0.0, 0.0, 0.2592066824436188, -0.7450546026229858, 0.0, -0.7913562059402466, -2.147273540496826, 0.0, 0.22922703623771667, 0.0, 0.0, 0.0, 1.2875455617904663, 0.0, 0.0, 1.479619026184082, 0.19513699412345886, -3.632373332977295, -1.458849549293518, 0.0, 0.0, 3.0625863075256348, 0.0, 4.0203728675842285, -0.48138120770454407, -1.4320319890975952, 0.0, 1.8962887525558472, 0.0, 0.0, 0.0, 0.0, -3.356109619140625, 0.0, 0.9220823645591736, 0.0, 0.0, 0.0, 0.0, 4.0572896003723145, 1.2288120985031128, 1.7310271263122559, 0.0, 1.9198864698410034, 0.0, -0.9063127636909485, 0.482198566198349, 0.30822479724884033, 0.0, 0.5578871965408325, 0.0, 0.5073034167289734, -0.9885820150375366, 0.0, 1.330124020576477, -1.3036680221557617, 0.0, 0.0, 0.0, 0.0, -1.9394383430480957, 0.0, 0.0, 5.012481212615967, -1.542784571647644, -1.5210628509521484, 1.3229856491088867, 0.0, 0.0, 0.0, -1.9384675025939941, 0.0, -3.1966400146484375, 0.0, -0.3483123779296875, 0.0, -2.244237184524536, 0.0, 0.0, 0.20889537036418915, -0.36931687593460083, 0.0, 1.1832847595214844, -1.4695624113082886], [0.6734529733657837, -1.9080055952072144, 0.0, -0.3419632911682129, 0.0, 0.8834132552146912, 0.0, 0.21293462812900543, 0.20807646214962006, 0.0, -0.2534739375114441, -4.440402507781982, 0.0, -0.7320156097412109, -0.5565549731254578, 2.8345022201538086, 0.0, 2.074124813079834, -0.7223010063171387, 0.0, 2.1912999153137207, 0.0, 0.0, 0.0, 0.0, -1.849541425704956, 0.0, 0.7271451950073242, -0.6101824045181274, -1.404534101486206, 0.6326953172683716, -1.1991037130355835, 1.9991786479949951, 3.7075603008270264, 0.8334941267967224, 0.0, 0.0, 3.921837091445923, 0.8404573798179626, 0.0, 0.0, 0.0, -0.9172224998474121, -0.5156241059303284, -0.756714940071106, 0.0, 4.6915602684021, 0.0, 0.0, 0.02834930829703808, 0.0, 0.0, 0.0, 0.0, 0.0, 0.6895018815994263, -0.7062206268310547, 1.1300915479660034, 0.0, 0.0, 0.0, 0.0, 0.0, -2.138801097869873, 1.1867235898971558, 4.555462837219238, -1.4225995540618896, -1.3369876146316528, 0.9485313892364502, 0.5453716516494751, 0.0, 0.0, 0.0, 1.2086100578308105, 0.0, 3.476475238800049, 1.8058497905731201, 0.0, 0.0, 0.0, 0.0, 4.275587558746338, 0.0, 0.0, 0.0, 0.0, 0.7911025285720825, -0.37519341707229614, -1.334099531173706, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -2.334778308868408, 1.5873210430145264, 1.027360200881958, 0.0, -3.5069563388824463, -0.42937514185905457, 0.0, -2.51603102684021, 0.0, 0.0, 0.0, -0.7688828110694885, 3.187155246734619, 2.216675043106079, 0.8931077122688293, 0.0, 1.55514395236969, 0.0, 0.13837884366512299, 0.0, 0.0, 0.0, 0.174440398812294, 0.0, 0.0, 0.0, 0.6728867888450623, 0.20782510936260223, 0.13384579122066498, -0.26605579257011414, 0.0, 0.0, 0.0, -3.3662607669830322, 0.0, 0.9030243754386902, 0.0, -0.8609316349029541, 2.646873712539673, 0.0, 0.0, 0.0, 0.0, 0.20360808074474335, 0.0, 4.098295211791992, -0.008156125433743, 0.0, -1.3727874755859375, -0.12344379723072052, 0.0, 0.0, 0.0, 0.0, 0.0, 2.7572085857391357, 0.0, 1.4944496154785156, 0.0, 0.0, 0.0, 2.7293777465820312, 0.0, 0.0, -0.4061228632926941, 2.2963435649871826, 0.0, 0.0, 0.0, -0.6259336471557617, 0.0, 0.5598344206809998, -1.3851178884506226, 1.8513802289962769, 0.6773930788040161, 0.0, 0.0, 0.0, 2.875795841217041, -0.123713418841362, 0.0, -0.8517659902572632, 0.0, -3.8445663452148438, -4.004470348358154, 0.0, 0.12935590744018555, 1.4033148288726807, 0.9443626999855042, -0.2580653429031372, 0.0, 0.0, 0.0, -0.0471859835088253, -0.15497469902038574, -0.9446114897727966, 0.0, 0.0, -2.5919477939605713, 0.4886161684989929, -0.3557316064834595, 0.0, 0.0, 0.0, -0.15224124491214752, 0.0, -1.6202104091644287, 0.0, -2.9970059394836426, 0.0, 0.0, 1.4260755777359009, 1.6464104652404785, 0.0, -0.48469308018684387, -0.6985577940940857, 2.0458498001098633, 0.25523290038108826, 0.0, 0.0, 0.0, 4.081955909729004, 0.0, 2.5031068325042725, 0.0, 0.0, 0.0, 0.5586416721343994, 1.7392388582229614, -2.4049930572509766, 0.48589077591896057, 0.0, -0.6636050939559937, 2.8395910263061523, -0.2698878049850464, 0.0, 0.0, 0.0, 2.0314505100250244, 0.0, -3.1142780780792236, 0.0, 0.0, 0.0, -1.597910761833191, 2.85512638092041, 0.0, 0.0, 0.0, 0.0, 0.0, 1.529112458229065, 0.0, -1.201748013496399, 4.408575534820557, 0.1652761548757553, 0.0, 0.06591492146253586, -2.504221200942993, 0.0, 1.9365555047988892, -1.1647998094558716, 2.002800703048706, -0.5814335942268372, 0.5764085650444031, -1.4831782579421997, -1.6016268730163574, 0.0, 0.4393410384654999, 0.0, 0.0, -2.257807493209839, -2.336066484451294, -1.2644883394241333, 0.0, 0.0, 0.0, 0.587382972240448, 0.0, 0.21156972646713257, 4.043642520904541, 0.0, 2.6624672412872314, 0.0, 2.347426176071167, 0.0, 0.48612499237060547, 0.0, -3.993509531021118, -2.7300117015838623, 0.0, 0.0, 0.0, 0.389252245426178, -5.807619571685791, 0.4975162744522095, 0.0, -1.0111581087112427, 0.0, 0.0, -1.9248111248016357, 2.4779860973358154, 0.8867815732955933, -1.3138271570205688, -0.8516109585762024, 0.0, 0.7537301778793335, 0.38154175877571106, -2.4618043899536133, -0.8787052631378174, -2.2810771465301514, 0.0, 0.0, 0.0, 0.0, 0.0, 1.4993820190429688, 4.170530319213867, 0.0, 1.8435726165771484, 0.0, 0.0, 0.0, 2.964111089706421, 0.0, -1.9040663242340088, -0.14943848550319672, -0.8863824009895325, 2.874516725540161, 0.0, 0.6643305420875549, 0.3405963182449341, 0.0, 0.0], [-1.8565235137939453, 0.0, 2.352069139480591, 0.0, 0.0, 0.0, 0.0, -1.9881231784820557, 0.0, -1.7142316102981567, 0.0, -2.462860345840454, 0.0, 0.0, 0.8299319744110107, 0.0, -2.821115732192993, 1.0982189178466797, 0.0, 0.0, -0.14813187718391418, 0.0, 0.0, -0.711398720741272, 2.27791690826416, 0.0, -1.752923846244812, 0.0, -0.31066617369651794, 1.4156029224395752, 0.0, 0.0, 0.0, 0.5589665174484253, 1.0605183839797974, -0.06808018684387207, 0.0, 0.8965147137641907, 0.0, 0.0, 0.0, -0.45257726311683655, 0.2702350318431854, 1.9773955345153809, 0.0, 0.0, 0.0, 0.0, 0.0, -0.5844877362251282, -1.2663681507110596, -1.0701826810836792, 0.0, 0.0, -3.722672462463379, 0.0, 0.0, 0.0, 1.9572376012802124, 5.643305778503418, 0.0, -1.6112465858459473, 0.0, 1.3144675493240356, 0.0, 0.0, -0.24280975759029388, 0.0, -2.846458673477173, 0.0, 0.0, 0.0, 0.0, -0.5593000650405884, 2.720790386199951, 1.174078106880188, 0.0, 0.0, 0.0, 3.9202330112457275, 3.255023717880249, -0.1973503977060318, 0.0, 0.3287176787853241, -1.8309704065322876, 0.0, 0.8122608065605164, 0.0, 0.0, 2.7816710472106934, 0.0, 0.0, 0.0, 0.7993678450584412, -0.2584025263786316, -0.4862240254878998, 3.6782565116882324, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.5837004780769348, 0.0, 0.0, 3.565852403640747, 1.3321975469589233, 0.0, 0.0, 0.0, 1.0727661848068237, 0.0, 1.8262372016906738, 0.0, -0.9403712153434753, 3.4160964488983154, 0.0, 0.0, 0.0, -0.290554016828537, 0.0, 0.0, 0.0, -2.6289520263671875, 0.1276417076587677, 0.0, -0.028274469077587128, -0.2993318438529968, 0.5610002279281616, 0.0, 0.0, 0.0, 0.0, -1.7620562314987183, -4.344112873077393, 0.0, 1.6090739965438843, 4.689886569976807, 0.0, 0.0, 0.0, -0.7552108764648438, 0.9789062142372131, 0.0, 0.4780441224575043, 0.0, -2.0913405418395996, 2.417515516281128, 0.0, 0.0, 1.6147295236587524, 0.0, -0.2626969516277313, 0.0, 0.0, 0.0, 0.4586419463157654, 0.0, 2.622195243835449, 0.4310700595378876, 0.0, 0.0, -0.8519706726074219, 0.20037688314914703, -1.4433884620666504, 0.0, 0.0, 1.0078588724136353, 2.7311134338378906, -1.748105764389038, 0.0, 0.0, 0.0, 2.266462802886963, 0.0, 0.0, 0.0, -1.7895901203155518, 0.0, -2.84828519821167, -2.3073410987854004, 0.0, 0.054213639348745346, 0.0, -3.8682425022125244, 0.0, 0.0, 0.0, 0.0, 0.0, 0.46832868456840515, 2.0797579288482666, 0.0, 0.0, 0.0, 0.0, 1.9038995504379272, 0.0, 0.0, 0.6360267400741577, 0.0, 0.0, -0.9250589609146118, 0.0, 1.020799994468689, -2.0223147869110107, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.4032444953918457, 0.0, 2.8885650634765625, 0.0, -3.283033847808838, 2.8873536586761475, 0.0, 0.0, -0.247262641787529, -0.416916161775589, -0.4015251696109772, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0364434719085693, 1.5115935802459717, 0.07752716541290283, 0.5010385513305664, 0.250488817691803, 0.0, 1.44754159450531, 0.0, 0.0, 0.0, 0.0, -2.0628340244293213, 0.842326283454895, 0.0, 0.0, -0.024385590106248856, 0.0, -1.3951374292373657, -1.3467090129852295, 2.0919010639190674, -2.1390793323516846, -1.8230904340744019, 0.0, 1.2933895587921143, 1.3025075197219849, -1.0019187927246094, -0.5667263865470886, 0.0, 0.0, -1.6756857633590698, -2.132868528366089, 0.0, 0.0, 5.754544258117676, 1.2890814542770386, -1.4532709121704102, 0.0, 0.0, 0.0, 0.0, 0.9475583434104919, 0.0, 0.0, 4.403178691864014, -0.8343247175216675, -0.7162457704544067, 1.457351565361023, 2.5278542041778564, 1.6734951734542847, 2.4691150188446045, 0.0, -0.41619905829429626, 0.0, 0.0, 3.376180410385132, 3.1498477458953857, -3.594994068145752, 1.592512607574463, 0.0, 0.0, 0.8209792971611023, 0.0, -0.6569370627403259, 0.0, 0.0, 0.0, 0.0, -0.18772366642951965, 0.2529924809932709, 0.0, 0.0, 0.0, -2.2946858406066895, 0.0, 0.28718301653862, 0.0, 0.0, -0.9853413105010986, 0.0, -3.592082977294922, 0.0, -0.7417434453964233, -0.11722595989704132, -0.6396371126174927, -0.5324878692626953, -2.3607616424560547, -1.1377936601638794, 0.0, 0.0, 0.0, -0.6928487420082092, 2.955287218093872, 0.0, 0.0, 0.0, 0.0, 0.0, -2.3422152996063232, 2.342348575592041, 1.453454852104187, 0.0], [1.3220537900924683, 0.919032633304596, 0.41689053177833557, 0.0, 0.0, -1.569533348083496, -2.0819146633148193, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.43407973647117615, 0.0, 0.8817319869995117, 4.21565055847168, -1.1811784505844116, 0.0, -0.7521303296089172, -5.593851089477539, 0.0, 0.0, 0.0, -1.6604212522506714, 0.0, 0.0, 0.0, -1.7714134454727173, -1.4980244636535645, -0.5868402123451233, 1.6431714296340942, 2.844273567199707, 0.0, 0.0, 0.1218298003077507, 0.0, 0.0, 0.0, 0.0, -2.5863447189331055, -0.9141381978988647, -1.6781452894210815, 3.3747291564941406, 0.0, 0.0, 1.044045329093933, -1.0387709140777588, 0.0, 2.8030641078948975, 0.0, -0.19804812967777252, 0.0, -0.1400398164987564, 2.674116373062134, 0.0, 0.0, 3.286515235900879, 0.0, 3.5054402351379395, 0.0, 0.0, 2.7991483211517334, 1.3509923219680786, 0.7392873167991638, -0.953903079032898, -2.8828179836273193, 0.0, 0.0, -1.4091578722000122, 1.7945139408111572, 0.0, -2.603886127471924, -1.0504463911056519, 0.0, 0.0, 1.100966453552246, 2.0096709728240967, -0.14256049692630768, 0.8147122859954834, -0.12980656325817108, 0.0, 0.8910359144210815, 0.7946394681930542, 0.0, 1.2823880910873413, -1.2845182418823242, 1.2786314487457275, 0.0, 0.0, -3.2761189937591553, 0.0, -1.206549882888794, 0.33733949065208435, -0.15679575502872467, -3.044222593307495, -3.584789991378784, 0.0, 0.0, 2.9744648933410645, -0.6879757046699524, 1.010570764541626, 0.0, -2.7528140544891357, 0.0, 1.4403917789459229, -0.5545571446418762, -0.6321425437927246, 1.917513132095337, 0.0, 0.0, 0.0, 0.0, -0.6956766247749329, 0.0, 0.0, -0.8208886981010437, 0.0, 3.6771175861358643, 2.571803331375122, 0.0, -0.016260797157883644, 0.4996618628501892, -2.7455251216888428, 0.0, 0.0, -0.28543779253959656, 0.9650159478187561, 0.0, 0.0, -2.987204074859619, 0.0, 0.0, -2.5192136764526367, -0.692271888256073, 0.12927106022834778, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.8927361965179443, 0.0, 1.1221246719360352, 0.0, 0.0, 0.0, 0.0, -1.568769931793213, 0.09645244479179382, 0.0, 0.0, -0.4629684090614319, 0.0, -2.644571542739868, -0.8027202486991882, 0.0, 2.0267245769500732, -0.959148108959198, 2.1334643363952637, 0.0, 0.0, -2.1023788452148438, 0.0, 0.0, 0.0, 0.0, 2.805906295776367, -1.2811037302017212, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.7068734765052795, -1.1166071891784668, 0.0, -1.9717978239059448, 0.0, 0.0, -0.5218767523765564, 1.57375967502594, 2.1382226943969727, -1.0848246812820435, -1.2224222421646118, 1.9589102268218994, -5.609011650085449, 2.6427693367004395, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.9387139081954956, 0.0, 0.0, 0.0, 0.0, -5.587138652801514, -0.37212592363357544, -1.3380200862884521, -0.15524999797344208, -0.9546727538108826, 0.0, 0.5695679783821106, 0.0, -1.5971815586090088, 3.318650722503662, -0.4606947898864746, 0.0, 0.0, 0.0, -2.9569647312164307, 1.7943631410598755, 0.6872258186340332, -1.1037445068359375, 0.0, 0.9578163027763367, 0.0, 1.459794282913208, 0.0, 0.0, 0.0, 0.0, 1.6616262197494507, 0.1102304458618164, 0.0, -1.9888414144515991, -0.6334438323974609, -3.561363458633423, 2.7317564487457275, 1.8986366987228394, -2.1437418460845947, -0.22070661187171936, 3.328434467315674, 0.7924187183380127, -1.9913007020950317, -4.233669281005859, 0.0, 2.3427436351776123, 0.0, -2.574277639389038, 1.393027901649475, -0.7141253352165222, 0.0, -1.0273284912109375, -0.9149823784828186, 3.29421067237854, 0.0, -0.7935258746147156, 0.0, 2.9703609943389893, -1.9357959032058716, 0.0, 2.178651809692383, 0.0, 0.0, 0.0, 1.348227620124817, -2.275449514389038, 0.0, 0.0, 0.8379835486412048, 0.0, 0.0, -0.20025008916854858, 0.0, -2.1149749755859375, 0.0, 0.0, 0.0, 0.0, 0.0, -3.0053062438964844, -0.6105784773826599, 0.0, 2.549206256866455, -1.8856407403945923, -0.890755295753479, 0.0, 1.579520583152771, 0.0, 0.0335240475833416, 0.0, 0.0, 0.6432019472122192, 0.0, -1.7626506090164185, 0.0, 0.0, 0.0, -1.2332597970962524, 0.0, -2.8754167556762695, 0.0, 0.0, -0.46902996301651, 0.532332181930542, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1527545303106308, 3.9130961894989014, 0.0, 0.5520901679992676, 4.460983753204346, 0.0, -1.6010173559188843, 0.0, 0.0, 1.025308609008789, 0.0, 1.2665200233459473, -0.8112320303916931, 0.0, 0.0, 2.342689037322998, -0.7929359078407288, 0.0, 2.1588213443756104, 0.0, 0.0, 0.0, -1.532109022140503]], "dtype": "float32"}}, {"a": [{"__nd__": [2.2121646404266357, -5.61783504486084, -5.613106727600098, -3.576307773590088, -6.677898406982422, -0.7687696814537048, -0.7498173713684082, -9.276634216308594, 13.849328994750977, 9.711793899536133, -4.431873321533203, 21.315513610839844, 24.124500274658203, 12.045334815979004, 11.593793869018555, -6.1464762687683105, -3.459005355834961, -16.09748649597168, -4.483409404754639, 9.449213027954102, -4.384552001953125, -17.053611755371094, 15.725517272949219, 19.372995376586914, 14.84926986694336, -15.543747901916504, 8.002057075500488, -8.549370765686035, -1.6470836400985718, 6.736114025115967, 15.062342643737793, -6.502536773681641, -3.729917526245117, -12.885120391845703, -0.16998517513275146, -9.998784065246582, -5.207630157470703, 7.3705339431762695, -0.3645455539226532, 0.965191125869751, 2.575528621673584, 7.807674407958984, 13.64545726776123, -11.89699935913086, -16.351634979248047, 4.939687728881836, 6.737595081329346, 13.723642349243164, 7.637546539306641, 17.13440704345703, 4.913666725158691, -6.58897590637207, 9.839157104492188, -9.16418743133545, -7.101773738861084, 3.877335786819458, -2.7784390449523926, 9.154777526855469, -3.963181734085083, 18.127233505249023, -10.465936660766602, 6.876965522766113, -4.849902153015137, 13.998412132263184, -5.95785665512085, 7.500109672546387, -31.9476318359375, -7.562859535217285, 16.440229415893555, 1.7951712608337402, 0.7237798571586609, 8.787222862243652, 24.086448669433594, -11.138269424438477, -0.8996239900588989, -4.6962456703186035, 6.988766670227051, -7.165192604064941, -15.770001411437988, -15.386096000671387, 6.577511787414551, 2.0536913871765137, 8.576554298400879, -16.141672134399414, -9.113469123840332, -5.524405479431152, 8.493219375610352, 4.43221378326416, -1.7584384679794312, -10.143233299255371, 1.8754111528396606, -4.811474323272705, 8.903257369995117, 17.524791717529297, 1.1806957721710205, -11.277342796325684, 23.92683982849121, 6.38670015335083, 5.40499210357666, 1.1709998846054077, 1.755434513092041, 0.9637994766235352, 2.6646008491516113, -15.029011726379395, -7.128811359405518, -0.856289803981781, 1.539600133895874, 2.0324649810791016, 4.386411190032959, -18.25263214111328, -9.652570724487305, -6.649162292480469, 1.4020893573760986, -1.863784909248352, 19.967880249023438, -3.7514877319335938, -2.374849319458008, -15.501016616821289, 18.39627456665039, 0.8958268761634827, -1.078182578086853, -6.406272888183594, 0.9485915899276733, -3.442650318145752, 12.290407180786133, -8.02987289428711, -4.270849704742432, -5.2542524337768555, 6.011941432952881, 3.424783945083618, 9.989286422729492, 5.548211097717285, 5.847760200500488, 0.6682098507881165, 9.215849876403809, 4.986637592315674, 3.4666147232055664, -15.984000205993652, -4.570258140563965, -6.7983717918396, -18.141340255737305, 13.253908157348633, 14.676054000854492, 11.02247428894043, 5.855725288391113, 15.040546417236328, -16.75930404663086, 12.876988410949707, -20.278217315673828, 1.6672346591949463, 1.2477067708969116, 5.207614421844482, 6.616611480712891, 7.193240642547607, -6.997896194458008, -6.765504837036133, -8.338171005249023, 1.3123352527618408, -3.4311630725860596, -15.966794967651367, 9.639090538024902, -11.453577041625977, -4.9327392578125, -7.468686580657959, -7.51369571685791, 5.793901443481445, -6.985334396362305, -14.447696685791016, -6.680954933166504, -10.391738891601562, -15.472881317138672, 9.108665466308594, 11.518431663513184, -14.524457931518555, 15.663437843322754, 18.31201934814453, -13.970926284790039, 4.144055366516113, -2.860534191131592, 8.263875007629395, -29.538803100585938, 6.184932708740234, -3.7554824352264404, 4.71701717376709, 8.81116771697998, -23.76124382019043, -0.569839358329773, -0.09758759289979935, 5.971779823303223, 11.342339515686035, 7.530909061431885, -7.797980785369873, 7.048905372619629, -7.647850513458252, 7.664387226104736, -1.7966587543487549, -1.8520007133483887, 1.293231725692749, -16.026756286621094, -3.256190061569214, -12.031436920166016, 9.245637893676758, 12.687865257263184, -14.544005393981934, -22.548145294189453, 2.345060110092163, -5.790863037109375, -9.400187492370605, 22.861101150512695, -4.227080821990967, -10.3682861328125, -2.8444271087646484, 2.816133975982666, 15.722278594970703, 3.0216619968414307, -12.60590934753418, -4.977386474609375, 18.389177322387695, -17.492385864257812, -0.5326255559921265, 11.751958847045898, 2.6453018188476562, -2.1864445209503174, -8.90854263305664, 1.5708130598068237, 5.220938682556152, 2.2065024375915527, 10.254951477050781, 2.0337820053100586, 12.955282211303711, 1.4253257513046265, 9.14061164855957, -3.2345261573791504, 17.500762939453125, 12.626352310180664, 6.977984428405762, -9.86756706237793, 1.9573814868927002, -29.404224395751953, -3.5798749923706055, 1.7086130380630493, -8.624042510986328, 1.6723747253417969, 6.749271869659424, 18.232044219970703, 0.2631077468395233, -15.613134384155273, -23.441028594970703, -1.5405339002609253, 9.138728141784668, 11.374052047729492, 1.1172823905944824, -6.495807647705078, -6.927495002746582, -1.27082097530365, 1.0574160814285278, 6.141836166381836, 1.3573172092437744, 20.286876678466797, 1.7361805438995361, 5.121679782867432, 10.653732299804688, 9.683578491210938, -6.292755603790283, -16.06220054626465, -12.859244346618652, 3.911088705062866, 6.1559672355651855, 3.6625311374664307, 8.153712272644043, 13.231335639953613, -7.4671311378479, 25.89240837097168, 26.949214935302734, 7.922901153564453, -1.4657868146896362, 2.844080686569214, -13.802837371826172, 0.5557444095611572, -7.875373840332031, -0.6569839715957642, -2.904240846633911, 21.011884689331055, -8.312980651855469, 3.424703598022461, -3.7476084232330322, 16.209943771362305, 10.977324485778809, -14.983114242553711, -9.542587280273438, 3.11742901802063, -5.88469934463501, 4.830966472625732, 1.1388503313064575, -10.498766899108887, -2.6755452156066895, 5.112425804138184, -0.39733749628067017, 1.7264677286148071, 4.0781049728393555, -0.8833014965057373, 13.859721183776855, -12.352266311645508, 5.062469959259033, -4.865581035614014, 4.394860744476318, 8.812254905700684, 6.743136405944824, 7.1210784912109375, 7.9852800369262695, 5.6930060386657715, -1.9642192125320435, 7.712723731994629, 10.11818790435791, 9.050741195678711, 3.1720919609069824, 3.173696517944336, -19.59058380126953, -7.865538120269775, -1.6779906749725342, 10.365452766418457, 12.323803901672363, 1.7361701726913452, -10.980147361755371, 5.173323631286621, 10.190629959106445, 7.952131271362305, -6.720214366912842, -10.542412757873535, -8.762338638305664, -10.533578872680664, 10.301188468933105, 3.390207052230835, 21.93131446838379, 16.805421829223633, 10.396211624145508, -6.603687286376953, 6.896167755126953, 4.772156715393066, -13.858195304870605, -12.504731178283691, -7.166083335876465, 12.884729385375977, -3.7086634635925293, -13.687017440795898, 1.0736982822418213, 6.2485127449035645, -11.979856491088867, 19.398509979248047, -7.331616401672363, 10.810818672180176, 7.048685550689697, -7.205103874206543, -1.022696614265442, 0.3224959373474121, -3.447049617767334, 7.717840671539307, 0.7062730193138123, -17.31240463256836, 1.0508065223693848, -11.43588924407959, -2.239546298980713, 2.4503917694091797, -1.0227534770965576, 10.354759216308594, -0.6736153960227966, -5.143167972564697, 16.705354690551758, 7.494338035583496, -1.2571276426315308, 0.18221870064735413, -3.955010414123535, 0.9125795364379883, -9.599751472473145, 17.82895851135254, 12.489776611328125, -22.50868034362793, -9.596999168395996, -0.4086044728755951, -6.454419136047363, -21.385156631469727, 7.045120716094971, -11.927255630493164, 6.926607131958008, 4.632445335388184, 15.36449909210205, -8.939252853393555, 18.404476165771484, -12.318965911865234, 15.152729988098145, 10.775054931640625, 4.454492568969727, -10.888444900512695, 16.32090187072754, 8.195083618164062, -16.687179565429688, -26.098535537719727, -6.617938041687012, -8.611835479736328, 7.232623100280762, 1.4572103023529053, 3.2750425338745117, -5.087993621826172, 9.652084350585938, -5.690624713897705, -15.160537719726562, -24.167041778564453, 5.338568687438965, -6.071176052093506, -17.231151580810547, -8.805272102355957, -3.883286952972412, 7.059786319732666, -21.403324127197266, -4.034608840942383, 7.570784568786621, 4.907650470733643, 8.411964416503906, 20.137027740478516, -8.454583168029785, -5.864386081695557, 11.090816497802734, 16.612815856933594, 3.758450508117676, -2.878715991973877, -11.306264877319336, 6.670629501342773, 3.9518797397613525, -10.37342643737793, -4.827656269073486, -9.444583892822266, 6.3578691482543945, -1.0015031099319458, 13.024002075195312, 14.127179145812988, 3.83890962600708, 12.55654239654541, 11.124869346618652, 15.714682579040527, -4.150358200073242, 7.018860816955566, 19.343826293945312, 7.39057731628418, -3.4310290813446045, 3.2830939292907715, 23.936368942260742, -3.0756754875183105, -5.4754486083984375, 10.030683517456055, -2.8803393840789795, -14.931410789489746, -15.056087493896484, 1.0545353889465332, -5.4253082275390625, 15.746673583984375, -5.132967948913574, -13.10616397857666, -6.902761459350586, -11.563756942749023, 12.794126510620117, -6.972624778747559, 16.17840003967285, 6.688474655151367, 21.549070358276367, -0.07260817289352417, 9.543844223022461, -20.188636779785156, -3.7623848915100098, -6.55595064163208, 12.682214736938477, -0.3068661093711853, 7.8272385597229, -6.256274223327637, -12.058167457580566, 0.17738285660743713, 14.272939682006836, 0.5251611471176147, 16.99496841430664, -2.2886011600494385, -2.5300803184509277, -2.5301716327667236, -1.5626628398895264, -5.937993049621582, -1.372635841369629, -7.169573783874512, -9.413731575012207, 16.49496841430664, 9.968984603881836, -1.9933563470840454, -5.074019432067871, -14.823898315429688, 7.030163764953613, -5.773409843444824, 20.625764846801758, -16.58045196533203, 4.538586616516113, 13.408435821533203, -4.336182594299316, 21.71350860595703, 11.325307846069336, 11.183666229248047, -7.60244607925415, 11.578311920166016, 11.399378776550293, 2.542656898498535, 19.444034576416016, 4.643674373626709, -21.76856803894043, 4.699362754821777, 19.025793075561523, -4.899548530578613, 8.83472728729248, -5.226644992828369, 1.2231082916259766, -26.298355102539062, -17.161182403564453, -1.827682375907898, -4.682173252105713, -17.798173904418945, 8.320234298706055, 4.708411693572998, -7.833366870880127, -19.936019897460938, 6.933661460876465, -3.9767532348632812, 8.431680679321289, 12.501337051391602, -2.5927624702453613, -1.4502745866775513, 8.58669376373291, -1.8249977827072144, -19.379154205322266, 6.856088161468506, 3.837844133377075, -5.266956806182861, 5.82409143447876, -3.408438205718994, -3.7375845909118652, 8.483831405639648, -13.691156387329102, -16.19403076171875, -1.302266001701355, -13.210888862609863, 9.007426261901855, -7.374932765960693, -17.465564727783203, -3.9401679039001465, -22.5650691986084, 14.078624725341797, -0.1488192081451416, 7.087767601013184, 5.431956768035889, 2.1947293281555176, -10.46723461151123, 8.152441024780273, -3.2314536571502686, -7.639811038970947, -20.60548973083496, 13.19373607635498, -2.256150007247925, -4.05363130569458, -19.26985740661621, 9.660433769226074, -0.9848111867904663, 10.420696258544922, -0.2242257446050644, -8.502372741699219, 5.933926582336426, -15.819890975952148, -15.64253044128418, -0.5472948551177979, 7.714531898498535, 11.48058032989502, -0.7887315154075623, -1.0494219064712524, 2.223360300064087, -1.9561129808425903, -4.960622310638428, 5.213789463043213, -4.985093116760254, -2.0568273067474365, 12.29482650756836, 2.3874406814575195, -5.123441219329834, 7.216011047363281, -1.2941162586212158, 7.01084566116333, -8.598970413208008, -7.705662727355957, -14.345697402954102, 11.511816024780273, 14.342113494873047, 12.175788879394531, 3.71462082862854, 2.1273796558380127, -2.275902509689331, 6.017061233520508, -1.7780863046646118, -10.162911415100098, 10.17501449584961, 9.154112815856934, 5.395534515380859, -0.5946754813194275, -3.7883617877960205, 1.0113999843597412, 2.1727662086486816, -11.721296310424805, -4.537254810333252, -10.62067985534668, 11.249536514282227, -13.5880765914917, -12.14269733428955, -8.891359329223633, 1.6064255237579346, -9.352792739868164, 17.630537033081055, 7.341064929962158, -24.15346908569336, 2.2065181732177734, 3.4455313682556152, -12.781314849853516, -0.9805023670196533, 0.7720964550971985, -6.5235795974731445, -1.117549180984497, 3.281989336013794, -5.277952194213867, -0.7486957907676697, -12.197714805603027, 6.183075428009033, -6.52726411819458, 12.086000442504883, 20.590763092041016, 16.71672248840332, -4.799756050109863, 4.185732364654541, -2.7594313621520996, 4.482318878173828, -8.487950325012207, -10.348352432250977, -0.6826993823051453, -8.22960090637207, 12.156171798706055, 7.111154079437256, 3.202765941619873, -1.653692603111267, -1.8788352012634277, 9.58360767364502, 0.591102659702301, -0.02214912697672844, 18.653867721557617, 1.2981654405593872, -2.73903489112854, -2.4073197841644287, 12.695266723632812, -5.098788738250732, -4.9135026931762695, 10.896235466003418, 11.587004661560059, 14.065893173217773, -8.701129913330078, 4.322880744934082, -5.786894798278809, 4.110245227813721, -15.94638442993164, -2.0742461681365967, 10.780086517333984, 1.9839812517166138, -24.289350509643555, 9.963857650756836, 3.6087453365325928, -2.7874746322631836, -0.08078908920288086, -3.286266803741455, 11.541324615478516, -0.3761252164840698, -16.129253387451172, -8.608098030090332, 16.373001098632812, 11.302027702331543, 8.020593643188477, -7.947816848754883, 5.445477485656738, 10.373910903930664, 1.024329423904419, 7.380441665649414, 11.478327751159668, 10.489896774291992, 4.65373420715332, 4.771973609924316, -12.310794830322266, -2.849235773086548, 8.489053726196289, 17.263504028320312, 1.9535505771636963, 11.044422149658203, 0.2590884268283844, -9.579198837280273, 4.506572246551514, 4.764017581939697, -8.810718536376953, -8.530171394348145, 0.6422373652458191, -4.961213111877441, 5.222740173339844, -13.480903625488281, 8.37613296508789, -6.021727561950684, 3.722848415374756, -15.494621276855469, 9.575895309448242, 2.2028679847717285, 21.379127502441406, -16.50518798828125, -18.201385498046875, 7.4002790451049805, 9.161568641662598, -17.75370979309082, 5.5282487869262695, 9.574748039245605, 14.334451675415039, 1.5199190378189087, -15.32264232635498, -19.284975051879883, 21.256649017333984, 4.3061323165893555, 10.098690032958984, -3.030620574951172, -2.12288761138916, -1.8911700248718262, -9.209693908691406, -2.721658229827881, -10.996479988098145, -3.979031562805176, 11.395179748535156, 0.45149651169776917, -6.598057746887207, 17.58956527709961, 13.343682289123535, 0.656252384185791, -17.313739776611328, 13.245672225952148, 13.405990600585938, -5.687099456787109, 14.602221488952637, -8.123390197753906, -2.4502410888671875, -4.398299217224121, 2.0743520259857178, -1.1514904499053955, 14.651955604553223, -3.279953956604004, 4.7001729011535645, 0.6889113187789917, 2.7046871185302734, -8.071578025817871, -0.12210383266210556, 14.180670738220215, 8.54702091217041, -11.564669609069824, -3.840153217315674, -9.913183212280273, -7.037746429443359, -6.275838851928711, -12.1030912399292, 4.894367218017578, 9.340826034545898, 1.7728506326675415, -4.563915252685547, 1.8330453634262085, -7.444700241088867, 10.7120361328125, 23.756732940673828, -26.689929962158203, 4.107498645782471, 0.7244969606399536, 2.201730251312256, -2.555675745010376, -14.880372047424316, -0.5739563703536987, -3.8199896812438965, -1.2715517282485962, -0.2480490803718567, -4.814864158630371, 23.225200653076172, -12.32297134399414, -3.705026149749756, -12.635740280151367, -1.4237279891967773, 15.762989044189453, -0.25904181599617004, -8.18883228302002, -13.169293403625488, 5.63325834274292, -6.978125095367432, 6.480230808258057, -6.826286792755127, -5.1831955909729, -0.43873775005340576, -16.282798767089844, 12.210603713989258, -11.338825225830078, 1.6451574563980103, 4.656142234802246, 23.799259185791016, 4.907276630401611, 8.572904586791992, -4.9660844802856445, -1.279146432876587, 0.0457908920943737, 10.845126152038574, -9.8168363571167, 4.211542129516602, 28.84079933166504, -9.504347801208496, -18.328567504882812, -5.068831443786621, -1.1693679094314575, 16.87701988220215, 5.061549186706543, 11.66906452178955, -6.943897247314453, -0.18139499425888062, 3.1467854976654053, -8.265490531921387, 3.0813074111938477, -2.5875296592712402, 2.528614044189453, -13.784524917602539, -20.226055145263672, 13.994863510131836, 0.7382532954216003, -0.630703330039978, 26.228683471679688, 10.543431282043457, 4.8895745277404785, -6.190235137939453, 4.179736137390137, 2.8269171714782715, -2.0488884449005127, 14.840276718139648, 8.115483283996582, 6.273366928100586, -7.6078338623046875, -6.314958572387695, 11.755646705627441, 2.97507905960083, -4.896024227142334, 3.8649282455444336, 0.9787987470626831, -3.617568016052246, -1.0614428520202637, 8.328702926635742, 4.090604305267334, -7.867615699768066, -6.648653030395508, 3.680727958679199, -9.736554145812988, 2.006025791168213, -0.522758960723877, -12.308465957641602, -23.317569732666016, -7.757451057434082, 0.4749618470668793, 0.5359082221984863, 13.120138168334961, 0.6025733947753906, -0.1958175152540207, 4.222832679748535, -3.8745548725128174, 17.69670867919922, -5.307775497436523, -8.15351390838623, -4.605798721313477, 12.504639625549316, -5.263204097747803, -7.219466686248779, 6.245077133178711, 12.140961647033691, -3.83146333694458, 7.301392078399658, -6.252620697021484, 14.276591300964355, -10.433752059936523, -2.4552741050720215, -4.014811038970947, -9.018205642700195, -0.5505795478820801, 6.937038421630859, -8.062141418457031, 19.24662208557129, 2.6688859462738037, 3.5484204292297363, -7.54914665222168, -0.49421989917755127, 9.0462646484375, -9.879617691040039, -6.7766923904418945, -8.168486595153809, -12.969193458557129, 6.404237270355225, -7.937796115875244, 16.559452056884766, -12.441452980041504, -3.3520963191986084, 0.521907389163971, -9.861715316772461, -21.32931900024414, -6.458339214324951, -0.1868724822998047, -9.457172393798828, 2.7028183937072754, -18.547204971313477, 3.0112650394439697, 20.945951461791992, 9.667203903198242, 10.001424789428711, -3.5242252349853516, -4.2210307121276855, 11.303417205810547, -3.2921111583709717, 6.475245475769043, -9.748710632324219, 3.8801722526550293, 1.7873729467391968, -22.193294525146484, -1.1452873945236206, 5.694127082824707, 5.43045711517334, 3.732353448867798, 12.871883392333984, -0.6303038597106934, -7.624683856964111, 9.119120597839355, 21.271995544433594, -3.38456130027771, 15.200472831726074, -1.7433481216430664, -14.85286808013916, -8.93088150024414, -8.595791816711426, 11.696990966796875, 1.0793589353561401, -5.477937698364258, 1.312652349472046, 14.064759254455566, -0.7602862119674683, -7.197099208831787, 3.6122665405273438, -8.339315414428711, 5.071964263916016, 9.953065872192383, 1.887162685394287, -4.301834583282471, 1.2163161039352417, -14.178251266479492, 15.762205123901367, 23.70199966430664, 0.20885497331619263, -20.66374969482422, -6.760558605194092, -14.468236923217773, -18.786060333251953, -1.4055075645446777, -1.6383153200149536, -6.311988830566406, 11.094221115112305, 0.37746739387512207, -9.807244300842285, 4.177740573883057, -12.65442943572998, -3.528878927230835, -8.345439910888672, 2.9617366790771484, -2.7068212032318115, -5.828347206115723, 2.651090621948242, 4.721800804138184, 2.443984270095825, 11.200489044189453, -10.498586654663086, 3.9946794509887695, 16.7941837310791, 0.12091109156608582, -22.16507911682129, 9.763059616088867, -2.0252254009246826, 6.506683349609375, 2.774111032485962, 6.222336292266846, 12.261150360107422, 1.3053227663040161, 7.917670249938965, -13.030365943908691, -8.942769050598145, -1.217942714691162, 15.871702194213867, -8.303033828735352, -3.493138551712036, 6.6786041259765625, -5.277369499206543, -13.024242401123047, -19.73811912536621, 7.885560512542725, -11.34699821472168, 3.4716179370880127, 5.138920783996582, -10.028000831604004, 4.457937240600586, -21.485260009765625, -1.1554404497146606, -10.481566429138184, 2.265120506286621, 1.9570748805999756, 1.780928134918213, 6.1868181228637695, -6.46340799331665, -14.562262535095215, -12.487845420837402, 9.943899154663086, -10.499481201171875, -3.560365676879883, 3.5017240047454834, -1.6506304740905762, -9.990633010864258, -21.240407943725586, 9.368873596191406, -5.864860534667969, -16.400039672851562, -6.5211944580078125, 0.41963353753089905, -19.781740188598633, 1.314176082611084, -0.0016628061421215534, 4.451232433319092, 0.039103660732507706, -10.072565078735352, -16.43920135498047, -2.0906965732574463, -5.352242469787598, 15.244277954101562, -3.022988796234131, 3.9656238555908203, 0.4386710822582245, -17.712501525878906, 7.427308559417725, -4.783039093017578, -14.675159454345703, 3.9544525146484375, 12.202258110046387, -1.3835670948028564, -3.02427077293396, -6.629498481750488, -4.497897624969482, 10.93004322052002, 0.6149920225143433, 4.0099077224731445, 5.549839019775391, -29.535722732543945, 1.4290266036987305, 5.476307392120361, 2.0034618377685547, -15.097211837768555, -8.090621948242188, 6.525032997131348, 3.908414602279663, 4.201511859893799, -11.687804222106934, 2.479555130004883, 14.195745468139648, 0.1649453490972519, 13.217453002929688, 8.392437934875488, 7.980894565582275, -6.977819442749023, -6.746792793273926, -6.980224609375, 11.194154739379883, 12.203770637512207, -10.626731872558594, -13.091612815856934, 0.9886866807937622, 12.790814399719238, 2.4412736892700195, 6.061949253082275, 4.554685592651367, 5.492851257324219, -1.607325792312622, 4.523470878601074, 6.561579704284668, -1.3490278720855713, 16.46564292907715, -10.842681884765625, -4.105921745300293, -8.67026138305664, 10.360666275024414, 10.55143928527832, -7.714958667755127, 0.4783846139907837, -4.769192218780518, -13.862018585205078, -8.306347846984863, 3.012254476547241, 5.330827713012695, -7.552490234375, -0.3091413974761963, 7.905978679656982, 13.442216873168945, 3.185622215270996, -8.346181869506836, -15.918752670288086, 0.40774160623550415, -3.3376052379608154, -6.050016403198242, -7.100210666656494, -5.8199357986450195, 1.2840741872787476, 6.462406158447266, -8.342623710632324, 18.225261688232422, 13.874354362487793, -2.6701042652130127, 2.8381636142730713, 14.84540843963623, -9.257749557495117, -6.303389072418213, -6.245606422424316, -10.1984224319458, 6.900027275085449, -2.331402540206909, 0.8102961778640747, 0.6527127027511597, -2.1388115882873535, 14.847816467285156, -1.7958650588989258, -8.6332426071167, -13.795823097229004, -6.87407112121582, 1.709261417388916, 1.56123948097229, 2.7062666416168213, -4.259446144104004, 4.201150894165039, 4.3961029052734375, -12.01119327545166, 3.8352978229522705, 4.016594886779785, -2.7972683906555176, 10.759517669677734, 0.4777170419692993, -7.545616626739502, 9.24808120727539, 17.460840225219727, -11.488367080688477, -0.7666337490081787, 8.670833587646484, 5.98801851272583, 10.015666961669922, 12.931705474853516, -17.753122329711914, 6.7886810302734375, -16.135080337524414, -9.415433883666992, -10.812965393066406, 4.695496082305908, -16.513622283935547, 0.9280555844306946, -15.056623458862305, 19.688434600830078, -5.322858810424805, -0.2527216672897339, -11.771740913391113, -5.816056728363037, 12.921346664428711, 2.34810209274292, -11.610928535461426, -5.089251518249512, -1.674823522567749, 17.299840927124023, -1.2696013450622559, 4.236573696136475, 10.366758346557617, 23.576128005981445, 6.331266403198242, -10.713767051696777, -9.935148239135742, 16.83951759338379, 3.969632148742676, -19.1993465423584, -5.187131881713867, -4.658158302307129, -3.505075216293335, 6.300398826599121, 6.5815839767456055, 13.225089073181152, -18.90669059753418, 2.8374547958374023, -2.126573085784912, -10.66925048828125, 8.28796672821045, -13.313398361206055, -13.914599418640137, 16.036161422729492, 4.001518726348877, 8.120513916015625, 13.026676177978516, -12.823548316955566, 13.609106063842773, 0.9827202558517456, 5.315954685211182, 8.025437355041504, 6.340173721313477, -3.8268179893493652, -0.16069330275058746, 1.6216100454330444, 10.53534984588623, -22.062482833862305, 14.157463073730469, -9.287504196166992, 21.780670166015625, 4.845901966094971, 17.579971313476562, -2.2272379398345947, 17.37596893310547, 4.2686238288879395, 4.245100975036621, 4.637980937957764, -24.856393814086914, -14.76109504699707, -16.162397384643555, 0.9130793809890747, -2.1644532680511475, 8.302789688110352, 3.5951366424560547, 12.495560646057129, -6.993480682373047, 3.584656238555908, -4.967894077301025, -11.635139465332031, -17.734594345092773, -0.9789464473724365, -9.294668197631836, -5.39682149887085, -9.694121360778809, -11.077970504760742, 28.215682983398438, 0.3299051523208618, -13.264369010925293, -2.078763723373413, 0.06386318057775497, -14.92090129852295, 10.465770721435547, -9.478848457336426, 5.050337791442871, -4.875583171844482, -8.40612506866455, 32.1939697265625, -6.49796199798584, -15.690971374511719, -5.6220011711120605, -10.435348510742188, 10.53407096862793, 3.9672555923461914, -12.380391120910645, 17.337230682373047, -2.763355016708374, -13.662318229675293, 10.538152694702148, 6.145570755004883, -10.708599090576172, -11.887474060058594, -2.072922468185425, -7.280892372131348, 5.042370796203613, 2.3555221557617188, -12.525991439819336, -11.462960243225098, -3.027575969696045, 12.613487243652344, -6.9764180183410645, 5.544565200805664, -7.039139270782471, 3.876507520675659, -6.302280902862549, 3.7212517261505127, -9.568120956420898, -8.11693286895752, 18.994279861450195, 8.772307395935059, -13.252819061279297, 11.294260025024414, -2.327122926712036, 13.065237998962402, -14.204946517944336, -1.6513700485229492, -5.009230613708496, -7.768837928771973, -12.44716739654541, 19.014657974243164, -0.709588885307312, -3.840634822845459, -17.398157119750977, 10.74223518371582, 10.80666732788086, -2.289198637008667, -14.487964630126953, 5.236842632293701, -0.5434355735778809, -15.309457778930664, 20.643442153930664, 6.866678237915039, -10.56003475189209, -8.713959693908691, 7.473781108856201, 11.624147415161133, 5.587146759033203, 4.920267105102539, -4.13533878326416, 20.747907638549805, -3.0260632038116455, 2.1041083335876465, 3.7403669357299805, -2.9591917991638184, 7.796346664428711, 15.64780330657959, -22.613544464111328, -2.176119804382324, 12.712224960327148, 14.267722129821777, -4.730127334594727, 6.842087745666504, 1.9561814069747925, 1.5219786167144775, -6.33561897277832, 10.245365142822266, 12.232884407043457, 16.274639129638672, 2.7963643074035645, 1.5032639503479004, 11.600956916809082, 33.81874465942383, -2.163236379623413, -0.6870324015617371, 18.0903377532959, -11.263532638549805, -25.445995330810547, 10.239418983459473, 14.436910629272461, 13.741816520690918, 10.512276649475098, 11.467656135559082, -6.672336578369141, 10.977593421936035, -9.740077018737793, -15.535337448120117, -1.9288737773895264, -12.74263858795166, -10.858068466186523, -6.286024570465088, -3.9524359703063965, 1.7288458347320557, -10.12811279296875, -5.847511291503906, -1.374248743057251, 1.7840032577514648, 4.975127696990967, 7.974522113800049, 2.489927053451538, 4.755643367767334, 6.9771342277526855, 0.014457613229751587, 2.0326180458068848, -9.042354583740234, 2.522614002227783, 10.12540054321289, -6.472926616668701, 13.506031036376953, -5.036961555480957, 11.080760955810547, 17.163166046142578, 2.4898712635040283, -3.774698257446289, 4.731132984161377, -3.8458192348480225, -9.006311416625977, 10.744184494018555, -19.24459457397461, 9.338224411010742, -7.7047295570373535, 9.60681438446045, -4.982018947601318, 18.66176414489746, -4.917170524597168, -22.599946975708008, 29.78435516357422, 0.4521271586418152, -1.1576651334762573, 0.017523912712931633, -3.133610486984253, 3.108090400695801, -7.5804853439331055, 22.210880279541016, -6.970045566558838, 6.661713600158691, 13.697785377502441, -2.1598658561706543, 8.932581901550293, -5.395350933074951, -10.014717102050781, -8.046008110046387, 11.096380233764648, 10.73211669921875, -9.138406753540039, 10.39710521697998, -9.1239013671875, 1.89992094039917, 7.1442437171936035, -20.846616744995117, 6.3408613204956055, -3.0367863178253174, -15.001897811889648, 6.895214080810547, 2.90187406539917, 3.5885848999023438, -2.852278232574463, -6.891574859619141, -13.454581260681152, -2.4443414211273193, -10.792104721069336, -4.6476263999938965, 3.9533982276916504, 7.925520896911621, 4.143782615661621, 2.722853660583496, -2.9439117908477783, 9.42686939239502, 11.742212295532227, -5.173035144805908, -4.598190784454346, -10.110496520996094, -3.1959500312805176, -4.605373382568359, 19.64137840270996, 13.580016136169434, -13.930630683898926, 7.666930675506592, -2.6176812648773193, -17.833871841430664, 2.3780405521392822, -7.0993332862854, 2.920093536376953, 5.076911449432373, -6.811123371124268, 14.949082374572754, 1.176584243774414, -8.154172897338867, -16.773466110229492, 22.332944869995117, -8.33442211151123, 0.43158435821533203, 2.6600093841552734, 8.788653373718262, -13.936615943908691, -24.9569091796875, 0.8027822971343994, -15.433825492858887, -3.151336193084717, 20.594995498657227, -6.358027935028076, 6.507047653198242, -11.6410493850708, 29.93269157409668, -4.191216945648193, 3.6753246784210205, -1.7892920970916748, -6.556334018707275, 10.462384223937988, -2.749662160873413, -9.755350112915039, -5.912735939025879, -3.5891172885894775, 0.43381693959236145, 0.8376941084861755, -8.874822616577148, -5.80961799621582, 20.380308151245117, -3.317410469055176, 13.073196411132812, -5.248797416687012, 2.8176522254943848, -7.194709777832031, -20.974201202392578, -6.521063327789307, 7.97422456741333, 11.90910816192627, 9.929388046264648, -6.637343406677246, -4.372632026672363, -12.729391098022461, 1.6843301057815552, 7.174457550048828, 1.1553632020950317, -8.739211082458496, 2.9160144329071045, -0.1732063889503479, 3.7369418144226074, -2.9953179359436035, 0.9373892545700073, -8.454690933227539, -9.601903915405273, -26.857036590576172, 1.0989495515823364, 0.582759439945221, -10.819015502929688, 14.65721607208252, 11.413497924804688, 16.510684967041016, -2.0629642009735107, -3.4398131370544434, 2.5329134464263916, 8.651223182678223, 14.587721824645996, -10.484423637390137, -2.4351258277893066, -21.337860107421875, -3.36630916595459, 8.604999542236328, 6.339088439941406, 23.372037887573242, 6.334481239318848, -4.919592380523682, -8.298519134521484, 2.100371837615967, 16.261672973632812, -13.371349334716797, -10.32762336730957, 0.25086140632629395, -4.029244899749756, 8.545266151428223, -5.29136848449707, -0.09014803916215897, -0.6661169528961182, 1.6549670696258545, 13.393021583557129, -13.765944480895996, -18.22177505493164, -15.767818450927734, -3.529852867126465, 0.6190512776374817, 4.896379470825195, 2.630121946334839, 0.7616981267929077, -18.717086791992188, -7.286749362945557, 3.8160016536712646, -13.192595481872559, -5.515843391418457, -5.002739429473877, 4.421051979064941, -9.547978401184082, -17.657363891601562, -4.340655326843262, -3.5335588455200195, 7.899689197540283, -6.737039566040039, -2.9653592109680176, -5.889847278594971, -5.518527984619141, 20.53152847290039, -26.49062156677246, 5.408820629119873, -8.725661277770996, 0.16237212717533112, -16.309011459350586, 1.4072800874710083, -8.877569198608398, 5.088758945465088, 13.2635498046875, -13.368071556091309, -16.99782371520996, 11.063584327697754, 19.278186798095703, -0.5285110473632812, -21.718814849853516, -16.5457763671875, 9.128987312316895, 17.292510986328125, -1.8968560695648193, -14.376659393310547, 11.139249801635742, -14.475881576538086, -7.572689533233643, -7.0426530838012695, -7.145585536956787, 4.91364860534668, -10.496646881103516, 0.909351110458374, 10.491998672485352, 10.507144927978516, 13.137116432189941, -4.407456398010254, -18.015480041503906, 1.3358325958251953, 1.7241652011871338, 2.3561742305755615, -4.527904987335205, 4.070209503173828, -11.038054466247559, 7.764595031738281, 17.485992431640625, 0.8690153956413269, 0.047645460814237595, 5.998391628265381, 5.120737552642822, 5.34487247467041, 1.6093966960906982, -3.3136863708496094, 15.568018913269043, -12.638693809509277, 10.990294456481934, -9.17424201965332, 4.393455505371094, -10.158429145812988, -16.204227447509766, -11.003679275512695, -10.37084674835205, 13.065292358398438, -6.657952785491943, -7.6836724281311035, -10.5027437210083, -0.8859392404556274, -0.8191591501235962, 5.108419418334961, -7.658571243286133, -12.843912124633789, -1.2737400531768799, 8.21761703491211, 6.956233501434326, -10.66698169708252, -5.683403968811035, 2.753450393676758, -6.508413314819336, -14.870471000671387, 1.3418471813201904, -4.243696212768555, 2.3659493923187256, 7.975165367126465, -5.60853385925293, -12.1419038772583, -8.32394790649414, 7.239495277404785, 0.39141416549682617, 7.273825645446777, -4.109683990478516, 6.90250825881958, -5.463948726654053, -7.055097579956055, -0.6361967325210571, -1.8097351789474487, -2.9834821224212646, -0.03516986221075058, 9.119952201843262, -7.357945919036865, -18.284801483154297, 3.651872158050537, 18.530616760253906, 10.115288734436035, -12.614543914794922, 4.060141563415527, 10.756749153137207, 8.565486907958984, 17.94461441040039, 8.098867416381836, -28.714447021484375, 5.320420265197754, -8.29715633392334, -9.859158515930176, -17.85846519470215, -13.476351737976074, -4.2551164627075195, 10.576810836791992, 11.060588836669922, 6.217541217803955, 12.037768363952637, -4.535131454467773, -9.027572631835938, 18.860925674438477, 4.213016986846924, -4.074059963226318, 7.1118011474609375, -8.056856155395508, 15.542035102844238, -13.761148452758789, -5.957056045532227, -2.3579084873199463, -2.4815468788146973, -13.189286231994629, 8.26842212677002, 5.8650641441345215, 0.13792791962623596, 5.435333251953125, -5.136580944061279, 9.42158317565918, -6.230356693267822, -11.259262084960938, -2.205669403076172, -2.3930721282958984, 3.2004735469818115, 6.50248908996582, 16.857913970947266, -13.858491897583008, -16.523212432861328, 2.572226047515869, 13.010530471801758, 10.230828285217285, -1.260302186012268, -2.575308084487915, -3.0810608863830566, 17.8699893951416, -2.176746368408203, 3.220515489578247, -12.440532684326172, 9.308568954467773, -1.323966383934021, -2.6432833671569824, -2.980254888534546, -1.2715882062911987, 27.487430572509766, 6.282689094543457, 6.794125556945801, -7.561183929443359, -14.15574836730957, -21.601852416992188, -5.80594539642334, 1.8420075178146362, 2.473328113555908, -14.554418563842773, 6.727168560028076, -5.505618572235107, 11.57945442199707, -2.4154911041259766, 0.6355035901069641, -7.196927547454834, -6.721878528594971, 0.5783593058586121, 6.017884254455566, 7.490167617797852, -6.902593612670898, -12.291961669921875, 4.5022292137146, 13.172357559204102, -4.7073259353637695, -4.045304298400879, -4.213801383972168, 3.6403579711914062, 9.893895149230957, -9.158307075500488, 4.649416446685791, 14.724106788635254, 6.297154426574707, -7.4063262939453125, -14.255361557006836, 0.7192729711532593, 13.948148727416992, 8.400725364685059, 14.867376327514648, -3.2522213459014893, 1.0585169792175293, -7.660594940185547, 5.992493629455566, -1.4863202571868896, -7.41799783706665, -3.3789408206939697, 6.171330451965332, 9.335886001586914, 8.403328895568848, 23.922805786132812, 8.165227890014648, -8.591352462768555, 11.455220222473145, -7.858669757843018, -10.318164825439453, 7.9080915451049805, -5.162892818450928, 0.7967934012413025, -4.320104598999023, -1.1373419761657715, -16.57813835144043, 9.761853218078613, 13.029470443725586, -1.6066728830337524, 13.299211502075195, 9.450321197509766, 12.839993476867676, 15.67655086517334, -11.377880096435547, 9.044015884399414, -0.5406059622764587, 8.512541770935059, -1.7136380672454834, 21.597118377685547, 17.578475952148438, 0.86379075050354, 5.442814350128174, 3.3111116886138916, 3.5708041191101074, 10.32878303527832, -8.683001518249512, -13.184637069702148, 12.249489784240723, 4.496832847595215, -3.4794139862060547, -5.71783447265625, -13.780083656311035, -1.7674264907836914, -16.24336051940918, 2.402635097503662, 3.808427095413208, -5.775590896606445, 11.048530578613281, 8.440472602844238, 2.816631317138672, -6.675247669219971, -5.778743743896484, -8.71447467803955, -18.923507690429688, 0.7026219367980957, 7.878701210021973, -14.671869277954102, 4.07660436630249, -2.0125527381896973, 1.5677152872085571, 7.603227615356445, 9.619693756103516, -6.466675281524658, 7.816687107086182, -3.8809988498687744, -25.40565299987793, 20.676639556884766, 0.611973226070404, -2.9892101287841797, 13.698674201965332, 4.776316165924072, -5.455913066864014, 2.3022689819335938, 3.4431254863739014, -1.8476812839508057, -23.74262237548828, 1.6578084230422974, -9.32030963897705, -10.39613151550293, 0.5797820687294006, 21.740402221679688, -11.930591583251953, 3.2004733085632324, -14.80193042755127, -16.09738540649414, -3.6137280464172363, 9.321054458618164, 2.891479015350342, 2.902588367462158, 12.358766555786133, 4.9786458015441895, -5.833588600158691, -9.53011703491211, -3.024844169616699, 3.584826946258545, -3.251102924346924, -2.355536699295044, -5.6017045974731445, -8.054054260253906, -12.025832176208496, -7.960137844085693, 17.952228546142578, 10.877887725830078, 13.96921157836914, 1.2348573207855225, -1.174640417098999, 8.24525260925293, -3.7256195545196533, 11.553492546081543, 1.4039040803909302, 8.06001091003418, -7.731200218200684, 18.374832153320312, 2.175226926803589, 6.798653602600098, -7.404027462005615, -6.841439247131348, 7.142606735229492, -16.42037010192871, 12.599407196044922, -5.883013725280762, 9.797707557678223, -6.494362831115723, -0.36781150102615356, 0.944292426109314, 9.307480812072754, -8.505483627319336, 5.646740913391113, 3.054027557373047, 7.877171039581299, -8.316673278808594, 0.13003620505332947, -7.302462577819824, -3.735976457595825, 10.866726875305176, 12.020883560180664, -9.263182640075684, -1.5392847061157227, -0.6508278846740723, -9.446142196655273, -3.857027530670166, -10.692161560058594, 5.665367126464844, -9.499868392944336, 14.262967109680176, -19.668659210205078, 5.781172752380371, 6.126323223114014, -6.519585132598877, -17.05522346496582, 1.7696192264556885, -12.30732536315918, 10.907870292663574, 1.062849521636963, -0.19988685846328735, 1.0582523345947266, 7.487205505371094, -0.2847309112548828, 5.428495407104492, 3.5895795822143555, -5.293632507324219, 10.973188400268555, 15.926375389099121, 11.011963844299316, 1.1075257062911987, 6.704763412475586, -10.772833824157715, -8.350327491760254, -8.93478775024414, -5.965683460235596, 8.187592506408691, -8.383974075317383, -6.531791687011719, -15.258895874023438, -7.094277858734131, -16.830585479736328, 8.691445350646973, -1.7881805896759033, 9.037130355834961, 13.391919136047363, 4.45680570602417, 3.1126930713653564, 1.1008727550506592, -4.493154048919678, -21.869714736938477, 1.5994871854782104, -7.1780548095703125, -10.121312141418457, -5.998141765594482, -0.4675552248954773, 6.67285680770874, -10.102673530578613, -12.763479232788086, 10.49275016784668, 5.278175354003906, 7.707357883453369, -2.9401443004608154, -14.055739402770996, 12.94279670715332, 12.141855239868164, -6.188318729400635, 3.9005706310272217, 10.656899452209473, 24.53302764892578, 11.820727348327637, 5.379936695098877, -2.256345510482788, -4.621392726898193, -0.6724455952644348, -6.644364356994629, 6.276237487792969, 1.3957810401916504, -2.3208742141723633, -10.015474319458008, 1.1586432456970215, -3.269801139831543, 3.9464902877807617, -9.314205169677734, 6.826131820678711, -1.0378201007843018, 19.17110824584961, -7.450455665588379, 19.73821449279785, 29.083477020263672, -0.0028865677304565907, 20.666091918945312, 11.740960121154785, -11.597368240356445, -5.03518533706665, 10.68405532836914, 3.7031893730163574, 9.395820617675781, 4.994577884674072, 15.294584274291992, -1.6030585765838623, -9.727108001708984, -5.851530075073242, 0.5032640099525452, -8.221158981323242, 0.4669341444969177, -1.3387868404388428, -2.3709568977355957, 0.31216976046562195, -3.136289596557617, 10.556334495544434, 1.7187737226486206, 8.368631362915039, -13.580202102661133, 0.9917325377464294, -4.0211381912231445, -18.6981201171875, 20.987585067749023, -12.398353576660156, 8.867931365966797, 8.180675506591797, 12.200325012207031, -3.6708319187164307, -24.717548370361328, 6.9890360832214355, -0.2993796467781067, 7.287926197052002, -0.8038773536682129, 13.613368034362793, 5.9411091804504395, -2.6882009506225586, 24.739282608032227, -5.093113899230957, -20.93318748474121, 2.0283093452453613, 5.635716438293457, 6.544750213623047, -4.475080966949463, 0.732944905757904, -7.4820780754089355, -3.252030849456787, 7.504348278045654, -4.292330265045166, 8.038114547729492, -1.5649889707565308, 1.7560428380966187, -2.33377742767334, 12.265642166137695, 2.0176353454589844, -6.944289207458496, 2.091364622116089, 12.609891891479492, 3.4603095054626465, -9.682608604431152, -6.714670181274414, -9.488351821899414, 14.280670166015625, 2.542559862136841, 9.41049575805664, 8.892770767211914, -2.567533493041992, 11.607757568359375, -0.4955710768699646, 3.827568531036377, -19.508756637573242, -0.46498751640319824, -3.6498024463653564, -7.218376159667969, 2.3988595008850098, -12.296954154968262, -2.388460636138916, -25.170467376708984, -2.072317123413086, -6.793715953826904, 17.3414306640625, 1.9419716596603394, 5.9972758293151855, -29.0871524810791, -4.324047088623047, -11.629846572875977, 13.057136535644531, 9.585745811462402, 1.1640645265579224, 1.5870033502578735, 0.9635245203971863, -0.6424090266227722, 6.358649253845215, 5.776117324829102, -1.619744062423706, -7.353265285491943, 15.984685897827148, -4.363072872161865, -7.327253341674805, 23.063257217407227, 1.0363831520080566, -1.4958680868148804, -13.462759017944336, -10.287841796875, 13.780327796936035, -3.0703048706054688, 11.67048454284668, -10.509072303771973, 8.481685638427734, 1.872300148010254, -3.641169309616089, -17.137012481689453, 13.9534273147583, 2.0208780765533447, 0.4718775153160095, 9.589982986450195, -9.22867488861084, 14.989435195922852, 8.958736419677734, 6.527225494384766, 6.9958930015563965, -5.604279041290283, -0.623505175113678, 5.486557483673096, -4.0024824142456055, 14.617118835449219, 9.311775207519531, 15.01348876953125, -13.87828254699707, 16.51399803161621, -3.774077892303467, -6.524196147918701, -1.7962327003479004, 15.23676872253418, 5.6947922706604, 5.118143558502197, 11.44406509399414, 11.888601303100586, 12.599984169006348, 10.509925842285156, 4.1798810958862305, 6.266095161437988, -12.459671020507812, -0.44397276639938354, -20.860130310058594, 0.0960903912782669, 2.0253469944000244, -6.545725345611572, -4.2364983558654785, 11.444493293762207, -2.5682520866394043, 1.0806108713150024, 19.836732864379883, 4.120782375335693, 10.019004821777344, 5.319943904876709, 6.46656608581543, 12.080531120300293, -1.3991421461105347, 8.20635986328125, -4.086930274963379, -3.396010160446167, 6.810110092163086, -5.471491813659668, -25.400362014770508, -1.7668509483337402, 13.30302619934082, -9.513330459594727, 6.268346786499023, -3.276087999343872, -14.665359497070312, -2.8464927673339844, -22.426860809326172, 12.057352066040039, -3.662947177886963, 3.9014930725097656, 0.06907379627227783, -14.626364707946777, -3.874476432800293, 11.737532615661621, 8.03826904296875, -3.707639217376709, -7.188538551330566, -0.7894748449325562, -12.910579681396484, 2.435590982437134, -16.27454948425293, -7.511404991149902, 8.243306159973145, -0.13393646478652954, -6.60272216796875, 1.310107707977295, -2.752469539642334, 2.3504385948181152, 10.58464527130127, -1.113935112953186, -16.913604736328125, 4.071018218994141, -8.030841827392578, 11.226521492004395, 9.57452392578125, -0.2555329203605652, 4.438603401184082, -14.52768325805664, -17.266481399536133, -3.48602032661438, 3.2993383407592773, 13.319068908691406, -6.690425872802734, -9.19741153717041, -8.47775650024414, -7.419293403625488, 1.6744201183319092, -14.131660461425781, 7.433253288269043, -13.025175094604492, 22.24525260925293, -1.3416956663131714, 0.8756642937660217, -16.38231658935547, 2.8032913208007812, -18.88037109375, 8.249201774597168, -6.379439353942871, 2.4069314002990723, 4.9669270515441895, 9.14667797088623, 13.243045806884766, 11.06640338897705, 7.297667503356934, -6.072576522827148, -11.238409042358398, 7.300339221954346, 3.2023837566375732, -5.161978721618652, -0.5750772356987, 9.222315788269043, 8.892369270324707, -5.959024429321289, -12.979263305664062, -18.92283821105957, -4.575497627258301, -3.128478765487671, -13.088539123535156, 10.442283630371094, 18.649673461914062, -7.577662467956543, 7.265689373016357, -1.1453369855880737, 5.635870933532715, 1.7221516370773315, -0.7910395264625549, 9.329011917114258, 6.940189361572266, -4.032718658447266, 4.826406478881836, -5.874603271484375, -0.3377031683921814, 6.02504825592041, 4.653539180755615, -15.217798233032227, 5.844485282897949, -11.7113676071167, 2.242191791534424, 7.006467819213867, -14.667991638183594, -10.477899551391602, 8.104785919189453, 3.1847519874572754, -20.3713436126709, 9.506916999816895, -5.979818820953369, -11.485213279724121, 2.606529712677002, -6.153468132019043, 0.47896599769592285, 2.6314382553100586, 23.96766471862793, 9.199122428894043, 2.6159260272979736, 1.0576539039611816, 3.496227264404297, 4.881161689758301, -6.077420711517334, 5.100893020629883, -6.709316253662109, -5.51377010345459, -0.8089156150817871, 10.089317321777344, 8.193795204162598, -29.086219787597656, 11.97705364227295, 0.6449573040008545, 0.8313966989517212, 14.402984619140625, -7.046485900878906, -10.244885444641113, -19.490570068359375, -11.599834442138672, 8.085563659667969, -7.991710186004639, -10.227645874023438, 3.812290668487549, -10.036171913146973, -7.539707183837891, -8.034273147583008, 3.8160014152526855, -6.949503421783447, -5.107419490814209, -1.9858124256134033, -18.276782989501953, -2.687878131866455, -18.02666664123535, -12.059110641479492, -4.967831611633301, 0.824545681476593, -2.859196662902832, 10.051136016845703, 0.7906292676925659, 5.242108345031738, -5.076442718505859, -6.729720115661621, 16.09638023376465, -16.544296264648438, 11.099711418151855, -7.811671257019043, 6.152987480163574, -13.111234664916992, 4.444192886352539, 2.686805248260498, -21.179386138916016, -6.750111103057861, 5.906826019287109, 9.568720817565918, 2.5018718242645264, -5.987778663635254, -0.9271835684776306, -13.182162284851074, -10.191627502441406, 1.7863613367080688, -0.4546092450618744, 6.512592315673828, 1.35926353931427, 5.327252388000488, -1.3969354629516602, 5.619922161102295, 23.755596160888672, -10.006117820739746, 1.0262534618377686, -4.864787578582764, 4.600545883178711, 4.2811279296875, -8.192378044128418, 1.3140957355499268, 1.8713603019714355, -1.255555272102356, 2.9425711631774902, 13.492740631103516, 5.886223793029785, -2.619988441467285, 2.304744243621826, 19.599382400512695, -6.097700595855713, 11.006658554077148, -6.559691429138184, -15.714859962463379, -6.787185192108154, 3.328805446624756, -2.335747480392456, -6.903722286224365, -7.720728397369385, -9.478202819824219, -10.478317260742188, -6.009049415588379, -24.532962799072266, 4.157017707824707, 2.7627015113830566, 0.8991283178329468, 8.858012199401855, -5.933233261108398, -0.18097712099552155, -25.66961669921875, 1.7304699420928955, -4.057566165924072, 5.809622764587402, 8.185375213623047, -1.8874536752700806, -12.56684398651123, 3.1475963592529297, 8.135854721069336, 8.782994270324707, -5.331432819366455, -9.245283126831055, -2.8697166442871094, -10.431670188903809, -21.386959075927734, 3.325772762298584, 4.038455009460449, 6.16662073135376, -8.691889762878418, -0.683792769908905, -12.22801399230957, -14.856975555419922, 4.274821758270264, -18.25299072265625, -3.7248945236206055, 22.7632999420166, 11.58902645111084, 6.275918006896973, 5.617126941680908, -2.3034820556640625, -6.13872766494751, 19.499242782592773, -2.22027325630188, -6.100919246673584, 6.269744873046875, -0.839924693107605, 12.5346097946167, -6.455514430999756, -8.344916343688965, -4.167993068695068, 15.541971206665039, 8.986211776733398, 2.4400932788848877, 0.7728871703147888, 3.276395559310913, -9.640780448913574, 4.929747104644775, -4.914599418640137, 2.2384085655212402, 0.17891407012939453, 2.966486930847168, -4.540072441101074, 9.353282928466797, 3.5238752365112305, -7.552578449249268, 4.334481239318848, -3.3276751041412354, 3.799407720565796, 8.360125541687012, -4.740096092224121, -0.26541218161582947, 0.7131298780441284, 0.43666765093803406, 20.023296356201172, 3.5110223293304443, -11.1228609085083, 1.1705121994018555, -5.285120487213135, 24.040081024169922, -6.101729393005371, -10.752943992614746, 7.309929847717285, -5.003285884857178, -3.6282472610473633, -2.1688232421875, -6.4908833503723145, -3.9235000610351562, -21.428680419921875, -0.837854266166687, 6.496987342834473, 18.510936737060547, 0.8373836278915405, 8.338748931884766, -6.8726677894592285, 14.57279109954834, 22.263490676879883, 3.5640344619750977, -3.046205520629883, 0.8164804577827454, 15.536808013916016, -11.863579750061035, 12.900900840759277, -9.396217346191406, -7.744821548461914, 3.01145601272583, 6.705164432525635, 9.870828628540039, 14.390365600585938, -10.709500312805176, 1.126265048980713, 7.982306480407715, 3.093750476837158, 11.48525333404541, 0.8010096549987793, -2.479430913925171, -21.567901611328125, -18.014751434326172, 8.152077674865723, -10.85824203491211, 3.425910472869873, 6.7903828620910645, -0.7553740739822388, 10.326658248901367, -5.946035861968994, 6.18825626373291, -0.8368228673934937, -14.207559585571289, -19.303482055664062, -1.5253784656524658, 16.347213745117188, 11.786102294921875, 5.028979778289795, 8.187976837158203, 9.215715408325195, -2.053086042404175, 13.241096496582031, 2.9382705688476562, -4.6524529457092285, 38.33388900756836, 6.923079967498779, -0.8148653507232666, 7.918899059295654, -11.58613395690918, -11.685585021972656, 19.26654815673828, 6.508156776428223, -10.298956871032715, -15.679426193237305, -0.7501665353775024, -10.047330856323242, 13.289936065673828, -18.80634880065918, 12.276891708374023, -9.925228118896484, -12.241965293884277, 0.5109072923660278, -12.460926055908203, -5.737154483795166, -5.488396167755127, -24.013187408447266, -11.77228832244873, 3.3652524948120117, -20.615205764770508, -13.591240882873535, -9.190975189208984, 9.03108024597168, 9.036334991455078, 14.703842163085938, -9.673333168029785, 7.9948039054870605, 2.2380642890930176, -13.03300666809082, 5.055145740509033, -0.21411177515983582, -6.742490768432617, -10.913237571716309, -16.228633880615234, 4.826313495635986, 4.25863790512085, -3.0266876220703125, -3.9761362075805664, -6.1249189376831055, 7.419820308685303, 1.7811942100524902, -17.924015045166016, 0.690670371055603, 4.531578063964844, -6.158227920532227, -4.897843360900879, -6.0502424240112305, 6.138628005981445, -4.174648284912109, -0.45291873812675476, 0.6949126720428467, 15.257841110229492, -15.172452926635742, 23.11041831970215, -12.187813758850098, -2.3357012271881104, -8.22674560546875, 1.1735174655914307, 5.367708683013916, 23.160619735717773, 11.799654960632324, -6.887692928314209, 2.0922741889953613, 1.0992969274520874, -6.741389274597168, 5.830865859985352, 2.5900611877441406, -4.518482685089111, -6.964600563049316, 7.013803482055664, -1.2325043678283691, 11.856736183166504, 8.570345878601074, -12.156368255615234, -21.076318740844727, 0.36544618010520935, -6.82961368560791, -10.441279411315918, -7.299304962158203, 14.397356033325195, -2.275217056274414, 18.934072494506836, 7.7495269775390625, -17.43705177307129, 4.407967567443848, 14.86622428894043, -17.85713005065918, -12.736778259277344, -1.129478096961975, -5.530622482299805, 16.932870864868164, -8.650228500366211, 17.041244506835938, -14.041033744812012, -8.592765808105469, 3.3124589920043945, 5.199960708618164, 1.5377488136291504, 16.155683517456055, 18.25128746032715, 19.812721252441406, -0.8932831287384033, 8.866400718688965, -2.6428442001342773, 14.463353157043457, -23.12720489501953, -6.410984516143799, -2.031255006790161, -4.770003795623779, 16.10353660583496, -13.821925163269043, -8.072461128234863, -11.923199653625488, -0.278558611869812, -8.981475830078125, 9.607440948486328, 9.103318214416504, -21.07053565979004, -21.23898696899414, 8.34752082824707, 4.06199312210083, -15.86720085144043, -0.22699898481369019, -12.320679664611816, 8.182940483093262, -13.135835647583008, -6.101572513580322, 4.875927448272705, 0.6040363311767578, 10.672761917114258, 4.305570125579834, -14.925125122070312, 2.4860377311706543, -11.222752571105957, 14.633657455444336, 15.961913108825684, 11.515100479125977, 22.17769432067871, 8.148351669311523, 14.929333686828613, 9.601288795471191, 9.013476371765137, 8.838883399963379, 3.0665669441223145, 18.231903076171875, -10.826767921447754, -15.640735626220703, -7.629724025726318, 16.04502296447754, -2.5894813537597656, -8.086374282836914, -8.409218788146973, 4.734431743621826, -0.4800945818424225, 12.12303638458252, -3.175053834915161, 9.22944164276123, -8.595868110656738, 5.50029993057251, 30.61394691467285, -1.8898417949676514, -1.1597009897232056, 9.877731323242188, -9.80807113647461, -4.487182140350342, -5.345897674560547, 2.678912878036499, -13.636619567871094, 14.487605094909668, 1.5017093420028687, 1.9881906509399414, -1.4986653327941895, -0.1973666399717331, -23.061668395996094, 12.944718360900879, -8.68383502960205, 8.428544044494629, -1.378119945526123, -1.937290906906128, 5.165108680725098, 17.04585075378418, -0.7840608358383179, -5.636632919311523, 4.869866371154785, -24.634483337402344, -8.471668243408203, -9.789077758789062, -3.4973599910736084, -13.591294288635254, 8.611140251159668, -2.966671943664551, 1.5227501392364502, 0.2769336998462677, -3.025834321975708, -6.439116477966309, 13.077569961547852, 8.445684432983398, 15.12481689453125, -1.472947359085083, 2.918360710144043, 3.7299883365631104, -3.162611484527588, -0.2419450581073761, -16.113435745239258, -11.798812866210938, -9.108086585998535, -11.331602096557617, 4.4651947021484375, 3.773686647415161, 4.210054397583008, -2.486112117767334, 10.26063346862793, 9.323232650756836, 5.85096549987793, -3.2275466918945312, 15.606303215026855, -11.798295974731445, -16.247568130493164, -12.947017669677734, 12.206097602844238, 1.0732914209365845, -5.788222312927246, -23.184776306152344, -2.1539697647094727, -9.839421272277832, -20.845340728759766, 11.035999298095703, 0.4369168281555176, 14.039590835571289, -0.05469033494591713, 5.749070644378662, -6.803360462188721, 6.3836140632629395, -3.908742904663086, 6.2367143630981445, 2.268226146697998, -10.840875625610352, 0.3708327114582062, 1.8300516605377197, -6.8610687255859375, -0.9805744290351868, 9.478889465332031, 12.539801597595215, 0.8142291307449341, -22.602306365966797, 7.70470666885376, 18.611013412475586, -2.147435188293457, -8.408586502075195, 3.231652021408081, 2.315141201019287, 9.043107032775879, -6.088000297546387, 17.710954666137695, 9.381174087524414, -9.93805980682373, 10.609821319580078, -1.0594998598098755, 7.176639556884766, 3.8858089447021484, 0.7712574005126953, 3.403460741043091, -3.9946160316467285, -7.133605480194092, -2.120486259460449, -2.610015392303467, 4.158411979675293, 2.120316505432129, -15.993993759155273, 8.605252265930176, -0.20705702900886536, 8.812701225280762, 7.051454544067383, 2.9629883766174316, 5.9102864265441895, -14.10346508026123, 15.769363403320312, 19.632675170898438, 6.630547523498535, -4.881267070770264, 7.236181259155273, -3.092947244644165, 16.58246421813965, -4.08927059173584, -11.304519653320312, -7.949172019958496, -7.1049394607543945, 7.527129173278809, -0.12892717123031616, 3.1007635593414307, -3.0646562576293945, 13.50705337524414, 0.49838703870773315, 2.5608255863189697, 16.092073440551758, 3.355323076248169, -4.842961311340332, -2.066542863845825, 11.757028579711914, 4.192192077636719, -7.341127395629883, 12.103219985961914, -14.174739837646484, -21.07917022705078, -9.41830062866211, -18.525941848754883, -1.5926752090454102, 0.7090162038803101, -2.3073463439941406, -5.370606899261475, -10.176719665527344, 9.621525764465332, -19.26976776123047, 7.605589866638184, 2.9146323204040527, -0.3614193797111511, 4.800506591796875, -15.220541954040527, 10.101322174072266, 5.413763523101807, 11.446836471557617, -20.28845977783203, 2.819187641143799, -1.0307307243347168, -22.71070098876953, -3.0628294944763184, 0.8147054314613342, -7.5994768142700195, 4.098702907562256, 5.910200119018555, -4.219757080078125, -3.2150862216949463, 7.944692611694336, -9.556724548339844, 5.2534050941467285, 13.531599998474121, -0.06563074886798859, 3.9866905212402344, 0.5092214941978455, 8.463603019714355, -14.78469467163086, -16.81634521484375, -8.324193954467773, -5.921304225921631, 9.941683769226074, 13.928519248962402, 6.04124116897583, 12.712066650390625, -3.844943046569824, -5.143406867980957, -15.172101020812988, 11.815425872802734, 5.565844535827637, -0.20654308795928955, -13.293350219726562, -10.018665313720703, 5.046748638153076, 0.0872751995921135, 11.12192153930664, 2.0154006481170654, -6.879197120666504, -11.664820671081543, 0.08583477139472961, 16.4652099609375, 19.9484806060791, -7.240232944488525, 5.2501115798950195, -7.109197616577148, 2.7330353260040283, -28.17744255065918, -6.222975730895996, -17.982080459594727, 14.626741409301758, -3.8226470947265625, 4.701855659484863, 0.3946186602115631, -13.294548034667969, 6.277060508728027, 10.894370079040527, 2.3233654499053955, 5.603334903717041, 11.198468208312988, 5.82099723815918, -1.0908255577087402, -16.19156265258789, -8.373111724853516, -7.518727779388428, 7.691905975341797, 13.536174774169922, 0.39096060395240784, 10.101228713989258, -23.735939025878906, 2.9785666465759277, -11.59293270111084, 9.621495246887207, 12.850132942199707, 11.524864196777344, 8.27475643157959, 11.75846004486084, 18.345626831054688, -4.733556270599365, 16.19015121459961, -2.697336435317993, -13.134208679199219, 6.572877883911133, 8.259572982788086, 3.881701946258545, -5.720902442932129, 6.822547912597656, -1.529786467552185, 25.28470230102539, 4.321746349334717, 4.887568950653076, -4.326805591583252, 6.833067893981934, 17.500221252441406, -6.304333209991455, -2.773444652557373, -18.23628807067871, 0.031074481084942818, 16.830718994140625, -11.20206356048584, -3.249037027359009, -6.865091800689697, -2.1969780921936035, -7.204011917114258, -3.0656299591064453, -10.661741256713867, 2.555793523788452, 10.121907234191895, 8.076433181762695, -18.18486976623535, 13.608558654785156, 7.594244956970215, 11.779828071594238, -8.331506729125977, -5.5043439865112305, 7.18806791305542, 17.881681442260742, -9.023353576660156, 3.221893072128296, 4.819029808044434, -3.0605742931365967, 3.9225354194641113, -4.6204938888549805, 13.229999542236328, -15.666685104370117, 3.5054337978363037, -4.670159339904785, -0.21030446887016296, -8.623443603515625, -29.482646942138672, 15.542974472045898, 7.670480728149414, 4.639073371887207, 0.17131400108337402, 3.6018588542938232, -2.5842227935791016, 1.7161781787872314, 14.4489107131958, -5.636434555053711, -8.000493049621582, 19.220312118530273, -4.234255313873291, 2.846989393234253, -1.0702451467514038, 6.556021690368652, 7.054518699645996, 1.8507235050201416, -9.005884170532227, -8.752399444580078, 1.3818531036376953, 18.781269073486328, -17.10064697265625, -0.4200473129749298, 5.238656997680664, -5.073614120483398, 2.890831470489502, -0.4559914767742157, 8.477416038513184, 17.046010971069336, -2.571526527404785, 3.5194382667541504, 4.98745584487915, 2.1244423389434814, 0.6138114333152771, -20.74981689453125, 1.4387589693069458, -10.118291854858398, -3.8702046871185303, 4.719986438751221, 6.042606353759766, -4.087169647216797, -4.2531657218933105, -2.4627685546875, -10.110137939453125, 15.69812297821045, 17.58948516845703, -1.572234034538269, 16.265615463256836, 7.979576110839844, -10.394262313842773, 9.38683795928955, 6.901969909667969, 1.8028591871261597, -14.21368408203125, -5.694106101989746, -22.630046844482422, -0.08343376219272614, -2.6638026237487793, -10.869746208190918, -12.21977424621582, 13.417274475097656, -14.870967864990234, 7.443385124206543, 1.2225160598754883, 12.452749252319336, -15.941515922546387, 1.2309424877166748, 12.5281982421875, -1.467529058456421, -13.856149673461914, -19.205612182617188, -6.800259590148926, 10.232075691223145, 15.349323272705078, 0.7670689821243286, 8.994540214538574, 10.140686988830566, 6.206455707550049, -8.130714416503906, 7.416524887084961, -9.79415512084961, 1.3317830562591553, 8.13228988647461, 10.970918655395508, -0.9275856614112854, 11.52590274810791, 15.436235427856445, 9.945304870605469, -0.11122141778469086, 3.463129997253418, 0.6085715889930725, 0.9377049803733826, 7.6398701667785645, 11.294589042663574, 0.5771226286888123, 7.854278564453125, 2.9791622161865234, 2.033588409423828, -9.207996368408203, 4.262668609619141, -0.6440503597259521, 3.3579447269439697, 10.479484558105469, 7.945723533630371, 7.575542449951172, -4.533879280090332, -15.445327758789062, -3.42429518699646, 0.25472545623779297, 2.3341712951660156, 13.81338119506836, 3.383624792098999, 14.562288284301758, -3.585787773132324, -10.314573287963867, 8.96695613861084, -10.724170684814453, -15.122075080871582, -5.124194622039795, 3.7125768661499023, 11.249388694763184, 0.4809894859790802, 9.333990097045898, 0.17461338639259338, -15.711647033691406, -10.505472183227539, -16.1345272064209, -1.646152138710022, -20.28632164001465, -13.805292129516602, -7.93372917175293, 9.120662689208984, 5.529918193817139, -3.5234129428863525, 2.183964967727661, -2.243137836456299, 27.160226821899414, -16.233379364013672, -19.894611358642578, -18.881179809570312, -14.000314712524414, -3.4188148975372314, 9.215479850769043, -2.3889293670654297, -2.709074020385742, -1.5989147424697876, 1.7409305572509766, -9.21617603302002, 5.97109317779541, -0.7427507638931274, 2.112496852874756, 5.111715793609619, 5.305777549743652, 7.455101490020752, -8.852299690246582, 4.303255558013916, -15.600704193115234, -20.971052169799805, -4.0904011726379395, 3.592693567276001, 4.279900074005127, 3.0637025833129883, 7.87289571762085, -17.81606101989746, -5.877272605895996, -2.737738847732544, -8.4498872756958, 2.687680244445801, 5.179176330566406, 2.5213193893432617, 0.6180009841918945, 22.16238784790039, 0.2508608102798462, -0.22403845191001892, 11.45654582977295, 17.376083374023438, -9.320382118225098, -10.123091697692871, 12.818708419799805, 10.605209350585938, 7.928616523742676, -2.848153591156006, 6.10848331451416, -14.101064682006836, -11.541781425476074, 14.878337860107422, -12.606805801391602, -1.8712509870529175, 5.83254337310791, 2.737053871154785, -11.001219749450684, -6.107578277587891, 5.3342180252075195, -4.617778301239014, 15.724886894226074, -1.7244513034820557, 17.362415313720703, -0.5623548030853271, 11.607171058654785, 3.8551292419433594, 0.24247092008590698, -13.632354736328125, 5.2030744552612305, 0.29419228434562683, 3.0107922554016113, -5.773565769195557, -9.693976402282715, -9.952601432800293, -1.5943669080734253, 8.31169605255127, -3.9645090103149414, -11.475869178771973, -8.829886436462402, -11.907100677490234, -16.668550491333008, -4.508635997772217, -7.218040943145752, -4.352396488189697, 17.613431930541992, 8.415000915527344, 13.412450790405273, -0.4520973861217499, 1.9186465740203857, -8.487166404724121, -17.864715576171875, -16.094865798950195, -4.24332332611084, 3.9700725078582764, 21.194889068603516, -1.4008420705795288, -15.613770484924316, -4.345522880554199, 4.290231227874756, -11.981008529663086, -5.363077163696289, 4.2729644775390625, 0.5321353077888489, 3.667325735092163, 0.0553673580288887, -0.7199224233627319, 12.848366737365723, 8.345264434814453, 2.0407817363739014, 4.757365703582764, -0.9110725522041321, -0.8917341232299805, 3.6655683517456055, 1.7493503093719482, 24.644657135009766, 2.2481391429901123, 3.939924716949463, 20.72984504699707, -9.06008243560791, 1.0827921628952026, 12.334348678588867, 2.1363983154296875, -0.3383258581161499, -8.046998023986816, 0.274068146944046, -6.811935901641846, 10.296911239624023, 5.625433444976807, -2.698225498199463, 11.375899314880371, -9.784324645996094, -8.2678804397583, 7.380451679229736, 2.843319892883301, -11.007064819335938, -7.163619041442871, -2.1582069396972656, -6.156961441040039, 2.116952657699585, -0.9733322262763977, 4.8360700607299805, -0.5584664344787598, 3.530271530151367, -4.12973690032959, 3.811481237411499, 3.447927474975586, 6.888537406921387, 3.429628849029541, -3.745681047439575, -0.577630877494812, 12.658740997314453, -2.502971887588501, 0.693289577960968, 15.41807746887207, 6.265005111694336, 6.709233283996582, -8.492145538330078, -8.6446533203125, 2.7890708446502686, 20.619182586669922, -17.392189025878906, -16.293476104736328, 5.455724716186523, 2.486962080001831, -9.500114440917969, 8.717657089233398, 1.8067963123321533, -4.682336330413818, -0.8714724779129028, -5.309726715087891, 2.8554186820983887, -6.890331268310547, -11.048365592956543, -7.068660259246826, 3.4206392765045166, 10.734498977661133, -6.271598815917969, -9.982123374938965, 2.7080905437469482, 10.617134094238281, -3.806757688522339, 14.461188316345215, -7.630359649658203, 13.684611320495605, 6.559687614440918, -16.44959259033203, 5.021836280822754, -2.240100860595703, -1.7884743213653564, -7.147639274597168, 0.08312755823135376, -6.40664529800415, 1.2135101556777954, 10.154914855957031, 13.152935028076172, 10.591121673583984, 8.364960670471191, 4.1285786628723145, -2.019721508026123, -10.862131118774414, 9.667871475219727, -4.318994522094727, 1.7120628356933594, -13.259135246276855, 4.162840366363525, 13.232906341552734, -0.972673773765564, 8.620404243469238, -1.5207324028015137, 1.510987401008606, -4.673773765563965, -25.803329467773438, 3.4266867637634277, 4.596075057983398, 5.498935699462891, 16.33660888671875, 8.784339904785156, 4.38743782043457, -6.405653953552246, 28.047954559326172, -13.055142402648926, -3.7236058712005615, -4.791158676147461, 10.326794624328613, 3.22346568107605, 5.201519012451172, -16.90268325805664, -5.052385330200195, 10.018428802490234, -4.872628688812256, -12.015911102294922, 2.0018575191497803, -5.655656814575195, 0.02837086282670498, 3.475757122039795, -9.466192245483398, -0.3748282194137573, 13.138708114624023, -2.4210081100463867, -1.9994109869003296, -5.363230228424072, 3.366718053817749, 5.135332107543945, -8.345147132873535, 1.389736533164978, 16.331527709960938, -15.747908592224121, 9.848960876464844, 8.855151176452637, -14.632586479187012, -15.810894966125488, -5.355047225952148, 13.533708572387695, -6.113272190093994, 0.386242538690567, 1.21139657497406, -0.7502232789993286, 19.369394302368164, 2.7826781272888184, -11.3154296875, 2.9179813861846924, 6.337240219116211, -10.397686958312988, 1.879101276397705, -5.815656661987305, -12.693501472473145, -8.813064575195312, 7.867149353027344, -1.6220149993896484, -16.03172492980957, 7.059803009033203, -3.021106719970703, -28.748153686523438, -10.617971420288086, -23.089582443237305, -6.149219512939453, 26.791664123535156, 19.831451416015625, 2.6026411056518555, -8.345130920410156, -11.522313117980957, -7.32444953918457, 2.3394362926483154, 12.281377792358398, 0.509806215763092, 12.233405113220215, -1.2431249618530273, -20.459728240966797, 3.1516644954681396, 10.013666152954102, 5.790005207061768, 10.85007095336914, -8.96401309967041, -2.529299736022949, 0.36958760023117065, 3.566554069519043, 5.744760036468506, 9.75991439819336, -3.835606098175049, -10.24438762664795, -4.2324347496032715, -1.8263612985610962, -4.257289886474609, -8.9885892868042, 8.70092487335205, 9.64513111114502, 9.8242769241333, 7.753336429595947, 5.416078090667725, -4.278483867645264, -3.8075523376464844, 2.9572501182556152, -17.95936393737793, 2.546489715576172, -19.199655532836914, 18.096012115478516, 6.311362266540527, -6.033995628356934, -0.11310340464115143, -4.877695560455322, 15.272730827331543, 11.060016632080078, 1.1147111654281616, -4.72451639175415, 6.034030914306641, 2.8547704219818115, -0.14470268785953522, -4.779598712921143, -14.868547439575195, -13.082122802734375, -6.875156402587891, -15.73824691772461, -8.995327949523926, 17.693485260009766, 0.5146524906158447, 7.070795059204102, -5.541579246520996, 13.362793922424316, 1.504472255706787, 8.11085319519043, 11.068685531616211, 4.586761474609375, 19.220623016357422, -8.800450325012207, -2.9444618225097656, 5.154225826263428, -0.6245794296264648, 4.778895854949951, 21.60106658935547, -2.6475870609283447, 2.966681957244873, -4.810849189758301, -7.743277072906494, -18.495075225830078, 4.959153175354004, 13.996840476989746, 0.6328519582748413, 12.144004821777344, -12.758761405944824, -1.5454211235046387, -1.2146533727645874, 7.469117164611816, 0.7183483839035034, 5.531536102294922, 14.722888946533203, -10.635772705078125, -0.9136283993721008, 4.064121723175049, -6.334905624389648, 3.8136820793151855, -17.04436683654785, -9.768339157104492, -2.404874324798584, -17.625619888305664, 21.30498695373535, 18.429473876953125, 12.406076431274414, 8.323545455932617, 9.03734016418457, -2.5080089569091797, -2.150698184967041, -19.941591262817383, 9.213897705078125, -1.7476239204406738, -19.315513610839844, -8.137489318847656, 4.981624126434326, -15.877156257629395, -5.100974082946777, -21.374431610107422, -4.8019700050354, -0.8139694929122925, -23.72612762451172, -7.722938537597656, 4.174458980560303, -3.7926971912384033, 14.308231353759766, 8.371541976928711, 11.151653289794922, -12.287271499633789, -9.39031982421875, 3.0961155891418457, 4.903312683105469, -5.383721351623535, -2.349964141845703, -9.717289924621582, -4.901102066040039, 3.7151238918304443, 13.979822158813477, 5.383389472961426, 8.586429595947266, 0.601891279220581, 4.971793174743652, 9.499808311462402, -8.959651947021484, 9.508575439453125, 5.8362650871276855, 9.934850692749023, 4.317758560180664, -6.656125068664551, -4.3343729972839355, 0.2773263454437256, -24.86429214477539, -8.22954273223877, -6.140705108642578, -5.512542724609375, -9.93578052520752, -10.74149227142334, -2.9830522537231445, -6.41399621963501, 12.586847305297852, -0.407539963722229, -1.0505380630493164, 7.831071853637695, -1.1213947534561157, -0.53873211145401, 11.145183563232422, 9.162491798400879, -2.295194625854492, -13.089630126953125, -6.196512699127197, 5.150289535522461, -16.23560905456543, -3.3018529415130615, -0.726199209690094, -14.884061813354492, 11.492429733276367, -2.6459712982177734, -15.151359558105469, -9.95240306854248, 3.9227020740509033, 13.153590202331543, 2.688488721847534, 1.670224666595459, -2.041313648223877, 15.545082092285156, 12.407716751098633, -15.700468063354492, 0.14021936058998108, 5.521881580352783, -1.9555331468582153, 4.262704372406006, 6.345681667327881, -8.856260299682617, 12.439119338989258, 1.432129144668579, 10.361272811889648, 2.6540687084198, -0.88874351978302, 13.035426139831543, 3.6471824645996094, 15.317996978759766, 5.881680488586426, 21.157493591308594, -7.620250701904297, 0.8054672479629517, -7.4286346435546875, -10.589210510253906, 1.84738290309906, -14.3463134765625, -0.9924871325492859, 12.553890228271484, 5.2023515701293945, -5.227870464324951, -2.4983465671539307, 6.896491527557373, 15.133230209350586, -2.5229172706604004, 1.1562632322311401, 4.707028865814209, -1.837339997291565, 3.410827875137329, 2.5855813026428223, 12.736414909362793, -5.782639503479004, 0.69342440366745, 3.2808897495269775, -7.085940361022949, -3.2794082164764404, -0.2965642213821411, -14.146102905273438, -4.795603275299072, -11.023181915283203, -5.014028549194336, -6.667389392852783, -4.848911285400391, 6.667374610900879, 11.848012924194336, 7.9013543128967285, -7.876648426055908, 3.0120315551757812, -12.357967376708984, 5.291801452636719, 6.963134288787842, -3.4653897285461426, 0.24436426162719727, 7.743401527404785, 6.8888444900512695, -5.530646324157715, -7.2142333984375, -16.408681869506836, -3.489349842071533, -5.934021472930908, -0.2934180200099945, 5.1538825035095215, -13.725027084350586, -7.573877334594727, -3.5734715461730957, -15.685077667236328, -2.1698670387268066, -5.083288192749023, 3.1052117347717285, 6.119309425354004, 6.392850875854492, -2.777722120285034, 2.0242197513580322, 1.6720154285430908, 9.758779525756836, -17.992637634277344, -2.7406539916992188, -7.657975196838379, 0.6749216318130493, 9.690409660339355, 1.7074265480041504, 0.9763889312744141, -7.054371356964111, 2.7263875007629395, 1.9292343854904175, -9.956684112548828, 5.7603349685668945, -4.805788993835449, -20.683300018310547, -2.177995443344116, 4.45976448059082, 3.9116122722625732, 0.8401883244514465, -0.8823317289352417, 5.912646293640137, 5.050131797790527, -17.956954956054688, -7.368210792541504, 0.461815744638443, -1.158712387084961, 7.477179527282715, 9.185932159423828, 0.9992852807044983, -4.067198753356934, 16.072803497314453, -8.565892219543457, -1.3972474336624146, 19.099300384521484, 5.701394557952881, 5.404655933380127, -0.40618225932121277, 11.245012283325195, 27.757856369018555, 2.769061326980591, -2.4795148372650146, 10.96125602722168, 18.099576950073242, 3.31923508644104, -8.170138359069824, 3.1348001956939697, -7.741110801696777, 9.081361770629883, -2.170832633972168, 4.84144926071167, 0.5936726331710815, 7.773089408874512, -19.6074275970459, 3.044421911239624, 1.6888775825500488, 2.6412734985351562, -14.100894927978516, 12.924214363098145, -6.424345016479492, 4.3242645263671875, -5.1252121925354, -2.08510684967041, -4.499283790588379, -15.755219459533691, -6.33029842376709, 12.478150367736816, -11.093277931213379, 1.0680733919143677, -17.084426879882812, 10.347875595092773, -13.42613410949707, -11.297205924987793, -1.6252909898757935, 5.106625556945801, -12.761529922485352, -1.0382558107376099, 5.077084541320801, 0.019973328337073326, -7.504131317138672, 4.225093841552734, 10.687294006347656, 22.26036262512207, 12.366414070129395, -16.47020721435547, 12.492666244506836, 9.947951316833496, 2.91593074798584, 6.700890064239502, 10.856518745422363, -2.4789068698883057, -11.86021614074707, -2.483541965484619, 6.486255645751953, -5.368561744689941, -6.242945671081543, -12.191120147705078, 5.268315315246582, -10.29853630065918, -1.9759645462036133, -6.972436428070068, 12.693361282348633, 9.857770919799805, 22.412540435791016, -6.689547538757324, -9.418767929077148, 2.6893184185028076, -13.23387336730957, -4.693850994110107, 6.715462684631348, 0.6460093259811401, 8.389328956604004, -7.092681407928467, -4.414528846740723, -1.1874971389770508, 8.289121627807617, 10.898362159729004, 11.9028902053833, 1.1823748350143433, -6.857298851013184, -3.897196054458618, 0.20892687141895294, -14.825176239013672, -8.921123504638672, -16.745208740234375, 2.9534246921539307, 2.273578643798828, -8.812061309814453, 9.413691520690918, 1.0422091484069824, -1.088782787322998, 6.606812953948975, 3.4164836406707764, -10.109027862548828, -6.370138645172119, -13.740169525146484, 6.069772243499756, -0.9287220239639282, 8.929533004760742, -6.731569290161133, -11.371563911437988, 10.920660018920898, 8.20090389251709, -13.984733581542969, 10.357625961303711, -6.829133033752441, 14.766778945922852, 4.975728511810303, -5.784604549407959, -2.180842876434326, -1.4952462911605835, -28.40904426574707, 6.649030685424805, -2.8774356842041016, 7.39600133895874, -27.259624481201172, 7.000762939453125, 0.3766692876815796, -10.806390762329102, 19.197830200195312, 8.084424018859863, -8.961612701416016, -7.664973735809326, 5.792583465576172, 17.449779510498047, 5.1670684814453125, -4.0314836502075195, 17.63963508605957, -5.608425140380859, 12.999395370483398, 6.742602348327637, -1.0125733613967896, -5.660837173461914, -5.327735900878906, -0.03256353735923767, 13.614930152893066, -0.4054352343082428, -8.319563865661621, 13.180963516235352, 10.218942642211914, 1.058619737625122, 12.317801475524902, 2.0810399055480957, 17.973726272583008, -0.7691915035247803, -6.874382495880127, -4.010448455810547, 1.4536858797073364, 1.7113529443740845, -5.252817153930664, -4.829272747039795, -12.437076568603516, -5.025095462799072, 15.999751091003418, -1.6669206619262695, -9.314016342163086, -6.326523780822754, -7.3243818283081055, -1.0183708667755127, -8.13725757598877, 11.383604049682617, -8.033729553222656, -2.4610519409179688, 14.226714134216309, -0.26943185925483704, 12.279324531555176, -1.8395049571990967, 7.69477653503418, 21.11790657043457, -8.896676063537598, 5.152464389801025, -10.646064758300781, -0.32607975602149963, -6.496189117431641, 1.208950161933899, -11.623730659484863, -20.607341766357422, 5.620739936828613, 1.012706995010376, 12.599133491516113, 1.9369388818740845, 0.969719409942627, -8.405143737792969, -13.44587230682373, 1.3203836679458618, -11.425366401672363, 13.497820854187012, 13.90512466430664, 5.234240531921387, 11.919998168945312, -0.8153055906295776, -5.2424187660217285, 23.606260299682617, -2.3769564628601074, 12.494120597839355, -7.87453556060791, 3.942775249481201, 8.378625869750977, -16.401962280273438, -14.80517578125, -6.9582695960998535, -3.5369160175323486, 0.5067298412322998, -7.463751792907715, -7.329478740692139, -10.243635177612305, -10.20290470123291, 3.5215678215026855, -5.182548522949219, -10.77608871459961, -11.637520790100098, -7.953490734100342, -15.931358337402344, 1.836498737335205, 3.823019504547119, -13.222779273986816, -5.463068008422852, -3.7494611740112305, 7.715019702911377, -7.45162296295166, -3.4903616905212402, -10.27260684967041, -9.680792808532715, -6.692418098449707, 32.16154479980469, 10.94521713256836, 3.3682010173797607, 2.5896506309509277, -15.174675941467285, -9.740742683410645, 7.332023620605469, 0.1277514547109604, 10.77327823638916, 0.04945862665772438, 5.3694891929626465, 1.115238904953003, 5.231359004974365, 3.656858444213867, 10.768354415893555, -5.149155139923096, -2.221626043319702, -6.7708258628845215, -0.11192019283771515, 4.514595031738281, 0.7386126518249512, -19.202268600463867, -6.890750408172607, -5.335263729095459, -3.250162124633789, -6.323789596557617, -3.6589298248291016, 8.981910705566406, 5.844911575317383, 14.006356239318848, -16.068389892578125, -10.360090255737305, -12.4788236618042, 9.89712142944336, 6.947521209716797, -5.739221572875977, -5.343330383300781, 2.6730802059173584, 12.964028358459473, 15.272317886352539, -6.343968391418457, 8.853652000427246, -1.8664456605911255, 15.798284530639648, 25.740434646606445, 5.4787211418151855, 6.094010353088379, -25.407869338989258, 5.653440952301025, -16.886533737182617, -5.860007286071777, 8.633628845214844, -15.704465866088867, 23.380512237548828, -5.351648330688477, 7.78794527053833, -5.348761081695557, 6.90106201171875, -5.7505340576171875, -0.531606912612915, -23.084484100341797, 9.556772232055664, 6.838652610778809, -3.446183443069458, -3.068847179412842, -10.840062141418457, -7.3312788009643555, 2.3541929721832275, 2.9895246028900146, 7.289599895477295, 11.492620468139648, 9.884173393249512, 4.050896167755127, 3.428751230239868, -1.9773675203323364, 12.304611206054688, -7.235345840454102, -7.840249061584473, 3.4171814918518066, -14.192761421203613, -3.128744602203369, -15.032843589782715, -8.497732162475586, 34.775146484375, -16.289003372192383, 9.839770317077637, -5.905306339263916, -11.38370418548584, -14.802390098571777, 6.03196382522583, -3.7947568893432617, 0.09253554791212082, 16.361371994018555, -13.57385540008545, -21.236141204833984, 1.2763501405715942, 13.585625648498535, 1.5973198413848877, 13.936482429504395, 11.294212341308594, 3.4961602687835693, -7.4443793296813965, 1.5526808500289917, 11.498977661132812, 5.135770797729492, 0.04641003906726837, 4.001218795776367, 0.09547155350446701, 8.390353202819824, -2.0528368949890137, 16.49645233154297, 4.227134704589844, 2.147636651992798, -17.358619689941406, -21.086078643798828, -1.4869649410247803, -4.163187026977539, 19.437301635742188, -7.103264808654785, -9.341580390930176, 8.141914367675781, -20.010435104370117, 0.9820413589477539, -12.762100219726562, -27.60112953186035, -17.880891799926758, -1.511062502861023, -0.17270037531852722, -2.7366137504577637, -9.28756332397461, 0.31258106231689453, 2.52874755859375, 0.30306103825569153, 15.897100448608398, 9.341832160949707, -9.510763168334961, -6.829599857330322, -9.697000503540039, 1.0961319208145142, 9.884286880493164, 15.291708946228027, 1.3411988019943237, -6.38883638381958, -10.926775932312012, -16.607295989990234, 16.834653854370117, -3.7307214736938477, -5.331791877746582, 17.933828353881836, -6.582093715667725, -1.1567832231521606, 4.044549465179443, -18.525461196899414, -13.59295654296875, 32.169857025146484, 3.495553493499756, -19.676607131958008, -5.855699062347412, -1.0901000499725342, 5.933902740478516, -1.0290929079055786, -11.749110221862793, -0.039907217025756836, 11.03833293914795, -5.984804153442383, -11.467243194580078, -10.895991325378418, -22.909448623657227, 12.322643280029297, 3.267489433288574, -0.9058361649513245, 1.693793773651123, 12.444228172302246, -3.7365598678588867, -13.185384750366211, 10.847840309143066, -22.834964752197266, -3.5510098934173584, 11.643357276916504, 15.284043312072754, -0.7085658311843872, -0.8153420686721802, -1.2313066720962524, -2.633232831954956, -1.1130733489990234, 13.857254028320312, 4.907212734222412, 17.084749221801758, 5.685029029846191, -3.585721492767334, -2.6217451095581055, 21.78135108947754, -6.028371334075928, 10.331937789916992, -9.473434448242188, 5.535529136657715], "dtype": "float32"}, {"__nd__": [0.2660057544708252, 0.4236275851726532, 0.3506370782852173, 0.924126148223877, 0.7792121767997742, 0.10881917923688889, 0.04100978747010231, 0.5438162088394165, 0.7910699248313904, 0.10795047134160995, 0.7704317569732666, 0.3609465956687927, 0.7406755089759827, 0.04420463368296623, 0.24603953957557678, 0.7487453818321228, 0.10730692744255066, 0.9708568453788757, 0.08647787570953369, 0.09200616180896759, 0.4307032525539398, 0.9007400274276733, 0.35369643568992615, 0.13562896847724915, 0.10926263779401779, 0.5825526118278503, 0.08596640825271606, 0.8369262218475342, 0.8101494312286377, 0.8154385685920715, 0.322940468788147, 0.3835891783237457, 0.7611138820648193, 0.7423444390296936, 0.3943481147289276, 0.748779296875, 0.9483068585395813, 0.8177439570426941, 0.8216028809547424, 0.6336407661437988, 0.38204872608184814, 0.6844070553779602, 0.16675272583961487, 0.9152923226356506, 0.6682839393615723, 0.22547848522663116, 0.0876694843173027, 0.1639522761106491, 0.9506192207336426, 0.7959543466567993, 0.8921461701393127, 0.3623059391975403, 0.26570767164230347, 0.19937098026275635, 0.3438121974468231, 0.012284043245017529, 0.5768453478813171, 0.40337637066841125, 0.81611567735672, 0.41072937846183777, 0.5076016783714294, 0.2892817258834839, 0.7240882515907288, 0.33254775404930115, 0.45432963967323303, 0.8193860650062561, 0.7610824704170227, 0.7920973896980286, 0.5761289596557617, 0.12865714728832245, 0.0874449759721756, 0.8632888793945312, 0.9780890345573425, 0.6359643936157227, 0.2631731331348419, 0.2059372067451477, 0.2391381412744522, 0.9081019759178162, 0.36522629857063293, 0.7092500329017639, 0.9315798282623291, 0.9090322256088257, 0.8907677531242371, 0.34116509556770325, 0.2908388078212738, 0.7174059748649597, 0.08014374226331711, 0.3217226564884186, 0.31510934233665466, 0.8252763152122498, 0.9942747354507446, 0.704056441783905, 0.6482495069503784, 0.23275022208690643, 0.40546274185180664, 0.5381215214729309, 0.19603754580020905, 0.7727921605110168, 0.5191839337348938, 0.876227080821991, 0.9701958894729614, 0.18775911629199982, 0.606905460357666, 0.7106768488883972, 0.1953921616077423, 0.4438197612762451, 0.6870137453079224, 0.12202213704586029, 0.0973527655005455, 0.4593282639980316, 0.4341605603694916, 0.8098678588867188, 0.11992866545915604, 0.23778176307678223, 0.4960147738456726, 0.7904883623123169, 0.4980132281780243, 0.6630871295928955, 0.2734590768814087, 0.5057179927825928, 0.38642990589141846, 0.9177792072296143, 0.6001386046409607, 0.5320951342582703, 0.9467105865478516, 0.1466452032327652, 0.36586594581604004, 0.9432383179664612, 0.9666717648506165, 0.34303733706474304, 0.8270103931427002, 0.9622431993484497, 0.34730347990989685, 0.08714896440505981, 0.2367742359638214, 0.9962295293807983, 0.8143619298934937, 0.09560010582208633, 0.8661293983459473, 0.2643122375011444, 0.2859843671321869, 0.6736975312232971, 0.2583792805671692, 0.44031617045402527, 0.1305869221687317, 0.2504851520061493, 0.3360730707645416, 0.8889848589897156, 0.1984710544347763, 0.02345944382250309, 0.9501971006393433, 0.22706098854541779, 0.38314393162727356, 0.38859203457832336, 0.6246010661125183, 0.9153117537498474, 0.5299570560455322, 0.8490588665008545, 0.2756374180316925, 0.5501114726066589, 0.5749443173408508, 0.20349113643169403, 0.963895320892334, 0.0852411687374115, 0.45343247056007385, 0.5186339020729065, 0.9821731448173523, 0.7833596467971802, 0.33594903349876404, 0.2513759434223175, 0.534990668296814, 0.4937661588191986, 0.9094027280807495, 0.12502546608448029, 0.900809645652771, 0.5136004090309143, 0.30958253145217896, 0.15048092603683472, 0.21154814958572388, 0.32357537746429443, 0.8892254829406738, 0.6816752552986145, 0.5456405282020569, 0.4200378954410553, 0.16631585359573364, 0.19897128641605377, 0.09841126948595047, 0.48528969287872314, 0.5072800517082214, 0.1525077521800995, 0.1115533709526062, 0.24450597167015076, 0.5562695860862732, 0.7280417084693909, 0.5528743863105774, 0.3470895290374756, 0.38102009892463684, 0.7952665686607361, 0.5207890272140503, 0.6939122676849365, 0.18558873236179352, 0.2969149351119995, 0.27734917402267456, 0.2114451378583908, 0.9836667776107788, 0.3274037539958954, 0.8448178172111511, 0.10437648743391037, 0.0839298814535141, 0.8753098845481873, 0.9093000888824463, 0.7295790314674377, 0.6298680901527405, 0.18407991528511047, 0.626954197883606, 0.22187626361846924, 0.5888833999633789, 0.7587122917175293, 0.14614902436733246, 0.6319961547851562, 0.7335208654403687, 0.32575613260269165, 0.3072481155395508, 0.5545536279678345, 0.6787793040275574, 0.9916217923164368, 0.6712506413459778, 0.24986623227596283, 0.7686812877655029, 0.0032970556057989597, 0.14807122945785522, 0.5936108827590942, 0.8522434830665588, 0.10597949475049973, 0.32787394523620605, 0.11703696101903915, 0.21224166452884674, 0.26909250020980835, 0.8244725465774536, 0.7230149507522583, 0.5345615744590759, 0.6873986124992371, 0.775684654712677, 0.6606000065803528, 0.2977618873119354, 0.1544136255979538, 0.9712681174278259, 0.149460569024086, 0.9848875999450684, 0.31326737999916077, 0.023935671895742416, 0.3661957383155823, 0.18496830761432648, 0.7195845246315002, 0.8589193224906921, 0.514030396938324, 0.3141162693500519, 0.7903642654418945, 0.2781018912792206, 0.4146733582019806, 0.9022590517997742, 0.8866438865661621, 0.9952213764190674, 0.7271355986595154, 0.280984491109848, 0.980461061000824, 0.4155435264110565, 0.7022086977958679, 0.7222272157669067, 0.26031479239463806, 0.35064324736595154, 0.3828156292438507, 0.7232757806777954, 0.9629202485084534, 0.47905293107032776, 0.014959724619984627, 0.5186759233474731, 0.04232507944107056, 0.6553053855895996, 0.8984163999557495, 0.5712012648582458, 0.5771833658218384, 0.8377147316932678, 0.5774445533752441, 0.17667530477046967, 0.9270405769348145, 0.21149185299873352, 0.757696270942688, 0.12205229699611664, 0.578681468963623, 0.1853468418121338, 0.49194464087486267, 0.481285035610199, 0.7375932931900024, 0.19312553107738495, 0.8057684898376465, 0.33411890268325806, 0.8301452398300171, 0.7685611844062805, 0.054797518998384476, 0.9434783458709717, 0.8262091875076294, 0.38400524854660034, 0.41295138001441956, 0.8524997234344482, 0.17535459995269775, 0.0443299263715744, 0.7671601176261902, 0.9643427729606628, 0.22739802300930023, 0.11530393362045288, 0.8112156987190247, 0.08349381387233734, 0.10143305361270905, 0.2777917981147766, 0.8048739433288574, 0.6123856902122498, 0.8541147708892822, 0.5748178362846375, 0.235418900847435, 0.6178773641586304, 0.47599709033966064, 0.12291369587182999, 0.8833335041999817, 0.8521943092346191, 0.6478726863861084, 0.5607239603996277, 0.3831326365470886, 0.4669809639453888, 0.14078180491924286, 0.8874576091766357, 0.7950994968414307, 0.2693604528903961, 0.6896816492080688, 0.05889860540628433, 0.44437676668167114, 0.7309287190437317, 0.8642504811286926, 0.8141197562217712, 0.39717546105384827, 0.8719901442527771, 0.36675727367401123, 0.4001391530036926, 0.048601288348436356, 0.1722145974636078, 0.52689528465271, 0.5391779541969299, 0.8454110026359558, 0.7665382027626038, 0.1853366494178772, 0.9835451245307922, 0.06368239969015121, 0.4560562074184418, 0.5503436326980591, 0.538657546043396, 0.11590007692575455, 0.19343778491020203, 0.0636284127831459, 0.07348250597715378, 0.6170181035995483, 0.3480835258960724, 0.2606629431247711, 0.0372074693441391, 0.9770415425300598, 0.8283706903457642, 0.22942495346069336, 0.0568021722137928, 0.027843382209539413, 0.20149926841259003, 0.41865795850753784, 0.7821207046508789, 0.5685088038444519, 0.7600597739219666, 0.08914248645305634, 0.375448614358902, 0.5350641012191772, 0.9867293834686279, 0.8006148934364319, 0.9533541202545166, 0.6553550362586975, 0.5498248338699341, 0.7067720890045166, 0.6762089729309082, 0.15561605989933014, 0.5577974915504456, 0.43644922971725464, 0.1525396704673767, 0.5332401990890503, 0.9530033469200134, 0.2138281613588333, 0.5749040842056274, 0.7026787400245667, 0.48240455985069275, 0.7034257054328918, 0.6613476276397705, 0.6758074164390564, 0.28222009539604187, 0.8081203103065491, 0.33214670419692993, 0.8897677659988403, 0.011132202111184597, 0.7126882672309875, 0.8412176370620728, 0.9784055352210999, 0.3601936995983124, 0.8511734008789062, 0.7327337265014648, 0.7495288848876953, 0.9307087063789368, 0.6170647144317627, 0.591137170791626, 0.8413367867469788, 0.7520758509635925, 0.05689668282866478, 0.9933019876480103, 0.944561779499054, 0.15202102065086365, 0.9245759844779968, 0.4431169927120209, 0.29240933060646057, 0.9674908518791199, 0.7172051072120667, 0.6334076523780823, 0.4296133518218994, 0.8951914310455322, 0.6540270447731018, 0.47602763772010803, 0.5972194075584412, 0.07913557440042496, 0.368334025144577, 0.7711092233657837, 0.05085229128599167, 0.6583399176597595, 0.18343716859817505, 0.6780135035514832, 0.9695332646369934, 0.4191122055053711, 0.875123918056488, 0.9194977283477783, 0.15264685451984406, 0.16324147582054138, 0.06749969720840454, 0.8774991631507874, 0.21665345132350922, 0.525446355342865, 0.27255359292030334, 0.9564409852027893, 0.03524268418550491, 0.8457226753234863, 0.47362738847732544, 0.7698276042938232, 0.5174761414527893, 0.2695475220680237, 0.8043872117996216, 0.23516413569450378, 0.07366207987070084, 0.2850838005542755, 0.3679588735103607, 0.06035705283284187, 0.38910555839538574, 0.008856162428855896, 0.9560197591781616, 0.30807337164878845, 0.466480553150177, 0.7993155121803284, 0.85443514585495, 0.05560808628797531, 0.36803731322288513, 0.889610230922699, 0.7786266803741455, 0.9160374402999878, 0.034805696457624435, 0.10728710889816284, 0.870404064655304, 0.8100001811981201, 0.34183990955352783, 0.017225265502929688, 0.3288067579269409, 0.6050236821174622, 0.19954045116901398, 0.26887521147727966, 0.522609293460846, 0.060108233243227005, 0.6436300277709961, 0.8647616505622864, 0.7936848998069763, 0.8052420020103455, 0.3704277575016022, 0.94818514585495, 0.9915753602981567, 0.3337843418121338, 0.433212012052536, 0.5057016015052795, 0.636756181716919, 0.3710307776927948, 0.8772314190864563, 0.9717168211936951, 0.30379247665405273, 0.9790176749229431, 0.9696781635284424, 0.4765324890613556, 0.5346453189849854, 0.8709740042686462, 0.5488941669464111, 0.8508033156394958, 0.8895717263221741, 0.034849245101213455, 0.9481601715087891, 0.24553737044334412, 0.09908463060855865, 0.6296212077140808, 0.106671541929245, 0.3647804260253906, 0.7015945315361023, 0.4307307004928589, 0.31234684586524963, 0.2151511162519455, 0.7855563163757324, 0.4276227355003357, 0.539506196975708, 0.11033280193805695, 0.19043686985969543, 0.11164207011461258, 0.8668954372406006, 0.7607191205024719, 0.8307375907897949, 0.9915419220924377, 0.7408565878868103, 0.8155269622802734, 0.84793621301651, 0.782015323638916, 0.9423034191131592, 0.8772864937782288, 0.41007065773010254, 0.9902739524841309, 0.9151018857955933, 0.49755820631980896, 0.7393563389778137, 0.8625404238700867, 0.8931896090507507, 0.3960830867290497, 0.13984400033950806, 0.6739130020141602, 0.6007572412490845, 0.5591346621513367, 0.6803200244903564, 0.5928894877433777, 0.6025978922843933, 0.656731367111206, 0.8691487908363342, 0.024286914616823196, 0.06208620220422745, 0.2730851173400879, 0.32960861921310425, 0.15385177731513977, 0.3172612488269806, 0.3855755925178528, 0.6348252296447754, 0.4549565315246582, 0.6981558799743652, 0.2759081721305847, 0.26672667264938354, 0.12709350883960724, 0.3325127065181732, 0.981511652469635, 0.31589946150779724, 0.7250937819480896, 0.4674178659915924, 0.7729999423027039, 0.2752753794193268, 0.4209360182285309, 0.00544150872156024, 0.6243379712104797, 0.12845709919929504, 0.4641348123550415, 0.11100391298532486, 0.7271870374679565, 0.7787081003189087, 0.8892343044281006, 0.8049025535583496, 0.00940952729433775, 0.765673816204071, 0.2868260145187378, 0.26153022050857544, 0.194157674908638, 0.34015336632728577, 0.25981977581977844, 0.08481919765472412, 0.6634452939033508, 0.6634200811386108, 0.7651416063308716, 0.3844802975654602, 0.3774614632129669, 0.8387318253517151, 0.9310141205787659, 0.11851514130830765, 0.7559093832969666, 0.8463473320007324, 0.528607189655304, 0.7488060593605042, 0.7700785398483276, 0.9510164260864258, 0.8729559779167175, 0.8553409576416016, 0.7014429569244385, 0.346417099237442, 0.10638006031513214, 0.44727012515068054, 0.22909429669380188, 0.2899360656738281, 0.2914639413356781, 0.0002976774994749576, 0.3416160047054291, 0.7751246094703674, 0.4383905231952667, 0.717795193195343, 0.7010199427604675, 0.7340289354324341, 0.041960347443819046, 0.49573999643325806, 0.8031290769577026, 0.6985761523246765, 0.7241753935813904, 0.9828311204910278, 0.08100529760122299, 0.4832932651042938, 0.4041139483451843, 0.19703936576843262, 0.2131621390581131, 0.16130776703357697, 0.30426138639450073, 0.034469276666641235, 0.7026517987251282, 0.9395295977592468, 0.4671134054660797, 0.20818737149238586, 0.3903975486755371, 0.33864831924438477, 0.28117454051971436, 0.8891060948371887, 0.9025325775146484, 0.9525701999664307, 0.24887821078300476, 0.819628894329071, 0.6772492527961731, 0.9745120406150818, 0.21261362731456757, 0.22539128363132477, 0.3442572355270386, 0.03334386274218559, 0.844377338886261, 0.6405439376831055, 0.25675666332244873, 0.5180960297584534, 0.33668357133865356, 0.6053627729415894, 0.6366696953773499, 0.2657368779182434, 0.5502299070358276, 0.4392865300178528, 0.5266090035438538, 0.20595179498195648, 0.23991335928440094, 0.7724243402481079, 0.7774140238761902, 0.31545549631118774, 0.2569585144519806, 0.11860884726047516, 0.7186013460159302, 0.7901931405067444, 0.481691837310791, 0.3666960597038269, 0.1128326952457428, 0.3268326222896576, 0.5492525100708008, 0.8967331647872925, 0.9652938842773438, 0.5440027713775635, 0.5002631545066833, 0.4643222391605377, 0.047206662595272064, 0.3597635328769684, 0.469332218170166, 0.1981952041387558, 0.38874006271362305, 0.3231460750102997, 0.022256188094615936, 0.9882652163505554, 0.5324826836585999, 0.5788266062736511, 0.5072566866874695, 0.9080799221992493, 0.6120731830596924, 0.39089247584342957, 0.2782159447669983, 0.202717587351799, 0.622094988822937, 0.9023083448410034, 0.8308289051055908, 0.07499327510595322, 0.9638260006904602, 0.6629894375801086, 0.2838289737701416, 0.38229015469551086, 0.6468256115913391, 0.7079780697822571, 0.1915561854839325, 0.21937774121761322, 0.698677659034729, 0.22964926064014435, 0.37349507212638855, 0.3967827558517456, 0.8164372444152832, 0.5140026211738586, 0.5273500084877014, 0.3149856626987457, 0.5935072302818298, 0.6845498085021973, 0.44334375858306885, 0.5133214592933655, 0.4285378158092499, 0.21706172823905945, 0.9666991233825684, 0.7425498366355896, 0.8372103571891785, 0.5941228866577148, 0.28810566663742065, 0.017916137352585793, 0.9564781188964844, 0.9722161293029785, 0.22738656401634216, 0.6015304327011108, 0.18206669390201569, 0.4334258437156677, 0.49283739924430847, 0.5456257462501526, 0.3221963047981262, 0.10472078621387482, 0.834112823009491, 0.9691426753997803, 0.688953161239624, 0.49444350600242615, 0.2368437498807907, 0.758697509765625, 0.5194975137710571, 0.6108176708221436, 0.08864790946245193, 0.4782608151435852, 0.7050712704658508, 0.7115392684936523, 0.13176095485687256, 0.22654078900814056, 0.5261735916137695, 0.7344887852668762, 0.5896318554878235, 0.99440598487854, 0.5197155475616455, 0.9834617972373962, 0.22310499846935272, 0.6985249519348145, 0.17139115929603577, 0.20233607292175293, 0.9231653809547424, 0.8735843896865845, 0.05254059657454491, 0.20746250450611115, 0.18265999853610992, 0.08976569026708603, 0.933799147605896, 0.6133856773376465, 0.318203866481781, 0.6003668308258057, 0.9702737927436829, 0.7589778304100037, 0.7731133699417114, 0.047735027968883514, 0.6345204710960388, 0.3664100766181946, 0.10783668607473373, 0.7223960757255554, 0.270779550075531, 0.2889990210533142, 0.39914101362228394, 0.1322382390499115, 0.9312881827354431, 0.025885622948408127, 0.6991064548492432, 0.7065218687057495, 0.6936691403388977, 0.6690081357955933, 0.5059058666229248, 0.33113324642181396, 0.8795977830886841, 0.5125837326049805, 0.21920877695083618, 0.46081650257110596, 0.4606691300868988, 0.12062907963991165, 0.6012259125709534, 0.9355900883674622, 0.7108470797538757, 0.6801722645759583, 0.11552251875400543, 0.7265963554382324, 0.8505553007125854, 0.6552565693855286, 0.5131726861000061, 0.6687400937080383, 0.22065655887126923, 0.13462740182876587, 0.10396449267864227, 0.07242493331432343, 0.8554644584655762, 0.8635722994804382, 0.4402865469455719, 0.34497517347335815, 0.30940526723861694, 0.18197956681251526, 0.6452114582061768, 0.3591403067111969, 0.07192216068506241, 0.30899855494499207, 0.7398064732551575, 0.27467048168182373, 0.7626494765281677, 0.2861131727695465, 0.5350988507270813, 0.48067420721054077, 0.3234449028968811, 0.8575061559677124, 0.8777509331703186, 0.56854248046875, 0.13627254962921143, 0.42704880237579346, 0.006110758055001497, 0.5479149222373962, 0.3457580506801605, 0.9262843132019043, 0.3330475091934204, 0.030801331624388695, 0.7799851298332214, 0.9033157825469971, 0.5175591707229614, 0.6881402730941772, 0.9302392601966858, 0.5976302027702332, 0.2354365736246109, 0.6152705550193787, 0.8780304789543152, 0.0741712749004364, 0.5543027520179749, 0.6992098689079285, 0.08031800389289856, 0.5782570838928223, 0.1168767511844635, 0.07463879883289337, 0.8432045578956604, 0.8333316445350647, 0.04798069968819618, 0.47173887491226196, 0.02080240100622177, 0.4538518488407135, 0.21783065795898438, 0.34028691053390503, 0.44799119234085083, 0.8519273996353149, 0.38439738750457764, 0.8570087552070618, 0.3683239221572876, 0.3045595586299896, 0.6196480393409729, 0.010530217550694942, 0.1322997361421585, 0.09445501118898392, 0.6278339624404907, 0.617018461227417, 0.19824160635471344, 0.3793453872203827, 0.41804608702659607, 0.23335234820842743, 0.4257863759994507, 0.7736108303070068, 0.23162512481212616, 0.5785799026489258, 0.9539305567741394, 0.8294884562492371, 0.3269427418708801, 0.49136829376220703, 0.12098058313131332, 0.8346433639526367, 0.4464397430419922, 0.9907945990562439, 0.9213454723358154, 0.26518014073371887, 0.8090763688087463, 0.6211034655570984, 0.33606866002082825, 0.24495932459831238, 8.222682663472369e-05, 0.008477546274662018, 0.6613810062408447, 0.4407716393470764, 0.37091299891471863, 0.21716463565826416, 0.8447997570037842, 0.23444834351539612, 0.550999641418457, 0.205527201294899, 0.336372971534729, 0.763994574546814, 0.8719373345375061, 0.21099916100502014, 0.18537817895412445, 0.6652551293373108, 0.8376200199127197, 0.12345091998577118, 0.6988412737846375, 0.04711225628852844, 0.14200322329998016, 0.9712839722633362, 0.067088782787323, 0.4151029884815216, 0.3150041997432709, 0.5949665904045105, 0.5390620827674866, 0.632023274898529, 0.5768275260925293, 0.22620642185211182, 0.42441526055336, 0.14514605700969696, 0.11808124929666519, 0.08239220827817917, 0.9761393666267395, 0.7503687739372253, 0.1763201653957367, 0.4856750965118408, 0.0008486761362291873, 0.6655470132827759, 0.2925952672958374, 0.8695288896560669, 0.6584517359733582, 0.9140321612358093, 0.8901126980781555, 0.5475007891654968, 0.11491897702217102, 0.44926634430885315, 0.12261654436588287, 0.08535995334386826, 0.7269819378852844, 0.9775198698043823, 0.9856923818588257, 0.8515381217002869, 0.66353440284729, 0.8740543723106384, 0.8172867298126221, 0.8193755745887756, 0.7152550220489502, 0.5717669725418091, 0.3998197019100189, 0.189031183719635, 0.6301335096359253, 0.46365174651145935, 0.9797085523605347, 0.8992190957069397, 0.9783174395561218, 0.1558716595172882, 0.7947774529457092, 0.06579779088497162, 0.10016164183616638, 0.16102787852287292, 0.6889259815216064, 0.7044187188148499, 0.44319137930870056, 0.2818930149078369, 0.13576027750968933, 0.08669428527355194, 0.3862282633781433, 0.16212841868400574, 0.22399753332138062, 0.2077665627002716, 0.7674604654312134, 0.04483041912317276, 0.11046364158391953, 0.19105243682861328, 0.491580992937088, 0.20815040171146393, 0.8336350321769714, 0.37680816650390625, 0.7323893308639526, 0.3024040460586548, 0.1289817839860916, 0.2580777406692505, 0.14174899458885193, 0.7329601049423218, 0.9865829348564148, 0.0799698680639267, 0.3270719349384308, 0.2791634500026703, 0.11478114873170853, 0.24365785717964172, 0.8607842922210693, 0.8860176205635071, 0.4680699110031128, 0.5168023109436035, 0.9737450480461121, 0.6375383138656616, 0.39100974798202515, 0.7032663226127625, 0.4298115670681, 0.5504629015922546, 0.9724414348602295, 0.2814900577068329, 0.5840701460838318, 0.9190154075622559, 0.675412118434906, 0.9716296195983887, 0.4155910015106201, 0.15416093170642853, 0.2247622162103653, 0.6623491644859314, 0.4734557569026947, 0.9462883472442627, 0.3050490617752075, 0.52238529920578, 0.4264751374721527, 0.0887916311621666, 0.6902790665626526, 0.7587624192237854, 0.7028307914733887, 0.6198051571846008, 0.7624268531799316, 0.5150243043899536, 0.5490632057189941, 0.14444151520729065, 0.01352651510387659, 0.6718292236328125, 0.3070935606956482, 0.08024746924638748, 0.3472537696361542, 0.6508254408836365, 0.8956190347671509, 0.7537306547164917, 0.8093793392181396, 0.2937842309474945, 0.21076925098896027, 0.011941472068428993, 0.9593214392662048, 0.980518639087677, 0.356092631816864, 0.34708938002586365, 0.6774687170982361, 0.8581849336624146, 0.0183004979044199, 0.554546594619751, 0.8566538095474243, 0.4482252895832062, 0.8755224943161011, 0.5405365824699402, 0.41715705394744873, 0.6104143857955933, 0.8567091822624207, 0.5157593488693237, 0.40559837222099304, 0.1940942406654358, 0.6930460333824158, 0.40499481558799744, 0.3144044280052185, 0.22695985436439514, 0.22339114546775818, 0.34848567843437195, 0.737480103969574, 0.7995271682739258, 0.16255557537078857, 0.7859286665916443, 0.6754563450813293, 0.30815452337265015, 0.30607423186302185, 0.9518287777900696, 0.5407779216766357, 0.8730341792106628, 0.4826352298259735, 0.04827602952718735, 0.41080722212791443, 0.4358096122741699, 0.4629516303539276, 0.8653131723403931, 0.4727345407009125, 0.18525874614715576, 0.8789411187171936, 0.15924011170864105, 0.12807919085025787, 0.3127606213092804, 0.08654256910085678, 0.4312642216682434, 0.3046952784061432, 0.8973302841186523, 0.6450260281562805, 0.45578378438949585, 0.8636833429336548, 0.4934898614883423, 0.8814685344696045, 0.602888286113739, 0.5106761455535889, 0.736255407333374, 0.5052317380905151, 0.11876410990953445, 0.3162735402584076, 0.17673391103744507, 0.29888904094696045, 0.9590402245521545, 0.2763476073741913, 0.18856550753116608, 0.5289380550384521, 0.340884268283844, 0.9700745344161987, 0.7842433452606201, 0.9037424325942993, 0.26192060112953186, 0.2277822345495224, 0.3811441659927368, 0.6877338290214539, 0.17779482901096344, 0.7269608378410339, 0.5096268653869629, 0.5254271030426025, 0.7680502533912659, 0.27185148000717163, 0.7585548758506775, 0.09664168208837509, 0.9318540692329407, 0.8838587403297424, 0.8122084736824036, 0.26390954852104187, 0.5293475985527039, 0.9491174817085266, 0.667970597743988, 0.9776396751403809, 0.11803331971168518, 0.1261272132396698, 0.016108794137835503, 0.6279722452163696, 0.8304476737976074, 0.25101733207702637, 0.832628607749939, 0.4104880392551422, 0.5541069507598877, 0.19530265033245087, 0.02648063935339451, 0.5336288213729858, 0.3084087371826172, 0.9870533347129822, 0.8062153458595276, 0.9812460541725159, 0.24628077447414398, 0.7785658240318298, 0.9786414504051208, 0.2106274515390396, 0.3466251492500305, 0.6028765439987183, 0.8296887278556824, 0.051784686744213104, 0.4198073446750641, 0.16761726140975952, 0.4148915708065033, 0.08509428054094315, 0.9092403054237366, 0.19666962325572968, 0.960835337638855, 0.7478723526000977, 0.9577791690826416, 0.20565947890281677, 0.22398047149181366, 0.49181655049324036, 0.2513585090637207, 0.7102336883544922, 0.0637887492775917, 0.14297768473625183, 0.9280500411987305, 0.24583914875984192, 0.8716920018196106, 0.8208464980125427, 0.761846125125885, 0.9399680495262146, 0.5596662163734436, 0.021943168714642525, 0.30284759402275085, 0.3922814428806305, 0.3006172478199005, 0.49095404148101807, 0.8609779477119446, 0.48787015676498413, 0.889515221118927, 0.023496026173233986, 0.02437400631606579, 0.4510049521923065, 0.5136367082595825, 0.23533430695533752, 0.3159598708152771, 0.9143742322921753, 0.8996893167495728, 0.15228544175624847, 0.9634649753570557, 0.5725879073143005, 0.5688029527664185, 0.08952446281909943, 0.11422644555568695, 0.09150183200836182, 0.2990027368068695, 0.424146443605423, 0.45027783513069153, 0.9941708445549011, 0.8426325917243958, 0.08190727233886719, 0.43099144101142883, 0.7635550498962402, 0.6684936881065369, 0.21608032286167145, 0.48473408818244934, 0.2844756543636322, 0.5252382159233093, 0.9167032241821289, 0.29816749691963196, 0.08586186170578003, 0.2830781936645508, 0.44056236743927, 0.9106733798980713, 0.7095191478729248, 0.08576162159442902, 0.9699405431747437, 0.67655348777771, 0.5466098189353943, 0.5454960465431213, 0.17753608524799347, 0.6047046780586243, 0.09448060393333435, 0.95699542760849, 0.09407731890678406, 0.466657429933548, 0.2205914855003357, 0.7735351324081421, 0.8042880892753601, 0.5591538548469543, 0.24540239572525024, 0.9038527607917786, 0.845089852809906, 0.023441413417458534, 0.07996097952127457, 0.615625262260437, 0.8299490809440613, 0.6552813053131104, 0.25713208317756653, 0.4488966464996338, 0.7515396475791931, 0.08111171424388885, 0.24902576208114624, 0.37985795736312866, 0.8979724049568176, 0.2390785664319992, 0.5618088245391846, 0.20057059824466705, 0.2154003083705902, 0.16650664806365967, 0.5784551501274109, 0.9128158092498779, 0.9513728022575378, 0.33588463068008423, 0.930438220500946, 0.3272557556629181, 0.6801947355270386, 0.2518180310726166, 0.2303372323513031, 0.10476580262184143, 0.3552004098892212, 0.29024502635002136, 0.9687011241912842, 0.9409134387969971, 0.35536226630210876, 0.41548052430152893, 0.8167426586151123, 0.21148455142974854, 0.26750561594963074, 0.1690511554479599, 0.9709122776985168, 0.16761305928230286, 0.12253352999687195, 0.06322221457958221, 0.9126946330070496, 0.9480729103088379, 0.9819046854972839, 0.5163095593452454, 0.9096974730491638, 0.1518280953168869, 0.4889798164367676, 0.06898663192987442, 0.1946755200624466, 0.053792402148246765, 0.25242552161216736, 0.16499830782413483, 0.9663448333740234, 0.7855656743049622, 0.17912566661834717, 0.4281694293022156, 0.2788926661014557, 0.11225234717130661, 0.26955553889274597, 0.4049660563468933, 0.8054551482200623, 0.7337632775306702, 0.42899027466773987, 0.03194522112607956, 0.9834302067756653, 0.7113877534866333, 0.8108801245689392, 0.7453737854957581, 0.5212824940681458, 0.7277568578720093, 0.7781110405921936, 0.7063745856285095, 0.8405695557594299, 0.7900668978691101, 0.007176021579653025, 0.6111454963684082, 0.059478793293237686, 0.3245147168636322, 0.8727613687515259, 0.6709330677986145, 0.20479969680309296, 0.6459711790084839, 0.4992871582508087, 0.5868306756019592, 0.31901851296424866, 0.16700106859207153, 0.1191367506980896, 0.08153928816318512, 0.9901773929595947, 0.37296900153160095, 0.38131970167160034, 0.6459577679634094, 0.1897510439157486, 0.6064891815185547, 0.28984516859054565, 0.35906535387039185, 0.7786655426025391, 0.6190908551216125, 0.17289048433303833, 0.03944341465830803, 0.9709838032722473, 0.2887404263019562, 0.8364527821540833, 0.040543656796216965, 0.6709787249565125, 0.6359795928001404, 0.40845662355422974, 0.8958817720413208, 0.5087583661079407, 0.8934817314147949, 0.2756897509098053, 0.16211940348148346, 0.9947434067726135, 0.7384184002876282, 0.03648703545331955, 0.38181963562965393, 0.9814416766166687, 0.12661075592041016, 0.356985479593277, 0.6897475719451904, 0.8294255137443542, 0.7989035844802856, 0.726713240146637, 0.9086818695068359, 0.17905616760253906, 0.23532304167747498, 0.0765666738152504, 0.3179074227809906, 0.20103666186332703, 0.7644559144973755, 0.3636563718318939, 0.0008187427883967757, 0.9635564088821411, 0.7589420676231384, 0.18659865856170654, 0.00890458095818758, 0.8020074367523193, 0.07300691306591034, 0.03054683282971382, 0.9208137392997742, 0.7240773439407349, 0.4985099732875824, 0.07388230413198471, 0.48334407806396484, 0.2871435880661011, 0.5148963332176208, 0.004493687767535448, 0.6094617247581482, 0.6557739973068237, 0.5987017750740051, 0.6206061840057373, 0.5992739200592041, 0.8292607665061951, 0.8533411622047424, 0.29498541355133057, 0.9646254777908325, 0.6597463488578796, 0.001286112004891038, 0.44111067056655884, 0.606395959854126, 0.7893610596656799, 0.2873963713645935, 0.38840287923812866, 0.14186188578605652, 0.225070059299469, 0.8098980784416199, 0.3326147198677063, 0.7664526700973511, 0.9375268816947937, 0.9989559054374695, 0.9571676850318909, 0.030913399532437325, 0.7384755611419678, 0.9300245642662048, 0.7882009744644165, 0.6984578371047974, 0.4086800217628479, 0.40671098232269287, 0.2784444987773895, 0.7395423650741577, 0.023874305188655853, 0.04333202540874481, 0.4980602264404297, 0.26706239581108093, 0.37900927662849426, 0.6401111483573914, 0.2731782793998718, 0.4859042465686798, 0.5072775483131409, 0.07990562170743942, 0.9737343192100525, 0.8165690302848816, 0.10650668293237686, 0.3658463656902313, 0.745922327041626, 0.6445350050926208, 0.06186074763536453, 0.09768985211849213, 0.5320152640342712, 0.015597695484757423, 0.630905270576477, 0.8264297842979431, 0.8904569745063782, 0.7703935503959656, 0.04834207519888878, 0.21741952002048492, 0.3867030739784241, 0.9445858001708984, 0.7771562337875366, 0.5161929130554199, 0.1765928566455841, 0.5826702117919922, 0.3249683678150177, 0.3631912171840668, 0.9647806286811829, 0.32598671317100525, 0.46099385619163513, 0.37116631865501404, 0.20383383333683014, 0.030053462833166122, 0.2654421329498291, 0.21192291378974915, 0.2154180109500885, 0.1821790188550949, 0.6770113110542297, 0.3994511365890503, 0.3765038847923279, 0.027872756123542786, 0.026224633678793907, 0.9794024229049683, 0.8890634179115295, 0.8744332790374756, 0.4733212888240814, 0.8352534174919128, 0.6491486430168152, 0.14752060174942017, 0.5721101760864258, 0.5689195394515991, 0.2719332277774811, 0.6093372702598572, 0.5947888493537903, 0.16097453236579895, 0.11479838937520981, 0.6410064697265625, 0.7625513672828674, 0.6610795259475708, 0.5754154324531555, 0.6436964273452759, 0.518768846988678, 0.3888949751853943, 0.11820566654205322, 0.5244782567024231, 0.9948239922523499, 0.7180789709091187, 0.13303549587726593, 0.6400940418243408, 0.1918383687734604, 0.6191362142562866, 0.9615280032157898, 0.9369659423828125, 0.4672185182571411, 0.0656210333108902, 0.32678309082984924, 0.3694632649421692, 0.45894870162010193, 0.3287845253944397, 0.8651084303855896, 0.62614905834198, 0.9768850803375244, 0.4794236421585083, 0.2717674970626831, 0.7132233381271362, 0.5313338041305542, 0.7951351404190063, 0.1588633805513382, 0.484035462141037, 0.9735333323478699, 0.10351027548313141, 0.1954653561115265, 0.24171845614910126, 0.5983945727348328, 0.7372978329658508, 0.5466060638427734, 0.3135066032409668, 0.9336974620819092, 0.09955480694770813, 0.30925506353378296, 0.2914789915084839, 0.2633344233036041, 0.14931868016719818, 0.9647917151451111, 0.9602885246276855, 0.6085832118988037, 0.8953322768211365, 0.28891652822494507, 0.8743504881858826, 0.7457916736602783, 0.7932841777801514, 0.6995820999145508, 0.010473714210093021, 0.7351267337799072, 0.3923746645450592, 0.6178876757621765, 0.240607351064682, 0.8952054381370544, 0.9354076385498047, 0.725481390953064, 0.9188556671142578, 0.631171703338623, 0.4097568690776825, 0.90645831823349, 0.8757175207138062, 0.18262580037117004, 0.5653454661369324, 0.05331034958362579, 0.007700800895690918, 0.12368379533290863, 0.8550013303756714, 0.18269677460193634, 0.06754174083471298, 0.9130263924598694, 0.09472358971834183, 0.4881419241428375, 0.4205332398414612, 0.5219034552574158, 0.10754068940877914, 0.47630563378334045, 0.1186157613992691, 0.3748316168785095, 0.7116519212722778, 0.14757040143013, 0.00023303687339648604, 0.6481360793113708, 0.8581138253211975, 0.15489719808101654, 0.07105018943548203, 0.17744918167591095, 0.10397642105817795, 0.5210103392601013, 0.9939813017845154, 0.5574523210525513, 0.11248631775379181, 0.22365409135818481, 0.7878069877624512, 0.526684582233429, 0.4993355870246887, 0.78182053565979, 0.3851538300514221, 0.16057735681533813, 0.13668616116046906, 0.5107280015945435, 0.9925965666770935, 0.6427397131919861, 0.7824155688285828, 0.7007373571395874, 0.697921097278595, 0.05096157640218735, 0.4162476360797882, 0.7179871797561646, 0.030605779960751534, 0.005006749648600817, 0.5874568223953247, 0.17741085588932037, 0.5618702173233032, 0.005006521008908749, 0.038765907287597656, 0.01735074818134308, 0.5940571427345276, 0.6156778335571289, 0.5498078465461731, 0.8958921432495117, 0.2725646197795868, 0.29406285285949707, 0.8256304860115051, 0.5663034915924072, 0.3317863345146179, 0.3567539155483246, 0.30094096064567566, 0.5093860030174255, 0.3192131221294403, 0.37736836075782776, 0.6393205523490906, 0.5638623833656311, 0.5065782070159912, 0.024121304973959923, 0.5576629042625427, 0.3984648883342743, 0.26017436385154724, 0.7891216278076172, 0.6970003247261047, 0.9171469807624817, 0.08463248610496521, 0.3824368119239807, 0.7503270506858826, 0.17398332059383392, 0.20202350616455078, 0.31285250186920166, 0.03680913522839546, 0.6855455040931702, 0.7896012663841248, 0.15561282634735107, 0.1116432175040245, 0.6497591733932495, 0.9159897565841675, 0.968562126159668, 0.5676835775375366, 0.5496447086334229, 0.9971755146980286, 0.06431721895933151, 0.388207346200943, 0.6939852833747864, 0.8714475631713867, 0.7020822763442993, 0.4926895499229431, 0.47815975546836853, 0.9099549651145935, 0.7483664751052856, 0.20494283735752106, 0.9272723197937012, 0.814460277557373, 0.6241686344146729, 0.8047693371772766, 0.7294739484786987, 0.34338298439979553, 0.7051926851272583, 0.3824775218963623, 0.031163927167654037, 0.3859282433986664, 0.5423403382301331, 0.7661423087120056, 0.5297741293907166, 0.11795370280742645, 0.3790757358074188, 0.7851400971412659, 0.3476777970790863, 0.6861715316772461, 0.316432386636734, 0.18711307644844055, 0.905517041683197, 0.1002214103937149, 0.7723281383514404, 0.6470057368278503, 0.05537436157464981, 0.014878904446959496, 0.2491852194070816, 0.6869716644287109, 0.7753961682319641, 0.36733582615852356, 0.8786413073539734, 0.11176830530166626, 0.7794857621192932, 0.7069529294967651, 0.3418949842453003, 0.9863511323928833, 0.3530251681804657, 0.2540108561515808, 0.6422074437141418, 0.5455431938171387, 0.17054383456707, 0.35077235102653503, 0.8144590258598328, 0.33953016996383667, 0.7679436206817627, 0.5846291184425354, 0.7677109241485596, 0.22992229461669922, 0.8760512471199036, 0.11465234309434891, 0.5834758281707764, 0.9652852416038513, 0.1354527324438095, 0.11177051067352295, 0.2021014392375946, 0.03076862171292305, 0.6835688948631287, 0.5236339569091797, 0.2643781006336212, 0.7102288603782654, 0.5105265378952026, 0.2872595489025116, 0.20918290317058563, 0.30513671040534973, 0.8530832529067993, 0.9979108572006226, 0.3667455315589905, 0.6096204519271851, 0.7545214891433716, 0.2581011652946472, 0.6802423000335693, 0.44810619950294495, 0.9171733260154724, 0.1571812629699707, 0.09072502702474594, 0.044896434992551804, 0.42907509207725525, 0.4308397173881531, 0.18997719883918762, 0.8053752183914185, 0.7566043138504028, 0.6843279600143433, 0.6580879092216492, 0.6959163546562195, 0.7797533273696899, 0.46763893961906433, 0.23499733209609985, 0.14172643423080444, 0.3240354657173157, 0.5402860045433044, 0.9704082608222961, 0.8557924032211304, 0.5343351364135742, 0.07095015794038773, 0.1449967622756958, 0.2277883142232895, 0.13041767477989197, 0.3837082087993622, 0.6870005130767822, 0.026149561628699303, 0.21798811852931976, 0.9100701808929443, 0.49047186970710754, 0.524482011795044, 0.5805748701095581, 0.6635432243347168, 0.22726313769817352, 0.6786112189292908, 0.09512127190828323, 0.7935870289802551, 0.783532977104187, 0.32249715924263, 0.7626902461051941, 0.13660641014575958, 0.8642449975013733, 0.483134001493454, 0.47078898549079895, 0.186228409409523, 0.7942401766777039, 0.486729234457016, 0.2786482572555542, 0.20869021117687225, 0.5819889903068542, 0.0499211885035038, 0.3732486069202423, 0.06104588508605957, 0.8755381107330322, 0.3859429657459259, 0.0715637281537056, 0.5110724568367004, 0.23167145252227783, 0.5983163118362427, 0.8091940879821777, 0.5715671181678772, 0.8385400772094727, 0.6005010604858398, 0.7990495562553406, 0.396036297082901, 0.6912102103233337, 0.32508018612861633, 0.7297304272651672, 0.8713208436965942, 0.29355552792549133, 0.48573675751686096, 0.43152326345443726, 0.1402360200881958, 0.34195202589035034, 0.3561127781867981, 0.8526967763900757, 0.6988563537597656, 0.13369803130626678, 0.1486973762512207, 0.2753114104270935, 0.6014986634254456, 0.2329014539718628, 0.1483919620513916, 0.6555902361869812, 0.3580239415168762, 0.47817784547805786, 0.8062513470649719, 0.9412527680397034, 0.6944795846939087, 0.7088432908058167, 0.6116666793823242, 0.542160153388977, 0.11584784835577011, 0.5435689687728882, 0.208143949508667, 0.19616734981536865, 0.30803531408309937, 0.5013759732246399, 0.8971788883209229, 0.13983522355556488, 0.3812050521373749, 0.1918186992406845, 0.7989968657493591, 0.23837047815322876, 0.463904470205307, 0.3921673893928528, 0.5968759655952454, 0.09799139946699142, 0.6477869749069214, 0.7075815796852112, 0.2718222141265869, 0.9746230840682983, 0.6891161799430847, 0.17049947381019592, 0.3271408975124359, 0.11870309710502625, 0.3885033130645752, 0.025246957316994667, 0.7011010646820068, 0.9743236303329468, 0.926925539970398, 0.1812467724084854, 0.4792729914188385, 0.6045773029327393, 0.5054469108581543, 0.5514475107192993, 0.07636073976755142, 0.49815303087234497, 0.5822334885597229, 0.5355525612831116, 0.9767348170280457, 0.5728470087051392, 0.533273458480835, 0.6169351935386658, 0.06760642677545547, 0.5833825469017029, 0.5892161130905151, 0.7874639630317688, 0.40685659646987915, 0.750145435333252, 0.4112694263458252, 0.12229601293802261, 0.3557819426059723, 0.9655289053916931, 0.5256759524345398, 0.6477558612823486, 0.03430268540978432, 0.8584151268005371, 0.5678431391716003, 0.32765600085258484, 0.8532149791717529, 0.6987049579620361, 0.0031943644862622023, 0.3623121976852417, 0.5975455641746521, 0.20113849639892578, 0.7119824886322021, 0.11310916393995285, 0.7215380072593689, 0.8144098520278931, 0.4433411955833435, 0.13164398074150085, 0.5665712952613831, 0.6667665243148804, 0.4726731479167938, 0.7707656025886536, 0.9867899417877197, 0.5095932483673096, 0.790142297744751, 0.9793840646743774, 0.950899600982666, 0.09398294985294342, 0.6768959164619446, 0.5737026333808899, 0.2212623804807663, 0.7115768194198608, 0.2907368540763855, 0.037764254957437515, 0.7739850282669067, 0.6743652820587158, 0.08061683923006058, 0.2852936387062073, 0.29703980684280396, 0.3913554847240448, 0.032537948340177536, 0.18208208680152893, 0.9808642864227295, 0.7343813180923462, 0.8163297176361084, 0.9385963678359985, 0.4381559193134308, 0.38801610469818115, 0.6515081524848938, 0.14437539875507355, 0.46417051553726196, 0.8753490447998047, 0.43648862838745117, 0.0400833897292614, 0.17385590076446533, 0.73163902759552, 0.9857021570205688, 0.3289182782173157, 0.6324306130409241, 0.9966672658920288, 0.7362010478973389, 0.015393220819532871, 0.08553699404001236, 0.6398949027061462, 0.5321750640869141, 0.5819774866104126, 0.04201794043183327, 0.4207676351070404, 0.5415789484977722, 0.27232009172439575, 0.4677390158176422, 0.4949042797088623, 0.4172450304031372, 0.18837407231330872, 0.9368378520011902, 0.9159206748008728, 0.43125614523887634, 0.35037240386009216, 0.717087984085083, 0.6991856098175049, 0.8340817093849182, 0.15064692497253418, 0.3450281620025635, 0.8343625068664551, 0.45126402378082275, 0.3388060927391052, 0.666428804397583, 0.7043118476867676, 0.2054724246263504, 0.1725424826145172, 0.11275314539670944, 0.09529959410429001, 0.1546076089143753, 0.5026674270629883, 0.14960666000843048, 0.08360112458467484, 0.9407680034637451, 0.2932255268096924, 0.3191361129283905, 0.049594148993492126, 0.47105836868286133, 0.6052212119102478, 0.7419657707214355, 0.8624295592308044, 0.8007417321205139, 0.039183881133794785, 0.12916435301303864, 0.4169394075870514, 0.6685163974761963, 0.9719251394271851, 0.9482248425483704, 0.9314942359924316, 0.7898297905921936, 0.5516141057014465, 0.10900916904211044, 0.8512142896652222, 0.06152890622615814, 0.6421718597412109, 0.7561092972755432, 0.5310484170913696, 0.5454187989234924, 0.7756978273391724, 0.2318541705608368, 0.8777578473091125, 0.7300761342048645, 0.6564140319824219, 0.9539640545845032, 0.7312569618225098, 0.8243494033813477, 0.3466114401817322, 0.21194609999656677, 0.04558247700333595, 0.8546208739280701, 0.877039909362793, 0.4973238408565521, 0.9228465557098389, 0.3223075568675995, 0.9644889831542969, 0.44767704606056213, 0.4866544306278229, 0.08640681952238083, 0.09793751686811447, 0.8331332206726074, 0.21260711550712585, 0.05702492594718933, 0.14946095645427704, 0.9456309080123901, 0.7103278040885925, 0.27993717789649963, 0.4153062403202057, 0.378960520029068, 0.8013745546340942, 0.9498568773269653, 0.17337852716445923, 0.11449059844017029, 0.28180450201034546, 0.8938900828361511, 0.2692805230617523, 0.36231449246406555, 0.6771113872528076, 0.8993242383003235, 0.19780555367469788, 0.25398993492126465, 0.6852348446846008, 0.9282827377319336, 0.6162441968917847, 0.3607117235660553, 0.842126727104187, 0.7833676338195801, 0.19243855774402618, 0.4299584925174713, 0.09973157942295074, 0.546484112739563, 0.517041027545929, 0.5993297696113586, 0.9472656846046448, 0.34528377652168274, 0.7978366613388062, 0.14842121303081512, 0.09452585875988007, 0.9329416751861572, 0.022174103185534477, 0.6309032440185547, 0.9863229393959045, 0.4141262471675873, 0.185638889670372, 0.06207801029086113, 0.19434630870819092, 0.3416388928890228, 0.6261455416679382, 0.4790705740451813, 0.008085755631327629, 0.24787643551826477, 0.8144711256027222, 0.7714508771896362, 0.47950735688209534, 0.18075604736804962, 0.004561024717986584, 0.20958183705806732, 0.5383457541465759, 0.40716752409935, 0.09239906817674637, 0.6139543056488037, 0.7097564339637756, 0.554331362247467, 0.7231042385101318, 0.2473989576101303, 0.5973542332649231, 0.8708032369613647, 0.10458270460367203, 0.8198606371879578, 0.025048719719052315, 0.04557843878865242, 0.5241162776947021, 0.5523884892463684, 0.39296504855155945, 0.7666637301445007, 0.36263492703437805, 0.2931644022464752, 0.25688567757606506, 0.051487669348716736, 0.5625641942024231, 0.7084251046180725, 0.563911497592926, 0.4206790030002594, 0.7357251644134521, 0.3500892221927643, 0.9792885184288025, 0.40045270323753357, 0.14562031626701355, 0.8986321687698364, 0.7387157082557678, 0.5736224055290222, 0.13604319095611572, 0.8501587510108948, 0.32879549264907837, 0.2654058635234833, 0.7550767064094543, 0.922020435333252, 0.900793194770813, 0.3039436340332031, 0.9780483245849609, 0.8563603758811951, 0.22826500236988068, 0.9650079011917114, 0.22959500551223755, 0.8104297518730164, 0.48500433564186096, 0.2864008843898773, 0.411014586687088, 0.003009229665622115, 0.9221926331520081, 0.18199752271175385, 0.25159531831741333, 0.6796389222145081, 0.6472251415252686, 0.8926112651824951, 0.6198925971984863, 0.8024680614471436, 0.5091697573661804, 0.14150740206241608, 0.8106271624565125, 0.2240770161151886, 0.7587316632270813, 0.42050090432167053, 0.10753881931304932, 0.5870817303657532, 0.37290695309638977, 0.3179636299610138, 0.4298681318759918, 0.7486674189567566, 0.6575325727462769, 0.738981306552887, 0.4697137176990509, 0.4894753694534302, 0.692173957824707, 0.8566477298736572, 0.445272833108902, 0.9635682106018066, 0.13439048826694489, 0.6866286396980286, 0.7117342352867126, 0.3432375192642212, 0.011509482748806477, 0.5412092208862305, 0.37601056694984436, 0.7931928634643555, 0.6280394196510315, 0.5898861289024353, 0.15545743703842163, 0.6762452721595764, 0.1890866458415985, 0.6659454107284546, 0.5315687656402588, 0.07049202919006348, 0.9393392205238342, 0.7847318649291992, 0.4960082471370697, 0.41349470615386963, 0.38474607467651367, 0.09189935773611069, 0.24141398072242737, 0.27025163173675537, 0.9769467115402222, 0.3807566165924072, 0.3193201422691345, 0.7007957100868225, 0.043957967311143875, 0.5498047471046448, 0.6508219242095947, 0.7339076399803162, 0.2591477930545807, 0.33223894238471985, 0.6511894464492798, 0.636759340763092, 0.6726405024528503, 0.5964179635047913, 0.08005835860967636, 0.5092598795890808, 0.06929855048656464, 0.1773887425661087, 0.8844868540763855, 0.38750404119491577, 0.3332779109477997, 0.9787976145744324, 0.664443850517273, 0.25424203276634216, 0.35396265983581543, 0.6557208299636841, 0.13523083925247192, 0.9735718965530396, 0.8428465723991394, 0.2698062062263489, 0.09978993237018585, 0.9966192245483398, 0.6972986459732056, 0.8230721950531006, 0.16106610000133514, 0.8695302605628967, 0.1901133954524994, 0.6873381733894348, 0.8717538118362427, 0.3285457193851471, 0.9528111219406128, 0.40297558903694153, 0.28898170590400696, 0.7796894907951355, 0.5873621702194214, 0.47944289445877075, 0.14224164187908173, 0.9301992654800415, 0.7525311708450317, 0.015570874325931072, 0.5021442174911499, 0.3900447487831116, 0.5824124813079834, 0.23208416998386383, 0.4087951183319092, 0.5847342014312744, 0.9323641657829285, 0.5552837252616882, 0.2836282253265381, 0.9640360474586487, 0.1080237552523613, 0.015403981320559978, 0.43273746967315674, 0.4908313751220703, 0.7529215812683105, 0.18806755542755127, 0.47323113679885864, 0.7908765077590942, 0.2514641284942627, 0.791296660900116, 0.9583561420440674, 0.8786466717720032, 0.5932881832122803, 0.8459792733192444, 0.3612440824508667, 0.945629358291626, 0.9560867547988892, 0.055572666227817535, 0.2847197651863098, 0.31953299045562744, 0.6273513436317444, 0.9512677788734436, 0.9846140146255493, 0.1652882844209671, 0.7458813786506653, 0.17416325211524963, 0.704048216342926, 0.6955793499946594, 0.2061309665441513, 0.5005195140838623, 0.7158782482147217, 0.8093441724777222, 0.9363358020782471, 0.43074706196784973, 0.9909219741821289, 0.9808026552200317, 0.4661129117012024, 0.5305717587471008, 0.45209425687789917, 0.12016288936138153, 0.4192180037498474, 0.3448858857154846, 0.0346250981092453, 0.07259491086006165, 0.48789238929748535, 0.9057658910751343, 0.6187138557434082, 0.5444079637527466, 0.11474158614873886, 0.938503623008728, 0.10054636746644974, 0.9347596168518066, 0.7936884164810181, 0.6956654787063599, 0.2850044369697571, 0.9431236386299133, 0.5300096273422241, 0.6876397132873535, 0.286198228597641, 0.5100085139274597, 0.2185852825641632, 0.21841232478618622, 0.543695867061615, 0.4412975013256073, 0.28890326619148254, 0.01650727167725563, 0.33911094069480896, 0.6981173157691956, 0.09997040778398514, 0.43057870864868164, 0.5970764756202698, 0.45396891236305237, 0.029184259474277496, 0.7771574854850769, 0.5632771253585815, 0.8984007835388184, 0.8857328295707703, 0.9628900289535522, 0.04223005473613739, 0.6181225180625916, 0.968259871006012, 0.3028021454811096, 0.9283866882324219, 0.2916489839553833, 0.466061532497406, 0.6907594203948975, 0.8283932209014893, 0.7523233294487, 0.043449170887470245, 0.6428838968276978, 0.7867997884750366, 0.27806657552719116, 0.26712560653686523, 0.9776860475540161, 0.9330361485481262, 0.6828828454017639, 0.7732816338539124, 0.9552027583122253, 0.0039123790338635445, 0.9220273494720459, 0.7763974666595459, 0.36094480752944946, 0.05689161270856857, 0.5947489142417908, 0.5265045166015625, 0.741292417049408, 0.7724258303642273, 0.9593226909637451, 0.24257224798202515, 0.13211116194725037, 0.1402757316827774, 0.7007315754890442, 0.9305914044380188, 0.62632155418396, 0.10682252049446106, 0.5123474597930908, 0.6516437530517578, 0.31217440962791443, 0.42070305347442627, 0.8067302107810974, 0.22038081288337708, 0.23960290849208832, 0.03424340486526489, 0.051960401237010956, 0.4910416901111603, 0.04294983297586441, 0.2828620374202728, 0.8008662462234497, 0.9880297780036926, 0.7264086008071899, 0.02686765231192112, 0.8478265404701233, 0.850648820400238, 0.32761338353157043, 0.6382791996002197, 0.5557798743247986, 0.2083444595336914, 0.28111740946769714, 0.29685425758361816, 0.9589194655418396, 0.46052515506744385, 0.8959262371063232, 0.9932572245597839, 0.14870135486125946, 0.5523000359535217, 0.7768145203590393, 0.6097898483276367, 0.21442241966724396, 0.746509313583374, 0.7221571207046509, 0.41415029764175415, 0.08767545968294144, 0.0780203565955162, 0.15576662123203278, 0.22516724467277527, 0.8209980726242065, 0.7976130247116089, 0.2270808219909668, 0.8179429173469543, 0.6067293882369995, 0.6974238753318787, 0.5519617795944214, 0.6589769124984741, 0.41809096932411194, 0.44224095344543457, 0.9146293997764587, 0.8527056574821472, 0.47070831060409546, 0.9385992884635925, 0.5639779567718506, 0.8237043023109436, 0.02126980386674404, 0.22569677233695984, 0.32627221941947937, 0.5276135802268982, 0.7885132431983948, 0.09077246487140656, 0.4141002893447876, 0.11810310930013657, 0.6033851504325867, 0.017984190955758095, 0.5304012894630432, 0.32030588388442993, 0.35743987560272217, 0.6991916298866272, 0.43643659353256226, 0.193308487534523, 0.3066942095756531, 0.3118257224559784, 0.7507823705673218, 0.23239411413669586, 0.9737842679023743, 0.5896444320678711, 0.8797208666801453, 0.44594910740852356, 0.4367978572845459, 0.642495334148407, 0.9206341505050659, 0.8373274207115173, 0.315371036529541, 0.09564829617738724, 0.25228801369667053, 0.0029064966365695, 0.8759225010871887, 0.9891892075538635, 0.8758864998817444, 0.9758851528167725, 0.49678364396095276, 0.6245081424713135, 0.965265691280365, 0.4909077286720276, 0.9789782762527466, 0.3992076516151428, 0.42468440532684326, 0.0683337077498436, 0.3270917236804962, 0.11282102763652802, 0.919273853302002, 0.22209300100803375, 0.5460421442985535, 0.4099186062812805, 0.6783708930015564, 0.613709032535553, 0.32989829778671265, 0.03903425857424736, 0.13050206005573273, 0.048337217420339584, 0.13508673012256622, 0.224458709359169, 0.251666396856308, 0.799347460269928, 0.7284941673278809, 0.31923943758010864, 0.5729951858520508, 0.038515567779541016, 0.7190775871276855, 0.869405210018158, 0.1836712658405304, 0.6400617957115173, 0.3451700806617737, 0.710354745388031, 0.14804309606552124, 0.8273427486419678, 0.7114790678024292, 0.9524996876716614, 0.4633249342441559, 0.4087139368057251, 0.26570677757263184, 0.46879690885543823, 0.19034746289253235, 0.7506648898124695, 0.9428501129150391, 0.7717186212539673, 0.4421142041683197, 0.5491084456443787, 0.6324731707572937, 0.3525136113166809, 0.4905804395675659, 0.8337755799293518, 0.37470611929893494, 0.07213751971721649, 0.6101300120353699, 0.18139083683490753, 0.4803506135940552, 0.5634022951126099, 0.05098860710859299, 0.8294437527656555, 0.284415602684021, 0.971829354763031, 0.035910796374082565, 0.22375336289405823, 0.5415759086608887, 0.5767087936401367, 0.012905100360512733, 0.5666805505752563, 0.7124167680740356, 0.5768573880195618, 0.3086830675601959, 0.3124832808971405, 0.07407257705926895, 0.29822564125061035, 0.25719860196113586, 0.8202449679374695, 0.2738224267959595, 0.7867559790611267, 0.41715967655181885, 0.49137285351753235, 0.5051929950714111, 0.24486181139945984, 0.15928037464618683, 0.4722904562950134, 0.29822731018066406, 0.9817669987678528, 0.16453275084495544, 0.530200183391571, 0.6418835520744324, 0.6378509998321533, 0.7918534278869629, 0.17734043300151825, 0.4329078197479248, 0.2509182095527649, 0.018200498074293137, 0.8678135275840759, 0.9222943782806396, 0.9606900811195374, 0.43833673000335693, 0.2552729547023773, 0.7508374452590942, 0.3122609257698059, 0.8624072074890137, 0.9354883432388306, 0.09079444408416748, 0.3581005036830902, 0.7811413407325745, 0.6141122579574585, 0.3534364104270935, 0.8364855051040649, 0.03735326603055, 0.9778231382369995, 0.6297768950462341, 0.12158774584531784, 0.6230711936950684, 0.0917077362537384, 0.7977938652038574, 0.6443830728530884, 0.7729229927062988, 0.651299774646759, 0.9889920949935913, 0.42100659012794495, 0.8803560733795166, 0.35090330243110657, 0.3771878480911255, 0.421126127243042, 0.2951626777648926, 0.394765704870224, 0.5640976428985596, 0.5077682137489319, 0.9714347124099731, 0.5583696961402893, 0.8307254314422607, 0.2771328389644623, 0.26917994022369385, 0.17673754692077637, 0.5081349015235901, 0.799315869808197, 0.4949423670768738, 0.6969718933105469, 0.047508299350738525, 0.30147069692611694, 0.9646141529083252, 0.8225095272064209, 0.32576635479927063, 0.8446427583694458, 0.7688252925872803, 0.9220697283744812, 0.48040035367012024, 0.609317421913147, 0.06911896914243698, 0.21773125231266022, 0.7433627247810364, 0.7412329912185669, 0.08589781075716019, 0.027398698031902313, 0.5306162238121033, 0.4590053856372833, 0.6673188209533691, 0.045671556144952774, 0.24531206488609314, 0.1883440464735031, 0.4789965748786926, 0.6905066967010498, 0.8841856718063354, 0.1106385588645935, 0.004680771380662918, 0.9298000931739807, 1.6911113561945967e-05, 0.727400004863739, 0.049587927758693695, 0.7463861107826233, 0.7509216666221619, 0.6231533288955688, 0.44863882660865784, 0.5916998982429504, 0.049099911004304886, 0.49892908334732056, 0.9647773504257202, 0.4903089702129364, 0.3449176251888275, 0.5282066464424133, 0.0859294906258583, 0.18840409815311432, 0.5597899556159973, 0.6596126556396484, 0.34318310022354126, 0.6470003128051758, 0.6662843823432922, 0.5045361518859863, 0.26837870478630066, 0.21507614850997925, 0.13861319422721863, 0.12954768538475037, 0.2612225115299225, 0.8919530510902405, 0.6299042105674744, 0.5515442490577698, 0.49868056178092957, 0.06888909637928009, 0.013529493473470211, 0.32914045453071594, 0.48575055599212646, 0.47434547543525696, 0.8092575073242188, 0.9896091222763062, 0.43792229890823364, 0.2814580798149109, 0.7860332131385803, 0.4240831732749939, 0.22188979387283325, 0.0930212214589119, 0.06541609764099121, 0.785818874835968, 0.7991752028465271, 0.8494357466697693, 0.538722813129425, 0.7911958694458008, 0.3842753767967224, 0.6703477501869202, 0.37984558939933777, 0.9463996291160583, 0.147075355052948, 0.13735967874526978, 0.8605766296386719, 0.7877001166343689, 0.3870798945426941, 0.6276857256889343, 0.2408646047115326, 0.13111652433872223, 0.4116672873497009, 0.7140874266624451, 0.8139200210571289, 0.16100537776947021, 0.9804995656013489, 0.15822386741638184, 0.04056898504495621, 0.8323507905006409, 0.588959276676178, 0.6783671379089355, 0.517625629901886, 0.15732187032699585, 0.9141514897346497, 0.09720084816217422, 0.4895170032978058, 0.29907700419425964, 0.0039342655800282955, 0.3909647464752197, 0.8633337020874023, 0.944267988204956, 0.9226267337799072, 0.27950096130371094, 0.714349627494812, 0.9666968584060669, 0.4287533164024353, 0.6301590800285339, 0.4088604748249054, 0.5028168559074402, 0.3474021255970001, 0.9382284283638, 0.401628702878952, 0.6624780297279358, 0.5906990766525269, 0.13478177785873413, 0.29114124178886414, 0.3979280889034271, 0.21809709072113037, 0.7674173712730408, 0.4857974648475647, 0.6539538502693176, 0.9161816239356995, 0.8816964030265808, 0.08917216956615448, 0.9945582747459412, 0.45655927062034607, 0.39518851041793823, 0.7985258102416992, 0.7831327319145203, 0.5415358543395996, 0.7126323580741882, 0.0343504324555397, 0.835049569606781, 0.38360947370529175, 0.32194581627845764, 0.0678001195192337, 0.28215742111206055, 0.22690437734127045, 0.8312362432479858, 0.7693303227424622, 0.9117900133132935, 0.6113033890724182, 0.9893987774848938, 0.7877971529960632, 0.16608358919620514, 0.2249237596988678, 0.8211060166358948, 0.053205594420433044, 0.6393399238586426, 0.421978235244751, 0.8051237463951111, 0.8473708033561707, 0.14881493151187897, 0.9690543413162231, 0.6324669122695923, 0.2111080139875412, 0.26186811923980713, 0.07721582055091858, 0.5052438378334045, 0.9956414699554443, 0.9646908044815063, 0.7913445234298706, 0.9202095866203308, 0.27302852272987366, 0.6788844466209412, 0.423942506313324, 0.93998783826828, 0.01974273845553398, 0.5538623929023743, 0.08988931030035019, 0.1658351719379425, 0.02918122708797455, 0.6176265478134155, 0.18671411275863647, 0.09429469704627991, 0.4882466495037079, 0.8832914233207703, 0.4027349352836609, 0.24045510590076447, 0.5255986452102661, 0.031867992132902145, 0.6498600244522095, 0.6490973830223083, 0.46228206157684326, 0.6946719288825989, 0.5782262682914734, 0.801605224609375, 0.7165390849113464, 0.9789670705795288, 0.8948028683662415, 0.21164658665657043, 0.6945992112159729, 0.2777940332889557, 0.28164035081863403, 0.6600114703178406, 0.9372580051422119, 0.9109795093536377, 0.2841285467147827, 0.6916967034339905, 0.7540954947471619, 0.7936809062957764, 0.5398197770118713, 0.836938202381134, 0.2753087878227234, 0.829516589641571, 0.932746410369873, 0.3462899923324585, 0.061558324843645096, 0.7058393955230713, 0.08508045226335526, 0.5033323168754578, 0.8993771076202393, 0.755714476108551, 0.9919977784156799, 0.10394683480262756, 0.5441237092018127, 0.7275006175041199, 0.12396439909934998, 0.18863296508789062, 0.7365084290504456, 0.31804507970809937, 0.15274859964847565, 0.732342541217804, 0.0920436754822731, 0.1479579508304596, 0.022385818883776665, 0.9332525134086609, 0.29682111740112305, 0.14672337472438812, 0.7995839715003967, 0.7588043212890625, 0.0330936536192894, 0.05279426649212837, 0.8923476934432983, 7.1894952270668e-05, 0.9199129343032837, 0.8807605504989624, 0.4080873727798462, 0.11139216274023056, 0.467075377702713, 0.6096627116203308, 0.7552684545516968, 0.22682221233844757, 0.8919454216957092, 0.9838911294937134, 0.7237839698791504, 0.5540103316307068, 0.44301965832710266, 0.8252536058425903, 0.8906070590019226, 0.26497647166252136, 0.14930972456932068, 0.0950251892209053, 0.37609171867370605, 0.3823259174823761, 0.6028618216514587, 0.39441654086112976, 0.6272997856140137, 0.2828689217567444, 0.9398900270462036, 0.5760259032249451, 0.6116429567337036, 0.8338510990142822, 0.6842412948608398, 0.3560830056667328, 0.21352127194404602, 0.0596136748790741, 0.32070785760879517, 0.07569658756256104, 0.5615549683570862, 0.10770434886217117, 0.8967456221580505, 0.07204613834619522, 0.9220309257507324, 0.2662216126918793, 0.7401138544082642, 0.319029837846756, 0.6845835447311401, 0.5871738195419312, 0.022521784529089928, 0.732012152671814, 0.68394535779953, 0.6374759078025818, 0.4443397521972656, 0.7261980772018433, 0.8928995728492737, 0.7859499454498291, 0.22277143597602844, 0.6203701496124268, 0.4074136018753052, 0.06488218903541565, 0.2634860575199127, 0.08556585758924484, 0.7049951553344727, 0.17509359121322632, 0.6151846647262573, 0.20567452907562256, 0.43659618496894836, 0.9002731442451477, 0.9642649292945862, 0.7986564636230469, 0.8384389877319336, 0.43425002694129944, 0.4989355206489563, 0.8010842204093933, 0.7263028025627136, 0.47579842805862427, 0.37474387884140015, 0.3884774446487427, 0.9774938225746155, 0.7410537600517273, 0.1350744366645813, 0.22090671956539154, 0.08488451689481735, 0.8926017880439758, 0.9831507802009583, 0.5571384429931641, 0.2569168210029602, 0.27584215998649597, 0.44903564453125, 0.006805851124227047, 0.38247326016426086, 0.8571653962135315, 0.26486849784851074, 0.10408977419137955, 0.17116239666938782, 0.045949339866638184, 0.2487986534833908, 0.442568838596344, 0.7291972637176514, 0.42494139075279236, 0.09354929625988007, 0.8204160332679749, 0.8776608109474182, 0.6571952700614929, 0.6829238533973694, 0.41171324253082275, 0.8403139710426331, 0.4215119481086731, 0.34833404421806335, 0.13813090324401855, 0.2901919186115265, 0.06544499844312668, 0.5583197474479675, 0.7107620239257812, 0.7709022164344788, 0.09663277864456177, 0.24907255172729492, 0.8599613904953003, 0.9114843606948853, 0.8583086729049683, 0.3638455867767334, 0.3980073630809784, 0.3314583897590637, 0.3583162724971771, 0.392098605632782, 0.06837595254182816, 0.37576404213905334, 0.49578818678855896, 0.8086593747138977, 0.7188633680343628, 0.3221847414970398, 0.565844714641571, 0.6358972191810608, 0.9667904376983643, 0.4322476089000702, 0.6248368620872498, 0.6939741373062134, 0.6194281578063965, 0.21365931630134583, 0.6378249526023865, 0.7356988191604614, 0.4554240107536316, 0.24848005175590515, 0.9157449007034302, 0.4944366216659546, 0.2610260248184204, 0.7388289570808411, 0.4543522894382477, 0.9978517889976501, 0.11826971173286438, 0.44581735134124756, 0.7215468287467957, 0.9946732521057129, 0.5638817548751831, 0.5314790606498718, 0.6986945271492004, 0.05566219240427017, 0.006587639916688204, 0.3367319405078888, 0.3315945565700531, 0.12203216552734375, 0.8368876576423645, 0.8683023452758789, 0.40047895908355713, 0.37121883034706116, 0.49071913957595825, 0.8040886521339417, 0.3354787230491638, 0.5695276260375977, 0.7916045784950256, 0.7151939272880554, 0.0803980603814125, 0.5146241784095764, 0.8945135474205017, 0.86479651927948, 0.05912301689386368, 0.5666549205780029, 0.8147019147872925, 0.3907007575035095, 0.6441724300384521, 0.6434133052825928, 0.050008926540613174, 0.7624967098236084, 0.5626544952392578, 0.4312736988067627, 0.21872185170650482, 0.5810427665710449, 0.8965630531311035, 0.03498411178588867, 0.6406720280647278, 0.47879230976104736, 0.1950347125530243, 0.2527526319026947, 0.04077976197004318, 0.6678227186203003, 0.7669256329536438, 0.8140836954116821, 0.3262244462966919, 0.5399377346038818, 0.598539412021637, 0.2639157772064209, 0.6843109726905823, 0.5444031953811646, 0.8723062872886658, 0.9127369523048401, 0.05484222620725632, 0.617845356464386, 0.8556427359580994, 0.09389445185661316, 0.2683807909488678, 0.04003046080470085, 0.5326588749885559, 0.09583715349435806, 0.553335428237915, 0.022929973900318146, 0.18221686780452728, 0.3555612862110138, 0.5211494565010071, 0.5034006237983704, 0.4695822596549988, 0.23797191679477692, 0.8128994703292847, 0.4782197177410126, 0.32320883870124817, 0.37791943550109863, 0.9908105731010437, 0.7776841521263123, 0.8518354296684265, 0.8190747499465942, 0.0016955938190221786, 0.20184046030044556, 0.611829936504364, 0.027151022106409073, 0.12515488266944885, 0.24860477447509766, 0.5448872447013855, 0.21914798021316528, 0.09907755255699158, 0.6585801839828491, 0.5840973258018494, 0.6216909885406494, 0.1385352909564972, 0.4552237093448639, 0.6344208121299744, 0.2596885859966278, 0.17397315800189972, 0.04518232122063637, 0.5479036569595337, 0.5722763538360596, 0.4540308117866516, 0.973793625831604, 0.144423708319664, 0.5769880414009094, 0.3037501871585846, 0.669597864151001, 0.1346026062965393, 0.4550277590751648, 0.9451115727424622, 0.4666772484779358, 0.6715540289878845, 0.8811736702919006, 0.34007424116134644, 0.8835718035697937, 0.761866569519043, 0.696013331413269, 0.2143772840499878, 0.536422848701477, 0.776221752166748, 0.09336177259683609, 0.2786756157875061, 0.698628306388855, 0.707255482673645, 0.307059109210968, 0.2031000405550003, 0.6530739665031433, 0.10782302170991898, 0.8772380352020264, 0.7783001661300659, 0.8044846653938293, 0.8153973817825317, 0.6760025024414062, 0.3270523250102997, 0.06182364001870155, 0.6506885290145874, 0.9049302935600281, 0.06513664871454239, 0.9988045692443848, 0.6250550150871277, 0.01357424259185791, 0.0941251665353775, 0.13027098774909973, 0.4648505449295044, 0.6702023148536682, 0.5792049765586853, 0.17990092933177948, 0.3077673017978668, 0.9127050638198853, 0.19983625411987305, 0.5320064425468445, 0.15222035348415375, 0.3859288692474365, 0.06280975043773651, 0.9870604872703552, 0.1031206026673317, 0.21608072519302368, 0.8104807138442993, 0.06651432067155838, 0.6368463635444641, 0.6757091283798218, 0.04577074572443962, 0.8316892981529236, 0.6462775468826294, 0.24143527448177338, 0.9549005627632141, 0.47500088810920715, 0.14861340820789337, 0.5050881505012512, 0.10526679456233978, 0.5566762685775757, 0.569614589214325, 0.7248668670654297, 0.5384058952331543, 0.7470081448554993, 0.40174049139022827, 0.0058653936721384525, 0.3134929835796356, 0.46336621046066284, 0.028525084257125854, 0.3437124490737915, 0.7349343299865723, 0.8462955355644226, 0.5185832381248474, 0.46302419900894165, 0.8141592144966125, 0.12984223663806915, 0.47839993238449097, 0.7627350687980652, 0.4038097560405731, 0.6650100350379944, 0.43243342638015747, 0.9501469135284424, 0.5944390892982483, 0.15201908349990845, 0.8178586959838867, 0.16258113086223602, 0.46009087562561035, 0.7027555108070374, 0.37391307950019836, 0.8658652901649475, 0.21961931884288788, 0.5303807854652405, 0.9120356440544128, 0.891808271408081, 0.11088303476572037, 0.671104371547699, 0.560310423374176, 0.6240249872207642, 0.6216413378715515, 0.7875102162361145, 0.511135995388031, 0.08801383525133133, 0.27240484952926636, 0.9483019709587097, 0.07095734030008316, 0.04666496068239212, 0.3771806061267853, 0.6085799336433411, 0.736826479434967, 0.7790738344192505, 0.7413734197616577, 0.9681142568588257, 0.8564391136169434, 0.3821365237236023, 0.15016043186187744, 0.3341678977012634, 0.07000547647476196, 0.6421515941619873, 0.7219261527061462, 0.6967993378639221, 0.6122775077819824, 0.36255353689193726, 0.780430793762207, 0.8137759566307068, 0.41916656494140625, 0.03943801298737526, 0.4532032608985901, 0.5750495195388794, 0.9653734564781189, 0.008796979673206806, 0.7151052951812744, 0.1303553581237793, 0.4086787700653076, 0.22716858983039856, 0.1051371693611145, 0.8884697556495667, 0.7172708511352539, 0.7333126664161682, 0.08829052746295929, 0.5999892950057983, 0.455824613571167, 0.38684773445129395, 0.5893387794494629, 0.09837870299816132, 0.5404369831085205, 0.7747655510902405, 0.49816766381263733, 0.43552184104919434, 0.9767277836799622, 0.5657196044921875, 0.7682461142539978, 0.11394665390253067, 0.45436298847198486, 0.4060755968093872, 0.18888144195079803, 0.3537626564502716, 0.6724698543548584, 0.7507468461990356, 0.28873518109321594, 0.12552765011787415, 0.3858155310153961, 0.8913101553916931, 0.524170994758606, 0.89774489402771, 0.16666880249977112, 0.8309289813041687, 0.1591445952653885, 0.7765098810195923, 0.06379684805870056, 0.6799254417419434, 0.5177570581436157, 0.08687259256839752, 0.6507042646408081, 0.1526898890733719, 0.7910671830177307, 0.49872589111328125, 0.7012225389480591, 0.8987475633621216, 0.38491278886795044, 0.2569122910499573, 0.876502513885498, 0.3899536430835724, 0.0029249037615954876, 0.7497068047523499, 0.1104496419429779, 0.5037100315093994, 0.444242000579834, 0.31344932317733765, 0.7470197081565857, 0.1107315793633461, 0.5563062429428101, 0.8668953776359558, 0.7335503101348877, 0.6985013484954834, 0.16499952971935272, 0.03881320729851723, 0.7141148447990417, 0.9034500122070312, 0.09686413407325745, 0.29001590609550476, 0.9970215559005737, 0.17603187263011932, 0.2725328207015991, 0.3109407424926758, 0.36787161231040955, 0.875202476978302, 0.5996602177619934, 0.1805698275566101, 0.4051714539527893, 0.847159743309021, 0.38840144872665405, 0.4862739145755768, 0.5958928465843201, 0.8097138404846191, 0.551223635673523, 0.663669228553772, 0.44801005721092224, 0.39999642968177795, 0.6036306023597717, 0.37626948952674866, 0.13562628626823425, 0.5422134399414062, 0.35516953468322754, 0.8009819388389587, 0.5469870567321777, 0.6119439601898193, 0.1642882227897644, 0.5970904231071472, 0.7883267998695374, 0.4202219545841217, 0.35030874609947205, 0.2732732594013214, 0.503050684928894, 0.336334228515625, 0.030062448233366013, 0.3677181601524353, 0.8760941624641418, 0.6922195553779602, 0.3907105624675751, 0.14500734210014343, 0.8521949648857117, 0.7370624542236328, 0.7621620893478394, 0.8822609186172485, 0.38386470079421997, 0.01631157658994198, 0.9980327486991882, 0.5040377974510193, 0.45285579562187195, 0.6342796683311462, 0.20782066881656647, 0.7239991426467896, 0.7204300165176392, 0.610612690448761, 0.7338566184043884, 0.5543389916419983, 0.185496523976326, 0.9505419135093689, 0.5936576128005981, 0.5255824327468872, 0.9868202209472656, 0.5876017212867737, 0.05852336436510086, 0.8286465406417847, 0.017169328406453133, 0.07961367070674896, 0.2578793466091156, 0.21026334166526794, 0.7086324095726013, 0.9293252825737, 0.6448321342468262, 0.6529466509819031, 0.5245940089225769, 0.2582060694694519, 0.5408245921134949, 0.10303814709186554, 0.4196634292602539, 0.4312453269958496, 0.7622382640838623, 0.6588921546936035, 0.044580455869436264, 0.2829076945781708, 0.5638899207115173, 0.49120476841926575, 0.11821810156106949, 0.03220420703291893, 0.6669183969497681, 0.450649619102478, 0.3238925337791443, 0.5715224146842957, 0.4696844220161438, 0.7611022591590881, 0.5670580267906189, 0.19480176270008087, 0.5962280035018921, 0.24956969916820526, 0.7363930940628052, 0.11940135061740875, 0.8459147810935974, 0.8138777017593384, 0.8224391937255859, 0.01552329771220684, 0.8441412448883057, 0.84389728307724, 0.2649814486503601, 0.7377529740333557, 0.9244007468223572, 0.5658540725708008, 0.9902888536453247, 0.07269027829170227, 0.4823518395423889, 0.703532338142395, 0.6447188854217529, 0.24738629162311554, 0.2329920381307602, 0.6562451124191284, 0.5263194441795349, 0.1442669928073883, 0.32833611965179443, 0.3758943974971771, 0.0444917231798172, 0.5690628886222839, 0.0276162289083004, 0.6387732028961182, 0.02297811582684517, 0.32461243867874146, 0.15168415009975433, 0.42747390270233154, 0.44609230756759644, 0.2989496886730194, 0.4839632511138916, 0.3106550872325897, 0.882148265838623, 0.4287964701652527, 0.07861906290054321, 0.4316985309123993, 0.4502381682395935, 0.41801875829696655, 0.45042169094085693, 0.5465556979179382, 0.4822470247745514, 0.9564229249954224, 0.11133545637130737, 0.9362841248512268, 0.03406434506177902, 0.40501856803894043, 0.6367758512496948, 0.08011291176080704, 0.3478008508682251, 0.03684720769524574, 0.1085452139377594, 0.09263560175895691, 0.8475481271743774, 0.9958163499832153, 0.22957105934619904, 0.6204482316970825, 0.7579832673072815, 0.6749944686889648, 0.46314916014671326, 0.4765543043613434, 0.6038123965263367, 0.8301118016242981, 0.308242529630661, 0.9095842838287354, 0.6080107092857361, 0.9905955195426941, 0.9308624863624573, 0.9258183836936951, 0.6443219780921936, 0.1858796775341034, 0.8796647787094116, 0.3967565596103668, 0.935205340385437, 0.8130718469619751, 0.2587619423866272, 0.916498601436615, 0.10769189149141312, 0.7225066423416138, 0.47571685910224915, 0.2563627362251282, 0.5886256098747253, 0.8355993628501892, 0.2215721160173416, 0.39305776357650757, 0.41064971685409546, 0.6143365502357483, 0.9922827482223511, 0.28386780619621277, 0.11759114265441895, 0.13634471595287323, 0.9377902746200562, 0.7973904013633728, 0.8901587724685669, 0.3409006595611572, 0.10434894263744354, 0.5208309888839722, 0.5182859897613525, 0.5112115740776062, 0.2098722904920578, 0.9273216128349304, 0.40746235847473145, 0.6794933080673218, 0.6099307537078857, 0.8933524489402771, 0.8338375687599182, 0.11964397877454758, 0.2195717692375183, 0.6597013473510742, 0.5159634351730347, 0.89566969871521, 0.20742446184158325, 0.38490256667137146, 0.7370928525924683, 0.17497888207435608, 0.5574150681495667, 0.09536270052194595, 0.4798455238342285, 0.5759686827659607, 0.4391953647136688, 0.2015315145254135, 0.8991942405700684, 0.94353848695755, 0.13890819251537323, 0.2034824937582016, 0.2776973247528076, 0.43580278754234314, 0.7069833874702454, 0.4434907138347626, 0.66874760389328, 0.17159031331539154, 0.9197117686271667, 0.3752596080303192, 0.8737481832504272, 0.7935830354690552, 0.4079011380672455, 0.8041059970855713, 0.7119358777999878, 0.44816115498542786, 0.0792212039232254, 0.24532362818717957, 0.12464446574449539, 0.5793103575706482, 0.47020164132118225, 0.7446225881576538, 0.9028775095939636, 0.27148136496543884, 0.1311958134174347, 0.7448316812515259, 0.17480187118053436, 0.15760648250579834, 0.8674253821372986, 0.4123516380786896, 0.49759814143180847, 0.6075113415718079, 0.8602694869041443, 0.38801518082618713, 0.9327232241630554, 0.5043127536773682, 0.5494574308395386, 0.4937790632247925, 0.39573928713798523, 0.6145573854446411, 0.22828766703605652, 0.0869055986404419, 0.43118199706077576, 0.8144258260726929, 0.6802943348884583, 0.2991809844970703, 0.1377718597650528, 0.2927544414997101, 0.2903098165988922, 0.09797755628824234, 0.4454524517059326, 0.6211563348770142, 0.4783971905708313, 0.019471749663352966, 0.14490661025047302, 0.43302789330482483, 0.5297582149505615, 0.42046937346458435, 0.3536824584007263, 0.7462149262428284, 0.5104460716247559, 0.8330911993980408, 0.9944958686828613, 0.4926053583621979, 0.6379671692848206, 0.8307327628135681, 0.7433109879493713, 0.12807412445545197, 0.35512569546699524, 0.509627640247345, 0.27010565996170044, 0.45681867003440857, 0.9570500254631042, 0.7337709665298462, 0.3912188708782196, 0.4197999835014343, 0.8539033532142639, 0.23222172260284424, 0.7027703523635864, 0.7048265933990479, 0.2174520641565323, 0.35447263717651367, 0.4990869164466858, 0.3263351023197174, 0.9182167649269104, 0.5943993330001831, 0.11149855703115463, 0.40111806988716125, 0.12474722415208817, 0.4512084126472473, 0.7392439246177673, 0.42335984110832214, 0.6644944548606873, 0.2809922993183136, 0.1540258675813675, 0.30826687812805176, 0.801666796207428, 0.17161402106285095, 0.45908448100090027, 0.6951395273208618, 0.5468441247940063, 0.1766081154346466, 0.5002010464668274, 0.9249138832092285, 0.4012935161590576, 0.1442907601594925, 0.5915204882621765, 0.5402388572692871, 0.8334885835647583, 0.7061454057693481, 0.7492089867591858, 0.5571103096008301, 0.41260454058647156, 0.6826881766319275, 0.9056636691093445, 0.16345393657684326, 0.9772413969039917, 0.7328954935073853, 0.8965637683868408, 0.4858008027076721, 0.14210981130599976, 0.22439837455749512, 0.5907618999481201, 0.17558585107326508, 0.911949634552002, 0.2562631368637085, 0.6722161769866943, 0.6086105704307556, 0.27542081475257874, 0.19589318335056305, 0.03280381113290787, 0.3923809230327606, 0.8996787667274475, 0.5459175705909729, 0.3689056634902954, 0.669145941734314, 0.675346314907074, 0.7412937879562378, 0.6133050918579102, 0.8746587634086609, 0.32650622725486755, 0.209279403090477, 0.44870004057884216, 0.9995328187942505, 0.6678334474563599, 0.6375728249549866, 0.1249593198299408, 0.41133373975753784, 0.63104248046875, 0.31471338868141174, 0.24933183193206787, 0.8523882627487183, 0.4515264928340912, 0.3445305824279785, 0.5585486888885498, 0.39071449637413025, 0.9337336421012878, 0.24861611425876617, 0.7282583117485046, 0.3543074131011963, 0.24881935119628906, 0.29737964272499084, 0.5416311025619507, 0.7499551177024841, 0.3528427481651306, 0.71368807554245, 0.9810038805007935, 0.6507205367088318, 0.7713175415992737, 0.8453613519668579, 0.47207269072532654, 0.25285643339157104, 0.9702923893928528, 0.1850365549325943, 0.8717139363288879, 0.35377177596092224, 0.1797192543745041, 0.770277202129364, 0.9448591470718384, 0.1707296520471573, 0.1784278154373169, 0.31538522243499756, 0.2160109579563141, 0.6504046320915222, 0.04548698663711548, 0.3848905563354492, 0.49310052394866943, 0.7725077271461487, 0.5573574900627136, 0.48665282130241394, 0.9129430055618286, 0.8565670251846313, 0.622481644153595, 0.6035676002502441, 0.3789544105529785, 0.8994088768959045, 0.5846242904663086, 0.2241332083940506, 0.3748190701007843, 0.9083943367004395, 0.8048214912414551, 0.7086234092712402, 0.7417979836463928, 0.8754225969314575, 0.008466498926281929, 0.6732107400894165, 0.6270536780357361, 0.7575973868370056, 0.3748481869697571, 0.22433504462242126, 0.5991922616958618, 0.9023239612579346, 0.9444584250450134, 0.010524366982281208, 0.3546407222747803, 0.8603734374046326, 0.12262582778930664, 0.7756876945495605, 0.9397547841072083, 0.7465996742248535, 0.8629698157310486, 0.007321374956518412, 0.04518692195415497, 0.8112183213233948, 0.4307677149772644, 0.6216421723365784, 0.40051332116127014, 0.04820377752184868, 0.12258488684892654, 0.30495455861091614, 0.03768029436469078, 0.09622320532798767, 0.13502880930900574, 0.2679380476474762, 0.9274431467056274, 0.98708575963974, 0.4871731102466583, 0.2073085606098175, 0.5984063148498535, 0.06967570632696152, 0.9276872873306274, 0.02771281823515892, 0.655768096446991, 0.019332610070705414, 0.09456212818622589, 0.552225649356842, 0.6543667912483215, 0.29540586471557617, 0.014268319122493267, 0.4825681746006012, 0.546559751033783, 0.23173603415489197, 0.909551739692688, 0.13205614686012268, 0.34829825162887573, 0.9207378029823303, 0.4394311308860779, 0.4103083312511444, 0.6511588096618652, 0.5926276445388794, 0.9298359751701355, 0.6512943506240845, 0.7349618077278137, 0.18835598230361938, 0.6298184990882874, 0.43614768981933594, 0.8671598434448242, 0.5932089686393738, 0.30257898569107056, 0.24489979445934296, 0.6998762488365173, 0.2220081090927124, 0.6090590357780457, 0.9085845351219177, 0.9176942706108093, 0.14096952974796295, 0.10240528732538223, 0.6351232528686523, 0.6586860418319702, 0.41300585865974426, 0.5851306319236755, 0.5555983781814575, 0.9061112999916077, 0.17706765234470367, 0.7305358052253723, 0.6187132596969604, 0.3259706199169159, 0.22101928293704987, 0.46049177646636963, 0.7620499730110168, 0.6861138343811035, 0.3744722306728363, 0.999613881111145, 0.9704091548919678, 0.9362959265708923, 0.2057315707206726, 0.5628861784934998, 0.4213607609272003, 0.2060900628566742, 0.7715415358543396, 0.9381442666053772, 0.19383293390274048, 0.5486329793930054, 0.7418684363365173, 0.9776108264923096, 0.7937912940979004, 0.772785484790802, 0.12147264927625656, 0.7161477208137512, 0.9724569916725159, 0.23737415671348572, 0.3075423836708069, 0.22197778522968292, 0.09996731579303741, 0.4888237416744232, 0.7654663920402527, 0.7509472370147705, 0.41140803694725037, 0.5811493992805481, 0.28440147638320923, 0.2847668528556824, 0.35303470492362976, 0.6948012709617615, 0.8514412641525269, 0.351897269487381, 0.8206146955490112, 0.8061862587928772, 0.8592726588249207, 0.6627108454704285, 0.3815425932407379, 0.9009790420532227, 0.4423043429851532, 0.6326383948326111, 0.05088408291339874, 0.24353240430355072, 0.37269270420074463, 0.8417269587516785, 0.986234724521637, 0.08671669661998749, 0.17877571284770966, 0.8101239204406738, 0.6806904673576355, 0.21926026046276093, 0.2917413115501404, 0.08160927891731262, 0.9914815425872803, 0.7321274280548096, 0.0663471445441246, 0.9717525243759155, 0.3011833727359772, 0.3600625991821289, 0.6322672963142395, 0.8500317335128784, 0.24560509622097015, 0.6675423383712769, 0.029216362163424492, 0.8062688708305359, 0.7742483019828796, 0.008939646184444427, 0.02310124970972538, 0.6603944897651672, 0.47763165831565857, 0.9901677966117859, 0.8786094784736633, 0.3123742341995239, 0.46522843837738037, 0.30460691452026367, 0.3886050879955292, 0.7949869632720947, 0.6164411306381226, 0.6255585551261902, 0.11978843808174133, 0.4088255763053894, 0.016568152233958244, 0.3064022362232208, 0.6482164859771729, 0.6011162400245667, 0.13761459290981293, 0.45553016662597656, 0.14301501214504242, 0.3718316853046417, 0.8470185995101929, 0.9969383478164673, 0.5945174098014832, 0.9945768713951111, 0.4135090708732605, 0.2207203209400177, 0.10090088099241257, 0.5490091443061829, 0.058611802756786346, 0.1353694647550583, 0.9574659466743469, 0.7388747334480286, 0.325268030166626, 0.5915972590446472, 0.580726146697998, 0.12643815577030182, 0.8555336594581604, 0.9520614743232727, 0.3591216802597046, 0.8569587469100952, 0.9860710501670837, 0.5911633968353271, 0.8841980695724487, 0.467696875333786, 0.7581477761268616, 0.5062493085861206, 0.023584749549627304, 0.2636990249156952, 0.4853028953075409, 0.22144849598407745, 0.8218105435371399, 0.9416132569313049, 0.224420428276062, 0.25212371349334717, 0.4596327245235443, 0.8283070921897888, 0.274640291929245, 0.583084762096405, 0.12327158451080322, 0.09584087878465652, 0.3126043379306793, 0.12343733012676239, 0.7359218001365662, 0.9391165375709534, 0.342061847448349, 0.8714927434921265, 0.5853124856948853, 0.13927051424980164, 0.5906739234924316, 0.8605359196662903, 0.24830657243728638, 0.4617246389389038, 0.2785212993621826, 0.6537293791770935, 0.2967531979084015, 0.43909281492233276, 0.493785560131073, 0.6822227835655212, 0.9617589116096497, 0.92945796251297, 0.3680136203765869, 0.4879632592201233, 0.0355449840426445, 0.3620149493217468, 0.4134310483932495, 0.7345288395881653, 0.7554688453674316, 0.8966485261917114, 0.8243072628974915, 0.13062021136283875, 0.4826793372631073, 0.6219698786735535, 0.1984664797782898, 0.7464526295661926, 0.8252413272857666, 0.7749122977256775, 0.32054588198661804, 0.23457616567611694, 0.12424392253160477, 0.016931388527154922, 0.07964829355478287, 0.07548029720783234, 0.926131010055542, 0.6795510649681091, 0.6923727989196777, 0.8656634092330933, 0.12932902574539185, 0.9213340282440186, 0.8964633941650391, 0.18040120601654053, 0.14799588918685913, 0.7139229774475098, 0.5183727741241455, 0.20890238881111145, 0.6877219676971436, 0.11614598333835602, 0.9502670764923096, 0.21816271543502808, 0.18791420757770538, 0.5980424880981445, 0.6165470480918884, 0.6106774210929871, 0.6212726831436157, 0.43208059668540955, 0.5163164734840393, 0.8450509905815125, 0.2595384120941162, 0.34291204810142517, 0.42927947640419006, 0.1648622304201126, 0.35913145542144775, 0.6078648567199707, 0.6124388575553894, 0.9927509427070618, 0.41569891571998596, 0.569775402545929, 0.7158535122871399, 0.812833309173584, 0.9490837454795837, 0.22709175944328308, 0.039590731263160706, 0.9842380881309509, 0.4060874581336975, 0.16010771691799164, 0.6837782859802246, 0.7786931395530701, 0.3443196713924408, 0.8194848299026489, 0.30312085151672363, 0.25366923213005066, 0.8191945552825928, 0.7377128601074219, 0.6147027015686035, 0.223000630736351, 0.34210166335105896, 0.7341318726539612, 0.6995079517364502, 0.7116117477416992, 0.605620265007019, 0.16455790400505066, 0.2901926338672638, 0.07841821759939194, 0.8957222700119019, 0.7144128680229187, 0.38057640194892883, 0.09881684184074402, 0.9822289347648621, 0.07705125212669373, 0.5121628046035767, 0.8898406624794006, 0.3904043734073639, 0.1676332801580429, 0.17656372487545013, 0.06336241960525513, 0.4260316491127014, 0.6226383447647095, 0.3004486560821533, 0.4514605402946472, 0.5810859203338623, 0.5998451709747314, 0.43016061186790466, 0.7520036101341248, 0.2669123113155365, 0.8802301287651062, 0.6177383065223694, 0.13708965480327606, 0.4338379204273224, 0.25934073328971863, 0.40812352299690247, 0.15464408695697784, 0.2783689796924591, 0.10701626539230347, 0.5673264861106873, 0.7148814797401428, 0.19050535559654236, 0.9920170903205872, 0.2892489433288574, 0.8443721532821655, 0.2579970955848694, 0.0314938984811306, 0.06784266233444214, 0.053422681987285614, 0.9174384474754333, 0.5211555361747742, 0.8302876353263855, 0.5649685263633728, 0.004056479781866074, 0.6491028070449829, 0.7836109399795532, 0.46154963970184326, 0.7379869222640991, 0.21718673408031464, 0.943882167339325, 0.971793532371521, 0.9473378658294678, 0.37976452708244324, 0.8509818315505981, 0.6258240938186646, 0.31534048914909363, 0.3029426634311676, 0.08575557172298431, 0.922808051109314, 0.21004299819469452], "dtype": "float32"}, 0.9], "e": {"__nd__": [0.0, 0.0, 0.0, -35.76307678222656, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -160.97486877441406, 0.0, 0.0, 0.0, -170.53611755371094, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -52.07630157470703, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -118.9699935913086, 0.0, 0.0, 0.0, 0.0, 76.3754653930664, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 240.86448669433594, 0.0, 0.0, 0.0, 0.0, -71.65192413330078, 0.0, 0.0, 65.77511596679688, 20.536914825439453, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 18.754112243652344, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 17.554344177246094, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -64.06272888183594, 0.0, 0.0, 122.90406799316406, 0.0, 0.0, -52.54252624511719, 60.119415283203125, 0.0, 0.0, 55.48210906982422, 0.0, 0.0, 0.0, 49.86637496948242, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 12.477067947387695, 0.0, 0.0, 0.0, 0.0, -67.65504455566406, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -49.327392578125, 0.0, 0.0, 0.0, -69.85334777832031, 0.0, 0.0, 0.0, 0.0, 0.0, 115.18431854248047, 0.0, 156.63438415527344, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -225.4814453125, 0.0, 0.0, 0.0, 0.0, 0.0, -103.682861328125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 52.209388732910156, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -156.13134765625, 0.0, -15.405339241027832, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 51.216796875, 0.0, 96.83578491210938, 0.0, 0.0, -128.59243774414062, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 269.4921569824219, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -37.4760856628418, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -8.833015441894531, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 71.21078491210938, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 108.10818481445312, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -10.227535247802734, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -225.08680725097656, 0.0, -4.086044788360596, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -123.18965911865234, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 96.52084350585938, 0.0, 0.0, 0.0, 0.0, -60.711761474609375, 0.0, 0.0, 0.0, 0.0, 0.0, -40.34608840942383, 75.70784759521484, 0.0, 84.11964416503906, 0.0, 0.0, -58.64385986328125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 38.389095306396484, 0.0, 0.0, 157.14682006835938, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -30.756755828857422, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 161.78399658203125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -3.0686612129211426, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -19.933563232421875, -50.740196228027344, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 134.0843505859375, 0.0, 217.1350860595703, 113.25308227539062, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -217.68568420410156, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 125.01336669921875, 0.0, 0.0, 0.0, 0.0, -193.79153442382812, 0.0, 0.0, -52.6695671081543, 58.24091339111328, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 104.20696258544922, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 21.27379608154297, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 53.955345153808594, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -65.23579406738281, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 167.16722106933594, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -6.826993942260742, -82.29600524902344, 0.0, 0.0, 0.0, -16.53692626953125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -161.29254150390625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 104.89897155761719, 0.0, 0.0, 0.0, -28.49235725402832, 0.0, 0.0, 0.0, 0.0, 0.0, -95.7919921875, 0.0, 0.0, -88.10718536376953, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -153.22642517089844, 0.0, 0.0, 0.0, 0.0, 0.0, -21.2288761138916, -18.911701202392578, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.56252384185791, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -80.71578216552734, 0.0, 141.80670166015625, 0.0, 0.0, 0.0, 0.0, -70.3774642944336, 0.0, 0.0, 0.0, 0.0, 0.0, -45.63915252685547, 0.0, 0.0, 0.0, 237.56732177734375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 232.25201416015625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -162.82798767089844, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 41.79736328125, 0.0, 0.0, 0.0, 81.15483093261719, 0.0, 0.0, -63.14958572387695, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -24.55274200439453, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 26.688859939575195, 35.48420333862305, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 113.03417205810547, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -76.24684143066406, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 116.96990966796875, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -83.39315795898438, 50.719642639160156, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -187.860595703125, 0.0, -16.383153915405273, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -89.42768859863281, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -113.46998596191406, 0.0, 0.0, 0.0, 0.0, 0.0, -11.554404258728027, 0.0, 0.0, 19.570749282836914, 0.0, 61.86817932128906, 0.0, 0.0, 0.0, 0.0, 0.0, -35.60365676879883, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -47.83039093017578, -146.7515869140625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 111.94154357910156, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -83.46182250976562, 0.0, 0.0, 0.0, 0.0, -71.00210571289062, 0.0, 12.840742111206055, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -23.31402587890625, 0.0, 0.0, 0.0, 0.0, -17.958650588989258, 0.0, -137.95823669433594, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -75.45616912841797, 0.0, 174.6083984375, 0.0, 0.0, 86.70833587646484, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -165.13623046875, 0.0, -150.5662384033203, 0.0, -53.22858810424805, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -16.74823570251465, 0.0, 0.0, 0.0, 0.0, 235.7612762451172, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -106.6925048828125, 0.0, 0.0, -139.14599609375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 80.2543716430664, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 48.45901870727539, 0.0, 0.0, 0.0, 0.0, 42.451011657714844, 0.0, 0.0, -147.61094665527344, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -69.93480682373047, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -96.94121551513672, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -123.80390930175781, 173.372314453125, 0.0, -136.62318420410156, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 23.555221557617188, -125.25991821289062, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 38.76507568359375, 0.0, 0.0, 0.0, -81.16932678222656, 189.9427947998047, 87.72307586669922, 0.0, 112.94259643554688, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 190.14657592773438, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 68.66677856445312, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 102.45365142822266, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 102.3941879272461, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -62.86024475097656, 0.0, 0.0, 0.0, -58.47511291503906, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 69.7713394165039, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 110.80760955810547, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 107.44184875488281, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 222.10879516601562, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -91.239013671875, 18.999210357666016, 71.44243621826172, 0.0, 0.0, -30.367862701416016, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 117.422119140625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 149.49082946777344, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 26.600093841552734, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -65.56333923339844, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -127.29391479492188, 0.0, 0.0, 0.0, 0.0, 0.0, -1.732063889503479, 37.36941909790039, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 146.57215881347656, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -104.84423828125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -49.1959228515625, 0.0, 0.0, 0.0, 0.0, 0.0, 2.5086140632629395, -40.292449951171875, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.616981506347656, 0.0, -72.86749267578125, 0.0, 0.0, -55.15843200683594, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -58.89847183227539, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 91.28987121582031, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 104.91998291015625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 82.1761703491211, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -83.2394790649414, 72.39495086669922, 0.0, 0.0, -41.096839904785156, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 91.19952392578125, 0.0, 0.0, 36.51872253417969, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -45.351314544677734, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.3792791366577148, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -12.603021621704102, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 67.94125366210938, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -145.544189453125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 36.40357971191406, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 148.67376708984375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -87.14474487304688, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 16.57808494567871, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -160.97384643554688, -36.13727951049805, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -32.51102828979492, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 115.53492736816406, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.3003621101379395, 0.0, 0.0, 108.66726684570312, 120.20883178710938, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -123.07325744628906, 0.0, 0.0, -1.9988685846328735, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 67.04763793945312, 0.0, 0.0, -89.3478775024414, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -44.931541442871094, -218.6971435546875, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 53.79936599731445, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 68.26132202148438, -10.37820053100586, 191.71109008789062, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -97.27108001708984, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 105.56334686279297, 0.0, 83.68630981445312, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 122.00325012207031, 0.0, 0.0, 0.0, 0.0, 0.0, -8.038773536682129, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -74.8207778930664, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 20.913646697998047, 0.0, 0.0, 0.0, 0.0, -94.88351440429688, 0.0, 0.0, 94.1049575805664, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 84.81685638427734, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 65.27225494384766, 69.95893096923828, 0.0, -6.23505163192749, 0.0, 0.0, 146.1711883544922, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -17.962326049804688, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 82.0635986328125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.6907379627227783, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -129.10580444335938, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -34.86020278930664, 0.0, 0.0, 0.0, 0.0, 0.0, -74.19293212890625, 0.0, 0.0, 0.0, -130.2517547607422, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 24.069313049316406, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -112.38409423828125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -189.22837829589844, 0.0, 0.0, -130.88539123535156, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -40.327186584472656, 0.0, 0.0, 0.0, 0.0, 46.53539276123047, -152.177978515625, 0.0, 0.0, 0.0, 0.0, -146.67991638183594, -104.77899169921875, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 26.314382553100586, 0.0, 91.99122619628906, 26.159259796142578, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 81.93795013427734, 0.0, 0.0, 0.0, 8.313966751098633, 0.0, -70.46485900878906, 0.0, 0.0, 0.0, 80.85563659667969, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -67.29720306396484, 0.0, 0.0, 110.99711608886719, 0.0, 61.529876708984375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -131.82162475585938, -101.91627502441406, 0.0, 0.0, 65.12592315673828, 0.0, 53.27252197265625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 42.811279296875, 0.0, 0.0, 0.0, 0.0, 29.42571258544922, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -104.78317260742188, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -40.575660705566406, 0.0, 0.0, -18.874536514282227, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 56.171268463134766, 0.0, 0.0, 194.992431640625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -75.52578735351562, 0.0, 0.0, 0.0, 0.0, 0.0, -2.6541218757629395, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -52.85120391845703, 0.0, -61.017295837402344, 0.0, 0.0, -50.032859802246094, 0.0, -21.688232421875, 0.0, 0.0, 0.0, 0.0, 0.0, 185.109375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 81.5207748413086, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 61.88256072998047, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 79.1889877319336, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 90.31080627441406, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -162.28634643554688, 48.26313400268555, 0.0, 0.0, 0.0, 0.0, 0.0, 17.81194305419922, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -41.746482849121094, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 11.735174179077148, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -45.1848258972168, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 143.9735565185547, 0.0, 0.0, 0.0, 0.0, 44.079673767089844, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 144.63352966308594, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 96.07440948486328, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 88.38883209228516, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 55.00299835205078, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -14.986653327941895, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 51.651084899902344, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -97.89077758789062, -34.97359848022461, 0.0, 0.0, -29.666719436645508, 0.0, 0.0, 0.0, 0.0, 0.0, 84.45684814453125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 44.651947021484375, 0.0, 0.0, -24.861122131347656, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 110.35999298095703, 0.0, 140.39590454101562, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -68.61068725585938, 0.0, 0.0, 0.0, 0.0, 0.0, 77.04706573486328, 186.11013793945312, 0.0, -84.08586120605469, 0.0, 0.0, 0.0, -60.8800048828125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 59.10286331176758, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -30.929471969604492, 165.82464599609375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 135.07052612304688, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 41.92192077636719, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -192.6976776123047, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -202.8845977783203, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -42.19757080078125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 139.28518676757812, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 20.154006958007812, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -161.91561889648438, -83.73111724853516, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 128.50132751464844, 0.0, 0.0, 0.0, 0.0, 0.0, 161.90151977539062, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -83.3150634765625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -86.23443603515625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -56.36434555053711, 0.0, 0.0, 0.0, 0.0, -10.702451705932617, 0.0, 0.0, 0.0, -90.058837890625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -138.56149291992188, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.0857157707214355, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 89.66956329345703, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 93.33990478515625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -7.427507400512695, 0.0, 51.117156982421875, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 30.637025833129883, 0.0, 0.0, 0.0, 0.0, 0.0, 26.876802444458008, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 128.1870880126953, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -57.73565673828125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -72.18041229248047, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 39.70072555541992, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 42.729644775390625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -90.60082244873047, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 21.169527053833008, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -22.40100860595703, 0.0, 0.0, 0.8312755823135376, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -3.7482821941375732, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 98.48960876464844, 0.0, 0.0, -158.10894775390625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -113.154296875, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 57.447601318359375, 0.0, -38.35606002807617, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -157.38247680664062, 0.0, 176.93484497070312, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -88.00450134277344, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -184.95074462890625, 0.0, 139.96839904785156, 6.328519821166992, 121.44004821777344, 0.0, 0.0, 0.0, 0.0, 7.183484077453613, 0.0, 0.0, -106.35772705078125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 124.06076049804688, 0.0, 0.0, 0.0, -21.506980895996094, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -213.74432373046875, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.0189127922058105, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -43.34373092651367, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -11.213947296142578, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 114.92430114746094, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -76.20250701904297, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -24.98346519470215, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 32.80889892578125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.4436426162719727, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -75.73876953125, 0.0, -156.85076904296875, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 16.72015380859375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.401883125305176, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -85.65892028808594, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 109.61256408691406, 0.0, 0.0, 0.0, 0.0, 0.0, 90.81361389160156, 0.0, 0.0, 0.0, 0.0, 0.0, 30.4442195892334, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 124.78150177001953, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -127.61529541015625, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 29.1593074798584, 67.00890350341797, 0.0, 0.0, 0.0, 0.0, 0.0, -53.68561553955078, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.4600934982299805, 83.8932876586914, 0.0, 0.0, 0.0, 0.0, 108.9836196899414, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 94.13691711425781, 0.0, 0.0, 66.06813049316406, 0.0, 0.0, 0.0, 0.0, 60.697723388671875, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -21.808429718017578, -14.952463150024414, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -108.06390380859375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -56.084251403808594, 129.99395751953125, 67.426025390625, 0.0, 0.0, 0.0, 0.0, 0.0, -4.054352283477783, 0.0, 0.0, 0.0, 10.586196899414062, 0.0, 0.0, 0.0, 0.0, -68.74382781982422, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -18.395050048828125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -64.9618911743164, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.69719409942627, 0.0, 0.0, 13.203836441040039, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -164.01962280273438, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 77.15019989013672, 0.0, -34.90361785888672, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 25.896507263183594, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 53.69489288330078, 0.0, 0.0, 36.56858444213867, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -32.50162124633789, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -53.43330383300781, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 86.33628845214844, -157.04466247558594, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 123.04611206054688, 0.0, 0.0, 0.0, 0.0, -31.287445068359375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 60.319637298583984, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 40.01218795776367, 0.0, 0.0, 0.0, 0.0, 42.27134704589844, 0.0, 0.0, -210.86077880859375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 98.84286499023438, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 123.22643280029297, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 108.47840118408203, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 138.57254028320312, 49.07212829589844, 170.8474884033203, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -94.73434448242188, 0.0], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(dropout, json.loads(_TESTS), 'triton', *[1e-06, 1e-07])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")